In [ ]:
# ============================================================
# STEP 24.1 — REVIEWS INVENTORY & TEMPORAL SCHEMA AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import gzip
import csv

print("=" * 80)
print("STEP 24.1 — REVIEWS INVENTORY & TEMPORAL SCHEMA AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
)

AIRBNB_DIR = (
    PROJECT_ROOT /
    "01_raw" /
    "airbnb"
)

snapshot_dates = [
    "2025-10-05",
    "2025-11-07",
    "2025-12-11",
    "2026-01-16",
    "2026-02-13",
    "2026-03-17",
    "2026-04-16",
    "2026-05-23",
    "2026-06-19",
    "2026-07-12",
    "2026-08-13"
]


# ------------------------------------------------------------
# 2. Locate review files
# ------------------------------------------------------------

records = []

for snapshot in snapshot_dates:

    snapshot_dir = AIRBNB_DIR / snapshot

    candidates = list(
        snapshot_dir.glob("reviews.csv*")
    )

    print(
        snapshot,
        "->",
        [p.name for p in candidates]
    )

    records.append({
        "snapshot_date": snapshot,
        "files_found": len(candidates),
        "file_names": "; ".join(
            p.name for p in candidates
        )
    })


inventory_files = pd.DataFrame(records)

print("\n1. FILE INVENTORY")
print("-" * 60)

display(inventory_files)

print(
    "\nSnapshots:",
    len(snapshot_dates)
)

print(
    "Snapshots with exactly one review file:",
    (
        inventory_files["files_found"] == 1
    ).sum()
)

In [ ]:
# ============================================================
# STEP 24.1B — REVIEW SCHEMA, SIZE & DATE COVERAGE AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 80)
print("STEP 24.1B — REVIEW SCHEMA, SIZE & DATE COVERAGE AUDIT")
print("=" * 80)

qc_records = []
schemas = {}

for snapshot in snapshot_dates:

    file_path = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    print(f"\nReading {snapshot} ...")

    # --------------------------------------------------------
    # Read only fields needed for this audit
    # --------------------------------------------------------

    df = pd.read_csv(
        file_path,
        compression="gzip",
        usecols=[
            "listing_id",
            "id",
            "date",
            "reviewer_id",
            "comments"
        ],
        low_memory=False
    )

    schemas[snapshot] = list(df.columns)

    # --------------------------------------------------------
    # Parse dates
    # --------------------------------------------------------

    review_date = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    snapshot_date = pd.Timestamp(snapshot)

    # --------------------------------------------------------
    # Basic QC
    # --------------------------------------------------------

    rows = len(df)

    missing_listing_id = df["listing_id"].isna().sum()
    missing_review_id = df["id"].isna().sum()
    missing_date = review_date.isna().sum()
    missing_comments = df["comments"].isna().sum()

    duplicate_review_id = df["id"].duplicated().sum()

    min_date = review_date.min()
    max_date = review_date.max()

    after_snapshot = (
        review_date > snapshot_date
    ).sum()

    on_snapshot = (
        review_date == snapshot_date
    ).sum()

    unique_listings = (
        df["listing_id"].nunique(
            dropna=True
        )
    )

    unique_reviews = (
        df["id"].nunique(
            dropna=True
        )
    )

    qc_records.append({
        "snapshot_date": snapshot,
        "rows": rows,
        "unique_reviews": unique_reviews,
        "unique_listings": unique_listings,
        "min_review_date": min_date,
        "max_review_date": max_date,
        "missing_listing_id": missing_listing_id,
        "missing_review_id": missing_review_id,
        "missing_review_date": missing_date,
        "missing_comments": missing_comments,
        "duplicate_review_id": duplicate_review_id,
        "reviews_on_snapshot_date": on_snapshot,
        "reviews_after_snapshot_date": after_snapshot
    })

    del df


# ------------------------------------------------------------
# 1. QC summary
# ------------------------------------------------------------

review_qc = pd.DataFrame(qc_records)

print("\n\n1. REVIEW FILE QC")
print("-" * 80)

display(review_qc)


# ------------------------------------------------------------
# 2. Schema consistency
# ------------------------------------------------------------

print("\n2. SCHEMA CONSISTENCY")
print("-" * 80)

unique_schemas = {
    tuple(v)
    for v in schemas.values()
}

print(
    "Number of distinct selected schemas:",
    len(unique_schemas)
)

for snapshot, cols in schemas.items():
    print(
        snapshot,
        "->",
        cols
    )


# ------------------------------------------------------------
# 3. Overall structural checks
# ------------------------------------------------------------

print("\n3. STRUCTURAL SUMMARY")
print("-" * 80)

print(
    "Total rows across snapshot files:",
    review_qc["rows"].sum()
)

print(
    "Snapshots with missing listing_id:",
    (review_qc["missing_listing_id"] > 0).sum()
)

print(
    "Snapshots with missing review ID:",
    (review_qc["missing_review_id"] > 0).sum()
)

print(
    "Snapshots with invalid/missing review date:",
    (review_qc["missing_review_date"] > 0).sum()
)

print(
    "Snapshots with duplicate review IDs:",
    (review_qc["duplicate_review_id"] > 0).sum()
)

print(
    "Snapshots containing review dates "
    "after nominal snapshot date:",
    (
        review_qc[
            "reviews_after_snapshot_date"
        ] > 0
    ).sum()
)


# ------------------------------------------------------------
# 4. Row-count evolution
# ------------------------------------------------------------

print("\n4. ROW-COUNT EVOLUTION")
print("-" * 80)

display(
    review_qc[
        [
            "snapshot_date",
            "rows",
            "unique_reviews",
            "unique_listings",
            "max_review_date",
            "reviews_after_snapshot_date"
        ]
    ]
)


print("\n" + "=" * 80)
print(
    "STEP 24.1B COMPLETE — "
    "AUDIT ONLY, NO REVIEW FEATURE CREATED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.2A — REVIEW DATE VS ACTUAL LISTING SCRAPE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 24.2A — REVIEW DATE VS ACTUAL LISTING SCRAPE AUDIT")
print("=" * 80)

audit_records = []

for snapshot in snapshot_dates:

    print(f"\nAuditing {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    # --------------------------------------------------------
    # 1. Read review identifiers + dates
    # --------------------------------------------------------

    reviews = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "listing_id",
            "id",
            "date"
        ],
        low_memory=False
    )

    reviews["date"] = pd.to_datetime(
        reviews["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 2. Read listing-specific actual scrape date
    # --------------------------------------------------------

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=[
            "id",
            "last_scraped"
        ],
        low_memory=False
    )

    listings = listings.rename(
        columns={
            "id": "listing_id",
            "last_scraped": "listing_last_scraped"
        }
    )

    listings[
        "listing_last_scraped"
    ] = pd.to_datetime(
        listings["listing_last_scraped"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 3. Merge review -> listing scrape date
    # --------------------------------------------------------

    merged = reviews.merge(
        listings,
        on="listing_id",
        how="left",
        validate="many_to_one"
    )

    nominal_date = pd.Timestamp(snapshot)

    # --------------------------------------------------------
    # 4. Temporal conditions
    # --------------------------------------------------------

    after_nominal = (
        merged["date"] > nominal_date
    )

    after_actual_scrape = (
        merged["date"] >
        merged["listing_last_scraped"]
    )

    missing_listing_match = (
        merged["listing_last_scraped"].isna()
    )

    # --------------------------------------------------------
    # 5. Focus specifically on reviews after nominal date
    # --------------------------------------------------------

    post_nominal = merged.loc[
        after_nominal
    ].copy()

    post_nominal_but_not_after_scrape = (
        (
            post_nominal["date"] <=
            post_nominal["listing_last_scraped"]
        )
    ).sum()

    post_nominal_and_after_scrape = (
        (
            post_nominal["date"] >
            post_nominal["listing_last_scraped"]
        )
    ).sum()

    # --------------------------------------------------------
    # 6. Record QC
    # --------------------------------------------------------

    audit_records.append({
        "snapshot_date": snapshot,

        "review_rows": len(merged),

        "matched_to_listing": (
            ~missing_listing_match
        ).sum(),

        "unmatched_to_listing": (
            missing_listing_match
        ).sum(),

        "reviews_after_nominal_date": (
            after_nominal.sum()
        ),

        "post_nominal_but_on_or_before_actual_scrape": (
            post_nominal_but_not_after_scrape
        ),

        "post_nominal_and_after_actual_scrape": (
            post_nominal_and_after_scrape
        ),

        "all_reviews_after_actual_scrape": (
            after_actual_scrape.sum()
        ),

        "min_listing_scrape_date": (
            merged[
                "listing_last_scraped"
            ].min()
        ),

        "max_listing_scrape_date": (
            merged[
                "listing_last_scraped"
            ].max()
        ),

        "max_review_date": (
            merged["date"].max()
        )
    })

    del reviews
    del listings
    del merged
    del post_nominal


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

scrape_audit = pd.DataFrame(
    audit_records
)

print("\n\n1. REVIEW DATE VS ACTUAL SCRAPE DATE")
print("-" * 80)

display(scrape_audit)


# ------------------------------------------------------------
# 8. Overall summary
# ------------------------------------------------------------

print("\n2. OVERALL TEMPORAL SUMMARY")
print("-" * 80)

print(
    "Total review-snapshot rows:",
    scrape_audit[
        "review_rows"
    ].sum()
)

print(
    "Total reviews after nominal snapshot date:",
    scrape_audit[
        "reviews_after_nominal_date"
    ].sum()
)

print(
    "Post-nominal reviews still on/before "
    "listing actual scrape date:",
    scrape_audit[
        "post_nominal_but_on_or_before_actual_scrape"
    ].sum()
)

print(
    "Post-nominal reviews also after "
    "listing actual scrape date:",
    scrape_audit[
        "post_nominal_and_after_actual_scrape"
    ].sum()
)

print(
    "All reviews after listing actual scrape date:",
    scrape_audit[
        "all_reviews_after_actual_scrape"
    ].sum()
)

print(
    "Total review rows without matching listing:",
    scrape_audit[
        "unmatched_to_listing"
    ].sum()
)


print("\n3. SNAPSHOT COUNTS")
print("-" * 80)

print(
    "Snapshots with any review after nominal date:",
    (
        scrape_audit[
            "reviews_after_nominal_date"
        ] > 0
    ).sum()
)

print(
    "Snapshots with any review after actual "
    "listing scrape date:",
    (
        scrape_audit[
            "all_reviews_after_actual_scrape"
        ] > 0
    ).sum()
)


print("\n" + "=" * 80)
print(
    "STEP 24.2A COMPLETE — "
    "TEMPORAL CUTOFF NOT YET FROZEN"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.2B — CONSECUTIVE REVIEW-ID OVERLAP AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 24.2B — CONSECUTIVE REVIEW-ID OVERLAP AUDIT")
print("=" * 80)

overlap_records = []

previous_ids = None
previous_snapshot = None

for snapshot in snapshot_dates:

    print(f"\nReading review IDs: {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    current = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=["id"],
        low_memory=False
    )

    current_ids = set(
        current["id"]
        .dropna()
        .astype("int64")
        .tolist()
    )

    if previous_ids is not None:

        retained = (
            previous_ids &
            current_ids
        )

        disappeared = (
            previous_ids -
            current_ids
        )

        newly_observed = (
            current_ids -
            previous_ids
        )

        overlap_records.append({
            "from_snapshot": previous_snapshot,
            "to_snapshot": snapshot,

            "previous_reviews": len(previous_ids),
            "current_reviews": len(current_ids),

            "retained_reviews": len(retained),
            "newly_observed_reviews": len(newly_observed),
            "disappeared_reviews": len(disappeared),

            "previous_retention_rate": (
                len(retained) /
                len(previous_ids)
                if len(previous_ids) > 0
                else np.nan
            ),

            "current_overlap_rate": (
                len(retained) /
                len(current_ids)
                if len(current_ids) > 0
                else np.nan
            )
        })

    previous_ids = current_ids
    previous_snapshot = snapshot

    del current


review_overlap = pd.DataFrame(
    overlap_records
)


# ------------------------------------------------------------
# 1. Consecutive overlap
# ------------------------------------------------------------

print("\n1. CONSECUTIVE SNAPSHOT OVERLAP")
print("-" * 80)

display(review_overlap)


# ------------------------------------------------------------
# 2. Percentage view
# ------------------------------------------------------------

print("\n2. RETENTION RATES")
print("-" * 80)

percentage_view = review_overlap[
    [
        "from_snapshot",
        "to_snapshot",
        "previous_retention_rate",
        "current_overlap_rate"
    ]
].copy()

percentage_view[
    "previous_retention_pct"
] = (
    percentage_view[
        "previous_retention_rate"
    ] * 100
)

percentage_view[
    "current_overlap_pct"
] = (
    percentage_view[
        "current_overlap_rate"
    ] * 100
)

display(
    percentage_view[
        [
            "from_snapshot",
            "to_snapshot",
            "previous_retention_pct",
            "current_overlap_pct"
        ]
    ]
)


# ------------------------------------------------------------
# 3. Structural reconciliation
# ------------------------------------------------------------

print("\n3. COUNT RECONCILIATION")
print("-" * 80)

review_overlap[
    "reconstructed_current"
] = (
    review_overlap[
        "retained_reviews"
    ] +
    review_overlap[
        "newly_observed_reviews"
    ]
)

review_overlap[
    "current_count_difference"
] = (
    review_overlap[
        "current_reviews"
    ] -
    review_overlap[
        "reconstructed_current"
    ]
)

display(
    review_overlap[
        [
            "from_snapshot",
            "to_snapshot",
            "current_reviews",
            "reconstructed_current",
            "current_count_difference"
        ]
    ]
)

print(
    "\nTransitions with count mismatch:",
    (
        review_overlap[
            "current_count_difference"
        ] != 0
    ).sum()
)


# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\n4. OVERALL SUMMARY")
print("-" * 80)

print(
    "Mean previous-snapshot retention (%):",
    review_overlap[
        "previous_retention_rate"
    ].mean() * 100
)

print(
    "Minimum previous-snapshot retention (%):",
    review_overlap[
        "previous_retention_rate"
    ].min() * 100
)

print(
    "Maximum previous-snapshot retention (%):",
    review_overlap[
        "previous_retention_rate"
    ].max() * 100
)

print(
    "Total newly observed across transitions:",
    review_overlap[
        "newly_observed_reviews"
    ].sum()
)

print(
    "Total disappeared across transitions:",
    review_overlap[
        "disappeared_reviews"
    ].sum()
)


print("\n" + "=" * 80)
print(
    "STEP 24.2B COMPLETE — "
    "CUMULATIVE STRUCTURE AUDITED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.3A — REVIEW OBSERVATION-WINDOW COVERAGE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 24.3A — REVIEW OBSERVATION-WINDOW COVERAGE AUDIT")
print("=" * 80)

WINDOWS = {
    "90d": 90,
    "180d": 180,
    "365d": 365,
    "730d": 730
}

coverage_records = []

for snapshot in snapshot_dates:

    print(f"\nAuditing {snapshot} ...")

    anchor_date = pd.Timestamp(snapshot)

    # --------------------------------------------------------
    # 1. Anchor listings
    # --------------------------------------------------------

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=["id"],
        low_memory=False
    )

    anchor_listing_ids = pd.Index(
        listings["id"]
        .dropna()
        .unique()
    )

    n_anchor = len(anchor_listing_ids)

    # --------------------------------------------------------
    # 2. Reviews available in THIS snapshot only
    # --------------------------------------------------------

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    reviews = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "listing_id",
            "date",
            "comments"
        ],
        low_memory=False
    )

    reviews["date"] = pd.to_datetime(
        reviews["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 3. Enforce frozen temporal cutoff
    # --------------------------------------------------------

    reviews = reviews.loc[
        reviews["date"] <= anchor_date
    ].copy()

    # Only reviews belonging to current anchor listings
    reviews = reviews.loc[
        reviews["listing_id"].isin(
            anchor_listing_ids
        )
    ].copy()

    # Text availability
    reviews["has_text"] = (
        reviews["comments"]
        .notna()
        &
        reviews["comments"]
        .astype(str)
        .str.strip()
        .ne("")
    )

    # --------------------------------------------------------
    # 4. Audit candidate windows
    # --------------------------------------------------------

    for window_name, days in WINDOWS.items():

        lower_bound = (
            anchor_date -
            pd.Timedelta(days=days)
        )

        # Interval:
        # (anchor-days, anchor]
        window_reviews = reviews.loc[
            (reviews["date"] > lower_bound)
            &
            (reviews["date"] <= anchor_date)
        ]

        review_counts = (
            window_reviews
            .groupby("listing_id")
            .size()
        )

        text_counts = (
            window_reviews.loc[
                window_reviews["has_text"]
            ]
            .groupby("listing_id")
            .size()
        )

        listings_with_review = (
            review_counts.index.nunique()
        )

        listings_with_text = (
            text_counts.index.nunique()
        )

        coverage_records.append({
            "snapshot_date": snapshot,
            "window": window_name,
            "window_days": days,

            "anchor_listings": n_anchor,

            "review_rows": len(
                window_reviews
            ),

            "text_review_rows": int(
                window_reviews[
                    "has_text"
                ].sum()
            ),

            "listings_with_review": (
                listings_with_review
            ),

            "listings_with_text_review": (
                listings_with_text
            ),

            "pct_anchor_with_review": (
                listings_with_review /
                n_anchor * 100
            ),

            "pct_anchor_with_text_review": (
                listings_with_text /
                n_anchor * 100
            ),

            "median_reviews_among_reviewed": (
                review_counts.median()
                if len(review_counts)
                else np.nan
            ),

            "mean_reviews_among_reviewed": (
                review_counts.mean()
                if len(review_counts)
                else np.nan
            ),

            "p90_reviews_among_reviewed": (
                review_counts.quantile(0.90)
                if len(review_counts)
                else np.nan
            )
        })

    # --------------------------------------------------------
    # 5. All-history comparison
    # --------------------------------------------------------

    all_counts = (
        reviews
        .groupby("listing_id")
        .size()
    )

    all_text_counts = (
        reviews.loc[
            reviews["has_text"]
        ]
        .groupby("listing_id")
        .size()
    )

    coverage_records.append({
        "snapshot_date": snapshot,
        "window": "all_history",
        "window_days": np.nan,

        "anchor_listings": n_anchor,

        "review_rows": len(reviews),

        "text_review_rows": int(
            reviews["has_text"].sum()
        ),

        "listings_with_review": (
            all_counts.index.nunique()
        ),

        "listings_with_text_review": (
            all_text_counts.index.nunique()
        ),

        "pct_anchor_with_review": (
            all_counts.index.nunique() /
            n_anchor * 100
        ),

        "pct_anchor_with_text_review": (
            all_text_counts.index.nunique() /
            n_anchor * 100
        ),

        "median_reviews_among_reviewed": (
            all_counts.median()
            if len(all_counts)
            else np.nan
        ),

        "mean_reviews_among_reviewed": (
            all_counts.mean()
            if len(all_counts)
            else np.nan
        ),

        "p90_reviews_among_reviewed": (
            all_counts.quantile(0.90)
            if len(all_counts)
            else np.nan
        )
    })

    del listings
    del reviews


# ------------------------------------------------------------
# 6. Full results
# ------------------------------------------------------------

window_qc = pd.DataFrame(
    coverage_records
)

print("\n1. FULL WINDOW COVERAGE RESULTS")
print("-" * 80)

display(window_qc)


# ------------------------------------------------------------
# 7. Mean coverage across snapshots
# ------------------------------------------------------------

print("\n2. MEAN COVERAGE ACROSS 11 SNAPSHOTS")
print("-" * 80)

window_summary = (
    window_qc
    .groupby("window", as_index=False)
    .agg(
        mean_pct_with_review=(
            "pct_anchor_with_review",
            "mean"
        ),
        mean_pct_with_text=(
            "pct_anchor_with_text_review",
            "mean"
        ),
        mean_review_rows=(
            "review_rows",
            "mean"
        ),
        median_of_median_reviews=(
            "median_reviews_among_reviewed",
            "median"
        ),
        mean_p90_reviews=(
            "p90_reviews_among_reviewed",
            "mean"
        )
    )
)

display(window_summary)


# ------------------------------------------------------------
# 8. Coverage by snapshot/window
# ------------------------------------------------------------

print("\n3. TEXT COVERAGE MATRIX (%)")
print("-" * 80)

coverage_matrix = (
    window_qc
    .pivot(
        index="snapshot_date",
        columns="window",
        values="pct_anchor_with_text_review"
    )
)

display(
    coverage_matrix.round(2)
)


print("\n" + "=" * 80)
print(
    "STEP 24.3A COMPLETE — "
    "WINDOW NOT YET SELECTED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.3B — PRIMARY 365-DAY TEXT-VOLUME AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 24.3B — PRIMARY 365-DAY TEXT-VOLUME AUDIT")
print("=" * 80)

text_volume_records = []

for snapshot in snapshot_dates:

    print(f"\nAuditing {snapshot} ...")

    anchor_date = pd.Timestamp(snapshot)

    # --------------------------------------------------------
    # 1. Current anchor listings
    # --------------------------------------------------------

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=["id"],
        low_memory=False
    )

    anchor_ids = set(
        listings["id"]
        .dropna()
        .astype("int64")
    )

    # --------------------------------------------------------
    # 2. Reviews
    # --------------------------------------------------------

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    reviews = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "listing_id",
            "date",
            "comments"
        ],
        low_memory=False
    )

    reviews["date"] = pd.to_datetime(
        reviews["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 3. Frozen 365-day window
    #
    # (anchor - 365 days, anchor]
    # --------------------------------------------------------

    lower_bound = (
        anchor_date -
        pd.Timedelta(days=365)
    )

    reviews = reviews.loc[
        (reviews["date"] > lower_bound)
        &
        (reviews["date"] <= anchor_date)
        &
        (reviews["listing_id"].isin(anchor_ids))
        &
        (reviews["comments"].notna())
    ].copy()

    reviews["comments"] = (
        reviews["comments"]
        .astype(str)
        .str.strip()
    )

    reviews = reviews.loc[
        reviews["comments"].ne("")
    ].copy()

    # --------------------------------------------------------
    # 4. Review-level text size
    # --------------------------------------------------------

    reviews["char_count"] = (
        reviews["comments"].str.len()
    )

    reviews["word_count"] = (
        reviews["comments"]
        .str.split()
        .str.len()
    )

    # --------------------------------------------------------
    # 5. Listing-level aggregation
    # --------------------------------------------------------

    listing_text = (
        reviews
        .groupby("listing_id")
        .agg(
            n_reviews=("comments", "size"),
            total_chars=("char_count", "sum"),
            total_words=("word_count", "sum"),
            median_review_chars=("char_count", "median"),
            max_review_chars=("char_count", "max")
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 6. Record distribution
    # --------------------------------------------------------

    text_volume_records.append({
        "snapshot_date": snapshot,

        "anchor_listings": len(anchor_ids),

        "listings_with_text": len(listing_text),

        "text_reviews": len(reviews),

        "review_char_median": (
            reviews["char_count"].median()
        ),

        "review_char_p90": (
            reviews["char_count"].quantile(0.90)
        ),

        "review_char_p99": (
            reviews["char_count"].quantile(0.99)
        ),

        "review_word_median": (
            reviews["word_count"].median()
        ),

        "review_word_p90": (
            reviews["word_count"].quantile(0.90)
        ),

        "listing_total_words_median": (
            listing_text["total_words"].median()
        ),

        "listing_total_words_p90": (
            listing_text["total_words"].quantile(0.90)
        ),

        "listing_total_words_p95": (
            listing_text["total_words"].quantile(0.95)
        ),

        "listing_total_words_p99": (
            listing_text["total_words"].quantile(0.99)
        ),

        "listing_reviews_median": (
            listing_text["n_reviews"].median()
        ),

        "listing_reviews_p90": (
            listing_text["n_reviews"].quantile(0.90)
        ),

        "listing_reviews_p99": (
            listing_text["n_reviews"].quantile(0.99)
        )
    })

    del listings
    del reviews
    del listing_text


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

text_volume_qc = pd.DataFrame(
    text_volume_records
)

print("\n1. TEXT-VOLUME DISTRIBUTION")
print("-" * 80)

display(text_volume_qc)


# ------------------------------------------------------------
# 8. Cross-snapshot summary
# ------------------------------------------------------------

print("\n2. CROSS-SNAPSHOT SUMMARY")
print("-" * 80)

summary_cols = [
    "review_char_median",
    "review_char_p90",
    "review_char_p99",
    "review_word_median",
    "review_word_p90",
    "listing_total_words_median",
    "listing_total_words_p90",
    "listing_total_words_p95",
    "listing_total_words_p99",
    "listing_reviews_median",
    "listing_reviews_p90",
    "listing_reviews_p99"
]

summary = (
    text_volume_qc[
        summary_cols
    ]
    .agg([
        "mean",
        "min",
        "max"
    ])
    .T
)

display(summary)


print("\n" + "=" * 80)
print(
    "STEP 24.3B COMPLETE — "
    "NO TEXT EMBEDDING CREATED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.4A — REVIEW-ID / TEXT STABILITY AUDIT
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import gc

print("=" * 80)
print("STEP 24.4A — REVIEW-ID / TEXT STABILITY AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# Helper: stable text hash
# ------------------------------------------------------------

def text_hash(x):

    if pd.isna(x):
        text = ""
    else:
        text = str(x)

    return hashlib.sha1(
        text.encode(
            "utf-8",
            errors="replace"
        )
    ).hexdigest()


# ------------------------------------------------------------
# Compare consecutive snapshots
# ------------------------------------------------------------

stability_records = []

previous = None
previous_snapshot = None


for snapshot in snapshot_dates:

    print(f"\nReading {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    current = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "listing_id",
            "date",
            "comments"
        ],
        low_memory=False
    )

    current["date"] = pd.to_datetime(
        current["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Hash text
    # --------------------------------------------------------

    current["text_hash"] = (
        current["comments"]
        .fillna("")
        .astype(str)
        .map(text_hash)
    )

    # Keep only fields required for comparison
    current_small = current[
        [
            "id",
            "listing_id",
            "date",
            "text_hash"
        ]
    ].copy()

    del current
    gc.collect()


    # --------------------------------------------------------
    # Compare with previous snapshot
    # --------------------------------------------------------

    if previous is not None:

        print(
            f"Comparing "
            f"{previous_snapshot} -> {snapshot} ..."
        )

        matched = previous.merge(
            current_small,
            on="id",
            how="inner",
            suffixes=(
                "_previous",
                "_current"
            ),
            validate="one_to_one"
        )

        same_text = (
            matched["text_hash_previous"]
            ==
            matched["text_hash_current"]
        )

        same_listing = (
            matched["listing_id_previous"]
            ==
            matched["listing_id_current"]
        )

        same_date = (
            matched["date_previous"]
            ==
            matched["date_current"]
        )

        stability_records.append({
            "from_snapshot":
                previous_snapshot,

            "to_snapshot":
                snapshot,

            "shared_review_ids":
                len(matched),

            "same_text":
                int(same_text.sum()),

            "changed_text":
                int((~same_text).sum()),

            "pct_same_text":
                float(
                    same_text.mean() * 100
                ),

            "changed_listing_id":
                int((~same_listing).sum()),

            "changed_review_date":
                int((~same_date).sum())
        })

        del matched
        gc.collect()


    previous = current_small
    previous_snapshot = snapshot


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

review_text_stability = pd.DataFrame(
    stability_records
)

print("\n1. CONSECUTIVE TEXT STABILITY")
print("-" * 80)

display(review_text_stability)


# ------------------------------------------------------------
# Overall weighted result
# ------------------------------------------------------------

print("\n2. OVERALL SUMMARY")
print("-" * 80)

total_shared = (
    review_text_stability[
        "shared_review_ids"
    ].sum()
)

total_same = (
    review_text_stability[
        "same_text"
    ].sum()
)

total_changed = (
    review_text_stability[
        "changed_text"
    ].sum()
)

print(
    "Total shared review-ID comparisons:",
    total_shared
)

print(
    "Same-text comparisons:",
    total_same
)

print(
    "Changed-text comparisons:",
    total_changed
)

print(
    "Weighted text stability (%):",
    total_same /
    total_shared *
    100
)

print(
    "Changed listing_id:",
    review_text_stability[
        "changed_listing_id"
    ].sum()
)

print(
    "Changed review date:",
    review_text_stability[
        "changed_review_date"
    ].sum()
)


print("\n" + "=" * 80)

if (
    total_changed == 0
    and
    review_text_stability[
        "changed_listing_id"
    ].sum() == 0
    and
    review_text_stability[
        "changed_review_date"
    ].sum() == 0
):

    print(
        "STEP 24.4A STATUS: "
        "EXACTLY STABLE ACROSS CONSECUTIVE SNAPSHOTS"
    )

else:

    print(
        "STEP 24.4A STATUS: "
        "CHANGES DETECTED — AUDIT REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.4B — AUDIT REVIEW IDs WITH CHANGED LISTING_ID
# ============================================================

import pandas as pd
import gc

print("=" * 80)
print("STEP 24.4B — AUDIT CHANGED REVIEW-ID -> LISTING-ID")
print("=" * 80)

anomaly_records = []

previous = None
previous_snapshot = None


for snapshot in snapshot_dates:

    print(f"\nReading {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    current = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "listing_id",
            "date",
            "reviewer_id",
            "comments"
        ],
        low_memory=False
    )

    current["date"] = pd.to_datetime(
        current["date"],
        errors="coerce"
    )

    if previous is not None:

        matched = previous.merge(
            current,
            on="id",
            how="inner",
            suffixes=(
                "_previous",
                "_current"
            ),
            validate="one_to_one"
        )

        changed = matched.loc[
            matched["listing_id_previous"]
            !=
            matched["listing_id_current"]
        ].copy()

        if len(changed) > 0:

            print(
                f"\nANOMALIES FOUND: "
                f"{previous_snapshot} -> {snapshot}: "
                f"{len(changed)}"
            )

            for _, row in changed.iterrows():

                anomaly_records.append({
                    "from_snapshot":
                        previous_snapshot,

                    "to_snapshot":
                        snapshot,

                    "review_id":
                        row["id"],

                    "listing_id_previous":
                        row["listing_id_previous"],

                    "listing_id_current":
                        row["listing_id_current"],

                    "review_date_previous":
                        row["date_previous"],

                    "review_date_current":
                        row["date_current"],

                    "reviewer_id_previous":
                        row["reviewer_id_previous"],

                    "reviewer_id_current":
                        row["reviewer_id_current"],

                    "text_same":
                        (
                            str(row["comments_previous"])
                            ==
                            str(row["comments_current"])
                        ),

                    "comment_preview":
                        str(
                            row["comments_current"]
                        )[:250]
                })

        del matched
        del changed
        gc.collect()

    previous = current
    previous_snapshot = snapshot


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

anomaly_df = pd.DataFrame(
    anomaly_records
)

print("\n\n1. LISTING-ID ANOMALIES")
print("-" * 80)

display(anomaly_df)


print("\n2. SUMMARY")
print("-" * 80)

print(
    "Total anomaly comparisons:",
    len(anomaly_df)
)

if len(anomaly_df) > 0:

    print(
        "Unique affected review IDs:",
        anomaly_df[
            "review_id"
        ].nunique()
    )

    print(
        "Unique previous listing IDs:",
        anomaly_df[
            "listing_id_previous"
        ].nunique()
    )

    print(
        "Unique current listing IDs:",
        anomaly_df[
            "listing_id_current"
        ].nunique()
    )

    print(
        "All review dates unchanged:",
        (
            anomaly_df[
                "review_date_previous"
            ]
            ==
            anomaly_df[
                "review_date_current"
            ]
        ).all()
    )

    print(
        "All reviewer IDs unchanged:",
        (
            anomaly_df[
                "reviewer_id_previous"
            ]
            ==
            anomaly_df[
                "reviewer_id_current"
            ]
        ).all()
    )

    print(
        "All text unchanged:",
        anomaly_df[
            "text_same"
        ].all()
    )


print("\n" + "=" * 80)
print(
    "STEP 24.4B COMPLETE — "
    "ANOMALIES IDENTIFIED, POLICY NOT YET APPLIED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.5A — PRIMARY-WINDOW UNIQUE REVIEW-ID REGISTRY AUDIT
# ============================================================

import pandas as pd
import numpy as np
import gc

print("=" * 80)
print("STEP 24.5A — PRIMARY-WINDOW UNIQUE REVIEW-ID REGISTRY AUDIT")
print("=" * 80)

all_eligible_review_ids = set()
snapshot_records = []

for snapshot in snapshot_dates:

    print(f"\nProcessing {snapshot} ...")

    anchor_date = pd.Timestamp(snapshot)
    lower_bound = anchor_date - pd.Timedelta(days=365)

    # --------------------------------------------------------
    # 1. Current anchor listings
    # --------------------------------------------------------

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=["id"],
        low_memory=False
    )

    anchor_ids = set(
        listings["id"]
        .dropna()
        .astype("int64")
    )

    # --------------------------------------------------------
    # 2. Current snapshot reviews
    # --------------------------------------------------------

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    reviews = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "listing_id",
            "date",
            "comments"
        ],
        low_memory=False
    )

    reviews["date"] = pd.to_datetime(
        reviews["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 3. Frozen primary eligibility rule
    #    (anchor - 365 days, anchor]
    # --------------------------------------------------------

    eligible = reviews.loc[
        (reviews["date"] > lower_bound)
        &
        (reviews["date"] <= anchor_date)
        &
        (reviews["listing_id"].isin(anchor_ids))
        &
        (reviews["comments"].notna())
    ].copy()

    eligible["comments"] = (
        eligible["comments"]
        .astype(str)
        .str.strip()
    )

    eligible = eligible.loc[
        eligible["comments"].ne("")
    ].copy()

    # --------------------------------------------------------
    # 4. Snapshot QC
    # --------------------------------------------------------

    n_rows = len(eligible)
    n_unique_ids = eligible["id"].nunique()
    n_duplicate_ids = n_rows - n_unique_ids

    ids_this_snapshot = set(
        eligible["id"].astype("int64")
    )

    union_before = len(
        all_eligible_review_ids
    )

    all_eligible_review_ids.update(
        ids_this_snapshot
    )

    union_after = len(
        all_eligible_review_ids
    )

    newly_added_to_union = (
        union_after - union_before
    )

    snapshot_records.append({
        "snapshot_date": snapshot,
        "eligible_review_rows": n_rows,
        "unique_review_ids": n_unique_ids,
        "duplicate_review_ids": n_duplicate_ids,
        "new_ids_added_to_union": newly_added_to_union,
        "cumulative_union_ids": union_after
    })

    del listings
    del reviews
    del eligible
    gc.collect()


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

registry_qc = pd.DataFrame(
    snapshot_records
)

print("\n1. SNAPSHOT-LEVEL REGISTRY AUDIT")
print("-" * 80)

display(registry_qc)


print("\n2. OVERALL SUMMARY")
print("-" * 80)

total_snapshot_eligible_rows = (
    registry_qc[
        "eligible_review_rows"
    ].sum()
)

final_unique_ids = len(
    all_eligible_review_ids
)

print(
    "Total eligible review-snapshot rows:",
    total_snapshot_eligible_rows
)

print(
    "Unique review IDs needed across "
    "all primary windows:",
    final_unique_ids
)

print(
    "Potential repeated encoding avoided:",
    total_snapshot_eligible_rows
    - final_unique_ids
)

print(
    "Reuse factor "
    "(review-snapshot rows / unique reviews):",
    total_snapshot_eligible_rows
    / final_unique_ids
)

print(
    "Snapshots with duplicate review IDs:",
    int(
        (
            registry_qc[
                "duplicate_review_ids"
            ] > 0
        ).sum()
    )
)


print("\n" + "=" * 80)
print(
    "STEP 24.5A COMPLETE — "
    "UNIQUE REVIEW REQUIREMENT AUDITED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.5B — GLOBAL REVIEW-ID IDENTITY CONSISTENCY AUDIT
# Primary 365-day review universe only
# ============================================================

import pandas as pd
import hashlib
import gc

print("=" * 80)
print("STEP 24.5B — GLOBAL REVIEW-ID IDENTITY CONSISTENCY AUDIT")
print("=" * 80)

eligible_id_set = all_eligible_review_ids

print(
    "Primary-window unique review IDs:",
    len(eligible_id_set)
)

# ------------------------------------------------------------
# Store first observed identity for each eligible review ID
# ------------------------------------------------------------

first_seen = {}

text_conflict_ids = set()
date_conflict_ids = set()
reviewer_conflict_ids = set()

# listing_id is deliberately NOT required to be globally stable
listing_change_ids = set()

observations_checked = 0


def stable_text_hash(x):

    if pd.isna(x):
        text = ""
    else:
        text = str(x)

    return hashlib.sha1(
        text.encode(
            "utf-8",
            errors="replace"
        )
    ).hexdigest()


# ------------------------------------------------------------
# Scan all 11 source snapshots
# ------------------------------------------------------------

for snapshot in snapshot_dates:

    print(f"\nChecking {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    df = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "listing_id",
            "date",
            "reviewer_id",
            "comments"
        ],
        low_memory=False
    )

    # Only IDs that are relevant to the frozen primary universe
    df = df.loc[
        df["id"].isin(eligible_id_set)
    ].copy()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df["text_hash"] = (
        df["comments"]
        .fillna("")
        .astype(str)
        .map(stable_text_hash)
    )

    observations_checked += len(df)

    # --------------------------------------------------------
    # Compare every occurrence against first observed version
    # --------------------------------------------------------

    for row in df[
        [
            "id",
            "listing_id",
            "date",
            "reviewer_id",
            "text_hash"
        ]
    ].itertuples(index=False):

        rid = int(row.id)

        current_identity = (
            row.text_hash,
            row.date,
            row.reviewer_id,
            row.listing_id
        )

        if rid not in first_seen:

            first_seen[rid] = current_identity

        else:

            (
                first_text,
                first_date,
                first_reviewer,
                first_listing
            ) = first_seen[rid]

            if row.text_hash != first_text:
                text_conflict_ids.add(rid)

            if row.date != first_date:
                date_conflict_ids.add(rid)

            if row.reviewer_id != first_reviewer:
                reviewer_conflict_ids.add(rid)

            if row.listing_id != first_listing:
                listing_change_ids.add(rid)

    del df
    gc.collect()


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n1. GLOBAL IDENTITY RESULTS")
print("-" * 80)

print(
    "Review observations checked:",
    observations_checked
)

print(
    "Unique eligible review IDs observed:",
    len(first_seen)
)

print(
    "Expected eligible review IDs:",
    len(eligible_id_set)
)

print(
    "Eligible IDs never found:",
    len(
        eligible_id_set -
        set(first_seen.keys())
    )
)

print(
    "\nReview IDs with TEXT conflict:",
    len(text_conflict_ids)
)

print(
    "Review IDs with DATE conflict:",
    len(date_conflict_ids)
)

print(
    "Review IDs with REVIEWER conflict:",
    len(reviewer_conflict_ids)
)

print(
    "Review IDs with LISTING-ID change:",
    len(listing_change_ids)
)


# ------------------------------------------------------------
# 2. Show listing-ID anomalies
# ------------------------------------------------------------

print("\n2. REVIEW IDs WITH LISTING-ID CHANGE")
print("-" * 80)

print(
    sorted(listing_change_ids)[:50]
)


# ------------------------------------------------------------
# 3. Final QC
# ------------------------------------------------------------

print("\n" + "=" * 80)

if (
    len(first_seen) == len(eligible_id_set)
    and
    len(text_conflict_ids) == 0
    and
    len(date_conflict_ids) == 0
    and
    len(reviewer_conflict_ids) == 0
):

    print(
        "STEP 24.5B STATUS: PASS — "
        "GLOBAL REVIEW IDENTITY IS STABLE"
    )

    print(
        "Note: listing_id changes are allowed "
        "and will remain snapshot-specific."
    )

else:

    print(
        "STEP 24.5B STATUS: "
        "CONFLICTS DETECTED — AUDIT REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.5C — BUILD FROZEN UNIQUE REVIEW-TEXT REGISTRY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import gc

print("=" * 80)
print("STEP 24.5C — BUILD FROZEN UNIQUE REVIEW-TEXT REGISTRY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Output location
# ------------------------------------------------------------

REVIEW_OUT_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "reviews"
)

REVIEW_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    REVIEW_OUT_DIR /
    "review_text_registry_primary365_frozen.parquet"
)

eligible_id_set = all_eligible_review_ids

print(
    "Expected unique review IDs:",
    len(eligible_id_set)
)

# ------------------------------------------------------------
# 2. Collect each eligible review exactly once
#
# IMPORTANT:
# We take the first observed occurrence only because Step 24.5B
# demonstrated globally stable:
# review_id -> date, reviewer_id, text
#
# listing_id is deliberately excluded.
# ------------------------------------------------------------

collected_ids = set()
registry_parts = []

collection_qc = []


for snapshot in snapshot_dates:

    print(f"\nScanning {snapshot} ...")

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    df = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "date",
            "reviewer_id",
            "comments"
        ],
        low_memory=False
    )

    # Eligible primary-universe IDs only
    df = df.loc[
        df["id"].isin(eligible_id_set)
    ].copy()

    # Only IDs not already collected
    new_mask = ~df["id"].isin(
        collected_ids
    )

    new_df = df.loc[
        new_mask
    ].copy()

    # Defensive duplicate check
    duplicate_ids_here = int(
        new_df["id"].duplicated().sum()
    )

    if duplicate_ids_here > 0:
        raise ValueError(
            f"Duplicate review IDs detected "
            f"within {snapshot}: "
            f"{duplicate_ids_here}"
        )

    # Add source provenance
    new_df["first_observed_snapshot"] = snapshot

    # Rename
    new_df = new_df.rename(
        columns={
            "id": "review_id",
            "date": "review_date"
        }
    )

    new_df["review_date"] = pd.to_datetime(
        new_df["review_date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Text QC fields
    # --------------------------------------------------------

    new_df["comments"] = (
        new_df["comments"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    new_df["char_count"] = (
        new_df["comments"]
        .str.len()
        .astype("int32")
    )

    new_df["word_count"] = (
        new_df["comments"]
        .str.split()
        .str.len()
        .astype("int32")
    )

    # --------------------------------------------------------
    # Update registry
    # --------------------------------------------------------

    ids_added = set(
        new_df["review_id"]
        .astype("int64")
    )

    collected_ids.update(
        ids_added
    )

    registry_parts.append(
        new_df[
            [
                "review_id",
                "review_date",
                "reviewer_id",
                "comments",
                "char_count",
                "word_count",
                "first_observed_snapshot"
            ]
        ]
    )

    collection_qc.append({
        "snapshot_date": snapshot,
        "eligible_ids_seen_in_file": len(df),
        "new_ids_collected": len(new_df),
        "cumulative_registry_ids": len(collected_ids)
    })

    del df
    del new_df
    gc.collect()


# ------------------------------------------------------------
# 3. Combine registry
# ------------------------------------------------------------

registry = pd.concat(
    registry_parts,
    ignore_index=True
)

del registry_parts
gc.collect()


# ------------------------------------------------------------
# 4. Structural QC
# ------------------------------------------------------------

print("\n1. COLLECTION QC")
print("-" * 80)

collection_qc_df = pd.DataFrame(
    collection_qc
)

display(collection_qc_df)


print("\n2. REGISTRY QC")
print("-" * 80)

print(
    "Registry rows:",
    len(registry)
)

print(
    "Unique review IDs:",
    registry[
        "review_id"
    ].nunique()
)

print(
    "Expected review IDs:",
    len(eligible_id_set)
)

print(
    "Duplicate review IDs:",
    registry[
        "review_id"
    ].duplicated().sum()
)

print(
    "Missing review_date:",
    registry[
        "review_date"
    ].isna().sum()
)

print(
    "Missing reviewer_id:",
    registry[
        "reviewer_id"
    ].isna().sum()
)

print(
    "Empty comments:",
    (
        registry[
            "comments"
        ].str.len() == 0
    ).sum()
)

print(
    "Missing expected IDs:",
    len(
        eligible_id_set -
        set(
            registry[
                "review_id"
            ].astype("int64")
        )
    )
)

print(
    "Unexpected IDs:",
    len(
        set(
            registry[
                "review_id"
            ].astype("int64")
        )
        -
        eligible_id_set
    )
)


# ------------------------------------------------------------
# 5. Distribution QC
# ------------------------------------------------------------

print("\n3. TEXT DISTRIBUTION")
print("-" * 80)

display(
    registry[
        [
            "char_count",
            "word_count"
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

registry = registry.sort_values(
    "review_id"
).reset_index(
    drop=True
)

registry.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_FILE
)


# ------------------------------------------------------------
# 7. Exact reload QC
# ------------------------------------------------------------

reloaded = pd.read_parquet(
    OUTPUT_FILE
)

print("\n4. SAVE / RELOAD QC")
print("-" * 80)

print(
    "Reloaded rows:",
    len(reloaded)
)

print(
    "Reloaded unique review IDs:",
    reloaded[
        "review_id"
    ].nunique()
)

print(
    "Columns identical:",
    list(reloaded.columns)
    ==
    list(registry.columns)
)

print(
    "Review IDs identical:",
    reloaded[
        "review_id"
    ].equals(
        registry[
            "review_id"
        ]
    )
)

print(
    "Review dates identical:",
    reloaded[
        "review_date"
    ].equals(
        registry[
            "review_date"
        ]
    )
)

print(
    "Comments identical:",
    reloaded[
        "comments"
    ].equals(
        registry[
            "comments"
        ]
    )
)


# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

structural_pass = (
    len(registry)
    == len(eligible_id_set)
    and
    registry["review_id"].nunique()
    == len(eligible_id_set)
    and
    registry["review_id"].duplicated().sum()
    == 0
    and
    registry["review_date"].isna().sum()
    == 0
    and
    (registry["comments"].str.len() == 0).sum()
    == 0
    and
    len(reloaded)
    == len(registry)
    and
    reloaded["review_id"].equals(
        registry["review_id"]
    )
    and
    reloaded["review_date"].equals(
        registry["review_date"]
    )
    and
    reloaded["comments"].equals(
        registry["comments"]
    )
)

print("\n" + "=" * 80)

if structural_pass:

    print(
        "STEP 24.5C STATUS: PASS — "
        "FROZEN UNIQUE REVIEW-TEXT REGISTRY CREATED"
    )

else:

    print(
        "STEP 24.5C STATUS: FAIL — "
        "DO NOT PROCEED TO EMBEDDING"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.6A — BUILD FROZEN TEMPORAL REVIEW MEMBERSHIP
# Primary window: (anchor - 365 days, anchor]
# ============================================================

import pandas as pd
from pathlib import Path
import gc

print("=" * 80)
print("STEP 24.6A — BUILD FROZEN TEMPORAL REVIEW MEMBERSHIP")
print("=" * 80)

REVIEW_OUT_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "reviews"
)

REVIEW_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    REVIEW_OUT_DIR /
    "review_membership_primary365_frozen.parquet"
)

membership_parts = []
qc_records = []

# T1 ... T11 source snapshots exist,
# but modelling anchors are T1 ... T10.
#
# The final Aug snapshot is the FUTURE snapshot for T10,
# not another modelling anchor.
anchor_snapshots = snapshot_dates[:10]

for idx, snapshot in enumerate(anchor_snapshots, start=1):

    transition = f"T{idx}"

    print(
        f"\nBuilding {transition}: {snapshot} ..."
    )

    anchor_date = pd.Timestamp(snapshot)
    lower_bound = (
        anchor_date -
        pd.Timedelta(days=365)
    )

    # --------------------------------------------------------
    # 1. Anchor listings
    # --------------------------------------------------------

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=["id"],
        low_memory=False
    )

    anchor_ids = set(
        listings["id"]
        .dropna()
        .astype("int64")
    )

    # --------------------------------------------------------
    # 2. Review membership from THIS snapshot only
    # --------------------------------------------------------

    review_file = (
        AIRBNB_DIR /
        snapshot /
        "reviews.csv.gz"
    )

    reviews = pd.read_csv(
        review_file,
        compression="gzip",
        usecols=[
            "id",
            "listing_id",
            "date",
            "comments"
        ],
        low_memory=False
    )

    reviews["date"] = pd.to_datetime(
        reviews["date"],
        errors="coerce"
    )

    reviews["comments"] = (
        reviews["comments"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # 3. Frozen eligibility rule
    #
    # review exists in this snapshot
    # AND belongs to current anchor listing
    # AND has non-empty text
    # AND:
    # (anchor - 365 days, anchor]
    # --------------------------------------------------------

    eligible = reviews.loc[
        (reviews["date"] > lower_bound)
        &
        (reviews["date"] <= anchor_date)
        &
        (reviews["listing_id"].isin(anchor_ids))
        &
        (reviews["comments"].ne(""))
    ][
        [
            "id",
            "listing_id"
        ]
    ].copy()

    eligible = eligible.rename(
        columns={
            "id": "review_id"
        }
    )

    eligible.insert(
        0,
        "snapshot_date",
        anchor_date
    )

    eligible.insert(
        0,
        "transition",
        transition
    )

    # --------------------------------------------------------
    # 4. QC
    # --------------------------------------------------------

    duplicate_review_ids = int(
        eligible[
            "review_id"
        ].duplicated().sum()
    )

    listings_with_text = (
        eligible[
            "listing_id"
        ].nunique()
    )

    qc_records.append({
        "transition": transition,
        "snapshot_date": snapshot,
        "anchor_listings": len(anchor_ids),
        "eligible_review_rows": len(eligible),
        "unique_review_ids": (
            eligible[
                "review_id"
            ].nunique()
        ),
        "duplicate_review_ids": (
            duplicate_review_ids
        ),
        "listings_with_text": (
            listings_with_text
        ),
        "pct_anchor_with_text": (
            listings_with_text /
            len(anchor_ids) *
            100
        )
    })

    membership_parts.append(
        eligible
    )

    del listings
    del reviews
    del eligible
    gc.collect()


# ------------------------------------------------------------
# 5. Combine
# ------------------------------------------------------------

membership = pd.concat(
    membership_parts,
    ignore_index=True
)

del membership_parts
gc.collect()


# ------------------------------------------------------------
# 6. QC
# ------------------------------------------------------------

membership_qc = pd.DataFrame(
    qc_records
)

print("\n1. TRANSITION-LEVEL QC")
print("-" * 80)

display(membership_qc)


print("\n2. OVERALL MEMBERSHIP QC")
print("-" * 80)

print(
    "Total membership rows:",
    len(membership)
)

print(
    "Unique review IDs represented:",
    membership[
        "review_id"
    ].nunique()
)

print(
    "Unique transition-review pairs:",
    membership[
        [
            "transition",
            "review_id"
        ]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate transition-review pairs:",
    membership.duplicated(
        subset=[
            "transition",
            "review_id"
        ]
    ).sum()
)

print(
    "Missing listing_id:",
    membership[
        "listing_id"
    ].isna().sum()
)

print(
    "Missing review_id:",
    membership[
        "review_id"
    ].isna().sum()
)


# ------------------------------------------------------------
# 7. Registry integrity
# ------------------------------------------------------------

registry_file = (
    REVIEW_OUT_DIR /
    "review_text_registry_primary365_frozen.parquet"
)

registry_ids = set(
    pd.read_parquet(
        registry_file,
        columns=["review_id"]
    )["review_id"]
    .astype("int64")
)

membership_ids = set(
    membership[
        "review_id"
    ].astype("int64")
)

print(
    "Membership IDs absent from registry:",
    len(
        membership_ids -
        registry_ids
    )
)


# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

membership = membership.sort_values(
    [
        "transition",
        "listing_id",
        "review_id"
    ]
).reset_index(
    drop=True
)

membership.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_FILE
)


# ------------------------------------------------------------
# 9. Reload QC
# ------------------------------------------------------------

reloaded = pd.read_parquet(
    OUTPUT_FILE
)

print("\n3. SAVE / RELOAD QC")
print("-" * 80)

print(
    "Reloaded rows:",
    len(reloaded)
)

print(
    "Columns identical:",
    list(reloaded.columns)
    ==
    list(membership.columns)
)

print(
    "Transition identical:",
    reloaded["transition"].equals(
        membership["transition"]
    )
)

print(
    "Listing IDs identical:",
    reloaded["listing_id"].equals(
        membership["listing_id"]
    )
)

print(
    "Review IDs identical:",
    reloaded["review_id"].equals(
        membership["review_id"]
    )
)


# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

expected_rows = int(
    membership_qc[
        "eligible_review_rows"
    ].sum()
)

structural_pass = (
    len(membership) == expected_rows
    and
    membership.duplicated(
        subset=[
            "transition",
            "review_id"
        ]
    ).sum() == 0
    and
    len(
        membership_ids -
        registry_ids
    ) == 0
    and
    len(reloaded) == len(membership)
    and
    reloaded["review_id"].equals(
        membership["review_id"]
    )
)

print("\n" + "=" * 80)

if structural_pass:

    print(
        "STEP 24.6A STATUS: PASS — "
        "TEMPORAL REVIEW MEMBERSHIP FROZEN"
    )

else:

    print(
        "STEP 24.6A STATUS: FAIL — "
        "DO NOT PROCEED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.6B — LISTING-LEVEL REVIEW COVERAGE AUDIT
# Frozen T1–T10 temporal membership
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 80)
print("STEP 24.6B — LISTING-LEVEL REVIEW COVERAGE AUDIT")
print("=" * 80)

REVIEW_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "reviews"
)

membership_file = (
    REVIEW_DIR /
    "review_membership_primary365_frozen.parquet"
)

membership = pd.read_parquet(
    membership_file
)

# ------------------------------------------------------------
# T1–T10 anchor snapshots only
# ------------------------------------------------------------

anchor_snapshots = snapshot_dates[:10]

records = []

for i, snapshot in enumerate(
    anchor_snapshots,
    start=1
):

    transition = f"T{i}"

    print(
        f"\nAuditing {transition}: {snapshot} ..."
    )

    # --------------------------------------------------------
    # All listings existing at this anchor
    # --------------------------------------------------------

    listing_file = (
        AIRBNB_DIR /
        snapshot /
        "listings.csv.gz"
    )

    listings = pd.read_csv(
        listing_file,
        compression="gzip",
        usecols=["id"]
    )

    listing_ids = (
        listings["id"]
        .dropna()
        .astype("int64")
        .drop_duplicates()
    )

    base = pd.DataFrame({
        "listing_id": listing_ids
    })

    # --------------------------------------------------------
    # Number of eligible reviews per listing
    # --------------------------------------------------------

    m = membership.loc[
        membership["transition"]
        == transition
    ]

    counts = (
        m.groupby("listing_id")
        .size()
        .rename("n_reviews_365d")
        .reset_index()
    )

    base = base.merge(
        counts,
        on="listing_id",
        how="left",
        validate="one_to_one"
    )

    base["n_reviews_365d"] = (
        base["n_reviews_365d"]
        .fillna(0)
        .astype("int32")
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    n_total = len(base)

    n_with = int(
        (base["n_reviews_365d"] > 0).sum()
    )

    n_without = int(
        (base["n_reviews_365d"] == 0).sum()
    )

    positive = base.loc[
        base["n_reviews_365d"] > 0,
        "n_reviews_365d"
    ]

    records.append({
        "transition": transition,
        "snapshot_date": snapshot,

        "total_anchor_listings":
            n_total,

        "listings_with_review_text":
            n_with,

        "listings_without_review_text":
            n_without,

        "pct_with_review_text":
            n_with / n_total * 100,

        "pct_without_review_text":
            n_without / n_total * 100,

        "median_reviews_all_listings":
            base["n_reviews_365d"].median(),

        "median_reviews_if_positive":
            positive.median(),

        "p90_reviews_if_positive":
            positive.quantile(0.90),

        "p95_reviews_if_positive":
            positive.quantile(0.95),

        "p99_reviews_if_positive":
            positive.quantile(0.99),

        "max_reviews":
            base["n_reviews_365d"].max()
    })


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

coverage_qc = pd.DataFrame(records)

print("\n1. TRANSITION-LEVEL COVERAGE")
print("-" * 80)

display(coverage_qc)


print("\n2. OVERALL SUMMARY")
print("-" * 80)

print(
    "Mean % listings with review text:",
    coverage_qc[
        "pct_with_review_text"
    ].mean()
)

print(
    "Range % with review text:",
    (
        coverage_qc[
            "pct_with_review_text"
        ].min(),
        coverage_qc[
            "pct_with_review_text"
        ].max()
    )
)

print(
    "Mean % listings without review text:",
    coverage_qc[
        "pct_without_review_text"
    ].mean()
)

print(
    "Median of transition medians "
    "(positive listings):",
    coverage_qc[
        "median_reviews_if_positive"
    ].median()
)

print(
    "Mean P90 reviews:",
    coverage_qc[
        "p90_reviews_if_positive"
    ].mean()
)

print(
    "Mean P99 reviews:",
    coverage_qc[
        "p99_reviews_if_positive"
    ].mean()
)


# ------------------------------------------------------------
# Integrity check against frozen membership
# ------------------------------------------------------------

expected_membership_rows = 7_334_068

print("\n3. MEMBERSHIP INTEGRITY")
print("-" * 80)

print(
    "Membership rows:",
    len(membership)
)

print(
    "Expected:",
    expected_membership_rows
)

print(
    "Exact match:",
    len(membership)
    == expected_membership_rows
)


print("\n" + "=" * 80)
print(
    "STEP 24.6B COMPLETE — "
    "LISTING-LEVEL REVIEW COVERAGE AUDITED"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.6C-0 — VERIFY CLEANED LISTING FILES BEFORE AUDIT
# ============================================================

import pandas as pd

print("=" * 80)
print("STEP 24.6C-0 — VERIFY CLEANED LISTING INPUT FILES")
print("=" * 80)

development_snapshots = snapshot_dates[:9]

qc = []

for i, snapshot in enumerate(
    development_snapshots,
    start=1
):

    listing_file = (
        PROJECT_ROOT /
        "02_intermediate" /
        "listings" /
        f"listings_{snapshot}.parquet"
    )

    exists = listing_file.exists()

    if exists:

        df = pd.read_parquet(
            listing_file
        )

        qc.append({
            "transition": f"T{i}",
            "snapshot_date": snapshot,
            "file_exists": True,
            "rows": len(df),
            "columns": len(df.columns),
            "has_listing_id":
                "listing_id" in df.columns,
            "has_number_of_reviews":
                "number_of_reviews" in df.columns,
            "duplicate_listing_id":
                (
                    df["listing_id"].duplicated().sum()
                    if "listing_id" in df.columns
                    else None
                )
        })

    else:

        qc.append({
            "transition": f"T{i}",
            "snapshot_date": snapshot,
            "file_exists": False,
            "rows": None,
            "columns": None,
            "has_listing_id": False,
            "has_number_of_reviews": False,
            "duplicate_listing_id": None
        })


qc_df = pd.DataFrame(qc)

display(qc_df)


print("\nSUMMARY")
print("-" * 80)

print(
    "Files found:",
    qc_df["file_exists"].sum(),
    "/",
    len(qc_df)
)

print(
    "Files with listing_id:",
    qc_df["has_listing_id"].sum(),
    "/",
    len(qc_df)
)

print(
    "Files with number_of_reviews:",
    qc_df["has_number_of_reviews"].sum(),
    "/",
    len(qc_df)
)

print(
    "Total duplicate listing IDs:",
    qc_df[
        "duplicate_listing_id"
    ].fillna(0).sum()
)


print("\n" + "=" * 80)

if (
    qc_df["file_exists"].all()
    and
    qc_df["has_listing_id"].all()
    and
    qc_df["has_number_of_reviews"].all()
    and
    qc_df[
        "duplicate_listing_id"
    ].fillna(0).sum() == 0
):

    print(
        "STEP 24.6C-0 STATUS: PASS — "
        "INPUT FILES VERIFIED"
    )

else:

    print(
        "STEP 24.6C-0 STATUS: "
        "CHECK REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.6C — REVIEW AVAILABILITY × LISTING LIFECYCLE AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Determine what "no eligible review text in the previous
# 365 days" actually represents:
#
# A. recent_review_text
#    -> >= 1 eligible review with non-empty text in the
#       frozen 365-day review membership
#
# B. historical_reviews_but_none_365d
#    -> no eligible review text in the 365-day window,
#       but number_of_reviews > 0 at the anchor
#
# C. no_review_history
#    -> no eligible review text in the 365-day window
#       AND number_of_reviews == 0
#
# IMPORTANT
# ------------------------------------------------------------
# DESIGN / DECISION DATA = T1–T9 ONLY.
# T10 remains locked final evaluation and is NOT used here
# to make feature-engineering decisions.
#
# This step DOES NOT create predictors.
# This step DOES NOT run MiniLM.
# This step DOES NOT drop zero-review listings.
# ============================================================


import pandas as pd
import numpy as np
from pathlib import Path


print("=" * 80)
print(
    "STEP 24.6C — "
    "REVIEW AVAILABILITY × LISTING LIFECYCLE AUDIT"
)
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
)

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

membership_file = (
    REVIEW_DIR
    / "review_membership_primary365_frozen.parquet"
)


# ============================================================
# 2. VERIFY FROZEN MEMBERSHIP FILE
# ============================================================

if not membership_file.exists():

    raise FileNotFoundError(
        f"Frozen membership file not found:\n"
        f"{membership_file}"
    )


membership = pd.read_parquet(
    membership_file,
    columns=[
        "transition",
        "listing_id",
        "review_id"
    ]
)


print(
    "\nFrozen membership rows:",
    len(membership)
)

print(
    "Frozen membership transitions:",
    sorted(
        membership[
            "transition"
        ].unique()
    )
)


# ------------------------------------------------------------
# Membership integrity check
# ------------------------------------------------------------

expected_membership_rows = 7_334_068

print(
    "Expected frozen membership rows:",
    expected_membership_rows
)

print(
    "Membership exact match:",
    len(membership)
    ==
    expected_membership_rows
)


if len(membership) != expected_membership_rows:

    raise ValueError(
        "Frozen membership row count does not match "
        "Step 24.6A. Stop before continuing."
    )


# ============================================================
# 3. DEVELOPMENT ANCHORS ONLY
#
# T1–T9:
# 2025-10-05 ... 2026-06-19
#
# T10 = 2026-07-12 is deliberately excluded from
# feature-design decisions.
# ============================================================

development_snapshots = snapshot_dates[:9]

print(
    "\nDevelopment snapshots used:"
)

for i, snapshot in enumerate(
    development_snapshots,
    start=1
):

    print(
        f"T{i}: {snapshot}"
    )


print(
    "\nT10 is NOT used in this design audit."
)


# ============================================================
# 4. CONTAINERS
# ============================================================

records = []
group_records = []


# ============================================================
# 5. LOOP THROUGH T1–T9
# ============================================================

for i, snapshot in enumerate(
    development_snapshots,
    start=1
):

    transition = f"T{i}"

    print(
        "\n" + "-" * 80
    )

    print(
        f"Auditing {transition}: "
        f"{snapshot}"
    )

    # --------------------------------------------------------
    # 5.1 Correct production cleaned-listing filename
    #
    # Verified in Step 24.6C-0:
    #
    # listings_2025-10-05.parquet
    # listings_2025-11-07.parquet
    # ...
    #
    # NOT:
    # listings_2025-10-05_clean.parquet
    # --------------------------------------------------------

    listing_file = (
        LISTINGS_DIR
        / f"listings_{snapshot}.parquet"
    )


    if not listing_file.exists():

        raise FileNotFoundError(
            f"Cleaned listing file not found:\n"
            f"{listing_file}"
        )


    # --------------------------------------------------------
    # 5.2 Read only required listing columns
    # --------------------------------------------------------

    listings = pd.read_parquet(
        listing_file,
        columns=[
            "listing_id",
            "number_of_reviews"
        ]
    )


    # --------------------------------------------------------
    # 5.3 Structural checks
    # --------------------------------------------------------

    if listings[
        "listing_id"
    ].isna().any():

        raise ValueError(
            f"{transition}: "
            "missing listing_id detected."
        )


    duplicate_listing_ids = int(
        listings[
            "listing_id"
        ].duplicated().sum()
    )


    if duplicate_listing_ids != 0:

        raise ValueError(
            f"{transition}: "
            f"{duplicate_listing_ids} duplicate "
            f"listing IDs detected."
        )


    # --------------------------------------------------------
    # 5.4 Select frozen membership for this transition
    # --------------------------------------------------------

    m = membership.loc[
        membership[
            "transition"
        ]
        ==
        transition
    ].copy()


    # --------------------------------------------------------
    # 5.5 Defensive membership checks
    # --------------------------------------------------------

    duplicate_transition_review = int(
        m[
            "review_id"
        ].duplicated().sum()
    )


    if duplicate_transition_review != 0:

        raise ValueError(
            f"{transition}: duplicate review IDs "
            f"within transition membership."
        )


    # --------------------------------------------------------
    # 5.6 Count eligible 365-day review texts per listing
    # --------------------------------------------------------

    review_counts = (
        m.groupby(
            "listing_id"
        )
        .size()
        .rename(
            "n_reviews_365d"
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # 5.7 Merge review counts onto ALL anchor listings
    #
    # LEFT JOIN is essential:
    # listings with no eligible review must remain.
    # --------------------------------------------------------

    df = listings.merge(
        review_counts,
        on="listing_id",
        how="left",
        validate="one_to_one"
    )


    df[
        "n_reviews_365d"
    ] = (
        df[
            "n_reviews_365d"
        ]
        .fillna(0)
        .astype("int32")
    )


    # --------------------------------------------------------
    # 5.8 Review availability
    #
    # This is an AUDIT variable at this stage,
    # not yet a frozen ML predictor.
    # --------------------------------------------------------

    df[
        "review_available_365d"
    ] = (
        df[
            "n_reviews_365d"
        ] > 0
    ).astype(
        "int8"
    )


    # --------------------------------------------------------
    # 5.9 Historical review evidence
    #
    # IMPORTANT:
    # Missing number_of_reviews is NOT converted to zero.
    # --------------------------------------------------------

    df[
        "number_of_reviews"
    ] = pd.to_numeric(
        df[
            "number_of_reviews"
        ],
        errors="coerce"
    )


    missing_history = int(
        df[
            "number_of_reviews"
        ]
        .isna()
        .sum()
    )


    # --------------------------------------------------------
    # 5.10 Logical consistency check
    #
    # A listing with eligible review text should normally
    # have number_of_reviews > 0.
    #
    # We audit rather than silently correct.
    # --------------------------------------------------------

    recent_text_but_zero_history = int(
        (
            (
                df[
                    "review_available_365d"
                ] == 1
            )
            &
            (
                df[
                    "number_of_reviews"
                ] == 0
            )
        ).sum()
    )


    # --------------------------------------------------------
    # 5.11 Lifecycle classification
    # --------------------------------------------------------

    conditions = [

        # A — recent eligible text exists
        (
            df[
                "review_available_365d"
            ] == 1
        ),

        # B — no recent text,
        # but listing has historical review evidence
        (
            (
                df[
                    "review_available_365d"
                ] == 0
            )
            &
            (
                df[
                    "number_of_reviews"
                ] > 0
            )
        ),

        # C — no recent text and no review history
        (
            (
                df[
                    "review_available_365d"
                ] == 0
            )
            &
            (
                df[
                    "number_of_reviews"
                ] == 0
            )
        )
    ]


    labels = [

        "recent_review_text",

        "historical_reviews_but_none_365d",

        "no_review_history"
    ]


    df[
        "review_lifecycle_group"
    ] = np.select(
        conditions,
        labels,
        default="unclassified"
    )


    # --------------------------------------------------------
    # 5.12 Count lifecycle groups
    # --------------------------------------------------------

    counts = (
        df[
            "review_lifecycle_group"
        ]
        .value_counts()
    )


    total = len(df)


    recent_n = int(
        counts.get(
            "recent_review_text",
            0
        )
    )


    historical_no_recent_n = int(
        counts.get(
            "historical_reviews_but_none_365d",
            0
        )
    )


    no_history_n = int(
        counts.get(
            "no_review_history",
            0
        )
    )


    unclassified_n = int(
        counts.get(
            "unclassified",
            0
        )
    )


    # --------------------------------------------------------
    # 5.13 Snapshot-level summary
    # --------------------------------------------------------

    records.append({

        "transition":
            transition,

        "snapshot_date":
            snapshot,

        "total_listings":
            total,

        "recent_review_text":
            recent_n,

        "historical_reviews_but_none_365d":
            historical_no_recent_n,

        "no_review_history":
            no_history_n,

        "unclassified":
            unclassified_n,

        "missing_number_of_reviews":
            missing_history,

        "recent_text_but_zero_number_of_reviews":
            recent_text_but_zero_history,

        "membership_rows":
            len(m)

    })


    # --------------------------------------------------------
    # 5.14 Group-level statistics
    # --------------------------------------------------------

    group_summary = (
        df.groupby(
            "review_lifecycle_group",
            observed=True
        )
        .agg(

            n_listings=(
                "listing_id",
                "size"
            ),

            median_number_of_reviews=(
                "number_of_reviews",
                "median"
            ),

            mean_number_of_reviews=(
                "number_of_reviews",
                "mean"
            ),

            median_reviews_365d=(
                "n_reviews_365d",
                "median"
            ),

            mean_reviews_365d=(
                "n_reviews_365d",
                "mean"
            ),

            p90_reviews_365d=(
                "n_reviews_365d",
                lambda x:
                    x.quantile(
                        0.90
                    )
            )

        )
        .reset_index()
    )


    group_summary[
        "transition"
    ] = transition


    group_summary[
        "snapshot_date"
    ] = snapshot


    group_records.append(
        group_summary
    )


    # --------------------------------------------------------
    # 5.15 Console snapshot summary
    # --------------------------------------------------------

    print(
        "Total listings:",
        total
    )

    print(
        "Recent review text:",
        recent_n
    )

    print(
        "Historical reviews but none 365d:",
        historical_no_recent_n
    )

    print(
        "No review history:",
        no_history_n
    )

    print(
        "Unclassified:",
        unclassified_n
    )

    print(
        "Missing number_of_reviews:",
        missing_history
    )

    print(
        "Recent text but number_of_reviews == 0:",
        recent_text_but_zero_history
    )


# ============================================================
# 6. COMBINE RESULTS
# ============================================================

lifecycle_qc = pd.DataFrame(
    records
)


lifecycle_groups = pd.concat(
    group_records,
    ignore_index=True
)


# ============================================================
# 7. CALCULATE PERCENTAGES
# ============================================================

for col in [

    "recent_review_text",

    "historical_reviews_but_none_365d",

    "no_review_history",

    "unclassified"

]:

    lifecycle_qc[
        f"pct_{col}"
    ] = (

        lifecycle_qc[
            col
        ]

        /

        lifecycle_qc[
            "total_listings"
        ]

        * 100
    )


# ============================================================
# 8. RECONCILIATION
# ============================================================

lifecycle_qc[
    "group_sum"
] = (

    lifecycle_qc[
        "recent_review_text"
    ]

    +

    lifecycle_qc[
        "historical_reviews_but_none_365d"
    ]

    +

    lifecycle_qc[
        "no_review_history"
    ]

    +

    lifecycle_qc[
        "unclassified"
    ]
)


lifecycle_qc[
    "reconciliation_difference"
] = (

    lifecycle_qc[
        "total_listings"
    ]

    -

    lifecycle_qc[
        "group_sum"
    ]
)


# ============================================================
# 9. DISPLAY — TRANSITION SUMMARY
# ============================================================

print(
    "\n\n1. DEVELOPMENT-ANCHOR LIFECYCLE SUMMARY"
)

print(
    "-" * 80
)


display(

    lifecycle_qc[
        [

            "transition",

            "snapshot_date",

            "total_listings",

            "recent_review_text",

            "historical_reviews_but_none_365d",

            "no_review_history",

            "unclassified",

            "pct_recent_review_text",

            "pct_historical_reviews_but_none_365d",

            "pct_no_review_history",

            "missing_number_of_reviews",

            "recent_text_but_zero_number_of_reviews"

        ]
    ]
)


# ============================================================
# 10. DISPLAY — GROUP DETAIL
# ============================================================

print(
    "\n2. GROUP-LEVEL DETAIL"
)

print(
    "-" * 80
)


display(

    lifecycle_groups[
        [

            "transition",

            "snapshot_date",

            "review_lifecycle_group",

            "n_listings",

            "median_number_of_reviews",

            "mean_number_of_reviews",

            "median_reviews_365d",

            "mean_reviews_365d",

            "p90_reviews_365d"

        ]
    ]
)


# ============================================================
# 11. DEVELOPMENT-ONLY SUMMARY
# ============================================================

print(
    "\n3. T1–T9 DEVELOPMENT SUMMARY"
)

print(
    "-" * 80
)


mean_recent = (
    lifecycle_qc[
        "pct_recent_review_text"
    ].mean()
)


mean_historical_no_recent = (
    lifecycle_qc[
        "pct_historical_reviews_but_none_365d"
    ].mean()
)


mean_no_history = (
    lifecycle_qc[
        "pct_no_review_history"
    ].mean()
)


print(
    "Mean % recent review text:",
    mean_recent
)


print(
    "Mean % historical reviews "
    "but none in 365d:",
    mean_historical_no_recent
)


print(
    "Mean % no review history:",
    mean_no_history
)


print(
    "Total unclassified:",
    int(
        lifecycle_qc[
            "unclassified"
        ].sum()
    )
)


print(
    "Total missing number_of_reviews:",
    int(
        lifecycle_qc[
            "missing_number_of_reviews"
        ].sum()
    )
)


print(
    "Total recent-text listings "
    "with number_of_reviews == 0:",
    int(
        lifecycle_qc[
            "recent_text_but_zero_number_of_reviews"
        ].sum()
    )
)


# ============================================================
# 12. ZERO-TEXT COMPOSITION
#
# Of listings without eligible review text,
# what proportion has historical review evidence?
# ============================================================

lifecycle_qc[
    "zero_text_total"
] = (

    lifecycle_qc[
        "historical_reviews_but_none_365d"
    ]

    +

    lifecycle_qc[
        "no_review_history"
    ]

    +

    lifecycle_qc[
        "unclassified"
    ]
)


lifecycle_qc[
    "pct_zero_text_with_historical_reviews"
] = np.where(

    lifecycle_qc[
        "zero_text_total"
    ] > 0,

    lifecycle_qc[
        "historical_reviews_but_none_365d"
    ]

    /

    lifecycle_qc[
        "zero_text_total"
    ]

    * 100,

    np.nan
)


lifecycle_qc[
    "pct_zero_text_no_review_history"
] = np.where(

    lifecycle_qc[
        "zero_text_total"
    ] > 0,

    lifecycle_qc[
        "no_review_history"
    ]

    /

    lifecycle_qc[
        "zero_text_total"
    ]

    * 100,

    np.nan
)


print(
    "\n4. COMPOSITION OF ZERO-TEXT LISTINGS"
)

print(
    "-" * 80
)


display(

    lifecycle_qc[
        [

            "transition",

            "zero_text_total",

            "historical_reviews_but_none_365d",

            "no_review_history",

            "pct_zero_text_with_historical_reviews",

            "pct_zero_text_no_review_history"

        ]
    ]
)


print(
    "\nMean % of zero-text listings "
    "with historical review evidence:",
    lifecycle_qc[
        "pct_zero_text_with_historical_reviews"
    ].mean()
)


print(
    "Mean % of zero-text listings "
    "with no review history:",
    lifecycle_qc[
        "pct_zero_text_no_review_history"
    ].mean()
)


# ============================================================
# 13. RECONCILIATION CHECK
# ============================================================

print(
    "\n5. RECONCILIATION"
)

print(
    "-" * 80
)


display(

    lifecycle_qc[
        [

            "transition",

            "total_listings",

            "group_sum",

            "reconciliation_difference",

            "membership_rows"

        ]
    ]
)


all_reconcile = (

    lifecycle_qc[
        "reconciliation_difference"
    ] == 0

).all()


print(
    "All transitions reconcile:",
    all_reconcile
)


# ============================================================
# 14. FINAL QC STATUS
# ============================================================

total_unclassified = int(

    lifecycle_qc[
        "unclassified"
    ].sum()
)


total_missing_history = int(

    lifecycle_qc[
        "missing_number_of_reviews"
    ].sum()
)


print(
    "\n" + "=" * 80
)


if (
    all_reconcile
    and
    total_unclassified == 0
    and
    total_missing_history == 0
):

    print(
        "STEP 24.6C STATUS: PASS — "
        "DEVELOPMENT-ONLY REVIEW LIFECYCLE "
        "AUDIT COMPLETE"
    )

else:

    print(
        "STEP 24.6C STATUS: "
        "AUDIT FINDINGS REQUIRE REVIEW "
        "BEFORE FEATURE DESIGN"
    )


print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.6D — ELIGIBLE REVIEW RECENCY DISTRIBUTION AUDIT
#
# PURPOSE:
# Determine whether review-level embeddings should later use:
#   (A) simple mean pooling, or
#   (B) recency-weighted pooling.
#
# DESIGN DATA:
# T1–T9 ONLY.
# T10 remains locked.
#
# NO MiniLM is run here.
# NO feature is created here.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 24.6D — ELIGIBLE REVIEW RECENCY DISTRIBUTION AUDIT")
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
)

membership_file = (
    REVIEW_DIR
    / "review_membership_primary365_frozen.parquet"
)

registry_file = (
    REVIEW_DIR
    / "review_text_registry_primary365_frozen.parquet"
)


# ============================================================
# 2. VERIFY INPUT FILES
# ============================================================

for f in [
    membership_file,
    registry_file
]:
    if not f.exists():
        raise FileNotFoundError(
            f"Required frozen file not found:\n{f}"
        )


# ============================================================
# 3. READ FROZEN MEMBERSHIP
# ============================================================

membership = pd.read_parquet(
    membership_file,
    columns=[
        "transition",
        "listing_id",
        "review_id"
    ]
)

print(
    "\nFrozen membership rows:",
    len(membership)
)

if len(membership) != 7_334_068:
    raise ValueError(
        "Membership row count does not match Step 24.6A."
    )


# ============================================================
# 4. READ REVIEW DATES FROM FROZEN REGISTRY
# ============================================================

registry = pd.read_parquet(
    registry_file,
    columns=[
        "review_id",
        "review_date"
    ]
)

print(
    "Registry rows:",
    len(registry)
)

print(
    "Registry unique review IDs:",
    registry["review_id"].nunique()
)

if registry["review_id"].duplicated().any():
    raise ValueError(
        "Duplicate review_id detected in frozen registry."
    )


registry["review_date"] = pd.to_datetime(
    registry["review_date"],
    errors="coerce"
)

if registry["review_date"].isna().any():
    raise ValueError(
        "Missing/invalid review_date detected."
    )


# ============================================================
# 5. DEVELOPMENT TRANSITIONS ONLY
# ============================================================

development_map = {
    "T1": "2025-10-05",
    "T2": "2025-11-07",
    "T3": "2025-12-11",
    "T4": "2026-01-16",
    "T5": "2026-02-13",
    "T6": "2026-03-17",
    "T7": "2026-04-16",
    "T8": "2026-05-23",
    "T9": "2026-06-19"
}

development_membership = membership.loc[
    membership["transition"].isin(
        development_map.keys()
    )
].copy()

print(
    "\nDevelopment membership rows (T1–T9):",
    len(development_membership)
)

print(
    "Transitions:",
    sorted(
        development_membership[
            "transition"
        ].unique()
    )
)


# ============================================================
# 6. MERGE REVIEW DATE
# ============================================================

df = development_membership.merge(
    registry,
    on="review_id",
    how="left",
    validate="many_to_one"
)

missing_dates = int(
    df["review_date"].isna().sum()
)

print(
    "\nMembership rows missing registry review date:",
    missing_dates
)

if missing_dates != 0:
    raise ValueError(
        "Membership review IDs are not fully represented "
        "in the frozen registry."
    )


# ============================================================
# 7. ATTACH ANCHOR DATE
# ============================================================

df["anchor_date"] = pd.to_datetime(
    df["transition"].map(
        development_map
    )
)

if df["anchor_date"].isna().any():
    raise ValueError(
        "Missing anchor date after transition mapping."
    )


# ============================================================
# 8. REVIEW AGE
# ============================================================

df["review_age_days"] = (
    df["anchor_date"]
    -
    df["review_date"]
).dt.days


# ============================================================
# 9. HARD TEMPORAL INTEGRITY CHECK
#
# Frozen window:
# (anchor - 365 days, anchor]
#
# Therefore:
# age must be:
# 0 <= age < 365
# ============================================================

negative_age = int(
    (df["review_age_days"] < 0).sum()
)

age_365_or_more = int(
    (df["review_age_days"] >= 365).sum()
)

print("\nTEMPORAL INTEGRITY")
print("-" * 80)

print(
    "Negative review ages:",
    negative_age
)

print(
    "Review ages >= 365 days:",
    age_365_or_more
)

print(
    "Minimum review age:",
    df["review_age_days"].min()
)

print(
    "Maximum review age:",
    df["review_age_days"].max()
)


# ============================================================
# 10. TRANSITION-LEVEL DISTRIBUTION
# ============================================================

records = []

for transition, anchor in development_map.items():

    x = df.loc[
        df["transition"] == transition,
        "review_age_days"
    ]

    records.append({

        "transition":
            transition,

        "anchor_date":
            anchor,

        "review_memberships":
            len(x),

        "mean_age_days":
            x.mean(),

        "p25_age_days":
            x.quantile(0.25),

        "median_age_days":
            x.median(),

        "p75_age_days":
            x.quantile(0.75),

        "p90_age_days":
            x.quantile(0.90),

        "p95_age_days":
            x.quantile(0.95),

        "pct_within_30d":
            (x <= 30).mean() * 100,

        "pct_within_90d":
            (x <= 90).mean() * 100,

        "pct_within_180d":
            (x <= 180).mean() * 100,

        "pct_181_to_364d":
            (
                (
                    (x > 180)
                    &
                    (x < 365)
                ).mean()
                * 100
            )
    })


recency_qc = pd.DataFrame(
    records
)


print(
    "\n1. TRANSITION-LEVEL RECENCY DISTRIBUTION"
)

print("-" * 80)

display(
    recency_qc
)


# ============================================================
# 11. POOLED DEVELOPMENT DISTRIBUTION
# ============================================================

x = df[
    "review_age_days"
]

print(
    "\n2. POOLED T1–T9 RECENCY SUMMARY"
)

print("-" * 80)

print(
    "Review memberships:",
    len(x)
)

print(
    "Mean age:",
    x.mean()
)

print(
    "P25 age:",
    x.quantile(0.25)
)

print(
    "Median age:",
    x.median()
)

print(
    "P75 age:",
    x.quantile(0.75)
)

print(
    "P90 age:",
    x.quantile(0.90)
)

print(
    "P95 age:",
    x.quantile(0.95)
)


for days in [
    30,
    60,
    90,
    180,
    270
]:

    pct = (
        (x <= days)
        .mean()
        * 100
    )

    print(
        f"% reviews within {days} days:",
        pct
    )


# ============================================================
# 12. AGE BANDS
# ============================================================

bins = [
    -1,
    30,
    90,
    180,
    270,
    364
]

labels = [
    "0-30d",
    "31-90d",
    "91-180d",
    "181-270d",
    "271-364d"
]

df[
    "review_age_band"
] = pd.cut(
    df[
        "review_age_days"
    ],
    bins=bins,
    labels=labels
)


age_band = (
    df[
        "review_age_band"
    ]
    .value_counts(
        sort=False
    )
    .rename(
        "n_review_memberships"
    )
    .reset_index()
)


age_band[
    "pct"
] = (
    age_band[
        "n_review_memberships"
    ]
    /
    len(df)
    * 100
)


print(
    "\n3. POOLED AGE BANDS"
)

print("-" * 80)

display(
    age_band
)


# ============================================================
# 13. STABILITY ACROSS T1–T9
# ============================================================

print(
    "\n4. ACROSS-TRANSITION STABILITY"
)

print("-" * 80)


for col in [
    "median_age_days",
    "p90_age_days",
    "pct_within_90d",
    "pct_within_180d"
]:

    print(
        col,
        "min =",
        recency_qc[col].min(),
        "| max =",
        recency_qc[col].max()
    )


# ============================================================
# 14. FINAL INTEGRITY STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

if (
    negative_age == 0
    and
    age_365_or_more == 0
):

    print(
        "STEP 24.6D STATUS: PASS — "
        "REVIEW RECENCY DISTRIBUTION AUDITED "
        "ON DEVELOPMENT ORIGINS T1–T9"
    )

else:

    print(
        "STEP 24.6D STATUS: FAIL — "
        "TEMPORAL WINDOW VIOLATION DETECTED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7A-0 — VERIFY DEVELOPMENT REVIEW UNIVERSE
#
# PURPOSE
# ------------------------------------------------------------
# Identify the exact UNIQUE review corpus that is eligible
# to influence model-development decisions.
#
# DEVELOPMENT ONLY: T1–T9
# T10 remains locked.
#
# No language detection.
# No MiniLM.
# No embedding.
# ============================================================

import pandas as pd

print("=" * 80)
print("STEP 24.7A-0 — VERIFY DEVELOPMENT REVIEW UNIVERSE")
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
)

membership_file = (
    REVIEW_DIR
    / "review_membership_primary365_frozen.parquet"
)

registry_file = (
    REVIEW_DIR
    / "review_text_registry_primary365_frozen.parquet"
)


# ============================================================
# 2. DEVELOPMENT TRANSITIONS ONLY
# ============================================================

development_transitions = [
    "T1", "T2", "T3",
    "T4", "T5", "T6",
    "T7", "T8", "T9"
]


# ============================================================
# 3. READ MEMBERSHIP
# ============================================================

membership = pd.read_parquet(
    membership_file,
    columns=[
        "transition",
        "review_id"
    ]
)

print(
    "\nFull frozen membership rows:",
    len(membership)
)


# ============================================================
# 4. FILTER T1–T9
# ============================================================

dev_membership = membership.loc[
    membership["transition"].isin(
        development_transitions
    )
].copy()

print(
    "T1–T9 membership rows:",
    len(dev_membership)
)


# ============================================================
# 5. UNIQUE DEVELOPMENT REVIEW IDS
# ============================================================

dev_review_ids = (
    dev_membership[
        ["review_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique review IDs in T1–T9:",
    len(dev_review_ids)
)

print(
    "Duplicate IDs after deduplication:",
    dev_review_ids[
        "review_id"
    ].duplicated().sum()
)


# ============================================================
# 6. READ FROZEN REGISTRY
# ============================================================

registry = pd.read_parquet(
    registry_file,
    columns=[
        "review_id",
        "review_date",
        "comments",
        "char_count",
        "word_count"
    ]
)

print(
    "\nFrozen registry rows:",
    len(registry)
)

print(
    "Frozen registry unique IDs:",
    registry[
        "review_id"
    ].nunique()
)


# ============================================================
# 7. JOIN DEVELOPMENT IDS TO REGISTRY
# ============================================================

dev_reviews = dev_review_ids.merge(
    registry,
    on="review_id",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 8. QC
# ============================================================

missing_registry = int(
    dev_reviews[
        "review_date"
    ].isna().sum()
)

missing_comments = int(
    dev_reviews[
        "comments"
    ].isna().sum()
)

empty_comments = int(
    (
        dev_reviews[
            "comments"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    ).sum()
)

duplicate_review_ids = int(
    dev_reviews[
        "review_id"
    ].duplicated().sum()
)


print("\nDEVELOPMENT CORPUS QC")
print("-" * 80)

print(
    "Development unique reviews:",
    len(dev_reviews)
)

print(
    "Missing from registry:",
    missing_registry
)

print(
    "Missing comments:",
    missing_comments
)

print(
    "Empty comments:",
    empty_comments
)

print(
    "Duplicate review IDs:",
    duplicate_review_ids
)


# ============================================================
# 9. TEXT SIZE DISTRIBUTION
# ============================================================

print("\nTEXT SIZE — DEVELOPMENT CORPUS")
print("-" * 80)

print(
    "Median words:",
    dev_reviews[
        "word_count"
    ].median()
)

print(
    "P90 words:",
    dev_reviews[
        "word_count"
    ].quantile(0.90)
)

print(
    "P95 words:",
    dev_reviews[
        "word_count"
    ].quantile(0.95)
)

print(
    "P99 words:",
    dev_reviews[
        "word_count"
    ].quantile(0.99)
)

print(
    "Maximum words:",
    dev_reviews[
        "word_count"
    ].max()
)


# ============================================================
# 10. CHECK DIFFERENCE FROM FULL REGISTRY
#
# Important:
# Reviews that exist only outside the development universe
# must not be used to choose the embedding model.
# ============================================================

outside_dev = (
    len(registry)
    -
    len(dev_reviews)
)

print("\nDEVELOPMENT BOUNDARY")
print("-" * 80)

print(
    "Full registry unique reviews:",
    len(registry)
)

print(
    "T1–T9 unique reviews:",
    len(dev_reviews)
)

print(
    "Registry reviews outside T1–T9 development universe:",
    outside_dev
)


# ============================================================
# 11. FINAL STATUS
# ============================================================

print("\n" + "=" * 80)

if (
    missing_registry == 0
    and
    missing_comments == 0
    and
    empty_comments == 0
    and
    duplicate_review_ids == 0
):

    print(
        "STEP 24.7A-0 STATUS: PASS — "
        "DEVELOPMENT REVIEW UNIVERSE VERIFIED"
    )

else:

    print(
        "STEP 24.7A-0 STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7A-1a — LANGUAGE DETECTION ENVIRONMENT CHECK
# ============================================================

import sys
import importlib.util
import platform

print("=" * 80)
print("STEP 24.7A-1a — LANGUAGE DETECTION ENVIRONMENT CHECK")
print("=" * 80)

print("Python version:", sys.version)
print("Platform:", platform.platform())

packages = [
    "fasttext",
    "fasttext_pybind",
    "lingua",
    "langdetect",
    "transformers",
    "sentence_transformers"
]

print("\nPACKAGE AVAILABILITY")
print("-" * 80)

for package in packages:
    found = importlib.util.find_spec(package) is not None
    print(f"{package:25s}: {found}")

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7A-1b — PILOT LANGUAGE DETECTION
#
# PURPOSE
# ------------------------------------------------------------
# Test multilingual language identification on a DEVELOPMENT-
# ONLY sample before processing the full 1,362,408-review
# development corpus.
#
# T1–T9 ONLY.
# T10 remains locked.
#
# This step:
#   - DOES NOT run MiniLM
#   - DOES NOT create ML predictors
#   - DOES NOT modify the frozen review registry
# ============================================================

import pandas as pd
import numpy as np
import torch
import time

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("=" * 80)
print("STEP 24.7A-1b — PILOT LANGUAGE DETECTION")
print("=" * 80)


# ============================================================
# 1. RECONSTRUCT EXACT DEVELOPMENT REVIEW UNIVERSE
# ============================================================

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
)

membership_file = (
    REVIEW_DIR
    / "review_membership_primary365_frozen.parquet"
)

registry_file = (
    REVIEW_DIR
    / "review_text_registry_primary365_frozen.parquet"
)

development_transitions = [
    "T1", "T2", "T3",
    "T4", "T5", "T6",
    "T7", "T8", "T9"
]


membership = pd.read_parquet(
    membership_file,
    columns=[
        "transition",
        "review_id"
    ]
)

dev_ids = (
    membership.loc[
        membership["transition"].isin(
            development_transitions
        ),
        ["review_id"]
    ]
    .drop_duplicates()
)


registry = pd.read_parquet(
    registry_file,
    columns=[
        "review_id",
        "comments",
        "word_count"
    ]
)


dev_reviews = dev_ids.merge(
    registry,
    on="review_id",
    how="left",
    validate="one_to_one"
)


print(
    "\nDevelopment unique reviews:",
    len(dev_reviews)
)

assert len(dev_reviews) == 1_362_408
assert dev_reviews["comments"].isna().sum() == 0


# ============================================================
# 2. FIXED PILOT SAMPLE
#
# random_state makes the audit reproducible.
# This sample is ONLY for checking whether the language model
# behaves sensibly before full-corpus inference.
# ============================================================

PILOT_N = 10_000
RANDOM_STATE = 612

pilot = (
    dev_reviews
    .sample(
        n=PILOT_N,
        random_state=RANDOM_STATE
    )
    .copy()
    .reset_index(drop=True)
)

print(
    "Pilot reviews:",
    len(pilot)
)

print(
    "Median pilot word count:",
    pilot["word_count"].median()
)


# ============================================================
# 3. LOAD LANGUAGE IDENTIFICATION MODEL
# ============================================================

MODEL_NAME = (
    "papluca/"
    "xlm-roberta-base-language-detection"
)

print(
    "\nLoading language model:",
    MODEL_NAME
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME
    )
)

model.eval()


# ============================================================
# 4. DEVICE
#
# CPU expected for your current machine.
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

print(
    "Device:",
    device
)


# ============================================================
# 5. LABEL MAP
# ============================================================

id2label = model.config.id2label

print(
    "\nModel labels:"
)

print(
    id2label
)


# ============================================================
# 6. BATCH INFERENCE
# ============================================================

BATCH_SIZE = 32

texts = (
    pilot["comments"]
    .astype(str)
    .tolist()
)

predicted_languages = []
predicted_confidences = []

start_time = time.time()


with torch.inference_mode():

    for start in range(
        0,
        len(texts),
        BATCH_SIZE
    ):

        batch_texts = texts[
            start:
            start + BATCH_SIZE
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(device)
            for k, v
            in encoded.items()
        }

        outputs = model(
            **encoded
        )

        probs = torch.softmax(
            outputs.logits,
            dim=-1
        )

        confidence, prediction = (
            probs.max(
                dim=-1
            )
        )

        predicted_languages.extend(
            [
                id2label[int(i)]
                for i in prediction.cpu()
            ]
        )

        predicted_confidences.extend(
            confidence.cpu().numpy()
        )


elapsed = (
    time.time()
    -
    start_time
)


# ============================================================
# 7. ATTACH RESULTS
# ============================================================

pilot[
    "detected_language"
] = predicted_languages

pilot[
    "language_confidence"
] = predicted_confidences


print(
    "\nInference seconds:",
    round(elapsed, 2)
)

print(
    "Reviews / second:",
    round(
        len(pilot) / elapsed,
        2
    )
)


# ============================================================
# 8. LANGUAGE DISTRIBUTION
# ============================================================

language_summary = (
    pilot
    .groupby(
        "detected_language"
    )
    .agg(

        n_reviews=(
            "review_id",
            "size"
        ),

        mean_confidence=(
            "language_confidence",
            "mean"
        ),

        median_confidence=(
            "language_confidence",
            "median"
        )

    )
    .reset_index()
)


language_summary[
    "pct_reviews"
] = (
    language_summary[
        "n_reviews"
    ]
    /
    len(pilot)
    * 100
)


language_summary = (
    language_summary
    .sort_values(
        "n_reviews",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n1. PILOT LANGUAGE DISTRIBUTION"
)

print("-" * 80)

display(
    language_summary
)


# ============================================================
# 9. ENGLISH VS NON-ENGLISH
# ============================================================

english_mask = (
    pilot[
        "detected_language"
    ]
    .astype(str)
    .str.lower()
    .isin([
        "en",
        "english"
    ])
)


english_pct = (
    english_mask.mean()
    * 100
)

non_english_pct = (
    (~english_mask).mean()
    * 100
)


print(
    "\n2. ENGLISH VS NON-ENGLISH"
)

print("-" * 80)

print(
    "English reviews:",
    int(
        english_mask.sum()
    )
)

print(
    "English %:",
    english_pct
)

print(
    "Non-English reviews:",
    int(
        (~english_mask).sum()
    )
)

print(
    "Non-English %:",
    non_english_pct
)


# ============================================================
# 10. CONFIDENCE AUDIT
# ============================================================

print(
    "\n3. CONFIDENCE AUDIT"
)

print("-" * 80)

print(
    "Mean confidence:",
    pilot[
        "language_confidence"
    ].mean()
)

print(
    "Median confidence:",
    pilot[
        "language_confidence"
    ].median()
)

print(
    "P10 confidence:",
    pilot[
        "language_confidence"
    ].quantile(0.10)
)

print(
    "% confidence < 0.50:",
    (
        pilot[
            "language_confidence"
        ] < 0.50
    ).mean() * 100
)

print(
    "% confidence < 0.70:",
    (
        pilot[
            "language_confidence"
        ] < 0.70
    ).mean() * 100
)

print(
    "% confidence < 0.90:",
    (
        pilot[
            "language_confidence"
        ] < 0.90
    ).mean() * 100
)


# ============================================================
# 11. LOW-CONFIDENCE EXAMPLES
#
# Display only a small number for manual sanity checking.
# ============================================================

low_conf_examples = (
    pilot
    .sort_values(
        "language_confidence"
    )
    [
        [
            "review_id",
            "word_count",
            "detected_language",
            "language_confidence",
            "comments"
        ]
    ]
    .head(20)
)


print(
    "\n4. LOW-CONFIDENCE EXAMPLES"
)

print("-" * 80)

display(
    low_conf_examples
)


# ============================================================
# 12. SHORT-TEXT CONFIDENCE
# ============================================================

pilot[
    "short_text"
] = (
    pilot[
        "word_count"
    ] < 5
)


short_text_summary = (
    pilot
    .groupby(
        "short_text"
    )
    .agg(

        n_reviews=(
            "review_id",
            "size"
        ),

        mean_confidence=(
            "language_confidence",
            "mean"
        ),

        median_confidence=(
            "language_confidence",
            "median"
        )

    )
    .reset_index()
)


print(
    "\n5. SHORT-TEXT CONFIDENCE"
)

print("-" * 80)

display(
    short_text_summary
)


# ============================================================
# 13. FINAL PILOT STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "STEP 24.7A-1b STATUS: "
    "PILOT LANGUAGE AUDIT COMPLETE"
)

print(
    "Do NOT process the full corpus yet."
)

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7A-1b — FAST PILOT LANGUAGE DETECTION
# CPU-SAFE VERSION
# ============================================================

import pandas as pd
import numpy as np
import torch
import time

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("=" * 80)
print("STEP 24.7A-1b — FAST PILOT LANGUAGE DETECTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. We already reconstructed dev_reviews previously.
#    Verify it is still available.
# ------------------------------------------------------------

print("Development unique reviews:", len(dev_reviews))

assert len(dev_reviews) == 1_362_408
assert dev_reviews["comments"].isna().sum() == 0


# ------------------------------------------------------------
# 2. Smaller reproducible pilot
# ------------------------------------------------------------

PILOT_N = 1_000
RANDOM_STATE = 612

pilot = (
    dev_reviews
    .sample(
        n=PILOT_N,
        random_state=RANDOM_STATE
    )
    .copy()
    .reset_index(drop=True)
)

print("Pilot reviews:", len(pilot))
print("Median word count:", pilot["word_count"].median())


# ------------------------------------------------------------
# 3. Reuse model if already loaded.
#    Otherwise load it.
# ------------------------------------------------------------

MODEL_NAME = "papluca/xlm-roberta-base-language-detection"

if "tokenizer" not in globals():

    print("\nLoading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )


if "model" not in globals():

    print("Loading model...")

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME
        )
    )


device = torch.device("cpu")

model.to(device)
model.eval()

id2label = model.config.id2label

print("Device:", device)
print("Model labels:", id2label)


# ------------------------------------------------------------
# 4. CPU settings
# ------------------------------------------------------------

BATCH_SIZE = 16
MAX_LENGTH = 192

texts = (
    pilot["comments"]
    .astype(str)
    .tolist()
)

predicted_languages = []
predicted_confidences = []

start_time = time.time()


# ------------------------------------------------------------
# 5. Inference WITH progress bar
# ------------------------------------------------------------

with torch.inference_mode():

    for start in tqdm(
        range(
            0,
            len(texts),
            BATCH_SIZE
        ),
        desc="Language detection"
    ):

        batch_texts = texts[
            start:
            start + BATCH_SIZE
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

        outputs = model(
            **encoded
        )

        probs = torch.softmax(
            outputs.logits,
            dim=-1
        )

        confidence, prediction = probs.max(
            dim=-1
        )

        predicted_languages.extend(
            [
                id2label[int(i)]
                for i in prediction.cpu()
            ]
        )

        predicted_confidences.extend(
            confidence.cpu().numpy()
        )


elapsed = time.time() - start_time


# ------------------------------------------------------------
# 6. Attach predictions
# ------------------------------------------------------------

pilot["detected_language"] = predicted_languages
pilot["language_confidence"] = predicted_confidences

print("\nInference complete.")
print("Seconds:", round(elapsed, 2))
print(
    "Reviews / second:",
    round(len(pilot) / elapsed, 2)
)


# ------------------------------------------------------------
# 7. Language distribution
# ------------------------------------------------------------

language_summary = (
    pilot
    .groupby("detected_language")
    .agg(
        n_reviews=("review_id", "size"),
        mean_confidence=("language_confidence", "mean"),
        median_confidence=("language_confidence", "median")
    )
    .reset_index()
)

language_summary["pct_reviews"] = (
    language_summary["n_reviews"]
    / len(pilot)
    * 100
)

language_summary = (
    language_summary
    .sort_values(
        "n_reviews",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n1. LANGUAGE DISTRIBUTION")
print("-" * 80)

display(language_summary)


# ------------------------------------------------------------
# 8. English vs non-English
# ------------------------------------------------------------

english_mask = (
    pilot["detected_language"] == "en"
)

print("\n2. ENGLISH VS NON-ENGLISH")
print("-" * 80)

print(
    "English:",
    int(english_mask.sum())
)

print(
    "English %:",
    english_mask.mean() * 100
)

print(
    "Non-English:",
    int((~english_mask).sum())
)

print(
    "Non-English %:",
    (~english_mask).mean() * 100
)


# ------------------------------------------------------------
# 9. Confidence
# ------------------------------------------------------------

print("\n3. CONFIDENCE")
print("-" * 80)

print(
    "Mean confidence:",
    pilot["language_confidence"].mean()
)

print(
    "Median confidence:",
    pilot["language_confidence"].median()
)

print(
    "% confidence < 0.50:",
    (
        pilot["language_confidence"] < 0.50
    ).mean() * 100
)

print(
    "% confidence < 0.70:",
    (
        pilot["language_confidence"] < 0.70
    ).mean() * 100
)

print(
    "% confidence < 0.90:",
    (
        pilot["language_confidence"] < 0.90
    ).mean() * 100
)


# ------------------------------------------------------------
# 10. Low-confidence examples
# ------------------------------------------------------------

print("\n4. LOW-CONFIDENCE EXAMPLES")
print("-" * 80)

display(
    pilot
    .sort_values("language_confidence")
    [
        [
            "review_id",
            "word_count",
            "detected_language",
            "language_confidence",
            "comments"
        ]
    ]
    .head(15)
)


# ------------------------------------------------------------
# 11. Status
# ------------------------------------------------------------

print("\n" + "=" * 80)
print(
    "STEP 24.7A-1b STATUS: "
    "FAST PILOT COMPLETE"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7B-0 — EMBEDDING ENVIRONMENT CHECK
# ============================================================

import torch
import transformers
import sentence_transformers
import os
import platform

print("=" * 80)
print("STEP 24.7B-0 — EMBEDDING ENVIRONMENT CHECK")
print("=" * 80)

print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print(
    "Sentence-Transformers version:",
    sentence_transformers.__version__
)

print("\nCOMPUTE")
print("-" * 80)

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CPU threads:", torch.get_num_threads())
print(
    "PyTorch interop threads:",
    torch.get_num_interop_threads()
)
print("Logical CPUs:", os.cpu_count())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("\n" + "=" * 80)
print(
    "STEP 24.7B-0 STATUS: "
    "ENVIRONMENT CHECK COMPLETE"
)
print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7B-1 — MULTILINGUAL EMBEDDING FEASIBILITY PILOT
#
# PURPOSE
# ------------------------------------------------------------
# Benchmark multilingual sentence embeddings on 1,000
# DEVELOPMENT reviews before full production encoding.
#
# DEVELOPMENT ONLY for model-design decision.
# T10 remains locked.
#
# NO embeddings are saved as production data here.
# ============================================================

import time
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

print("=" * 80)
print("STEP 24.7B-1 — MULTILINGUAL EMBEDDING FEASIBILITY PILOT")
print("=" * 80)


# ============================================================
# 1. VERIFY DEVELOPMENT CORPUS
# ============================================================

assert "dev_reviews" in globals(), (
    "dev_reviews is not available. "
    "Run Step 24.7A-0 / 24.7A-1b first."
)

assert len(dev_reviews) == 1_362_408
assert dev_reviews["comments"].isna().sum() == 0
assert dev_reviews["review_id"].duplicated().sum() == 0

print(
    "\nDevelopment unique reviews:",
    len(dev_reviews)
)


# ============================================================
# 2. REPRODUCIBLE 1,000-REVIEW PILOT
# ============================================================

PILOT_N = 1_000
RANDOM_STATE = 612

embedding_pilot = (
    dev_reviews
    .sample(
        n=PILOT_N,
        random_state=RANDOM_STATE
    )
    .copy()
    .reset_index(drop=True)
)

texts = (
    embedding_pilot["comments"]
    .astype(str)
    .tolist()
)

print(
    "Pilot reviews:",
    len(embedding_pilot)
)

print(
    "Median words:",
    embedding_pilot["word_count"].median()
)

print(
    "P90 words:",
    embedding_pilot["word_count"].quantile(0.90)
)

print(
    "Maximum words:",
    embedding_pilot["word_count"].max()
)


# ============================================================
# 3. MODEL
# ============================================================

MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print(
    "\nEmbedding model:",
    MODEL_NAME
)

print(
    "Loading model..."
)

embedding_model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

embedding_dimension = (
    embedding_model
    .get_sentence_embedding_dimension()
)

print(
    "Embedding dimension:",
    embedding_dimension
)

print(
    "Device:",
    embedding_model.device
)


# ============================================================
# 4. SMALL WARM-UP
#
# Avoid including one-time initialization overhead
# in the main benchmark.
# ============================================================

print(
    "\nRunning warm-up..."
)

_ = embedding_model.encode(
    texts[:32],
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=False
)

print(
    "Warm-up complete."
)


# ============================================================
# 5. BENCHMARK
# ============================================================

BATCH_SIZE = 32

print(
    "\nRunning 1,000-review embedding benchmark..."
)

start_time = time.time()

embeddings = embedding_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

elapsed = (
    time.time()
    -
    start_time
)

reviews_per_second = (
    len(texts)
    /
    elapsed
)


# ============================================================
# 6. BASIC OUTPUT QC
# ============================================================

print(
    "\n1. EMBEDDING OUTPUT"
)

print("-" * 80)

print(
    "Embedding shape:",
    embeddings.shape
)

print(
    "Embedding dtype:",
    embeddings.dtype
)

print(
    "NaN values:",
    int(
        np.isnan(
            embeddings
        ).sum()
    )
)

print(
    "Infinite values:",
    int(
        np.isinf(
            embeddings
        ).sum()
    )
)


row_norms = np.linalg.norm(
    embeddings,
    axis=1
)

print(
    "Zero-norm embeddings:",
    int(
        (row_norms == 0).sum()
    )
)

print(
    "Mean vector norm:",
    float(
        row_norms.mean()
    )
)

print(
    "Median vector norm:",
    float(
        np.median(
            row_norms
        )
    )
)


# ============================================================
# 7. SPEED
# ============================================================

print(
    "\n2. SPEED"
)

print("-" * 80)

print(
    "Elapsed seconds:",
    round(
        elapsed,
        2
    )
)

print(
    "Reviews / second:",
    round(
        reviews_per_second,
        2
    )
)


# ============================================================
# 8. ESTIMATED PRODUCTION RUNTIME
#
# Two estimates:
#
# A. Development corpus only
# B. Full frozen registry
#
# This is ONLY an estimate from pilot throughput.
# ============================================================

N_DEV = 1_362_408
N_REGISTRY = 1_447_835

dev_seconds = (
    N_DEV
    /
    reviews_per_second
)

registry_seconds = (
    N_REGISTRY
    /
    reviews_per_second
)


print(
    "\n3. ESTIMATED RUNTIME"
)

print("-" * 80)

print(
    "Estimated T1–T9 development corpus hours:",
    round(
        dev_seconds / 3600,
        2
    )
)

print(
    "Estimated full frozen registry hours:",
    round(
        registry_seconds / 3600,
        2
    )
)


# ============================================================
# 9. STORAGE ESTIMATE
#
# float32:
# N × embedding_dimension × 4 bytes
# ============================================================

bytes_per_float32 = 4

dev_storage_gb = (
    N_DEV
    *
    embedding_dimension
    *
    bytes_per_float32
    /
    (1024 ** 3)
)

registry_storage_gb = (
    N_REGISTRY
    *
    embedding_dimension
    *
    bytes_per_float32
    /
    (1024 ** 3)
)


print(
    "\n4. RAW EMBEDDING STORAGE ESTIMATE"
)

print("-" * 80)

print(
    "Embedding dimension:",
    embedding_dimension
)

print(
    "Estimated development float32 GB:",
    round(
        dev_storage_gb,
        3
    )
)

print(
    "Estimated full registry float32 GB:",
    round(
        registry_storage_gb,
        3
    )
)


# ============================================================
# 10. PILOT LENGTH × SPEED CONTEXT
# ============================================================

embedding_pilot[
    "word_count_band"
] = pd.cut(
    embedding_pilot[
        "word_count"
    ],
    bins=[
        -1,
        10,
        30,
        70,
        150,
        np.inf
    ],
    labels=[
        "0-10",
        "11-30",
        "31-70",
        "71-150",
        "151+"
    ]
)

length_summary = (
    embedding_pilot[
        "word_count_band"
    ]
    .value_counts(
        sort=False
    )
    .rename(
        "n_reviews"
    )
    .reset_index()
)

length_summary[
    "pct_reviews"
] = (
    length_summary[
        "n_reviews"
    ]
    /
    len(
        embedding_pilot
    )
    *
    100
)


print(
    "\n5. PILOT TEXT-LENGTH COMPOSITION"
)

print("-" * 80)

display(
    length_summary
)


# ============================================================
# 11. FINAL STATUS
# ============================================================

qc_pass = (
    embeddings.shape[0] == PILOT_N
    and
    embeddings.shape[1] == embedding_dimension
    and
    np.isnan(embeddings).sum() == 0
    and
    np.isinf(embeddings).sum() == 0
    and
    (row_norms == 0).sum() == 0
)


print(
    "\n" + "=" * 80
)

if qc_pass:

    print(
        "STEP 24.7B-1 STATUS: PASS — "
        "MULTILINGUAL EMBEDDING PILOT COMPLETE"
    )

else:

    print(
        "STEP 24.7B-1 STATUS: "
        "QC REVIEW REQUIRED"
    )

print(
    "Do NOT run full-registry encoding yet."
)

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7B-2A — TOKEN-LENGTH / TRUNCATION AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Determine whether review texts risk truncation under the
# selected multilingual MiniLM encoder.
#
# DEVELOPMENT ONLY.
# NO production embeddings are created.
# ============================================================

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("STEP 24.7B-2A — TOKEN-LENGTH / TRUNCATION AUDIT")
print("=" * 80)


# ============================================================
# 1. VERIFY MODEL AND DEVELOPMENT CORPUS
# ============================================================

assert "embedding_model" in globals(), (
    "embedding_model not found. "
    "Run Step 24.7B-1 first."
)

assert "dev_reviews" in globals()

assert len(dev_reviews) == 1_362_408

print(
    "\nDevelopment unique reviews:",
    len(dev_reviews)
)


# ============================================================
# 2. MODEL MAX SEQUENCE LENGTH
# ============================================================

model_max_seq_length = (
    embedding_model.max_seq_length
)

print(
    "Model max sequence length:",
    model_max_seq_length
)


# ============================================================
# 3. REPRODUCIBLE DEVELOPMENT SAMPLE
# ============================================================

SAMPLE_N = 10_000
RANDOM_STATE = 612

token_sample = (
    dev_reviews
    .sample(
        n=SAMPLE_N,
        random_state=RANDOM_STATE
    )
    .copy()
    .reset_index(drop=True)
)

texts = (
    token_sample["comments"]
    .astype(str)
    .tolist()
)

print(
    "Token audit sample:",
    len(token_sample)
)

print(
    "Median words:",
    token_sample["word_count"].median()
)

print(
    "P90 words:",
    token_sample["word_count"].quantile(0.90)
)

print(
    "P99 words:",
    token_sample["word_count"].quantile(0.99)
)


# ============================================================
# 4. ACCESS TOKENIZER
# ============================================================

tokenizer = embedding_model.tokenizer


# ============================================================
# 5. TOKENIZE WITHOUT TRUNCATION
#
# We need the natural token length BEFORE truncation.
# ============================================================

print(
    "\nTokenizing without truncation..."
)

start_time = time.time()

encoded = tokenizer(
    texts,
    add_special_tokens=True,
    truncation=False,
    padding=False,
    return_attention_mask=False
)

token_lengths = np.array(
    [
        len(ids)
        for ids in encoded["input_ids"]
    ],
    dtype=np.int32
)

elapsed = time.time() - start_time


token_sample[
    "token_length"
] = token_lengths


print(
    "Tokenization seconds:",
    round(elapsed, 2)
)


# ============================================================
# 6. DISTRIBUTION
# ============================================================

print(
    "\n1. TOKEN-LENGTH DISTRIBUTION"
)

print("-" * 80)

print(
    "Mean tokens:",
    token_lengths.mean()
)

print(
    "Median tokens:",
    np.median(token_lengths)
)

print(
    "P90 tokens:",
    np.quantile(token_lengths, 0.90)
)

print(
    "P95 tokens:",
    np.quantile(token_lengths, 0.95)
)

print(
    "P99 tokens:",
    np.quantile(token_lengths, 0.99)
)

print(
    "Maximum tokens:",
    token_lengths.max()
)


# ============================================================
# 7. TRUNCATION RISK
# ============================================================

would_truncate = (
    token_lengths
    >
    model_max_seq_length
)

n_truncate = int(
    would_truncate.sum()
)

pct_truncate = (
    would_truncate.mean()
    * 100
)


print(
    "\n2. MODEL-LIMIT TRUNCATION RISK"
)

print("-" * 80)

print(
    "Model max sequence length:",
    model_max_seq_length
)

print(
    "Reviews exceeding model limit:",
    n_truncate
)

print(
    "% exceeding model limit:",
    pct_truncate
)


# ============================================================
# 8. ADDITIONAL THRESHOLDS
# ============================================================

print(
    "\n3. TOKEN THRESHOLDS"
)

print("-" * 80)

for threshold in [
    64,
    128,
    192,
    256,
    384,
    512
]:

    n = int(
        (
            token_lengths
            >
            threshold
        ).sum()
    )

    pct = (
        (
            token_lengths
            >
            threshold
        ).mean()
        * 100
    )

    print(
        f"> {threshold:3d} tokens: "
        f"{n:5d} "
        f"({pct:.3f}%)"
    )


# ============================================================
# 9. LONGEST EXAMPLES
# ============================================================

print(
    "\n4. LONGEST TOKENIZED REVIEWS"
)

print("-" * 80)

display(
    token_sample
    .sort_values(
        "token_length",
        ascending=False
    )
    [
        [
            "review_id",
            "word_count",
            "token_length",
            "comments"
        ]
    ]
    .head(15)
)


# ============================================================
# 10. WORD/TOKEN RELATIONSHIP
# ============================================================

valid_word_mask = (
    token_sample[
        "word_count"
    ] > 0
)

token_sample.loc[
    valid_word_mask,
    "tokens_per_word"
] = (
    token_sample.loc[
        valid_word_mask,
        "token_length"
    ]
    /
    token_sample.loc[
        valid_word_mask,
        "word_count"
    ]
)


print(
    "\n5. TOKEN / WORD RELATIONSHIP"
)

print("-" * 80)

print(
    "Median tokens per word:",
    token_sample[
        "tokens_per_word"
    ].median()
)

print(
    "P90 tokens per word:",
    token_sample[
        "tokens_per_word"
    ].quantile(0.90)
)

print(
    "P99 tokens per word:",
    token_sample[
        "tokens_per_word"
    ].quantile(0.99)
)


# ============================================================
# 11. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "STEP 24.7B-2A STATUS: PASS — "
    "TOKEN-LENGTH AUDIT COMPLETE"
)

print(
    "Do NOT run full-registry encoding yet."
)

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7B-2B — 128 vs 256 TOKEN FEASIBILITY BENCHMARK
#
# PURPOSE
# ------------------------------------------------------------
# Test whether increasing the operational sequence length
# from 128 to 256 is technically valid and computationally
# feasible for the selected multilingual MiniLM encoder.
#
# DEVELOPMENT PILOT ONLY.
# NO production embeddings are saved.
# ============================================================

import time
import numpy as np

print("=" * 80)
print("STEP 24.7B-2B — 128 vs 256 TOKEN FEASIBILITY BENCHMARK")
print("=" * 80)


# ============================================================
# 1. MODEL CONFIGURATION
# ============================================================

print("\nMODEL CONFIGURATION")
print("-" * 80)

print(
    "SentenceTransformer max_seq_length:",
    embedding_model.max_seq_length
)

transformer_module = embedding_model[0]

print(
    "Underlying transformer:",
    transformer_module.auto_model.config.model_type
)

print(
    "max_position_embeddings:",
    transformer_module.auto_model.config.max_position_embeddings
)


# ============================================================
# 2. USE SAME REPRODUCIBLE 1,000-REVIEW PILOT
# ============================================================

PILOT_N = 1000
RANDOM_STATE = 612
BATCH_SIZE = 32

benchmark_sample = (
    dev_reviews
    .sample(
        n=PILOT_N,
        random_state=RANDOM_STATE
    )
    .copy()
    .reset_index(drop=True)
)

texts = (
    benchmark_sample["comments"]
    .astype(str)
    .tolist()
)

print(
    "\nBenchmark reviews:",
    len(texts)
)


# ============================================================
# 3. BENCHMARK FUNCTION
# ============================================================

def benchmark_length(max_seq_length):

    embedding_model.max_seq_length = max_seq_length

    # warm-up
    _ = embedding_model.encode(
        texts[:32],
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

    start = time.time()

    emb = embedding_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

    elapsed = time.time() - start

    speed = len(texts) / elapsed

    norms = np.linalg.norm(
        emb,
        axis=1
    )

    return {
        "max_seq_length": max_seq_length,
        "seconds": elapsed,
        "reviews_per_second": speed,
        "shape": emb.shape,
        "nan": int(np.isnan(emb).sum()),
        "inf": int(np.isinf(emb).sum()),
        "zero_norm": int((norms == 0).sum()),
        "embeddings": emb
    }


# ============================================================
# 4. RUN 128
# ============================================================

print("\nRunning max_seq_length = 128 ...")

result_128 = benchmark_length(
    128
)


# ============================================================
# 5. RUN 256
# ============================================================

print("\nRunning max_seq_length = 256 ...")

result_256 = benchmark_length(
    256
)


# ============================================================
# 6. RESULTS
# ============================================================

print("\n1. SPEED COMPARISON")
print("-" * 80)

for r in [
    result_128,
    result_256
]:

    print(
        f"max_seq_length={r['max_seq_length']}: "
        f"{r['seconds']:.2f} sec | "
        f"{r['reviews_per_second']:.2f} reviews/sec"
    )


speed_change_pct = (
    (
        result_256[
            "reviews_per_second"
        ]
        /
        result_128[
            "reviews_per_second"
        ]
    )
    - 1
) * 100

print(
    "\n256 vs 128 throughput change:",
    round(
        speed_change_pct,
        2
    ),
    "%"
)


# ============================================================
# 7. OUTPUT QC
# ============================================================

print("\n2. OUTPUT QC")
print("-" * 80)

for r in [
    result_128,
    result_256
]:

    print(
        "\nmax_seq_length =",
        r["max_seq_length"]
    )

    print(
        "Shape:",
        r["shape"]
    )

    print(
        "NaN:",
        r["nan"]
    )

    print(
        "Inf:",
        r["inf"]
    )

    print(
        "Zero norm:",
        r["zero_norm"]
    )


# ============================================================
# 8. EMBEDDING DIFFERENCE
#
# This is diagnostic only.
# Reviews <=128 tokens should be essentially unaffected,
# whereas long reviews may change because more text is retained.
# ============================================================

emb128 = result_128[
    "embeddings"
]

emb256 = result_256[
    "embeddings"
]

numerator = np.sum(
    emb128 * emb256,
    axis=1
)

denominator = (
    np.linalg.norm(
        emb128,
        axis=1
    )
    *
    np.linalg.norm(
        emb256,
        axis=1
    )
)

cosine_similarity = (
    numerator
    /
    denominator
)

benchmark_sample[
    "cosine_128_vs_256"
] = cosine_similarity


# Natural token lengths for the same sample
encoded = embedding_model.tokenizer(
    texts,
    add_special_tokens=True,
    truncation=False,
    padding=False,
    return_attention_mask=False
)

benchmark_sample[
    "token_length"
] = [
    len(ids)
    for ids in encoded[
        "input_ids"
    ]
]


short_mask = (
    benchmark_sample[
        "token_length"
    ]
    <= 128
)

long_mask = (
    benchmark_sample[
        "token_length"
    ]
    > 128
)


print(
    "\n3. 128 vs 256 EMBEDDING CONSISTENCY"
)

print("-" * 80)

print(
    "Reviews <=128 tokens:",
    int(
        short_mask.sum()
    )
)

print(
    "Mean cosine similarity <=128:",
    benchmark_sample.loc[
        short_mask,
        "cosine_128_vs_256"
    ].mean()
)

print(
    "\nReviews >128 tokens:",
    int(
        long_mask.sum()
    )
)

print(
    "Mean cosine similarity >128:",
    benchmark_sample.loc[
        long_mask,
        "cosine_128_vs_256"
    ].mean()
)


# ============================================================
# 9. FULL-REGISTRY RUNTIME ESTIMATE AT 256
# ============================================================

N_REGISTRY = 1_447_835

estimated_hours_256 = (
    N_REGISTRY
    /
    result_256[
        "reviews_per_second"
    ]
    /
    3600
)

print(
    "\n4. PRODUCTION ESTIMATE"
)

print("-" * 80)

print(
    "Estimated full-registry hours at 256:",
    round(
        estimated_hours_256,
        2
    )
)


# ============================================================
# 10. RESTORE 128 TEMPORARILY
#
# Do not freeze 256 until results are reviewed.
# ============================================================

embedding_model.max_seq_length = 128

print(
    "\nModel temporarily restored to:",
    embedding_model.max_seq_length
)


# ============================================================
# 11. STATUS
# ============================================================

print("\n" + "=" * 80)

print(
    "STEP 24.7B-2B STATUS: "
    "128-vs-256 FEASIBILITY BENCHMARK COMPLETE"
)

print(
    "Do NOT run production encoding yet."
)

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7B-2C — EXACT DEVELOPMENT TOKEN-LENGTH AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Obtain exact token-length / truncation statistics across
# ALL 1,362,408 unique T1–T9 development reviews.
#
# Selected operational sequence length candidate = 256.
#
# This step performs TOKENIZATION ONLY.
# NO embeddings are generated.
# T10 remains locked.
# ============================================================

import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

print("=" * 80)
print("STEP 24.7B-2C — EXACT DEVELOPMENT TOKEN-LENGTH AUDIT")
print("=" * 80)


# ============================================================
# 1. VERIFY DEVELOPMENT CORPUS
# ============================================================

assert "dev_reviews" in globals()
assert "embedding_model" in globals()

assert len(dev_reviews) == 1_362_408
assert dev_reviews["review_id"].duplicated().sum() == 0
assert dev_reviews["comments"].isna().sum() == 0

print(
    "\nDevelopment unique reviews:",
    len(dev_reviews)
)


# ============================================================
# 2. TOKENIZER
# ============================================================

tokenizer = embedding_model.tokenizer

OPERATIONAL_MAX_LENGTH = 256

print(
    "Operational max sequence length:",
    OPERATIONAL_MAX_LENGTH
)


# ============================================================
# 3. BATCH TOKENIZATION
#
# Tokenize without truncation so that natural token length
# can be measured exactly.
#
# We retain ONLY token lengths, not token IDs.
# ============================================================

BATCH_SIZE = 10_000

texts = (
    dev_reviews["comments"]
    .astype(str)
    .tolist()
)

token_lengths = np.empty(
    len(texts),
    dtype=np.int32
)

start_time = time.time()


for start in tqdm(
    range(
        0,
        len(texts),
        BATCH_SIZE
    ),
    desc="Exact token-length audit"
):

    end = min(
        start + BATCH_SIZE,
        len(texts)
    )

    batch_texts = texts[
        start:end
    ]

    encoded = tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False
    )

    token_lengths[
        start:end
    ] = [
        len(ids)
        for ids in encoded["input_ids"]
    ]


elapsed = time.time() - start_time


print(
    "\nTokenization complete."
)

print(
    "Elapsed seconds:",
    round(elapsed, 2)
)


# ============================================================
# 4. BASIC DISTRIBUTION
# ============================================================

print(
    "\n1. EXACT TOKEN-LENGTH DISTRIBUTION — T1–T9"
)

print("-" * 80)

print(
    "N reviews:",
    len(token_lengths)
)

print(
    "Mean tokens:",
    token_lengths.mean()
)

print(
    "Median tokens:",
    np.median(token_lengths)
)

print(
    "P90 tokens:",
    np.quantile(token_lengths, 0.90)
)

print(
    "P95 tokens:",
    np.quantile(token_lengths, 0.95)
)

print(
    "P99 tokens:",
    np.quantile(token_lengths, 0.99)
)

print(
    "P99.5 tokens:",
    np.quantile(token_lengths, 0.995)
)

print(
    "P99.9 tokens:",
    np.quantile(token_lengths, 0.999)
)

print(
    "Maximum tokens:",
    token_lengths.max()
)


# ============================================================
# 5. EXACT THRESHOLD COUNTS
# ============================================================

print(
    "\n2. EXACT TOKEN THRESHOLDS"
)

print("-" * 80)

threshold_results = []

for threshold in [
    64,
    128,
    192,
    256,
    384,
    512
]:

    n = int(
        (
            token_lengths
            >
            threshold
        ).sum()
    )

    pct = (
        n
        /
        len(token_lengths)
        *
        100
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "n_reviews_above": n,
            "pct_reviews_above": pct
        }
    )

threshold_table = pd.DataFrame(
    threshold_results
)

display(
    threshold_table
)


# ============================================================
# 6. PRIMARY PRODUCTION TRUNCATION RATE
# ============================================================

n_truncated_256 = int(
    (
        token_lengths
        >
        OPERATIONAL_MAX_LENGTH
    ).sum()
)

pct_truncated_256 = (
    n_truncated_256
    /
    len(token_lengths)
    *
    100
)


print(
    "\n3. PROPOSED PRODUCTION TRUNCATION — 256 TOKENS"
)

print("-" * 80)

print(
    "Reviews exceeding 256 tokens:",
    n_truncated_256
)

print(
    "% exceeding 256 tokens:",
    pct_truncated_256
)


# ============================================================
# 7. COMPARE 128 vs 256 EXACTLY
# ============================================================

n_128 = int(
    (
        token_lengths > 128
    ).sum()
)

pct_128 = (
    n_128
    /
    len(token_lengths)
    *
    100
)

print(
    "\n4. EXACT 128 vs 256 COMPARISON"
)

print("-" * 80)

print(
    "Would truncate at 128:",
    n_128,
    f"({pct_128:.4f}%)"
)

print(
    "Would truncate at 256:",
    n_truncated_256,
    f"({pct_truncated_256:.4f}%)"
)

reduction = (
    n_128
    -
    n_truncated_256
)

print(
    "Additional reviews preserved by 256:",
    reduction
)


# ============================================================
# 8. LONGEST REVIEWS
# ============================================================

audit = dev_reviews[
    [
        "review_id",
        "word_count",
        "comments"
    ]
].copy()

audit[
    "token_length"
] = token_lengths


print(
    "\n5. LONGEST DEVELOPMENT REVIEWS"
)

print("-" * 80)

display(
    audit
    .nlargest(
        20,
        "token_length"
    )
    [
        [
            "review_id",
            "word_count",
            "token_length",
            "comments"
        ]
    ]
)


# ============================================================
# 9. INTERNAL QC
# ============================================================

qc_pass = (
    len(token_lengths)
    ==
    len(dev_reviews)
    and
    token_lengths.min() > 0
)


print(
    "\n6. INTERNAL QC"
)

print("-" * 80)

print(
    "Length count matches development corpus:",
    len(token_lengths)
    ==
    len(dev_reviews)
)

print(
    "Non-positive token lengths:",
    int(
        (
            token_lengths <= 0
        ).sum()
    )
)


# ============================================================
# 10. STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

if qc_pass:

    print(
        "STEP 24.7B-2C STATUS: PASS — "
        "EXACT DEVELOPMENT TOKEN AUDIT COMPLETE"
    )

else:

    print(
        "STEP 24.7B-2C STATUS: QC REVIEW REQUIRED"
    )

print(
    "Do NOT run production embeddings yet."
)

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7C-0 — PRODUCTION EMBEDDING PRE-FLIGHT
#
# NO production encoding is performed in this step.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import torch

print("=" * 80)
print("STEP 24.7C-0 — PRODUCTION EMBEDDING PRE-FLIGHT")
print("=" * 80)


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

REGISTRY_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "review_text_registry_primary365_frozen.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "embedding_chunks_minilm256"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\n1. PATH CHECK")
print("-" * 80)

print("Registry:")
print(REGISTRY_PATH)

print("\nRegistry exists:", REGISTRY_PATH.exists())

print("\nCheckpoint folder:")
print(OUTPUT_DIR)

print("Checkpoint folder exists:", OUTPUT_DIR.exists())


# ------------------------------------------------------------
# 2. LOAD FROZEN REGISTRY
# ------------------------------------------------------------

registry = pd.read_parquet(
    REGISTRY_PATH,
    columns=[
        "review_id",
        "comments"
    ]
)

print("\n2. FROZEN REGISTRY QC")
print("-" * 80)

print("Rows:", len(registry))
print(
    "Unique review IDs:",
    registry["review_id"].nunique()
)

print(
    "Duplicate review IDs:",
    registry["review_id"].duplicated().sum()
)

print(
    "Missing comments:",
    registry["comments"].isna().sum()
)

empty_comments = (
    registry["comments"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty comments:",
    empty_comments
)


# ------------------------------------------------------------
# 3. EXPECTED REGISTRY SIZE
# ------------------------------------------------------------

EXPECTED_N = 1_447_835

print(
    "\nExpected registry rows:",
    EXPECTED_N
)

print(
    "Registry row count correct:",
    len(registry) == EXPECTED_N
)


# ------------------------------------------------------------
# 4. MODEL CHECK
# ------------------------------------------------------------

print("\n3. MODEL CHECK")
print("-" * 80)

print(
    "Model class:",
    type(embedding_model).__name__
)

print(
    "Embedding dimension:",
    embedding_model.get_embedding_dimension()
)

print(
    "Current max_seq_length:",
    embedding_model.max_seq_length
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


# ------------------------------------------------------------
# 5. SET FROZEN PRODUCTION LENGTH
# ------------------------------------------------------------

embedding_model.max_seq_length = 256

print(
    "\nProduction max_seq_length set to:",
    embedding_model.max_seq_length
)


# ------------------------------------------------------------
# 6. CHUNK DESIGN
# ------------------------------------------------------------

CHUNK_SIZE = 5_000

n_chunks = int(
    np.ceil(
        len(registry)
        /
        CHUNK_SIZE
    )
)

print("\n4. CHECKPOINT DESIGN")
print("-" * 80)

print(
    "Chunk size:",
    CHUNK_SIZE
)

print(
    "Expected chunks:",
    n_chunks
)


# ------------------------------------------------------------
# 7. EXISTING CHECKPOINTS
# ------------------------------------------------------------

existing_chunks = sorted(
    OUTPUT_DIR.glob(
        "embedding_chunk_*.parquet"
    )
)

print(
    "Existing completed chunks:",
    len(existing_chunks)
)

if existing_chunks:

    print(
        "First existing chunk:",
        existing_chunks[0].name
    )

    print(
        "Last existing chunk:",
        existing_chunks[-1].name
    )


# ------------------------------------------------------------
# 8. STORAGE ESTIMATE
# ------------------------------------------------------------

EMBED_DIM = 384

raw_embedding_bytes = (
    len(registry)
    *
    EMBED_DIM
    *
    4
)

raw_embedding_gb = (
    raw_embedding_bytes
    /
    1024**3
)

disk = shutil.disk_usage(
    OUTPUT_DIR
)

free_gb = (
    disk.free
    /
    1024**3
)

print("\n5. STORAGE CHECK")
print("-" * 80)

print(
    "Raw float32 embedding estimate (GB):",
    round(raw_embedding_gb, 3)
)

print(
    "Free disk space (GB):",
    round(free_gb, 2)
)


# ------------------------------------------------------------
# 9. FINAL PRE-FLIGHT QC
# ------------------------------------------------------------

preflight_pass = all(
    [
        REGISTRY_PATH.exists(),
        len(registry) == EXPECTED_N,
        registry["review_id"].nunique()
            == EXPECTED_N,
        registry["review_id"].duplicated().sum()
            == 0,
        registry["comments"].isna().sum()
            == 0,
        empty_comments == 0,
        embedding_model.get_embedding_dimension()
            == 384,
        embedding_model.max_seq_length
            == 256,
        free_gb
            > raw_embedding_gb * 2
    ]
)


print("\n" + "=" * 80)

if preflight_pass:

    print(
        "STEP 24.7C-0 STATUS: PASS — "
        "READY FOR CHECKPOINTED PRODUCTION ENCODING"
    )

else:

    print(
        "STEP 24.7C-0 STATUS: "
        "REVIEW REQUIRED — DO NOT START PRODUCTION"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7C-1 — CHECKPOINTED PRODUCTION REVIEW ENCODING
#
# Frozen specification:
#   Encoder:
#     paraphrase-multilingual-MiniLM-L12-v2
#   Embedding dimension: 384
#   max_seq_length: 256
#   Encoding unit: individual review
#   Output dtype: float32
#
# Design:
#   - 5,000 reviews per checkpoint
#   - completed chunks are skipped automatically
#   - temporary file is validated before final rename
#   - safe to resume after interruption
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


print("=" * 80)
print("STEP 24.7C-1 — CHECKPOINTED PRODUCTION REVIEW ENCODING")
print("=" * 80)


# ============================================================
# 1. FROZEN SETTINGS
# ============================================================

EXPECTED_N = 1_447_835
EMBED_DIM = 384
MAX_SEQ_LENGTH = 256

CHUNK_SIZE = 5_000
ENCODE_BATCH_SIZE = 32


embedding_model.max_seq_length = MAX_SEQ_LENGTH


# ============================================================
# 2. INPUT / OUTPUT CHECK
# ============================================================

assert len(registry) == EXPECTED_N
assert registry["review_id"].nunique() == EXPECTED_N
assert registry["review_id"].duplicated().sum() == 0
assert registry["comments"].isna().sum() == 0

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

n_chunks = int(
    np.ceil(
        len(registry) / CHUNK_SIZE
    )
)


print("\nFrozen registry rows:", len(registry))
print("Embedding dimension:", EMBED_DIM)
print("max_seq_length:", embedding_model.max_seq_length)
print("Chunk size:", CHUNK_SIZE)
print("Encoding batch size:", ENCODE_BATCH_SIZE)
print("Total expected chunks:", n_chunks)


# ============================================================
# 3. HELPER — VALIDATE EXISTING CHUNK
# ============================================================

def validate_chunk(
    path,
    expected_ids
):

    try:

        df = pd.read_parquet(path)

        expected_rows = len(
            expected_ids
        )

        if len(df) != expected_rows:
            return False

        if "review_id" not in df.columns:
            return False

        embedding_cols = [
            f"emb_{i:03d}"
            for i in range(
                EMBED_DIM
            )
        ]

        expected_cols = (
            ["review_id"]
            +
            embedding_cols
        )

        if list(df.columns) != expected_cols:
            return False

        if df["review_id"].duplicated().any():
            return False

        if (
            df["review_id"]
            .astype(str)
            .tolist()
            !=
            pd.Series(
                expected_ids
            )
            .astype(str)
            .tolist()
        ):
            return False

        matrix = df[
            embedding_cols
        ].to_numpy(
            dtype=np.float32
        )

        if matrix.shape != (
            expected_rows,
            EMBED_DIM
        ):
            return False

        if not np.isfinite(
            matrix
        ).all():
            return False

        norms = np.linalg.norm(
            matrix,
            axis=1
        )

        if (
            norms == 0
        ).any():
            return False

        return True

    except Exception:

        return False


# ============================================================
# 4. PRODUCTION LOOP
# ============================================================

production_start = time.time()

completed_this_run = 0
skipped_existing = 0


for chunk_idx in range(
    n_chunks
):

    start = (
        chunk_idx
        *
        CHUNK_SIZE
    )

    end = min(
        start + CHUNK_SIZE,
        len(registry)
    )

    chunk = (
        registry
        .iloc[start:end]
        .copy()
        .reset_index(drop=True)
    )

    expected_ids = (
        chunk["review_id"]
        .tolist()
    )


    final_path = (
        OUTPUT_DIR
        /
        f"embedding_chunk_{chunk_idx:04d}.parquet"
    )

    temp_path = (
        OUTPUT_DIR
        /
        f"embedding_chunk_{chunk_idx:04d}.tmp.parquet"
    )


    # --------------------------------------------------------
    # Resume logic
    # --------------------------------------------------------

    if final_path.exists():

        if validate_chunk(
            final_path,
            expected_ids
        ):

            skipped_existing += 1

            print(
                f"[{chunk_idx + 1:03d}/{n_chunks}] "
                f"SKIP — valid checkpoint exists"
            )

            continue

        else:

            print(
                f"[{chunk_idx + 1:03d}/{n_chunks}] "
                f"Existing checkpoint failed validation. "
                f"Rebuilding..."
            )

            final_path.unlink()


    # Remove incomplete temporary file from an interrupted run
    if temp_path.exists():

        temp_path.unlink()


    # --------------------------------------------------------
    # Encode
    # --------------------------------------------------------

    texts = (
        chunk["comments"]
        .astype(str)
        .tolist()
    )

    chunk_start = time.time()


    embeddings = embedding_model.encode(
        texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False
    )


    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # In-memory QC
    # --------------------------------------------------------

    assert embeddings.shape == (
        len(chunk),
        EMBED_DIM
    )

    assert np.isfinite(
        embeddings
    ).all()

    norms = np.linalg.norm(
        embeddings,
        axis=1
    )

    assert not (
        norms == 0
    ).any()


    # --------------------------------------------------------
    # Build checkpoint
    # --------------------------------------------------------

    embedding_cols = [
        f"emb_{i:03d}"
        for i in range(
            EMBED_DIM
        )
    ]


    output_chunk = pd.DataFrame(
        embeddings,
        columns=embedding_cols
    )

    output_chunk.insert(
        0,
        "review_id",
        chunk["review_id"].values
    )


    # --------------------------------------------------------
    # Save TEMPORARY checkpoint first
    # --------------------------------------------------------

    output_chunk.to_parquet(
        temp_path,
        index=False
    )


    # --------------------------------------------------------
    # Validate temporary checkpoint
    # --------------------------------------------------------

    if not validate_chunk(
        temp_path,
        expected_ids
    ):

        raise RuntimeError(
            f"Chunk {chunk_idx:04d} "
            f"failed post-save validation."
        )


    # --------------------------------------------------------
    # Promote TEMP -> FINAL
    # --------------------------------------------------------

    temp_path.replace(
        final_path
    )


    completed_this_run += 1

    elapsed_chunk = (
        time.time()
        -
        chunk_start
    )

    speed = (
        len(chunk)
        /
        elapsed_chunk
    )


    total_elapsed = (
        time.time()
        -
        production_start
    )


    chunks_processed = (
        completed_this_run
        +
        skipped_existing
    )


    remaining_chunks = (
        n_chunks
        -
        chunks_processed
    )


    estimated_remaining_hours = (
        (
            total_elapsed
            /
            max(
                completed_this_run,
                1
            )
        )
        *
        remaining_chunks
        /
        3600
    )


    print(
        f"[{chunk_idx + 1:03d}/{n_chunks}] "
        f"SAVED | "
        f"rows={len(chunk):,} | "
        f"{elapsed_chunk:.1f}s | "
        f"{speed:.2f} reviews/s | "
        f"ETA~{estimated_remaining_hours:.2f}h"
    )


    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    del embeddings
    del output_chunk
    del chunk
    del texts

    gc.collect()


# ============================================================
# 5. END-OF-RUN SUMMARY
# ============================================================

total_seconds = (
    time.time()
    -
    production_start
)


final_chunks = sorted(
    OUTPUT_DIR.glob(
        "embedding_chunk_*.parquet"
    )
)


temp_chunks = sorted(
    OUTPUT_DIR.glob(
        "*.tmp.parquet"
    )
)


print("\n" + "=" * 80)
print("PRODUCTION ENCODING RUN SUMMARY")
print("=" * 80)

print(
    "Completed during this run:",
    completed_this_run
)

print(
    "Valid existing chunks skipped:",
    skipped_existing
)

print(
    "Final checkpoint files present:",
    len(final_chunks)
)

print(
    "Temporary files remaining:",
    len(temp_chunks)
)

print(
    "Elapsed hours:",
    round(
        total_seconds / 3600,
        2
    )
)


if (
    len(final_chunks)
    ==
    n_chunks
    and
    len(temp_chunks)
    ==
    0
):

    print(
        "\nSTEP 24.7C-1 STATUS: "
        "ENCODING COMPLETE — "
        "READY FOR FINAL QC"
    )

else:

    print(
        "\nSTEP 24.7C-1 STATUS: "
        "PARTIAL / INTERRUPTED — "
        "SAFE TO RESUME"
    )


print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7C-2 — FINAL PRODUCTION EMBEDDING QC
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time

print("=" * 80)
print("STEP 24.7C-2 — FINAL PRODUCTION EMBEDDING QC")
print("=" * 80)

EXPECTED_N = 1_447_835
EXPECTED_CHUNKS = 290
EXPECTED_DIM = 384
EXPECTED_LAST_CHUNK = 2_835

embedding_cols = [
    f"emb_{i:03d}"
    for i in range(EXPECTED_DIM)
]

files = sorted(
    OUTPUT_DIR.glob("embedding_chunk_*.parquet")
)

print("\n1. CHECKPOINT INVENTORY")
print("-" * 80)

print("Checkpoint files:", len(files))
print("Expected:", EXPECTED_CHUNKS)

assert len(files) == EXPECTED_CHUNKS

expected_names = [
    f"embedding_chunk_{i:04d}.parquet"
    for i in range(EXPECTED_CHUNKS)
]

actual_names = [
    f.name
    for f in files
]

print(
    "Continuous chunk sequence:",
    actual_names == expected_names
)

assert actual_names == expected_names


# ============================================================
# 2. SCAN ALL CHECKPOINTS
# ============================================================

start_time = time.time()

all_ids = []

total_rows = 0
total_nan = 0
total_inf = 0
total_zero_norm = 0

wrong_schema = []
wrong_rows = []

for i, path in enumerate(files):

    df = pd.read_parquet(path)

    expected_rows = (
        EXPECTED_LAST_CHUNK
        if i == EXPECTED_CHUNKS - 1
        else 5_000
    )

    if len(df) != expected_rows:
        wrong_rows.append(
            (path.name, len(df), expected_rows)
        )

    expected_columns = (
        ["review_id"] + embedding_cols
    )

    if list(df.columns) != expected_columns:
        wrong_schema.append(path.name)
        continue

    all_ids.append(
        df["review_id"].copy()
    )

    X = df[
        embedding_cols
    ].to_numpy(
        dtype=np.float32
    )

    total_rows += len(df)

    total_nan += int(
        np.isnan(X).sum()
    )

    total_inf += int(
        np.isinf(X).sum()
    )

    norms = np.linalg.norm(
        X,
        axis=1
    )

    total_zero_norm += int(
        (norms == 0).sum()
    )

    if (
        (i + 1) % 25 == 0
        or i == EXPECTED_CHUNKS - 1
    ):

        print(
            f"Checked {i + 1}/{EXPECTED_CHUNKS} chunks"
        )


# ============================================================
# 3. REVIEW-ID QC
# ============================================================

ids = pd.concat(
    all_ids,
    ignore_index=True
)

print("\n2. GLOBAL EMBEDDING QC")
print("-" * 80)

print("Total embedding rows:", total_rows)
print("Expected rows:", EXPECTED_N)

print(
    "Unique review IDs:",
    ids.nunique()
)

print(
    "Duplicate review IDs:",
    ids.duplicated().sum()
)

print(
    "Missing review IDs:",
    ids.isna().sum()
)

print("NaN embedding values:", total_nan)
print("Inf embedding values:", total_inf)
print("Zero-norm embeddings:", total_zero_norm)

print(
    "Wrong-schema chunks:",
    len(wrong_schema)
)

print(
    "Wrong-row-count chunks:",
    len(wrong_rows)
)

if wrong_schema:
    print("Wrong schema:", wrong_schema)

if wrong_rows:
    print("Wrong rows:", wrong_rows)


# ============================================================
# 4. REGISTRY COVERAGE
# ============================================================

registry_ids = (
    registry["review_id"]
    .astype(str)
)

embedding_ids = (
    ids
    .astype(str)
)

registry_set = set(
    registry_ids
)

embedding_set = set(
    embedding_ids
)

missing_from_embeddings = (
    registry_set
    -
    embedding_set
)

unexpected_embeddings = (
    embedding_set
    -
    registry_set
)

print("\n3. FROZEN REGISTRY COVERAGE")
print("-" * 80)

print(
    "Registry IDs:",
    len(registry_set)
)

print(
    "Embedding IDs:",
    len(embedding_set)
)

print(
    "Registry IDs missing embeddings:",
    len(missing_from_embeddings)
)

print(
    "Unexpected embedding IDs:",
    len(unexpected_embeddings)
)


# ============================================================
# 5. FINAL PASS / FAIL
# ============================================================

qc_pass = all([
    len(files) == EXPECTED_CHUNKS,
    actual_names == expected_names,
    total_rows == EXPECTED_N,
    ids.nunique() == EXPECTED_N,
    ids.duplicated().sum() == 0,
    ids.isna().sum() == 0,
    total_nan == 0,
    total_inf == 0,
    total_zero_norm == 0,
    len(wrong_schema) == 0,
    len(wrong_rows) == 0,
    len(missing_from_embeddings) == 0,
    len(unexpected_embeddings) == 0
])

elapsed = time.time() - start_time

print("\n4. QC RUNTIME")
print("-" * 80)

print(
    "Elapsed seconds:",
    round(elapsed, 2)
)

print("\n" + "=" * 80)

if qc_pass:
    print(
        "STEP 24.7C-2 STATUS: PASS — "
        "PRODUCTION REVIEW EMBEDDING STORE COMPLETE"
    )
else:
    print(
        "STEP 24.7C-2 STATUS: FAIL — "
        "REVIEW QC OUTPUT BEFORE PROCEEDING"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7D-0 — TEMPORAL AGGREGATION PRE-FLIGHT
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 80)
print("STEP 24.7D-0 — TEMPORAL AGGREGATION PRE-FLIGHT")
print("=" * 80)

MEMBERSHIP_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "review_membership_primary365_frozen.parquet"
)

print("\nMembership file:")
print(MEMBERSHIP_PATH)

print(
    "Exists:",
    MEMBERSHIP_PATH.exists()
)

membership = pd.read_parquet(
    MEMBERSHIP_PATH
)

print("\n1. SCHEMA")
print("-" * 80)

print(
    "Columns:",
    membership.columns.tolist()
)

print(
    "\nShape:",
    membership.shape
)

print(
    "\nDtypes:"
)

print(
    membership.dtypes
)


print("\n2. KEY QC")
print("-" * 80)

print(
    "Rows:",
    len(membership)
)

print(
    "Unique review IDs:",
    membership["review_id"].nunique()
)

print(
    "Missing review_id:",
    membership["review_id"].isna().sum()
)

print(
    "Missing listing_id:",
    membership["listing_id"].isna().sum()
)


# ------------------------------------------------------------
# Identify transition/origin column
# ------------------------------------------------------------

candidate_cols = [
    c
    for c in membership.columns
    if (
        "transition" in c.lower()
        or
        "anchor" in c.lower()
        or
        "snapshot" in c.lower()
    )
]

print(
    "\nPotential temporal-key columns:",
    candidate_cols
)


print("\n3. SAMPLE")
print("-" * 80)

display(
    membership.head(10)
)


print("\n4. REVIEW-ID COVERAGE AGAINST EMBEDDING STORE")
print("-" * 80)

membership_ids = set(
    membership["review_id"]
    .astype(str)
)

embedding_ids = set(
    embedding_ids
)

print(
    "Membership unique review IDs:",
    len(membership_ids)
)

print(
    "Embedding unique review IDs:",
    len(embedding_ids)
)

print(
    "Membership IDs missing embeddings:",
    len(
        membership_ids
        -
        embedding_ids
    )
)


print("\n" + "=" * 80)

if (
    len(membership) == 7_334_068
    and
    membership["review_id"].isna().sum() == 0
    and
    membership["listing_id"].isna().sum() == 0
    and
    len(
        membership_ids
        -
        embedding_ids
    ) == 0
):

    print(
        "STEP 24.7D-0 STATUS: PASS — "
        "READY TO DESIGN TEMPORAL MEAN POOLING"
    )

else:

    print(
        "STEP 24.7D-0 STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.7D-1 — TEMPORAL LISTING-LEVEL MEAN POOLING
#
# PURPOSE
# ------------------------------------------------------------
# Convert review-level MiniLM embeddings into one semantic
# representation per:
#
#   transition × snapshot_date × listing_id
#
# Primary aggregation:
#   unweighted mean of eligible 365-day review embeddings
#
# IMPORTANT
# ------------------------------------------------------------
# - Processes one transition at a time.
# - Does NOT construct a 7.33M × 384 merged matrix at once.
# - review_count_365d is retained for QC/provenance.
# - Zero-review listings are NOT created here.
# - Zero-review listings will be restored later when merging
#   with the complete listing-origin backbone.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time


print("=" * 80)
print("STEP 24.7D-1 — TEMPORAL LISTING-LEVEL MEAN POOLING")
print("=" * 80)


# ============================================================
# 1. FROZEN SETTINGS
# ============================================================

EXPECTED_MEMBERSHIP_ROWS = 7_334_068
EXPECTED_EMBEDDING_ROWS = 1_447_835
EXPECTED_EMBEDDING_CHUNKS = 290

EMBED_DIM = 384


embedding_cols = [
    f"emb_{i:03d}"
    for i in range(
        EMBED_DIM
    )
]


# ============================================================
# 2. INPUT / OUTPUT PATHS
# ============================================================

MEMBERSHIP_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "review_membership_primary365_frozen.parquet"
)


EMBEDDING_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "embedding_chunks_minilm256"
)


AGG_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "listing_review_embeddings_mean365"
)


AGG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nMembership:")
print(MEMBERSHIP_PATH)

print("\nEmbedding folder:")
print(EMBEDDING_DIR)

print("\nOutput folder:")
print(AGG_DIR)


# ============================================================
# 3. LOAD / VERIFY MEMBERSHIP
# ============================================================

membership = pd.read_parquet(
    MEMBERSHIP_PATH
)


required_membership_cols = [
    "transition",
    "snapshot_date",
    "review_id",
    "listing_id"
]


missing_membership_cols = [
    c
    for c in required_membership_cols
    if c not in membership.columns
]


if missing_membership_cols:

    raise RuntimeError(
        "Missing membership columns: "
        + str(missing_membership_cols)
    )


assert len(membership) == EXPECTED_MEMBERSHIP_ROWS


assert (
    membership[
        required_membership_cols
    ]
    .isna()
    .sum()
    .sum()
    ==
    0
)


duplicate_membership_pairs = (
    membership[
        [
            "transition",
            "review_id"
        ]
    ]
    .duplicated()
    .sum()
)


print("\nMembership rows:")
print(f"{len(membership):,}")

print(
    "Unique membership review IDs:",
    f"{membership['review_id'].nunique():,}"
)

print(
    "Duplicate transition-review pairs:",
    duplicate_membership_pairs
)


if duplicate_membership_pairs != 0:

    raise RuntimeError(
        "Duplicate transition-review pairs detected."
    )


# ============================================================
# 4. VERIFY EMBEDDING CHECKPOINT INVENTORY
# ============================================================

files = sorted(
    EMBEDDING_DIR.glob(
        "embedding_chunk_*.parquet"
    )
)


print(
    "\nEmbedding checkpoint files:",
    len(files)
)


if len(files) != EXPECTED_EMBEDDING_CHUNKS:

    raise RuntimeError(
        f"Expected {EXPECTED_EMBEDDING_CHUNKS} "
        f"embedding chunks, found {len(files)}."
    )


expected_names = [
    f"embedding_chunk_{i:04d}.parquet"
    for i in range(
        EXPECTED_EMBEDDING_CHUNKS
    )
]


actual_names = [
    path.name
    for path in files
]


if actual_names != expected_names:

    raise RuntimeError(
        "Embedding chunk sequence is incomplete "
        "or incorrectly ordered."
    )


print(
    "Embedding chunk sequence:",
    "PASS"
)


# ============================================================
# 5. TRANSITION ORDER
# ============================================================

transitions = [
    f"T{i}"
    for i in range(
        1,
        11
    )
]


print(
    "\nTransitions:",
    transitions
)

print(
    "Embedding dimensions:",
    EMBED_DIM
)


# ============================================================
# 6. HELPER:
# LOAD ONLY EMBEDDINGS REQUIRED FOR ONE TRANSITION
# ============================================================

def load_embeddings_for_ids(
    required_ids
):

    required_ids = set(
        int(x)
        for x in required_ids
    )


    pieces = []


    for path in files:

        # -----------------------------------------------
        # First read review_id only.
        # -----------------------------------------------

        ids_part = pd.read_parquet(
            path,
            columns=[
                "review_id"
            ]
        )


        mask = (
            ids_part[
                "review_id"
            ]
            .isin(
                required_ids
            )
        )


        if not mask.any():

            del ids_part
            continue


        # -----------------------------------------------
        # Only load the full chunk if it contains
        # reviews required by this transition.
        # -----------------------------------------------

        full_part = pd.read_parquet(
            path
        )


        full_part = (
            full_part.loc[
                full_part[
                    "review_id"
                ].isin(
                    required_ids
                )
            ]
            .copy()
        )


        pieces.append(
            full_part
        )


        del ids_part
        del full_part


    if not pieces:

        raise RuntimeError(
            "No embeddings found for required review IDs."
        )


    out = pd.concat(
        pieces,
        ignore_index=True
    )


    # -----------------------------------------------
    # Internal embedding QC
    # -----------------------------------------------

    if (
        out["review_id"]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            "Duplicate review IDs detected "
            "in loaded embeddings."
        )


    return out


# ============================================================
# 7. PROCESS T1–T10
# ============================================================

summary_rows = []

overall_start = time.time()


for transition in transitions:

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "Processing",
        transition
    )

    print(
        "=" * 80
    )


    transition_start = time.time()


    # ========================================================
    # 7A. MEMBERSHIP FOR THIS TRANSITION
    # ========================================================

    m = (
        membership.loc[
            membership[
                "transition"
            ]
            ==
            transition,
            [
                "transition",
                "snapshot_date",
                "listing_id",
                "review_id"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    membership_rows = len(m)


    unique_review_ids = (
        m[
            "review_id"
        ]
        .nunique()
    )


    listing_groups = (
        m[
            [
                "snapshot_date",
                "listing_id"
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )


    snapshot_dates = (
        m[
            "snapshot_date"
        ]
        .drop_duplicates()
    )


    if len(snapshot_dates) != 1:

        raise RuntimeError(
            f"{transition}: expected exactly "
            f"one snapshot_date, found "
            f"{len(snapshot_dates)}."
        )


    snapshot_date = (
        snapshot_dates.iloc[0]
    )


    print(
        "Membership rows:",
        f"{membership_rows:,}"
    )

    print(
        "Unique reviews:",
        f"{unique_review_ids:,}"
    )

    print(
        "Listing groups with text:",
        f"{listing_groups:,}"
    )

    print(
        "Snapshot date:",
        snapshot_date
    )


    # ========================================================
    # 7B. LOAD REQUIRED EMBEDDINGS
    # ========================================================

    required_ids = (
        m[
            "review_id"
        ]
        .unique()
    )


    emb = load_embeddings_for_ids(
        required_ids
    )


    # ========================================================
    # 7C. EMBEDDING COVERAGE QC
    # ========================================================

    loaded_unique_reviews = (
        emb[
            "review_id"
        ]
        .nunique()
    )


    print(
        "Loaded embedding rows:",
        f"{len(emb):,}"
    )


    if (
        loaded_unique_reviews
        !=
        unique_review_ids
    ):

        raise RuntimeError(
            f"{transition}: embedding coverage mismatch. "
            f"Expected {unique_review_ids:,}, "
            f"loaded {loaded_unique_reviews:,}."
        )


    if (
        emb[
            "review_id"
        ]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            f"{transition}: duplicate review IDs "
            f"in loaded embeddings."
        )


    # ========================================================
    # 7D. JOIN MEMBERSHIP -> EMBEDDINGS
    # ========================================================

    joined = m.merge(
        emb,
        on="review_id",
        how="left",
        validate="many_to_one"
    )


    if len(joined) != membership_rows:

        raise RuntimeError(
            f"{transition}: row count changed "
            f"after membership-embedding merge."
        )


    missing_embedding_rows = (
        joined[
            embedding_cols
        ]
        .isna()
        .any(
            axis=1
        )
        .sum()
    )


    if missing_embedding_rows != 0:

        raise RuntimeError(
            f"{transition}: "
            f"{missing_embedding_rows:,} membership rows "
            f"have missing embeddings after merge."
        )


    # ========================================================
    # 7E. REVIEW COUNT PER LISTING-ORIGIN
    # ========================================================

    group_keys = [
        "transition",
        "snapshot_date",
        "listing_id"
    ]


    counts = (
        joined
        .groupby(
            group_keys,
            sort=False
        )
        .size()
        .rename(
            "review_count_365d"
        )
        .reset_index()
    )


    # ========================================================
    # 7F. PRIMARY AGGREGATION:
    # UNWEIGHTED MEAN POOLING
    # ========================================================

    pooled_embeddings = (
        joined
        .groupby(
            group_keys,
            sort=False
        )[
            embedding_cols
        ]
        .mean()
        .reset_index()
    )


    # ========================================================
    # 7G. JOIN COUNTS + POOLED EMBEDDINGS
    #
    # IMPORTANT FIX:
    # how="left"
    # validate="one_to_one"
    # ========================================================

    pooled = counts.merge(
        pooled_embeddings,
        on=group_keys,
        how="left",
        validate="one_to_one"
    )


    # ========================================================
    # 7H. OUTPUT KEY QC
    # ========================================================

    if len(pooled) != listing_groups:

        raise RuntimeError(
            f"{transition}: pooled row count mismatch. "
            f"Expected {listing_groups:,}, "
            f"found {len(pooled):,}."
        )


    duplicate_listing_origin_keys = (
        pooled[
            group_keys
        ]
        .duplicated()
        .sum()
    )


    if duplicate_listing_origin_keys != 0:

        raise RuntimeError(
            f"{transition}: duplicate "
            f"listing-origin keys detected."
        )


    # ========================================================
    # 7I. EMBEDDING QC
    # ========================================================

    X = (
        pooled[
            embedding_cols
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    nan_values = int(
        np.isnan(
            X
        )
        .sum()
    )


    inf_values = int(
        np.isinf(
            X
        )
        .sum()
    )


    norms = np.linalg.norm(
        X,
        axis=1
    )


    zero_norm = int(
        (
            norms == 0
        )
        .sum()
    )


    if nan_values != 0:

        raise RuntimeError(
            f"{transition}: "
            f"NaN values detected in pooled embeddings."
        )


    if inf_values != 0:

        raise RuntimeError(
            f"{transition}: "
            f"Inf values detected in pooled embeddings."
        )


    if zero_norm != 0:

        raise RuntimeError(
            f"{transition}: "
            f"zero-norm pooled embeddings detected."
        )


    # ========================================================
    # 7J. STORE EMBEDDINGS AS FLOAT32
    # ========================================================

    pooled[
        embedding_cols
    ] = (
        pooled[
            embedding_cols
        ]
        .astype(
            np.float32
        )
    )


    pooled[
        "review_count_365d"
    ] = (
        pooled[
            "review_count_365d"
        ]
        .astype(
            np.int32
        )
    )


    # ========================================================
    # 7K. SAVE TRANSITION OUTPUT
    # ========================================================

    output_path = (
        AGG_DIR
        /
        f"listing_review_mean365_{transition}.parquet"
    )


    pooled.to_parquet(
        output_path,
        index=False
    )


    # ========================================================
    # 7L. SAVE / RELOAD VALIDATION
    # ========================================================

    check = pd.read_parquet(
        output_path
    )


    if len(check) != len(pooled):

        raise RuntimeError(
            f"{transition}: "
            f"save/reload row mismatch."
        )


    if (
        check[
            group_keys
        ]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            f"{transition}: duplicate keys "
            f"after save/reload."
        )


    if (
        check[
            embedding_cols
        ]
        .isna()
        .any()
        .any()
    ):

        raise RuntimeError(
            f"{transition}: missing embedding values "
            f"after save/reload."
        )


    # ========================================================
    # 7M. SUMMARY STATISTICS
    # ========================================================

    elapsed = (
        time.time()
        -
        transition_start
    )


    mean_reviews = (
        membership_rows
        /
        listing_groups
    )


    median_reviews = float(
        pooled[
            "review_count_365d"
        ]
        .median()
    )


    p90_reviews = float(
        pooled[
            "review_count_365d"
        ]
        .quantile(
            0.90
        )
    )


    p99_reviews = float(
        pooled[
            "review_count_365d"
        ]
        .quantile(
            0.99
        )
    )


    max_reviews = int(
        pooled[
            "review_count_365d"
        ]
        .max()
    )


    summary_rows.append(
        {
            "transition":
                transition,

            "snapshot_date":
                snapshot_date,

            "membership_rows":
                membership_rows,

            "unique_reviews":
                unique_review_ids,

            "listing_groups_with_text":
                listing_groups,

            "mean_reviews_per_listing":
                mean_reviews,

            "median_reviews_per_listing":
                median_reviews,

            "p90_reviews_per_listing":
                p90_reviews,

            "p99_reviews_per_listing":
                p99_reviews,

            "max_reviews_per_listing":
                max_reviews,

            "nan_values":
                nan_values,

            "inf_values":
                inf_values,

            "zero_norm":
                zero_norm,

            "elapsed_seconds":
                elapsed
        }
    )


    # ========================================================
    # 7N. PRINT TRANSITION RESULT
    # ========================================================

    print(
        "\nSaved:",
        output_path.name
    )


    print(
        "Pooled rows:",
        f"{len(pooled):,}"
    )


    print(
        "Mean reviews/listing:",
        round(
            mean_reviews,
            2
        )
    )


    print(
        "Median reviews/listing:",
        median_reviews
    )


    print(
        "P90 reviews/listing:",
        p90_reviews
    )


    print(
        "P99 reviews/listing:",
        p99_reviews
    )


    print(
        "Max reviews/listing:",
        max_reviews
    )


    print(
        "NaN:",
        nan_values
    )


    print(
        "Inf:",
        inf_values
    )


    print(
        "Zero-norm pooled vectors:",
        zero_norm
    )


    print(
        "Elapsed seconds:",
        round(
            elapsed,
            2
        )
    )


    # ========================================================
    # 7O. MEMORY CLEANUP
    # ========================================================

    del m
    del emb
    del joined
    del counts
    del pooled_embeddings
    del pooled
    del check
    del X
    del norms
    del required_ids

    gc.collect()


# ============================================================
# 8. GLOBAL SUMMARY
# ============================================================

summary = pd.DataFrame(
    summary_rows
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "TEMPORAL MEAN-POOLING SUMMARY"
)

print(
    "=" * 80
)


display(
    summary
)


total_memberships_processed = int(
    summary[
        "membership_rows"
    ]
    .sum()
)


total_listing_origins_with_text = int(
    summary[
        "listing_groups_with_text"
    ]
    .sum()
)


print(
    "\nTotal membership rows processed:",
    f"{total_memberships_processed:,}"
)


print(
    "Expected membership rows:",
    f"{EXPECTED_MEMBERSHIP_ROWS:,}"
)


print(
    "Total listing-origin rows with text:",
    f"{total_listing_origins_with_text:,}"
)


print(
    "Total NaN values:",
    int(
        summary[
            "nan_values"
        ]
        .sum()
    )
)


print(
    "Total Inf values:",
    int(
        summary[
            "inf_values"
        ]
        .sum()
    )
)


print(
    "Total zero-norm vectors:",
    int(
        summary[
            "zero_norm"
        ]
        .sum()
    )
)


total_elapsed_minutes = (
    (
        time.time()
        -
        overall_start
    )
    /
    60
)


print(
    "Total elapsed minutes:",
    round(
        total_elapsed_minutes,
        2
    )
)


# ============================================================
# 9. OUTPUT FILE INVENTORY
# ============================================================

output_files = sorted(
    AGG_DIR.glob(
        "listing_review_mean365_T*.parquet"
    )
)


print(
    "\nOutput transition files:",
    len(output_files)
)


for path in output_files:

    print(
        " -",
        path.name
    )


# ============================================================
# 10. FINAL STATUS
# ============================================================

pass_status = all(
    [
        total_memberships_processed
            ==
            EXPECTED_MEMBERSHIP_ROWS,

        int(
            summary[
                "nan_values"
            ]
            .sum()
        )
            ==
            0,

        int(
            summary[
                "inf_values"
            ]
            .sum()
        )
            ==
            0,

        int(
            summary[
                "zero_norm"
            ]
            .sum()
        )
            ==
            0,

        len(output_files)
            ==
            10
    ]
)


print(
    "\n"
    +
    "=" * 80
)


if pass_status:

    print(
        "STEP 24.7D-1 STATUS: PASS — "
        "TEMPORAL MEAN POOLING COMPLETE"
    )

else:

    print(
        "STEP 24.7D-1 STATUS: "
        "REVIEW REQUIRED"
    )


print(
    "=" * 80
)

In [ ]:
# ============================================================
# STEP 24.7D-2 — FINAL POOLED REVIEW EMBEDDING QC & FREEZE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time

print("=" * 80)
print("STEP 24.7D-2 — FINAL POOLED REVIEW EMBEDDING QC")
print("=" * 80)

EXPECTED_FILES = 10
EXPECTED_ROWS = 398_989
EXPECTED_MEMBERSHIPS = 7_334_068
EMBED_DIM = 384

embedding_cols = [
    f"emb_{i:03d}"
    for i in range(EMBED_DIM)
]

group_keys = [
    "transition",
    "snapshot_date",
    "listing_id"
]

expected_dates = {
    "T1":  "2025-10-05",
    "T2":  "2025-11-07",
    "T3":  "2025-12-11",
    "T4":  "2026-01-16",
    "T5":  "2026-02-13",
    "T6":  "2026-03-17",
    "T7":  "2026-04-16",
    "T8":  "2026-05-23",
    "T9":  "2026-06-19",
    "T10": "2026-07-12"
}

expected_rows_by_transition = {
    "T1": 38321,
    "T2": 38870,
    "T3": 39602,
    "T4": 38810,
    "T5": 40216,
    "T6": 40630,
    "T7": 40543,
    "T8": 40149,
    "T9": 40810,
    "T10": 41038
}


# ============================================================
# 1. FILE INVENTORY
# ============================================================

files_pooled = [
    AGG_DIR / f"listing_review_mean365_T{i}.parquet"
    for i in range(1, 11)
]

print("\n1. FILE INVENTORY")
print("-" * 80)

print(
    "Expected files:",
    EXPECTED_FILES
)

print(
    "Files found:",
    sum(
        path.exists()
        for path in files_pooled
    )
)

missing_files = [
    path.name
    for path in files_pooled
    if not path.exists()
]

print(
    "Missing files:",
    len(missing_files)
)

if missing_files:
    print(missing_files)


# ============================================================
# 2. GLOBAL QC
# ============================================================

start = time.time()

qc_rows = []

all_keys = []

total_rows = 0
total_review_counts = 0

total_nan = 0
total_inf = 0
total_zero_norm = 0

wrong_schema = []
wrong_dates = []
wrong_row_counts = []
wrong_dtypes = []


for i in range(1, 11):

    transition = f"T{i}"

    path = (
        AGG_DIR
        /
        f"listing_review_mean365_{transition}.parquet"
    )

    df = pd.read_parquet(path)


    # --------------------------------------------------------
    # Expected schema
    # --------------------------------------------------------

    expected_columns = (
        group_keys
        +
        ["review_count_365d"]
        +
        embedding_cols
    )

    if list(df.columns) != expected_columns:
        wrong_schema.append(transition)


    # --------------------------------------------------------
    # Expected row count
    # --------------------------------------------------------

    expected_n = (
        expected_rows_by_transition[
            transition
        ]
    )

    if len(df) != expected_n:

        wrong_row_counts.append(
            (
                transition,
                len(df),
                expected_n
            )
        )


    # --------------------------------------------------------
    # Key uniqueness
    # --------------------------------------------------------

    duplicate_keys = int(
        df[
            group_keys
        ]
        .duplicated()
        .sum()
    )


    # --------------------------------------------------------
    # Transition/date consistency
    # --------------------------------------------------------

    transition_values = (
        df["transition"]
        .dropna()
        .unique()
        .tolist()
    )

    dates = (
        pd.to_datetime(
            df["snapshot_date"]
        )
        .dt.strftime("%Y-%m-%d")
        .dropna()
        .unique()
        .tolist()
    )


    if (
        transition_values
        !=
        [transition]
    ):
        wrong_dates.append(
            (
                transition,
                "transition mismatch",
                transition_values
            )
        )


    if (
        dates
        !=
        [
            expected_dates[
                transition
            ]
        ]
    ):
        wrong_dates.append(
            (
                transition,
                "date mismatch",
                dates
            )
        )


    # --------------------------------------------------------
    # Review-count QC
    # --------------------------------------------------------

    missing_review_count = int(
        df[
            "review_count_365d"
        ]
        .isna()
        .sum()
    )


    nonpositive_review_count = int(
        (
            df[
                "review_count_365d"
            ]
            <=
            0
        )
        .sum()
    )


    review_count_sum = int(
        df[
            "review_count_365d"
        ]
        .sum()
    )


    # --------------------------------------------------------
    # Embedding QC
    # --------------------------------------------------------

    X = (
        df[
            embedding_cols
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    nan_values = int(
        np.isnan(X).sum()
    )

    inf_values = int(
        np.isinf(X).sum()
    )

    zero_norm = int(
        (
            np.linalg.norm(
                X,
                axis=1
            )
            ==
            0
        )
        .sum()
    )


    # --------------------------------------------------------
    # Dtype QC
    # --------------------------------------------------------

    embedding_dtype_ok = all(
        str(df[col].dtype)
        ==
        "float32"
        for col in embedding_cols
    )


    if not embedding_dtype_ok:
        wrong_dtypes.append(
            transition
        )


    # --------------------------------------------------------
    # Global key collection
    # --------------------------------------------------------

    all_keys.append(
        df[
            group_keys
        ].copy()
    )


    # --------------------------------------------------------
    # Totals
    # --------------------------------------------------------

    total_rows += len(df)

    total_review_counts += (
        review_count_sum
    )

    total_nan += nan_values
    total_inf += inf_values
    total_zero_norm += zero_norm


    qc_rows.append(
        {
            "transition":
                transition,

            "rows":
                len(df),

            "expected_rows":
                expected_n,

            "duplicate_keys":
                duplicate_keys,

            "review_count_sum":
                review_count_sum,

            "missing_review_count":
                missing_review_count,

            "nonpositive_review_count":
                nonpositive_review_count,

            "nan_values":
                nan_values,

            "inf_values":
                inf_values,

            "zero_norm":
                zero_norm,

            "float32_embeddings":
                embedding_dtype_ok
        }
    )


# ============================================================
# 3. GLOBAL KEY QC
# ============================================================

all_keys = pd.concat(
    all_keys,
    ignore_index=True
)

global_duplicate_keys = int(
    all_keys[
        group_keys
    ]
    .duplicated()
    .sum()
)


qc = pd.DataFrame(
    qc_rows
)


print("\n2. TRANSITION QC")
print("-" * 80)

display(qc)


print("\n3. GLOBAL RECONCILIATION")
print("-" * 80)

print(
    "Total pooled rows:",
    f"{total_rows:,}"
)

print(
    "Expected pooled rows:",
    f"{EXPECTED_ROWS:,}"
)

print(
    "Global duplicate listing-origin keys:",
    global_duplicate_keys
)

print(
    "Sum of review_count_365d:",
    f"{total_review_counts:,}"
)

print(
    "Expected membership rows:",
    f"{EXPECTED_MEMBERSHIPS:,}"
)

print(
    "Total NaN embedding values:",
    total_nan
)

print(
    "Total Inf embedding values:",
    total_inf
)

print(
    "Total zero-norm vectors:",
    total_zero_norm
)

print(
    "Wrong-schema transitions:",
    len(wrong_schema)
)

print(
    "Wrong-row-count transitions:",
    len(wrong_row_counts)
)

print(
    "Wrong transition/date mappings:",
    len(wrong_dates)
)

print(
    "Wrong embedding dtypes:",
    len(wrong_dtypes)
)


if wrong_schema:
    print(
        "Wrong schema:",
        wrong_schema
    )

if wrong_row_counts:
    print(
        "Wrong row counts:",
        wrong_row_counts
    )

if wrong_dates:
    print(
        "Wrong dates:",
        wrong_dates
    )

if wrong_dtypes:
    print(
        "Wrong dtypes:",
        wrong_dtypes
    )


# ============================================================
# 4. FINAL PASS / FAIL
# ============================================================

qc_pass = all(
    [
        len(missing_files) == 0,

        total_rows
        ==
        EXPECTED_ROWS,

        global_duplicate_keys
        ==
        0,

        total_review_counts
        ==
        EXPECTED_MEMBERSHIPS,

        int(
            qc[
                "duplicate_keys"
            ].sum()
        )
        ==
        0,

        int(
            qc[
                "missing_review_count"
            ].sum()
        )
        ==
        0,

        int(
            qc[
                "nonpositive_review_count"
            ].sum()
        )
        ==
        0,

        total_nan
        ==
        0,

        total_inf
        ==
        0,

        total_zero_norm
        ==
        0,

        len(wrong_schema)
        ==
        0,

        len(wrong_row_counts)
        ==
        0,

        len(wrong_dates)
        ==
        0,

        len(wrong_dtypes)
        ==
        0
    ]
)


print("\n4. QC RUNTIME")
print("-" * 80)

print(
    "Elapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if qc_pass:

    print(
        "STEP 24.7D-2 STATUS: PASS — "
        "POOLED REVIEW REPRESENTATION READY TO FREEZE"
    )

else:

    print(
        "STEP 24.7D-2 STATUS: FAIL — "
        "REVIEW QC OUTPUT BEFORE PROCEEDING"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8A-0 — LISTING-ORIGIN BACKBONE PREFLIGHT
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("STEP 24.8A-0 — LISTING-ORIGIN BACKBONE PREFLIGHT")
print("=" * 80)

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

anchor_map = {
    "T1":  "2025-10-05",
    "T2":  "2025-11-07",
    "T3":  "2025-12-11",
    "T4":  "2026-01-16",
    "T5":  "2026-02-13",
    "T6":  "2026-03-17",
    "T7":  "2026-04-16",
    "T8":  "2026-05-23",
    "T9":  "2026-06-19",
    "T10": "2026-07-12"
}

expected_rows = {
    "T1": 47752,
    "T2": 48624,
    "T3": 49899,
    "T4": 46819,
    "T5": 48692,
    "T6": 49076,
    "T7": 48951,
    "T8": 49305,
    "T9": 50926,
    "T10": 51096
}

qc_rows = []

for transition, snapshot in anchor_map.items():

    path = (
        LISTINGS_DIR
        / f"listings_{snapshot}.parquet"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing listing file: {path}"
        )

    df = pd.read_parquet(path)

    if "listing_id" not in df.columns:
        raise RuntimeError(
            f"{transition}: listing_id missing."
        )

    n_rows = len(df)

    unique_ids = df["listing_id"].nunique(dropna=True)

    duplicate_ids = int(
        df["listing_id"].duplicated().sum()
    )

    missing_ids = int(
        df["listing_id"].isna().sum()
    )

    expected_n = expected_rows[transition]

    qc_rows.append(
        {
            "transition": transition,
            "snapshot_date": snapshot,
            "rows": n_rows,
            "expected_rows": expected_n,
            "unique_listing_ids": unique_ids,
            "duplicate_listing_ids": duplicate_ids,
            "missing_listing_ids": missing_ids,
            "columns": len(df.columns)
        }
    )

qc = pd.DataFrame(qc_rows)

display(qc)


# ============================================================
# GLOBAL CHECK
# ============================================================

total_rows = int(qc["rows"].sum())
expected_total = int(
    sum(expected_rows.values())
)

print("\n" + "=" * 80)
print("GLOBAL BACKBONE PREFLIGHT")
print("=" * 80)

print(
    "Total T1-T10 listing-origin rows:",
    f"{total_rows:,}"
)

print(
    "Expected total:",
    f"{expected_total:,}"
)

print(
    "Total duplicate listing IDs within snapshots:",
    int(qc["duplicate_listing_ids"].sum())
)

print(
    "Total missing listing IDs:",
    int(qc["missing_listing_ids"].sum())
)

print(
    "All row counts match:",
    bool(
        (qc["rows"] == qc["expected_rows"]).all()
    )
)

print(
    "All snapshots have 40 columns:",
    bool(
        (qc["columns"] == 40).all()
    )
)


pass_status = all(
    [
        total_rows == expected_total,
        int(qc["duplicate_listing_ids"].sum()) == 0,
        int(qc["missing_listing_ids"].sum()) == 0,
        bool(
            (qc["rows"] == qc["expected_rows"]).all()
        ),
        bool(
            (qc["columns"] == 40).all()
        )
    ]
)

print("\n" + "=" * 80)

if pass_status:
    print(
        "STEP 24.8A-0 STATUS: PASS — "
        "LISTING SNAPSHOTS READY FOR BACKBONE CONSTRUCTION"
    )
else:
    print(
        "STEP 24.8A-0 STATUS: FAIL — "
        "DO NOT BUILD BACKBONE YET"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8A-1 — BUILD COMPLETE LISTING-ORIGIN BACKBONE
# UPDATED VERSION
#
# IMPORTANT:
# - Cleaned listing files already contain snapshot_date.
# - We validate the existing snapshot_date against the frozen
#   transition anchor.
# - We add ONLY the transition column.
# - 40 source columns + 1 transition column = 41 columns.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8A-1 — BUILD COMPLETE LISTING-ORIGIN BACKBONE")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTPUT_DIR
    / "listing_origin_backbone_T1_T10.parquet"
)


print("\nListings directory:")
print(LISTINGS_DIR)

print("\nOutput:")
print(OUTPUT_PATH)


# ============================================================
# 2. FROZEN TRANSITION / ANCHOR MAP
# ============================================================

anchor_map = {
    "T1":  "2025-10-05",
    "T2":  "2025-11-07",
    "T3":  "2025-12-11",
    "T4":  "2026-01-16",
    "T5":  "2026-02-13",
    "T6":  "2026-03-17",
    "T7":  "2026-04-16",
    "T8":  "2026-05-23",
    "T9":  "2026-06-19",
    "T10": "2026-07-12"
}


expected_rows = {
    "T1": 47752,
    "T2": 48624,
    "T3": 49899,
    "T4": 46819,
    "T5": 48692,
    "T6": 49076,
    "T7": 48951,
    "T8": 49305,
    "T9": 50926,
    "T10": 51096
}


EXPECTED_TOTAL = 491_140

EXPECTED_SOURCE_COLUMNS = 40

EXPECTED_BACKBONE_COLUMNS = 41


# ============================================================
# 3. LOAD EACH ANCHOR SNAPSHOT
# ============================================================

parts = []

qc_rows = []

reference_listing_columns = None


for transition, snapshot in anchor_map.items():

    print("\n" + "-" * 80)

    print(
        f"Loading {transition} — {snapshot}"
    )


    path = (
        LISTINGS_DIR
        / f"listings_{snapshot}.parquet"
    )


    # --------------------------------------------------------
    # 3A. File existence
    # --------------------------------------------------------

    if not path.exists():

        raise FileNotFoundError(
            f"{transition}: listing file not found:\n"
            f"{path}"
        )


    # --------------------------------------------------------
    # 3B. Load source
    # --------------------------------------------------------

    df = pd.read_parquet(
        path
    )


    print(
        "Source rows:",
        f"{len(df):,}"
    )

    print(
        "Source columns:",
        len(df.columns)
    )


    # --------------------------------------------------------
    # 3C. Expected row count
    # --------------------------------------------------------

    expected_n = (
        expected_rows[
            transition
        ]
    )


    if len(df) != expected_n:

        raise RuntimeError(
            f"{transition}: row-count mismatch. "
            f"Expected {expected_n:,}, "
            f"found {len(df):,}."
        )


    # --------------------------------------------------------
    # 3D. Expected source column count
    # --------------------------------------------------------

    if len(df.columns) != EXPECTED_SOURCE_COLUMNS:

        raise RuntimeError(
            f"{transition}: expected "
            f"{EXPECTED_SOURCE_COLUMNS} source columns, "
            f"found {len(df.columns)}."
        )


    # --------------------------------------------------------
    # 3E. listing_id QC
    # --------------------------------------------------------

    if "listing_id" not in df.columns:

        raise RuntimeError(
            f"{transition}: listing_id missing."
        )


    missing_listing_ids = int(
        df[
            "listing_id"
        ]
        .isna()
        .sum()
    )


    duplicate_listing_ids = int(
        df[
            "listing_id"
        ]
        .duplicated()
        .sum()
    )


    if missing_listing_ids != 0:

        raise RuntimeError(
            f"{transition}: "
            f"{missing_listing_ids:,} missing listing IDs."
        )


    if duplicate_listing_ids != 0:

        raise RuntimeError(
            f"{transition}: "
            f"{duplicate_listing_ids:,} duplicate listing IDs."
        )


    # --------------------------------------------------------
    # 3F. Confirm listing schemas are identical
    #
    # Compare the ORIGINAL 40-column cleaned schemas before
    # adding transition.
    # --------------------------------------------------------

    current_listing_columns = list(
        df.columns
    )


    if reference_listing_columns is None:

        reference_listing_columns = (
            current_listing_columns.copy()
        )

    else:

        if (
            current_listing_columns
            !=
            reference_listing_columns
        ):

            raise RuntimeError(
                f"{transition}: cleaned listing schema/order "
                f"differs from T1."
            )


    # ========================================================
    # 4. VALIDATE EXISTING snapshot_date
    # ========================================================

    if "snapshot_date" not in df.columns:

        raise RuntimeError(
            f"{transition}: snapshot_date is missing "
            f"from cleaned listing source."
        )


    # Convert source snapshot_date safely to datetime.
    source_snapshot_dates = pd.to_datetime(
        df[
            "snapshot_date"
        ],
        errors="coerce"
    )


    invalid_snapshot_dates = int(
        source_snapshot_dates
        .isna()
        .sum()
    )


    if invalid_snapshot_dates != 0:

        raise RuntimeError(
            f"{transition}: "
            f"{invalid_snapshot_dates:,} invalid/missing "
            f"snapshot_date values."
        )


    # Normalize dates to midnight.
    source_snapshot_dates = (
        source_snapshot_dates
        .dt
        .normalize()
    )


    unique_source_dates = (
        source_snapshot_dates
        .drop_duplicates()
        .tolist()
    )


    expected_date = (
        pd.Timestamp(snapshot)
        .normalize()
    )


    # There must be exactly one source snapshot date.
    if len(unique_source_dates) != 1:

        raise RuntimeError(
            f"{transition}: expected exactly one "
            f"snapshot_date in source, found "
            f"{len(unique_source_dates)}: "
            f"{unique_source_dates}"
        )


    observed_date = (
        pd.Timestamp(
            unique_source_dates[0]
        )
        .normalize()
    )


    # Existing source date must agree with frozen anchor.
    if observed_date != expected_date:

        raise RuntimeError(
            f"{transition}: source snapshot_date "
            f"{observed_date.date()} does not match "
            f"expected anchor {expected_date.date()}."
        )


    # Standardise dtype while PRESERVING source provenance.
    df[
        "snapshot_date"
    ] = source_snapshot_dates


    print(
        "Validated snapshot_date:",
        observed_date.date()
    )


    # ========================================================
    # 5. ADD transition ONLY
    # ========================================================

    if "transition" in df.columns:

        raise RuntimeError(
            f"{transition}: transition already exists "
            f"in cleaned listing source."
        )


    df.insert(
        0,
        "transition",
        transition
    )


    # --------------------------------------------------------
    # Column count after adding transition
    # --------------------------------------------------------

    if len(df.columns) != EXPECTED_BACKBONE_COLUMNS:

        raise RuntimeError(
            f"{transition}: expected "
            f"{EXPECTED_BACKBONE_COLUMNS} columns after "
            f"adding transition, found {len(df.columns)}."
        )


    # --------------------------------------------------------
    # Transition-specific key QC
    # --------------------------------------------------------

    transition_key_cols = [
        "transition",
        "snapshot_date",
        "listing_id"
    ]


    transition_duplicate_keys = int(
        df[
            transition_key_cols
        ]
        .duplicated()
        .sum()
    )


    transition_missing_keys = int(
        df[
            transition_key_cols
        ]
        .isna()
        .sum()
        .sum()
    )


    if transition_duplicate_keys != 0:

        raise RuntimeError(
            f"{transition}: duplicate listing-origin keys."
        )


    if transition_missing_keys != 0:

        raise RuntimeError(
            f"{transition}: missing listing-origin key values."
        )


    # --------------------------------------------------------
    # Transition QC record
    # --------------------------------------------------------

    qc_rows.append(
        {
            "transition":
                transition,

            "snapshot_date":
                observed_date,

            "rows":
                len(df),

            "expected_rows":
                expected_n,

            "unique_listing_ids":
                df[
                    "listing_id"
                ].nunique(),

            "duplicate_listing_ids":
                duplicate_listing_ids,

            "missing_listing_ids":
                missing_listing_ids,

            "duplicate_origin_keys":
                transition_duplicate_keys,

            "missing_origin_keys":
                transition_missing_keys,

            "columns":
                len(df.columns)
        }
    )


    parts.append(
        df
    )


# ============================================================
# 6. CONCATENATE COMPLETE BACKBONE
# ============================================================

print("\n" + "=" * 80)

print(
    "CONSTRUCTING COMPLETE BACKBONE"
)

print("=" * 80)


backbone = pd.concat(
    parts,
    ignore_index=True
)


del parts

gc.collect()


# ============================================================
# 7. GLOBAL KEY QC
# ============================================================

key_cols = [
    "transition",
    "snapshot_date",
    "listing_id"
]


duplicate_keys = int(
    backbone[
        key_cols
    ]
    .duplicated()
    .sum()
)


missing_key_values = int(
    backbone[
        key_cols
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "\nBackbone rows:",
    f"{len(backbone):,}"
)

print(
    "Expected rows:",
    f"{EXPECTED_TOTAL:,}"
)

print(
    "Backbone columns:",
    len(backbone.columns)
)

print(
    "Expected columns:",
    EXPECTED_BACKBONE_COLUMNS
)

print(
    "Duplicate listing-origin keys:",
    duplicate_keys
)

print(
    "Missing key values:",
    missing_key_values
)


# ============================================================
# 8. TRANSITION ROW-COUNT QC
# ============================================================

observed_counts = (
    backbone
    .groupby(
        "transition",
        sort=False
    )
    .size()
)


wrong_counts = []


for transition, expected_n in expected_rows.items():

    observed_n = int(
        observed_counts.get(
            transition,
            0
        )
    )


    if observed_n != expected_n:

        wrong_counts.append(
            (
                transition,
                observed_n,
                expected_n
            )
        )


print(
    "Wrong transition row counts:",
    len(wrong_counts)
)


if wrong_counts:

    print(
        wrong_counts
    )


# ============================================================
# 9. TRANSITION / DATE MAPPING QC
# ============================================================

wrong_dates = []


for transition, expected_date_string in anchor_map.items():

    observed_dates = (
        backbone.loc[
            backbone[
                "transition"
            ]
            ==
            transition,
            "snapshot_date"
        ]
        .drop_duplicates()
        .tolist()
    )


    expected_timestamp = (
        pd.Timestamp(
            expected_date_string
        )
        .normalize()
    )


    if len(observed_dates) != 1:

        wrong_dates.append(
            (
                transition,
                observed_dates,
                expected_timestamp
            )
        )

        continue


    observed_timestamp = (
        pd.Timestamp(
            observed_dates[0]
        )
        .normalize()
    )


    if observed_timestamp != expected_timestamp:

        wrong_dates.append(
            (
                transition,
                observed_timestamp,
                expected_timestamp
            )
        )


print(
    "Wrong transition/date mappings:",
    len(wrong_dates)
)


if wrong_dates:

    print(
        wrong_dates
    )


# ============================================================
# 10. GLOBAL LISTING-ID STRUCTURE
# ============================================================
#
# IMPORTANT:
#
# listing_id alone is NOT expected to be globally unique.
# The same listing may legitimately persist across several
# snapshots/transitions.
#
# Required unique observation key:
#
# transition + snapshot_date + listing_id
#
# ============================================================

unique_listing_ids_global = int(
    backbone[
        "listing_id"
    ]
    .nunique()
)


repeated_listing_occurrences = (
    len(backbone)
    -
    unique_listing_ids_global
)


print(
    "\nUnique listing IDs across T1-T10:",
    f"{unique_listing_ids_global:,}"
)

print(
    "Repeated listing occurrences across transitions:",
    f"{repeated_listing_occurrences:,}"
)

print(
    "(Repeated listing IDs across transitions are expected.)"
)


# ============================================================
# 11. TRANSITION QC TABLE
# ============================================================

qc = pd.DataFrame(
    qc_rows
)


print("\n" + "=" * 80)
print("TRANSITION QC")
print("=" * 80)


display(
    qc
)


# ============================================================
# 12. GLOBAL PRE-SAVE QC
# ============================================================

row_count_match = (
    len(backbone)
    ==
    EXPECTED_TOTAL
)


column_count_match = (
    len(backbone.columns)
    ==
    EXPECTED_BACKBONE_COLUMNS
)


all_transition_rows_match = bool(
    (
        qc[
            "rows"
        ]
        ==
        qc[
            "expected_rows"
        ]
    )
    .all()
)


all_transition_columns_match = bool(
    (
        qc[
            "columns"
        ]
        ==
        EXPECTED_BACKBONE_COLUMNS
    )
    .all()
)


all_unique_within_snapshot = bool(
    (
        qc[
            "duplicate_listing_ids"
        ]
        ==
        0
    )
    .all()
)


all_listing_ids_present = bool(
    (
        qc[
            "missing_listing_ids"
        ]
        ==
        0
    )
    .all()
)


all_transition_keys_unique = bool(
    (
        qc[
            "duplicate_origin_keys"
        ]
        ==
        0
    )
    .all()
)


all_transition_keys_complete = bool(
    (
        qc[
            "missing_origin_keys"
        ]
        ==
        0
    )
    .all()
)


print("\n" + "=" * 80)
print("GLOBAL PRE-SAVE QC")
print("=" * 80)


print(
    "Row count match:",
    row_count_match
)

print(
    "Column count match:",
    column_count_match
)

print(
    "All transition row counts match:",
    all_transition_rows_match
)

print(
    "All transition column counts match:",
    all_transition_columns_match
)

print(
    "All listing IDs unique within snapshots:",
    all_unique_within_snapshot
)

print(
    "All listing IDs present:",
    all_listing_ids_present
)

print(
    "All transition keys unique:",
    all_transition_keys_unique
)

print(
    "All transition keys complete:",
    all_transition_keys_complete
)

print(
    "Global duplicate listing-origin keys:",
    duplicate_keys
)

print(
    "Global missing key values:",
    missing_key_values
)


pre_save_pass = all(
    [
        row_count_match,
        column_count_match,
        all_transition_rows_match,
        all_transition_columns_match,
        all_unique_within_snapshot,
        all_listing_ids_present,
        all_transition_keys_unique,
        all_transition_keys_complete,
        duplicate_keys == 0,
        missing_key_values == 0,
        len(wrong_counts) == 0,
        len(wrong_dates) == 0
    ]
)


if not pre_save_pass:

    raise RuntimeError(
        "STEP 24.8A-1 pre-save QC failed. "
        "Backbone will NOT be frozen."
    )


# ============================================================
# 13. SAVE BACKBONE
# ============================================================

print("\n" + "=" * 80)
print("SAVING BACKBONE")
print("=" * 80)


backbone.to_parquet(
    OUTPUT_PATH,
    index=False
)


print(
    "Saved:",
    OUTPUT_PATH
)


# ============================================================
# 14. SAVE / RELOAD QC
# ============================================================

check = pd.read_parquet(
    OUTPUT_PATH
)


reload_duplicate_keys = int(
    check[
        key_cols
    ]
    .duplicated()
    .sum()
)


reload_missing_keys = int(
    check[
        key_cols
    ]
    .isna()
    .sum()
    .sum()
)


same_rows = (
    len(check)
    ==
    len(backbone)
)


same_columns = (
    list(check.columns)
    ==
    list(backbone.columns)
)


same_dtypes = (
    check.dtypes.astype(str).tolist()
    ==
    backbone.dtypes.astype(str).tolist()
)


reload_transition_counts = (
    check
    .groupby(
        "transition"
    )
    .size()
)


reload_counts_match = True


for transition, expected_n in expected_rows.items():

    if int(
        reload_transition_counts.get(
            transition,
            0
        )
    ) != expected_n:

        reload_counts_match = False
        break


print("\n" + "=" * 80)
print("SAVE / RELOAD QC")
print("=" * 80)


print(
    "Reload rows:",
    f"{len(check):,}"
)

print(
    "Expected rows:",
    f"{EXPECTED_TOTAL:,}"
)

print(
    "Reload columns:",
    len(check.columns)
)

print(
    "Expected columns:",
    EXPECTED_BACKBONE_COLUMNS
)

print(
    "Same row count:",
    same_rows
)

print(
    "Same column order:",
    same_columns
)

print(
    "Same dtypes:",
    same_dtypes
)

print(
    "Transition counts preserved:",
    reload_counts_match
)

print(
    "Duplicate keys after reload:",
    reload_duplicate_keys
)

print(
    "Missing key values after reload:",
    reload_missing_keys
)


# ============================================================
# 15. FINAL STATUS
# ============================================================

final_pass = all(
    [
        pre_save_pass,
        same_rows,
        same_columns,
        same_dtypes,
        reload_counts_match,
        len(check) == EXPECTED_TOTAL,
        len(check.columns)
            ==
            EXPECTED_BACKBONE_COLUMNS,
        reload_duplicate_keys == 0,
        reload_missing_keys == 0
    ]
)


elapsed = (
    time.time()
    -
    start
)


print(
    "\nElapsed seconds:",
    round(
        elapsed,
        2
    )
)


print("\n" + "=" * 80)


if final_pass:

    print(
        "STEP 24.8A-1 STATUS: PASS — "
        "COMPLETE LISTING-ORIGIN BACKBONE CREATED"
    )

else:

    print(
        "STEP 24.8A-1 STATUS: FAIL — "
        "DO NOT PROCEED TO MULTIMODAL MERGING"
    )


print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8B-0 — GEOGRAPHY / TA INTEGRATION PREFLIGHT
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("STEP 24.8B-0 — GEOGRAPHY / TA INTEGRATION PREFLIGHT")
print("=" * 80)

SEARCH_ROOTS = [
    PROJECT_ROOT / "02_intermediate",
    PROJECT_ROOT / "03_processed"
]

candidate_files = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*.parquet"):

        name = path.name.lower()

        if any(
            term in name
            for term in [
                "geograph",
                "ta_",
                "_ta",
                "territorial",
                "listing_ta",
                "spatial"
            ]
        ):
            candidate_files.append(path)


candidate_files = sorted(
    set(candidate_files)
)


print(
    "\nCandidate geography/TA parquet files:",
    len(candidate_files)
)


for i, path in enumerate(
    candidate_files,
    start=1
):

    print("\n" + "-" * 80)
    print(f"[{i}] {path}")

    try:

        df = pd.read_parquet(path)

        print(
            "Rows:",
            f"{len(df):,}"
        )

        print(
            "Columns:",
            len(df.columns)
        )

        print(
            "Column names:"
        )

        print(
            list(df.columns)
        )

        # ---------------------------------------------
        # Potential key fields
        # ---------------------------------------------

        potential_keys = [
            c
            for c in df.columns
            if (
                "listing" in c.lower()
                or
                "snapshot" in c.lower()
                or
                "transition" in c.lower()
                or
                c.lower().startswith("ta")
                or
                "territorial" in c.lower()
            )
        ]

        print(
            "Potential geography/key columns:"
        )

        print(
            potential_keys
        )


        # ---------------------------------------------
        # Small preview of relevant columns
        # ---------------------------------------------

        if potential_keys:

            preview_cols = (
                potential_keys[:10]
            )

            display(
                df[
                    preview_cols
                ].head()
            )

    except Exception as e:

        print(
            "Could not inspect:",
            repr(e)
        )


print("\n" + "=" * 80)

if len(candidate_files) == 0:

    print(
        "No candidate TA/geography parquet file "
        "was automatically identified."
    )

else:

    print(
        "STEP 24.8B-0 INVENTORY COMPLETE"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8B-1 — MERGE TA GEOGRAPHY INTO LISTING BACKBONE
# ============================================================

from pathlib import Path
import pandas as pd
import time
import gc

print("=" * 80)
print("STEP 24.8B-1 — MERGE TA GEOGRAPHY INTO LISTING BACKBONE")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

BACKBONE_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

GEOGRAPHY_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "geography"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_TA.parquet"
)


# ============================================================
# 2. FROZEN TRANSITION / DATE MAP
# ============================================================

anchor_map = {
    "T1":  "2025-10-05",
    "T2":  "2025-11-07",
    "T3":  "2025-12-11",
    "T4":  "2026-01-16",
    "T5":  "2026-02-13",
    "T6":  "2026-03-17",
    "T7":  "2026-04-16",
    "T8":  "2026-05-23",
    "T9":  "2026-06-19",
    "T10": "2026-07-12"
}

expected_rows = {
    "T1": 47752,
    "T2": 48624,
    "T3": 49899,
    "T4": 46819,
    "T5": 48692,
    "T6": 49076,
    "T7": 48951,
    "T8": 49305,
    "T9": 50926,
    "T10": 51096
}

EXPECTED_TOTAL = 491_140


# ============================================================
# 3. LOAD BACKBONE
# ============================================================

backbone = pd.read_parquet(
    BACKBONE_PATH
)

print("\nBackbone rows:", f"{len(backbone):,}")
print("Backbone columns:", len(backbone.columns))


if len(backbone) != EXPECTED_TOTAL:
    raise RuntimeError(
        f"Backbone row mismatch: "
        f"{len(backbone):,} != {EXPECTED_TOTAL:,}"
    )


backbone["snapshot_date"] = pd.to_datetime(
    backbone["snapshot_date"]
).dt.normalize()


backbone_key = [
    "snapshot_date",
    "listing_id"
]


backbone_duplicate_keys = int(
    backbone[
        backbone_key
    ]
    .duplicated()
    .sum()
)


backbone_missing_keys = int(
    backbone[
        backbone_key
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Backbone duplicate snapshot/listing keys:",
    backbone_duplicate_keys
)

print(
    "Backbone missing snapshot/listing keys:",
    backbone_missing_keys
)


if backbone_duplicate_keys != 0:
    raise RuntimeError(
        "Backbone snapshot_date + listing_id "
        "is not globally unique."
    )


if backbone_missing_keys != 0:
    raise RuntimeError(
        "Backbone contains missing geography merge keys."
    )


# ============================================================
# 4. LOAD T1-T10 GEOGRAPHY FILES
# ============================================================

geo_parts = []

geo_qc_rows = []


for transition, snapshot in anchor_map.items():

    path = (
        GEOGRAPHY_DIR
        / f"listing_ta_{snapshot}.parquet"
    )

    print("\n" + "-" * 80)
    print(f"Loading {transition}: {path.name}")


    if not path.exists():
        raise FileNotFoundError(
            f"Missing geography file: {path}"
        )


    geo = pd.read_parquet(
        path
    )


    required_cols = [
        "snapshot_date",
        "listing_id",
        "ta_code",
        "ta_name",
        "ta_name_ascii"
    ]


    missing_cols = [
        c
        for c in required_cols
        if c not in geo.columns
    ]


    if missing_cols:
        raise RuntimeError(
            f"{transition}: missing geography columns "
            f"{missing_cols}"
        )


    # Keep only frozen geography fields
    geo = geo[
        required_cols
    ].copy()


    geo["snapshot_date"] = pd.to_datetime(
        geo["snapshot_date"],
        errors="coerce"
    ).dt.normalize()


    expected_date = pd.Timestamp(
        snapshot
    ).normalize()


    unique_dates = (
        geo["snapshot_date"]
        .dropna()
        .unique()
    )


    if (
        len(unique_dates) != 1
        or pd.Timestamp(unique_dates[0]) != expected_date
    ):
        raise RuntimeError(
            f"{transition}: geography snapshot_date "
            f"does not match expected anchor {snapshot}."
        )


    if len(geo) != expected_rows[transition]:
        raise RuntimeError(
            f"{transition}: geography row mismatch. "
            f"Expected {expected_rows[transition]:,}, "
            f"found {len(geo):,}."
        )


    duplicate_keys = int(
        geo[
            ["snapshot_date", "listing_id"]
        ]
        .duplicated()
        .sum()
    )


    missing_keys = int(
        geo[
            ["snapshot_date", "listing_id"]
        ]
        .isna()
        .sum()
        .sum()
    )


    missing_ta_code = int(
        geo[
            "ta_code"
        ]
        .isna()
        .sum()
    )


    missing_ta_name = int(
        geo[
            "ta_name"
        ]
        .isna()
        .sum()
    )


    if duplicate_keys != 0:
        raise RuntimeError(
            f"{transition}: duplicate geography keys."
        )


    if missing_keys != 0:
        raise RuntimeError(
            f"{transition}: missing geography merge keys."
        )


    if missing_ta_code != 0:
        raise RuntimeError(
            f"{transition}: missing ta_code detected."
        )


    if missing_ta_name != 0:
        raise RuntimeError(
            f"{transition}: missing ta_name detected."
        )


    geo_qc_rows.append(
        {
            "transition": transition,
            "rows": len(geo),
            "expected_rows": expected_rows[transition],
            "duplicate_keys": duplicate_keys,
            "missing_keys": missing_keys,
            "missing_ta_code": missing_ta_code,
            "missing_ta_name": missing_ta_name
        }
    )


    geo_parts.append(
        geo
    )


# ============================================================
# 5. CONCATENATE GEOGRAPHY
# ============================================================

geography = pd.concat(
    geo_parts,
    ignore_index=True
)

del geo_parts
gc.collect()


print("\n" + "=" * 80)
print("GEOGRAPHY GLOBAL QC")
print("=" * 80)

print(
    "Geography rows:",
    f"{len(geography):,}"
)

print(
    "Expected geography rows:",
    f"{EXPECTED_TOTAL:,}"
)


geo_global_duplicates = int(
    geography[
        backbone_key
    ]
    .duplicated()
    .sum()
)


geo_global_missing_keys = int(
    geography[
        backbone_key
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Global duplicate geography keys:",
    geo_global_duplicates
)

print(
    "Global missing geography keys:",
    geo_global_missing_keys
)


if len(geography) != EXPECTED_TOTAL:
    raise RuntimeError(
        "Global geography row count mismatch."
    )


if geo_global_duplicates != 0:
    raise RuntimeError(
        "Global geography keys are not unique."
    )


# ============================================================
# 6. NORMALISE ta_code SAFELY
# ============================================================
#
# Preserve official three-digit TA codes such as:
# 001, 021, 076.
#
# TA999 must remain TA999.
# ============================================================

geography["ta_code"] = (
    geography["ta_code"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 7. MERGE GEOGRAPHY INTO BACKBONE
# ============================================================

rows_before = len(backbone)


merged = backbone.merge(
    geography,
    on=[
        "snapshot_date",
        "listing_id"
    ],
    how="left",
    validate="one_to_one",
    indicator="_geo_merge"
)


rows_after = len(merged)


print("\n" + "=" * 80)
print("MERGE QC")
print("=" * 80)

print(
    "Rows before merge:",
    f"{rows_before:,}"
)

print(
    "Rows after merge:",
    f"{rows_after:,}"
)


merge_counts = (
    merged[
        "_geo_merge"
    ]
    .value_counts(
        dropna=False
    )
)


print("\nMerge status:")
print(merge_counts)


unmatched = int(
    (
        merged[
            "_geo_merge"
        ]
        !=
        "both"
    )
    .sum()
)


missing_ta_after_merge = int(
    merged[
        "ta_code"
    ]
    .isna()
    .sum()
)


duplicate_origin_keys = int(
    merged[
        [
            "transition",
            "snapshot_date",
            "listing_id"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\nUnmatched backbone rows:",
    unmatched
)

print(
    "Missing ta_code after merge:",
    missing_ta_after_merge
)

print(
    "Duplicate listing-origin keys after merge:",
    duplicate_origin_keys
)


if rows_after != rows_before:
    raise RuntimeError(
        "Geography merge changed backbone row count."
    )


if unmatched != 0:
    raise RuntimeError(
        f"{unmatched:,} backbone rows did not match geography."
    )


if missing_ta_after_merge != 0:
    raise RuntimeError(
        "Missing TA assignments after geography merge."
    )


if duplicate_origin_keys != 0:
    raise RuntimeError(
        "Duplicate listing-origin keys created by geography merge."
    )


# Merge indicator no longer needed
merged = merged.drop(
    columns="_geo_merge"
)


# ============================================================
# 8. TA DISTRIBUTION / TA999 AUDIT
# ============================================================

unique_ta_codes = int(
    merged[
        "ta_code"
    ]
    .nunique()
)


ta999_rows = int(
    (
        merged[
            "ta_code"
        ]
        ==
        "TA999"
    )
    .sum()
)


print("\n" + "=" * 80)
print("TA ASSIGNMENT AUDIT")
print("=" * 80)

print(
    "Unique TA codes in T1-T10 backbone:",
    unique_ta_codes
)

print(
    "TA999 listing-origin rows:",
    f"{ta999_rows:,}"
)


ta_qc = (
    merged
    .groupby(
        "transition",
        sort=False
    )
    .agg(
        rows=(
            "listing_id",
            "size"
        ),
        unique_ta_codes=(
            "ta_code",
            "nunique"
        ),
        ta999_rows=(
            "ta_code",
            lambda x: int(
                (x == "TA999").sum()
            )
        )
    )
    .reset_index()
)


display(
    ta_qc
)


# ============================================================
# 9. TRANSITION ↔ SNAPSHOT CONSISTENCY AFTER MERGE
# ============================================================

wrong_dates = []


for transition, snapshot in anchor_map.items():

    observed_dates = (
        merged.loc[
            merged[
                "transition"
            ]
            ==
            transition,
            "snapshot_date"
        ]
        .drop_duplicates()
        .tolist()
    )


    expected_date = pd.Timestamp(
        snapshot
    ).normalize()


    if (
        len(observed_dates) != 1
        or
        pd.Timestamp(
            observed_dates[0]
        ).normalize()
        !=
        expected_date
    ):

        wrong_dates.append(
            transition
        )


print(
    "Wrong transition/date mappings after merge:",
    len(wrong_dates)
)


# ============================================================
# 10. FINAL PRE-SAVE PASS
# ============================================================

pre_save_pass = all(
    [
        rows_before == EXPECTED_TOTAL,
        rows_after == EXPECTED_TOTAL,
        unmatched == 0,
        missing_ta_after_merge == 0,
        duplicate_origin_keys == 0,
        len(wrong_dates) == 0
    ]
)


if not pre_save_pass:
    raise RuntimeError(
        "STEP 24.8B-1 pre-save QC failed."
    )


# ============================================================
# 11. SAVE
# ============================================================

merged.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\nSaved:")
print(OUTPUT_PATH)


# ============================================================
# 12. SAVE / RELOAD QC
# ============================================================

check = pd.read_parquet(
    OUTPUT_PATH
)


reload_duplicates = int(
    check[
        [
            "transition",
            "snapshot_date",
            "listing_id"
        ]
    ]
    .duplicated()
    .sum()
)


reload_missing_ta = int(
    check[
        "ta_code"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("SAVE / RELOAD QC")
print("=" * 80)

print(
    "Reload rows:",
    f"{len(check):,}"
)

print(
    "Reload columns:",
    len(check.columns)
)

print(
    "Duplicate listing-origin keys:",
    reload_duplicates
)

print(
    "Missing ta_code:",
    reload_missing_ta
)


# ============================================================
# 13. FINAL STATUS
# ============================================================

final_pass = all(
    [
        len(check) == EXPECTED_TOTAL,
        reload_duplicates == 0,
        reload_missing_ta == 0
    ]
)


print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if final_pass:

    print(
        "STEP 24.8B-1 STATUS: PASS — "
        "TA GEOGRAPHY INTEGRATED INTO COMPLETE BACKBONE"
    )

else:

    print(
        "STEP 24.8B-1 STATUS: FAIL — "
        "DO NOT PROCEED TO CONTEXTUAL MERGING"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8B-2 — AUDIT 68 TA CATEGORIES / OUTSIDE-TA CODE
# ============================================================

import pandas as pd

print("=" * 80)
print("STEP 24.8B-2 — AUDIT 68 TA CATEGORIES / OUTSIDE-TA CODE")
print("=" * 80)

PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_TA.parquet"
)

df = pd.read_parquet(
    PATH,
    columns=[
        "transition",
        "snapshot_date",
        "listing_id",
        "ta_code",
        "ta_name",
        "ta_name_ascii"
    ]
)

print("\nRows:", f"{len(df):,}")


# ============================================================
# 1. EXACT TA CODE INVENTORY
# ============================================================

df["ta_code_check"] = (
    df["ta_code"]
    .astype("string")
    .str.strip()
)

inventory = (
    df[
        [
            "ta_code_check",
            "ta_name",
            "ta_name_ascii"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "ta_code_check"
    )
    .reset_index(
        drop=True
    )
)

print("\nUnique TA codes:")
print(
    df["ta_code_check"].nunique()
)

print("\nTA code/name inventory:")
display(inventory)


# ============================================================
# 2. FREQUENCY BY TA CATEGORY
# ============================================================

freq = (
    df
    .groupby(
        [
            "ta_code_check",
            "ta_name",
            "ta_name_ascii"
        ],
        dropna=False
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(
        "ta_code_check"
    )
)

print("\nTA frequencies across T1-T10:")
display(freq)


# ============================================================
# 3. IDENTIFY ANY 999-LIKE CATEGORY
# ============================================================

mask_999 = (
    df["ta_code_check"]
    .str.contains(
        "999",
        na=False
    )
)

rows_999_like = df.loc[
    mask_999,
    [
        "transition",
        "snapshot_date",
        "listing_id",
        "ta_code",
        "ta_name",
        "ta_name_ascii"
    ]
].copy()


print("\n" + "=" * 80)
print("999-LIKE CATEGORY AUDIT")
print("=" * 80)

print(
    "Rows with ta_code containing '999':",
    f"{len(rows_999_like):,}"
)

print(
    "Unique raw ta_code values:",
    rows_999_like["ta_code"]
    .astype("string")
    .unique()
    .tolist()
)

print(
    "Associated TA names:",
    rows_999_like[
        [
            "ta_code",
            "ta_name",
            "ta_name_ascii"
        ]
    ]
    .drop_duplicates()
    .to_dict("records")
)


# ============================================================
# 4. 999-LIKE ROWS BY TRANSITION
# ============================================================

if len(rows_999_like) > 0:

    by_transition = (
        rows_999_like
        .groupby(
            "transition",
            sort=False
        )
        .size()
        .rename(
            "outside_ta_rows"
        )
        .reset_index()
    )

    print("\n999-like rows by transition:")
    display(by_transition)


# ============================================================
# 5. LOOK FOR SUSPICIOUS CATEGORY NAMES
# ============================================================

name_text = (
    df["ta_name"]
    .astype("string")
    .fillna("")
    +
    " "
    +
    df["ta_name_ascii"]
    .astype("string")
    .fillna("")
)

mask_name = (
    name_text
    .str.lower()
    .str.contains(
        r"outside|area outside|not applicable|undefined|999",
        regex=True,
        na=False
    )
)

suspicious_names = (
    df.loc[
        mask_name,
        [
            "ta_code",
            "ta_name",
            "ta_name_ascii"
        ]
    ]
    .drop_duplicates()
)

print("\nSuspicious outside-TA category names:")
display(suspicious_names)


# ============================================================
# 6. FINAL INFORMATION
# ============================================================

print("\n" + "=" * 80)
print("STEP 24.8B-2 AUDIT COMPLETE")
print("=" * 80)

print(
    "Unique TA categories:",
    df["ta_code_check"].nunique()
)

print(
    "999-like rows:",
    f"{len(rows_999_like):,}"
)

print(
    "\nDo NOT modify geography or merge contextual features yet."
)

In [ ]:
# ============================================================
# STEP 24.8C-0 — CONTEXTUAL TABLES PRE-MERGE QC
# Construction + Rental Market + Seasonality
# ============================================================

import pandas as pd

print("=" * 80)
print("STEP 24.8C-0 — CONTEXTUAL TABLES PRE-MERGE QC")
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

CONSTRUCTION_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "construction"
    / "construction_features_ta_transition_frozen.parquet"
)

RENTAL_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "property_market"
    / "rental_market_features_ta_transition_frozen.parquet"
)

SEASONALITY_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "seasonality"
    / "seasonality_features_ta_transition_merge_ready_frozen.parquet"
)


# ============================================================
# 2. EXPECTED STRUCTURE
# ============================================================

EXPECTED_TRANSITIONS = [
    f"T{i}"
    for i in range(1, 11)
]

EXPECTED_CONSTRUCTION_ROWS = 670
EXPECTED_RENTAL_ROWS = 670
EXPECTED_SEASONALITY_ROWS = 680

KEY = [
    "transition",
    "ta_code"
]


# ============================================================
# 3. HELPER
# ============================================================

def audit_context_table(
    name,
    path,
    expected_rows,
    expect_999
):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if not path.exists():
        raise FileNotFoundError(
            f"{name}: file not found:\n{path}"
        )

    df = pd.read_parquet(path)

    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Expected rows:",
        f"{expected_rows:,}"
    )

    print(
        "Columns:",
        len(df.columns)
    )


    # --------------------------------------------------------
    # Required keys
    # --------------------------------------------------------

    missing_key_columns = [
        c
        for c in KEY
        if c not in df.columns
    ]

    if missing_key_columns:
        raise RuntimeError(
            f"{name}: missing key columns "
            f"{missing_key_columns}"
        )


    # --------------------------------------------------------
    # Normalise key formats for QC
    # --------------------------------------------------------

    df = df.copy()

    df["transition"] = (
        df["transition"]
        .astype("string")
        .str.strip()
    )

    df["ta_code"] = (
        df["ta_code"]
        .astype("string")
        .str.strip()
    )


    # --------------------------------------------------------
    # Key QC
    # --------------------------------------------------------

    duplicate_keys = int(
        df[
            KEY
        ]
        .duplicated()
        .sum()
    )

    missing_keys = int(
        df[
            KEY
        ]
        .isna()
        .sum()
        .sum()
    )


    # --------------------------------------------------------
    # Transition inventory
    # --------------------------------------------------------

    transitions = (
        df[
            "transition"
        ]
        .drop_duplicates()
        .tolist()
    )


    transition_set_ok = (
        set(transitions)
        ==
        set(EXPECTED_TRANSITIONS)
    )


    # --------------------------------------------------------
    # TA inventory
    # --------------------------------------------------------

    unique_ta_codes = int(
        df[
            "ta_code"
        ]
        .nunique()
    )

    has_999 = bool(
        (
            df[
                "ta_code"
            ]
            ==
            "999"
        )
        .any()
    )

    rows_999 = int(
        (
            df[
                "ta_code"
            ]
            ==
            "999"
        )
        .sum()
    )


    # --------------------------------------------------------
    # Rows per transition
    # --------------------------------------------------------

    rows_by_transition = (
        df
        .groupby(
            "transition"
        )
        .size()
        .reindex(
            EXPECTED_TRANSITIONS
        )
    )


    print(
        "Duplicate transition+TA keys:",
        duplicate_keys
    )

    print(
        "Missing transition+TA key values:",
        missing_keys
    )

    print(
        "Transitions T1-T10 exactly:",
        transition_set_ok
    )

    print(
        "Unique TA codes:",
        unique_ta_codes
    )

    print(
        "Contains code 999:",
        has_999
    )

    print(
        "Rows with code 999:",
        rows_999
    )

    print("\nRows per transition:")
    display(
        rows_by_transition
        .rename("rows")
        .reset_index()
    )


    # --------------------------------------------------------
    # Expected 999 behaviour
    # --------------------------------------------------------

    if expect_999:

        expected_ta_count = 68
        expected_rows_per_transition = 68

        category_rule_ok = (
            has_999
            and
            unique_ta_codes == 68
            and
            rows_999 == 10
        )

    else:

        expected_ta_count = 67
        expected_rows_per_transition = 67

        category_rule_ok = (
            (not has_999)
            and
            unique_ta_codes == 67
            and
            rows_999 == 0
        )


    rows_per_transition_ok = bool(
        (
            rows_by_transition
            ==
            expected_rows_per_transition
        )
        .all()
    )


    row_count_ok = (
        len(df)
        ==
        expected_rows
    )


    pass_status = all(
        [
            row_count_ok,
            duplicate_keys == 0,
            missing_keys == 0,
            transition_set_ok,
            category_rule_ok,
            rows_per_transition_ok
        ]
    )


    print("\nExpected TA count:", expected_ta_count)

    print(
        "Expected rows/transition:",
        expected_rows_per_transition
    )

    print(
        "Category rule OK:",
        category_rule_ok
    )

    print(
        "Rows/transition OK:",
        rows_per_transition_ok
    )

    print(
        f"{name} STATUS:",
        "PASS" if pass_status else "FAIL"
    )


    return df, {
        "table": name,
        "rows": len(df),
        "expected_rows": expected_rows,
        "unique_ta_codes": unique_ta_codes,
        "contains_999": has_999,
        "rows_999": rows_999,
        "duplicate_keys": duplicate_keys,
        "missing_keys": missing_keys,
        "transition_set_ok": transition_set_ok,
        "rows_per_transition_ok": rows_per_transition_ok,
        "status": "PASS" if pass_status else "FAIL"
    }


# ============================================================
# 4. CONSTRUCTION
# ============================================================

construction, construction_qc = audit_context_table(
    name="CONSTRUCTION",
    path=CONSTRUCTION_PATH,
    expected_rows=EXPECTED_CONSTRUCTION_ROWS,
    expect_999=False
)


# ============================================================
# 5. RENTAL MARKET
# ============================================================

rental, rental_qc = audit_context_table(
    name="RENTAL MARKET",
    path=RENTAL_PATH,
    expected_rows=EXPECTED_RENTAL_ROWS,
    expect_999=False
)


# ============================================================
# 6. SEASONALITY
# ============================================================

seasonality, seasonality_qc = audit_context_table(
    name="SEASONALITY MERGE-READY",
    path=SEASONALITY_PATH,
    expected_rows=EXPECTED_SEASONALITY_ROWS,
    expect_999=True
)


# ============================================================
# 7. CROSS-TABLE KEY COMPARISON
# ============================================================

print("\n" + "=" * 80)
print("CROSS-TABLE KEY COMPARISON")
print("=" * 80)


construction_keys = set(
    map(
        tuple,
        construction[
            KEY
        ].to_numpy()
    )
)

rental_keys = set(
    map(
        tuple,
        rental[
            KEY
        ].to_numpy()
    )
)

seasonality_keys = set(
    map(
        tuple,
        seasonality[
            KEY
        ].to_numpy()
    )
)


print(
    "Construction keys:",
    len(construction_keys)
)

print(
    "Rental keys:",
    len(rental_keys)
)

print(
    "Seasonality keys:",
    len(seasonality_keys)
)


construction_vs_rental = (
    construction_keys
    ==
    rental_keys
)


extra_seasonality_keys = (
    seasonality_keys
    -
    construction_keys
)


missing_from_seasonality = (
    construction_keys
    -
    seasonality_keys
)


print(
    "\nConstruction keys == Rental keys:",
    construction_vs_rental
)

print(
    "Extra seasonality keys:",
    len(extra_seasonality_keys)
)

print(
    "Construction keys missing from seasonality:",
    len(missing_from_seasonality)
)


extra_seasonality_999 = all(
    key[1] == "999"
    for key in extra_seasonality_keys
)


print(
    "All extra seasonality keys are code 999:",
    extra_seasonality_999
)


if extra_seasonality_keys:

    print(
        "\nExtra seasonality keys:"
    )

    print(
        sorted(
            extra_seasonality_keys
        )
    )


# ============================================================
# 8. SUMMARY
# ============================================================

summary = pd.DataFrame(
    [
        construction_qc,
        rental_qc,
        seasonality_qc
    ]
)


print("\n" + "=" * 80)
print("CONTEXTUAL PRE-MERGE QC SUMMARY")
print("=" * 80)

display(summary)


# ============================================================
# 9. FINAL STATUS
# ============================================================

FINAL_PASS = all(
    [
        construction_qc["status"] == "PASS",
        rental_qc["status"] == "PASS",
        seasonality_qc["status"] == "PASS",

        construction_vs_rental,

        len(extra_seasonality_keys) == 10,

        extra_seasonality_999,

        len(missing_from_seasonality) == 0
    ]
)


print("\n" + "=" * 80)

if FINAL_PASS:

    print(
        "STEP 24.8C-0 STATUS: PASS — "
        "CONTEXTUAL TABLES READY FOR CONTROLLED MERGE"
    )

else:

    print(
        "STEP 24.8C-0 STATUS: FAIL — "
        "DO NOT MERGE CONTEXTUAL FEATURES YET"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8C-1 — CONTROLLED MERGE OF CONTEXTUAL CHANNELS
#
# Channels:
#   1. Construction / development pipeline
#   2. Local residential rental-market context
#   3. Historical accommodation seasonality
#
# Merge key:
#   transition + ta_code
#
# IMPORTANT:
# - Preserve all 491,140 listing-origin rows.
# - TA code 999 is intentionally absent from construction/rental.
# - TA code 999 IS present in merge-ready seasonality.
# - Join failure is audited separately from feature-level NaN.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8C-1 — CONTROLLED CONTEXTUAL FEATURE MERGE")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

BACKBONE_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_TA.parquet"
)

CONSTRUCTION_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "construction"
    / "construction_features_ta_transition_frozen.parquet"
)

RENTAL_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "property_market"
    / "rental_market_features_ta_transition_frozen.parquet"
)

SEASONALITY_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "seasonality"
    / "seasonality_features_ta_transition_merge_ready_frozen.parquet"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context.parquet"
)


EXPECTED_TOTAL = 491_140

EXPECTED_OUTSIDE_TA_ROWS = 2_448

KEY = [
    "transition",
    "ta_code"
]

ORIGIN_KEY = [
    "transition",
    "snapshot_date",
    "listing_id"
]


# ============================================================
# 2. LOAD BACKBONE
# ============================================================

backbone = pd.read_parquet(
    BACKBONE_PATH
)


print("\nBackbone rows:", f"{len(backbone):,}")
print("Backbone columns:", len(backbone.columns))


if len(backbone) != EXPECTED_TOTAL:

    raise RuntimeError(
        f"Backbone row mismatch: "
        f"{len(backbone):,} != {EXPECTED_TOTAL:,}"
    )


# Standardise merge-key representation
backbone["transition"] = (
    backbone["transition"]
    .astype("string")
    .str.strip()
)

backbone["ta_code"] = (
    backbone["ta_code"]
    .astype("string")
    .str.strip()
)


# Listing-origin integrity
duplicate_origin_keys = int(
    backbone[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)

missing_origin_keys = int(
    backbone[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Duplicate listing-origin keys:",
    duplicate_origin_keys
)

print(
    "Missing listing-origin key values:",
    missing_origin_keys
)


if duplicate_origin_keys != 0:

    raise RuntimeError(
        "Backbone contains duplicate listing-origin keys."
    )


if missing_origin_keys != 0:

    raise RuntimeError(
        "Backbone contains missing listing-origin keys."
    )


# Outside-TA count
outside_mask = (
    backbone["ta_code"]
    ==
    "999"
)

outside_rows = int(
    outside_mask.sum()
)


print(
    "Outside-TA rows (999):",
    f"{outside_rows:,}"
)


if outside_rows != EXPECTED_OUTSIDE_TA_ROWS:

    raise RuntimeError(
        f"Expected {EXPECTED_OUTSIDE_TA_ROWS:,} "
        f"outside-TA rows, found {outside_rows:,}."
    )


# ============================================================
# 3. LOAD CONTEXTUAL TABLES
# ============================================================

construction = pd.read_parquet(
    CONSTRUCTION_PATH
)

rental = pd.read_parquet(
    RENTAL_PATH
)

seasonality = pd.read_parquet(
    SEASONALITY_PATH
)


for df in [
    construction,
    rental,
    seasonality
]:

    df["transition"] = (
        df["transition"]
        .astype("string")
        .str.strip()
    )

    df["ta_code"] = (
        df["ta_code"]
        .astype("string")
        .str.strip()
    )


# ============================================================
# 4. PRE-MERGE KEY SAFETY CHECK
# ============================================================

for name, df, expected_n in [

    (
        "construction",
        construction,
        670
    ),

    (
        "rental",
        rental,
        670
    ),

    (
        "seasonality",
        seasonality,
        680
    )
]:

    if len(df) != expected_n:

        raise RuntimeError(
            f"{name}: expected {expected_n:,} rows, "
            f"found {len(df):,}."
        )


    dup = int(
        df[
            KEY
        ]
        .duplicated()
        .sum()
    )


    miss = int(
        df[
            KEY
        ]
        .isna()
        .sum()
        .sum()
    )


    if dup != 0:

        raise RuntimeError(
            f"{name}: duplicate transition+ta_code keys."
        )


    if miss != 0:

        raise RuntimeError(
            f"{name}: missing transition+ta_code keys."
        )


# ============================================================
# 5. PREPARE CONSTRUCTION FEATURES
# ============================================================

construction_feature_cols = [
    "latest_available_consent_period",

    "construction_dwelling_3m_log",
    "construction_dwelling_12m_log",
    "construction_momentum_3m_vs_prior9m",
    "construction_multiunit_share_12m",

    "construction_dwelling_6m_log",
    "no_dwelling_12m"
]


missing_construction_cols = [
    c
    for c in construction_feature_cols
    if c not in construction.columns
]


if missing_construction_cols:

    raise RuntimeError(
        "Missing construction columns: "
        f"{missing_construction_cols}"
    )


construction_merge = construction[
    KEY
    +
    construction_feature_cols
].copy()


# Explicit join-success flag
construction_merge[
    "_construction_key_present"
] = 1


# ============================================================
# 6. PREPARE RENTAL FEATURES
# ============================================================

rental_feature_cols = [
    "eligible_rental_month",

    "rental_geomean_rent_log",
    "rental_geomean_rent_change_12m_log",

    "rental_geomean_rent_change_3m_log",

    "rental_current_available",
    "rental_change_12m_available"
]


missing_rental_cols = [
    c
    for c in rental_feature_cols
    if c not in rental.columns
]


if missing_rental_cols:

    raise RuntimeError(
        "Missing rental columns: "
        f"{missing_rental_cols}"
    )


rental_merge = rental[
    KEY
    +
    rental_feature_cols
].copy()


rental_merge[
    "_rental_key_present"
] = 1


# ============================================================
# 7. PREPARE SEASONALITY FEATURES
# ============================================================

seasonality_feature_cols = [
    "anchor_date",
    "future_date",
    "forecast_horizon_days",
    "historical_seasonality_index_horizon",
    "seasonality_source"
]


missing_seasonality_cols = [
    c
    for c in seasonality_feature_cols
    if c not in seasonality.columns
]


if missing_seasonality_cols:

    raise RuntimeError(
        "Missing seasonality columns: "
        f"{missing_seasonality_cols}"
    )


seasonality_merge = seasonality[
    KEY
    +
    seasonality_feature_cols
].copy()


seasonality_merge[
    "_seasonality_key_present"
] = 1


# ============================================================
# 8. CONSTRUCTION MERGE
# ============================================================

print("\n" + "=" * 80)
print("CONSTRUCTION MERGE")
print("=" * 80)


rows_before = len(
    backbone
)


merged = backbone.merge(
    construction_merge,
    on=KEY,
    how="left",
    validate="many_to_one"
)


rows_after = len(
    merged
)


construction_key_unmatched = int(
    merged[
        "_construction_key_present"
    ]
    .isna()
    .sum()
)


construction_unmatched_999 = int(
    (
        merged[
            "_construction_key_present"
        ]
        .isna()
        &
        (
            merged[
                "ta_code"
            ]
            ==
            "999"
        )
    )
    .sum()
)


construction_unmatched_actual_ta = int(
    (
        merged[
            "_construction_key_present"
        ]
        .isna()
        &
        (
            merged[
                "ta_code"
            ]
            !=
            "999"
        )
    )
    .sum()
)


print(
    "Rows before:",
    f"{rows_before:,}"
)

print(
    "Rows after:",
    f"{rows_after:,}"
)

print(
    "Unmatched construction keys:",
    f"{construction_key_unmatched:,}"
)

print(
    "Unmatched construction keys — code 999:",
    f"{construction_unmatched_999:,}"
)

print(
    "Unmatched construction keys — actual TA:",
    f"{construction_unmatched_actual_ta:,}"
)


if rows_after != rows_before:

    raise RuntimeError(
        "Construction merge changed row count."
    )


if construction_unmatched_999 != EXPECTED_OUTSIDE_TA_ROWS:

    raise RuntimeError(
        "Construction outside-TA unmatched count "
        "is not as expected."
    )


if construction_unmatched_actual_ta != 0:

    raise RuntimeError(
        "Construction has unmatched actual TA rows."
    )


# ============================================================
# 9. RENTAL MERGE
# ============================================================

print("\n" + "=" * 80)
print("RENTAL MARKET MERGE")
print("=" * 80)


rows_before_rental = len(
    merged
)


merged = merged.merge(
    rental_merge,
    on=KEY,
    how="left",
    validate="many_to_one"
)


rows_after_rental = len(
    merged
)


rental_key_unmatched = int(
    merged[
        "_rental_key_present"
    ]
    .isna()
    .sum()
)


rental_unmatched_999 = int(
    (
        merged[
            "_rental_key_present"
        ]
        .isna()
        &
        (
            merged[
                "ta_code"
            ]
            ==
            "999"
        )
    )
    .sum()
)


rental_unmatched_actual_ta = int(
    (
        merged[
            "_rental_key_present"
        ]
        .isna()
        &
        (
            merged[
                "ta_code"
            ]
            !=
            "999"
        )
    )
    .sum()
)


print(
    "Rows before:",
    f"{rows_before_rental:,}"
)

print(
    "Rows after:",
    f"{rows_after_rental:,}"
)

print(
    "Unmatched rental keys:",
    f"{rental_key_unmatched:,}"
)

print(
    "Unmatched rental keys — code 999:",
    f"{rental_unmatched_999:,}"
)

print(
    "Unmatched rental keys — actual TA:",
    f"{rental_unmatched_actual_ta:,}"
)


if rows_after_rental != rows_before_rental:

    raise RuntimeError(
        "Rental merge changed row count."
    )


if rental_unmatched_999 != EXPECTED_OUTSIDE_TA_ROWS:

    raise RuntimeError(
        "Rental outside-TA unmatched count "
        "is not as expected."
    )


if rental_unmatched_actual_ta != 0:

    raise RuntimeError(
        "Rental has unmatched actual TA rows."
    )


# ============================================================
# 10. SEASONALITY MERGE
# ============================================================

print("\n" + "=" * 80)
print("SEASONALITY MERGE")
print("=" * 80)


rows_before_seasonality = len(
    merged
)


merged = merged.merge(
    seasonality_merge,
    on=KEY,
    how="left",
    validate="many_to_one"
)


rows_after_seasonality = len(
    merged
)


seasonality_key_unmatched = int(
    merged[
        "_seasonality_key_present"
    ]
    .isna()
    .sum()
)


seasonality_unmatched_999 = int(
    (
        merged[
            "_seasonality_key_present"
        ]
        .isna()
        &
        (
            merged[
                "ta_code"
            ]
            ==
            "999"
        )
    )
    .sum()
)


print(
    "Rows before:",
    f"{rows_before_seasonality:,}"
)

print(
    "Rows after:",
    f"{rows_after_seasonality:,}"
)

print(
    "Unmatched seasonality keys:",
    seasonality_key_unmatched
)

print(
    "Unmatched seasonality code-999 rows:",
    seasonality_unmatched_999
)


if rows_after_seasonality != rows_before_seasonality:

    raise RuntimeError(
        "Seasonality merge changed row count."
    )


if seasonality_key_unmatched != 0:

    raise RuntimeError(
        "Seasonality should match all 68 TA categories."
    )


# ============================================================
# 11. FEATURE-LEVEL MISSINGNESS AUDIT
# ============================================================
#
# IMPORTANT:
# Key-level unmatched != feature-level missing.
#
# A contextual key may match successfully while a particular
# source feature is legitimately unavailable.
# ============================================================

print("\n" + "=" * 80)
print("FEATURE-LEVEL MISSINGNESS AUDIT")
print("=" * 80)


audit_features = [

    # Construction primary
    "construction_dwelling_3m_log",
    "construction_dwelling_12m_log",
    "construction_momentum_3m_vs_prior9m",
    "construction_multiunit_share_12m",

    # Rental primary
    "rental_geomean_rent_log",
    "rental_geomean_rent_change_12m_log",

    # Seasonality primary
    "historical_seasonality_index_horizon",
    "forecast_horizon_days"
]


missingness_rows = []


for col in audit_features:

    n_missing = int(
        merged[
            col
        ]
        .isna()
        .sum()
    )

    pct_missing = (
        n_missing
        /
        len(merged)
        *
        100
    )

    n_missing_999 = int(
        (
            merged[
                col
            ]
            .isna()
            &
            (
                merged[
                    "ta_code"
                ]
                ==
                "999"
            )
        )
        .sum()
    )

    n_missing_actual_ta = (
        n_missing
        -
        n_missing_999
    )


    missingness_rows.append(
        {
            "feature": col,

            "missing_rows":
                n_missing,

            "missing_pct":
                pct_missing,

            "missing_code_999":
                n_missing_999,

            "missing_actual_TA":
                n_missing_actual_ta
        }
    )


missingness = pd.DataFrame(
    missingness_rows
)


display(
    missingness
)


# ============================================================
# 12. SEASONALITY STRICT FEATURE QC
# ============================================================

seasonality_primary_missing = int(
    merged[
        [
            "historical_seasonality_index_horizon",
            "forecast_horizon_days"
        ]
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "\nMissing values in primary seasonality features:",
    seasonality_primary_missing
)


if seasonality_primary_missing != 0:

    raise RuntimeError(
        "Primary seasonality features contain missing values."
    )


# ============================================================
# 13. SEASONALITY 999 FALLBACK AUDIT
# ============================================================

seasonality_999 = merged.loc[
    merged[
        "ta_code"
    ]
    ==
    "999",
    [
        "transition",
        "ta_code",
        "historical_seasonality_index_horizon",
        "forecast_horizon_days",
        "seasonality_source"
    ]
].copy()


print("\n" + "=" * 80)
print("CODE 999 SEASONALITY FALLBACK AUDIT")
print("=" * 80)


print(
    "Code 999 rows:",
    f"{len(seasonality_999):,}"
)

print(
    "Missing historical seasonality:",
    int(
        seasonality_999[
            "historical_seasonality_index_horizon"
        ]
        .isna()
        .sum()
    )
)

print(
    "Missing forecast horizon:",
    int(
        seasonality_999[
            "forecast_horizon_days"
        ]
        .isna()
        .sum()
    )
)


print("\nSeasonality source(s) for code 999:")

display(
    seasonality_999[
        [
            "transition",
            "seasonality_source"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "transition"
    )
)


# ============================================================
# 14. TRANSITION-LEVEL JOIN QC
# ============================================================

transition_qc = (
    merged
    .groupby(
        "transition",
        sort=False
    )
    .agg(

        rows=(
            "listing_id",
            "size"
        ),

        outside_ta_rows=(
            "ta_code",
            lambda x: int(
                (x == "999").sum()
            )
        ),

        construction_key_missing=(
            "_construction_key_present",
            lambda x: int(
                x.isna().sum()
            )
        ),

        rental_key_missing=(
            "_rental_key_present",
            lambda x: int(
                x.isna().sum()
            )
        ),

        seasonality_key_missing=(
            "_seasonality_key_present",
            lambda x: int(
                x.isna().sum()
            )
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("TRANSITION-LEVEL CONTEXT JOIN QC")
print("=" * 80)

display(
    transition_qc
)


# Construction/rental key missingness should equal outside TA
transition_rule_ok = bool(
    (
        transition_qc[
            "construction_key_missing"
        ]
        ==
        transition_qc[
            "outside_ta_rows"
        ]
    ).all()

    and

    (
        transition_qc[
            "rental_key_missing"
        ]
        ==
        transition_qc[
            "outside_ta_rows"
        ]
    ).all()

    and

    (
        transition_qc[
            "seasonality_key_missing"
        ]
        ==
        0
    ).all()
)


print(
    "Transition-level structural missingness rule OK:",
    transition_rule_ok
)


# ============================================================
# 15. FINAL ORIGIN-KEY INTEGRITY
# ============================================================

final_duplicate_origin_keys = int(
    merged[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


final_missing_origin_keys = int(
    merged[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("FINAL BACKBONE INTEGRITY")
print("=" * 80)

print(
    "Rows:",
    f"{len(merged):,}"
)

print(
    "Duplicate listing-origin keys:",
    final_duplicate_origin_keys
)

print(
    "Missing listing-origin key values:",
    final_missing_origin_keys
)


# ============================================================
# 16. REMOVE TEMPORARY JOIN FLAGS
# ============================================================

merged = merged.drop(
    columns=[
        "_construction_key_present",
        "_rental_key_present",
        "_seasonality_key_present"
    ]
)


# ============================================================
# 17. PRE-SAVE PASS
# ============================================================

pre_save_pass = all(
    [
        len(merged) == EXPECTED_TOTAL,

        construction_key_unmatched
            ==
            EXPECTED_OUTSIDE_TA_ROWS,

        construction_unmatched_actual_ta
            ==
            0,

        rental_key_unmatched
            ==
            EXPECTED_OUTSIDE_TA_ROWS,

        rental_unmatched_actual_ta
            ==
            0,

        seasonality_key_unmatched
            ==
            0,

        seasonality_primary_missing
            ==
            0,

        transition_rule_ok,

        final_duplicate_origin_keys
            ==
            0,

        final_missing_origin_keys
            ==
            0
    ]
)


if not pre_save_pass:

    raise RuntimeError(
        "STEP 24.8C-1 pre-save QC failed. "
        "Context-integrated backbone will NOT be frozen."
    )


# ============================================================
# 18. SAVE
# ============================================================

merged.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\nSaved:")
print(OUTPUT_PATH)


# ============================================================
# 19. SAVE / RELOAD QC
# ============================================================

check = pd.read_parquet(
    OUTPUT_PATH
)


reload_duplicates = int(
    check[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


reload_missing_keys = int(
    check[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


reload_seasonality_missing = int(
    check[
        [
            "historical_seasonality_index_horizon",
            "forecast_horizon_days"
        ]
    ]
    .isna()
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("SAVE / RELOAD QC")
print("=" * 80)

print(
    "Reload rows:",
    f"{len(check):,}"
)

print(
    "Reload columns:",
    len(check.columns)
)

print(
    "Duplicate listing-origin keys:",
    reload_duplicates
)

print(
    "Missing listing-origin key values:",
    reload_missing_keys
)

print(
    "Missing primary seasonality values:",
    reload_seasonality_missing
)


# ============================================================
# 20. FINAL STATUS
# ============================================================

final_pass = all(
    [
        len(check)
            ==
            EXPECTED_TOTAL,

        reload_duplicates
            ==
            0,

        reload_missing_keys
            ==
            0,

        reload_seasonality_missing
            ==
            0
    ]
)


print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if final_pass:

    print(
        "STEP 24.8C-1 STATUS: PASS — "
        "CONTEXTUAL CHANNELS INTEGRATED"
    )

else:

    print(
        "STEP 24.8C-1 STATUS: FAIL — "
        "DO NOT PROCEED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8D-0 — REVIEW POOLED REPRESENTATION PRE-MERGE QC
#
# Purpose:
#   Verify the 10 frozen listing-level review embedding files
#   against the complete T1-T10 modelling backbone BEFORE
#   merging 384 embedding dimensions.
#
# Expected:
#   Backbone rows                  = 491,140
#   Review-pooled rows             = 398,989
#   No-review listing-origin rows  =  92,151
#   Membership represented         = 7,334,068
#
# No files are modified in this step.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8D-0 — REVIEW POOLED REPRESENTATION PRE-MERGE QC")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

BACKBONE_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context.parquet"
)

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "listing_review_embeddings_mean365"
)


EXPECTED_BACKBONE_ROWS = 491_140

EXPECTED_REVIEW_ROWS = 398_989

EXPECTED_NO_REVIEW_ROWS = (
    EXPECTED_BACKBONE_ROWS
    -
    EXPECTED_REVIEW_ROWS
)

EXPECTED_MEMBERSHIPS = 7_334_068

EXPECTED_EMBEDDING_DIM = 384


anchor_map = {
    "T1":  "2025-10-05",
    "T2":  "2025-11-07",
    "T3":  "2025-12-11",
    "T4":  "2026-01-16",
    "T5":  "2026-02-13",
    "T6":  "2026-03-17",
    "T7":  "2026-04-16",
    "T8":  "2026-05-23",
    "T9":  "2026-06-19",
    "T10": "2026-07-12"
}


expected_review_rows = {
    "T1": 38321,
    "T2": 38870,
    "T3": 39602,
    "T4": 38810,
    "T5": 40216,
    "T6": 40630,
    "T7": 40543,
    "T8": 40149,
    "T9": 40810,
    "T10": 41038
}


expected_memberships = {
    "T1": 702695,
    "T2": 709654,
    "T3": 714539,
    "T4": 704488,
    "T5": 722691,
    "T6": 750513,
    "T7": 762091,
    "T8": 755301,
    "T9": 756271,
    "T10": 755825
}


# ============================================================
# 2. LOAD BACKBONE KEYS ONLY
# ============================================================

backbone = pd.read_parquet(
    BACKBONE_PATH,
    columns=[
        "transition",
        "snapshot_date",
        "listing_id"
    ]
)


backbone["transition"] = (
    backbone["transition"]
    .astype("string")
    .str.strip()
)

backbone["snapshot_date"] = pd.to_datetime(
    backbone["snapshot_date"]
).dt.normalize()


print(
    "\nBackbone rows:",
    f"{len(backbone):,}"
)


if len(backbone) != EXPECTED_BACKBONE_ROWS:

    raise RuntimeError(
        "Unexpected backbone row count."
    )


ORIGIN_KEY = [
    "transition",
    "snapshot_date",
    "listing_id"
]


backbone_duplicates = int(
    backbone[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


backbone_missing = int(
    backbone[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Backbone duplicate origin keys:",
    backbone_duplicates
)

print(
    "Backbone missing origin keys:",
    backbone_missing
)


if backbone_duplicates != 0:
    raise RuntimeError(
        "Backbone origin keys are not unique."
    )


if backbone_missing != 0:
    raise RuntimeError(
        "Backbone contains missing origin keys."
    )


# ============================================================
# 3. LOAD / AUDIT EACH POOLED REVIEW FILE
# ============================================================

review_parts = []

qc_rows = []

reference_columns = None


for transition, snapshot in anchor_map.items():

    print("\n" + "-" * 80)
    print(
        f"Auditing {transition} — {snapshot}"
    )


    path = (
        REVIEW_DIR
        / f"listing_review_mean365_{transition}.parquet"
    )


    if not path.exists():

        raise FileNotFoundError(
            f"{transition}: missing review pooled file:\n"
            f"{path}"
        )


    df = pd.read_parquet(
        path
    )


    # --------------------------------------------------------
    # Required non-embedding columns
    # --------------------------------------------------------

    required_cols = [
        "transition",
        "snapshot_date",
        "listing_id",
        "review_count_365d"
    ]


    missing_required = [
        c
        for c in required_cols
        if c not in df.columns
    ]


    if missing_required:

        raise RuntimeError(
            f"{transition}: missing columns "
            f"{missing_required}"
        )


    # --------------------------------------------------------
    # Embedding columns
    # --------------------------------------------------------

    embedding_cols = [
        c
        for c in df.columns
        if c.startswith("emb_")
    ]


    expected_embedding_cols = [
        f"emb_{i:03d}"
        for i in range(
            EXPECTED_EMBEDDING_DIM
        )
    ]


    if embedding_cols != expected_embedding_cols:

        raise RuntimeError(
            f"{transition}: embedding schema/order "
            f"is not exactly emb_000 ... emb_383."
        )


    # --------------------------------------------------------
    # Complete schema consistency across T1-T10
    # --------------------------------------------------------

    if reference_columns is None:

        reference_columns = list(
            df.columns
        )

    elif list(df.columns) != reference_columns:

        raise RuntimeError(
            f"{transition}: pooled review schema "
            f"differs from T1."
        )


    # --------------------------------------------------------
    # Normalise temporal keys
    # --------------------------------------------------------

    df["transition"] = (
        df["transition"]
        .astype("string")
        .str.strip()
    )


    df["snapshot_date"] = pd.to_datetime(
        df["snapshot_date"]
    ).dt.normalize()


    expected_date = pd.Timestamp(
        snapshot
    ).normalize()


    # --------------------------------------------------------
    # Transition/date mapping
    # --------------------------------------------------------

    wrong_transition = int(
        (
            df[
                "transition"
            ]
            !=
            transition
        )
        .sum()
    )


    wrong_date = int(
        (
            df[
                "snapshot_date"
            ]
            !=
            expected_date
        )
        .sum()
    )


    # --------------------------------------------------------
    # Origin-key QC
    # --------------------------------------------------------

    duplicate_keys = int(
        df[
            ORIGIN_KEY
        ]
        .duplicated()
        .sum()
    )


    missing_keys = int(
        df[
            ORIGIN_KEY
        ]
        .isna()
        .sum()
        .sum()
    )


    # --------------------------------------------------------
    # Review-count QC
    # --------------------------------------------------------

    missing_review_count = int(
        df[
            "review_count_365d"
        ]
        .isna()
        .sum()
    )


    nonpositive_review_count = int(
        (
            df[
                "review_count_365d"
            ]
            <=
            0
        )
        .sum()
    )


    review_count_sum = int(
        df[
            "review_count_365d"
        ]
        .sum()
    )


    # --------------------------------------------------------
    # Embedding numerical QC
    # --------------------------------------------------------

    emb = df[
        embedding_cols
    ].to_numpy(
        dtype=np.float32,
        copy=False
    )


    nan_values = int(
        np.isnan(
            emb
        ).sum()
    )


    inf_values = int(
        np.isinf(
            emb
        ).sum()
    )


    norms = np.linalg.norm(
        emb,
        axis=1
    )


    zero_norm_rows = int(
        (
            norms
            ==
            0
        )
        .sum()
    )


    # --------------------------------------------------------
    # Expected counts
    # --------------------------------------------------------

    row_count_ok = (
        len(df)
        ==
        expected_review_rows[
            transition
        ]
    )


    membership_count_ok = (
        review_count_sum
        ==
        expected_memberships[
            transition
        ]
    )


    transition_pass = all(
        [
            row_count_ok,
            membership_count_ok,
            wrong_transition == 0,
            wrong_date == 0,
            duplicate_keys == 0,
            missing_keys == 0,
            missing_review_count == 0,
            nonpositive_review_count == 0,
            nan_values == 0,
            inf_values == 0,
            zero_norm_rows == 0
        ]
    )


    qc_rows.append(
        {
            "transition":
                transition,

            "rows":
                len(df),

            "expected_rows":
                expected_review_rows[
                    transition
                ],

            "review_count_sum":
                review_count_sum,

            "expected_memberships":
                expected_memberships[
                    transition
                ],

            "duplicate_keys":
                duplicate_keys,

            "missing_keys":
                missing_keys,

            "NaN_emb":
                nan_values,

            "Inf_emb":
                inf_values,

            "zero_norm":
                zero_norm_rows,

            "status":
                "PASS"
                if transition_pass
                else "FAIL"
        }
    )


    review_parts.append(
        df
    )


# ============================================================
# 4. COMBINE T1-T10 REVIEW REPRESENTATIONS
# ============================================================

reviews = pd.concat(
    review_parts,
    ignore_index=True
)

del review_parts
gc.collect()


print("\n" + "=" * 80)
print("GLOBAL REVIEW REPRESENTATION QC")
print("=" * 80)


print(
    "Pooled review rows:",
    f"{len(reviews):,}"
)

print(
    "Expected pooled review rows:",
    f"{EXPECTED_REVIEW_ROWS:,}"
)


global_duplicates = int(
    reviews[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


global_missing_keys = int(
    reviews[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


global_memberships = int(
    reviews[
        "review_count_365d"
    ]
    .sum()
)


print(
    "Global duplicate review origin keys:",
    global_duplicates
)

print(
    "Global missing review origin keys:",
    global_missing_keys
)

print(
    "Total represented memberships:",
    f"{global_memberships:,}"
)

print(
    "Expected memberships:",
    f"{EXPECTED_MEMBERSHIPS:,}"
)


# ============================================================
# 5. VERIFY EVERY REVIEW REPRESENTATION BELONGS TO BACKBONE
# ============================================================

review_keys = reviews[
    ORIGIN_KEY
].copy()


membership_check = review_keys.merge(
    backbone,
    on=ORIGIN_KEY,
    how="left",
    validate="one_to_one",
    indicator=True
)


review_keys_not_in_backbone = int(
    (
        membership_check[
            "_merge"
        ]
        !=
        "both"
    )
    .sum()
)


print(
    "\nReview pooled keys absent from backbone:",
    review_keys_not_in_backbone
)


# ============================================================
# 6. DETERMINE BACKBONE COVERAGE
# ============================================================

coverage = backbone.merge(
    review_keys.assign(
        review_available=1
    ),
    on=ORIGIN_KEY,
    how="left",
    validate="one_to_one"
)


with_review = int(
    coverage[
        "review_available"
    ]
    .notna()
    .sum()
)


without_review = int(
    coverage[
        "review_available"
    ]
    .isna()
    .sum()
)


coverage_pct = (
    with_review
    /
    len(coverage)
    *
    100
)


print("\n" + "=" * 80)
print("BACKBONE REVIEW COVERAGE")
print("=" * 80)


print(
    "Backbone rows:",
    f"{len(coverage):,}"
)

print(
    "Rows with pooled review representation:",
    f"{with_review:,}"
)

print(
    "Expected with review:",
    f"{EXPECTED_REVIEW_ROWS:,}"
)

print(
    "Rows without current 365d review representation:",
    f"{without_review:,}"
)

print(
    "Expected without review:",
    f"{EXPECTED_NO_REVIEW_ROWS:,}"
)

print(
    "Overall review representation coverage:",
    f"{coverage_pct:.4f}%"
)


# ============================================================
# 7. COVERAGE BY TRANSITION
# ============================================================

coverage_transition = (
    coverage
    .assign(
        review_available=lambda x:
            x[
                "review_available"
            ]
            .notna()
            .astype(int)
    )
    .groupby(
        "transition",
        sort=False
    )
    .agg(

        backbone_rows=(
            "listing_id",
            "size"
        ),

        with_review=(
            "review_available",
            "sum"
        )
    )
    .reset_index()
)


coverage_transition[
    "without_review"
] = (
    coverage_transition[
        "backbone_rows"
    ]
    -
    coverage_transition[
        "with_review"
    ]
)


coverage_transition[
    "coverage_pct"
] = (
    coverage_transition[
        "with_review"
    ]
    /
    coverage_transition[
        "backbone_rows"
    ]
    *
    100
)


print("\nCoverage by transition:")

display(
    coverage_transition
)


# ============================================================
# 8. QC TABLE
# ============================================================

qc = pd.DataFrame(
    qc_rows
)


print("\n" + "=" * 80)
print("POOLED REVIEW FILE QC")
print("=" * 80)

display(
    qc
)


# ============================================================
# 9. FINAL STATUS
# ============================================================

all_transition_pass = bool(
    (
        qc[
            "status"
        ]
        ==
        "PASS"
    )
    .all()
)


FINAL_PASS = all(
    [
        all_transition_pass,

        len(reviews)
            ==
            EXPECTED_REVIEW_ROWS,

        global_duplicates
            ==
            0,

        global_missing_keys
            ==
            0,

        global_memberships
            ==
            EXPECTED_MEMBERSHIPS,

        review_keys_not_in_backbone
            ==
            0,

        with_review
            ==
            EXPECTED_REVIEW_ROWS,

        without_review
            ==
            EXPECTED_NO_REVIEW_ROWS
    ]
)


print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if FINAL_PASS:

    print(
        "STEP 24.8D-0 STATUS: PASS — "
        "POOLED REVIEW REPRESENTATION READY FOR BACKBONE MERGE"
    )

else:

    print(
        "STEP 24.8D-0 STATUS: FAIL — "
        "DO NOT MERGE REVIEW EMBEDDINGS"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8D-1 — MERGE FROZEN REVIEW REPRESENTATION
#
# IMPORTANT:
# - Left join into COMPLETE 491,140-row backbone.
# - Keep all zero-review listing-origin rows.
# - Do NOT zero-fill embeddings here.
# - review_available = 1 if a valid 365d pooled representation exists,
#                      0 otherwise.
# - review_count_365d = 0 is safe for no-review rows because this is
#   a directly defined count, not an imputed semantic value.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8D-1 — MERGE FROZEN REVIEW REPRESENTATION")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS / EXPECTATIONS
# ============================================================

BACKBONE_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context.parquet"
)

REVIEW_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "reviews"
    / "listing_review_embeddings_mean365"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context_reviews.parquet"
)


EXPECTED_TOTAL = 491_140
EXPECTED_WITH_REVIEW = 398_989
EXPECTED_WITHOUT_REVIEW = 92_151
EXPECTED_MEMBERSHIPS = 7_334_068
EXPECTED_EMBEDDING_DIM = 384


ORIGIN_KEY = [
    "transition",
    "snapshot_date",
    "listing_id"
]


TRANSITIONS = [
    f"T{i}"
    for i in range(1, 11)
]


# ============================================================
# 2. LOAD BACKBONE
# ============================================================

backbone = pd.read_parquet(
    BACKBONE_PATH
)

backbone["transition"] = (
    backbone["transition"]
    .astype("string")
    .str.strip()
)

backbone["snapshot_date"] = pd.to_datetime(
    backbone["snapshot_date"]
).dt.normalize()


print("\nBackbone rows:", f"{len(backbone):,}")
print("Backbone columns:", len(backbone.columns))


if len(backbone) != EXPECTED_TOTAL:
    raise RuntimeError(
        "Unexpected backbone row count."
    )


duplicate_backbone = int(
    backbone[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


missing_backbone_key = int(
    backbone[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Duplicate backbone origin keys:",
    duplicate_backbone
)

print(
    "Missing backbone origin keys:",
    missing_backbone_key
)


if duplicate_backbone != 0:
    raise RuntimeError(
        "Backbone origin keys are not unique."
    )

if missing_backbone_key != 0:
    raise RuntimeError(
        "Backbone contains missing origin keys."
    )


# ============================================================
# 3. LOAD FROZEN T1-T10 POOLED REVIEW REPRESENTATIONS
# ============================================================

review_parts = []


for transition in TRANSITIONS:

    path = (
        REVIEW_DIR
        / f"listing_review_mean365_{transition}.parquet"
    )

    print(
        f"Loading {transition}: {path.name}"
    )

    df = pd.read_parquet(
        path
    )

    df["transition"] = (
        df["transition"]
        .astype("string")
        .str.strip()
    )

    df["snapshot_date"] = pd.to_datetime(
        df["snapshot_date"]
    ).dt.normalize()

    review_parts.append(
        df
    )


reviews = pd.concat(
    review_parts,
    ignore_index=True
)

del review_parts
gc.collect()


print(
    "\nReview pooled rows:",
    f"{len(reviews):,}"
)


if len(reviews) != EXPECTED_WITH_REVIEW:
    raise RuntimeError(
        "Unexpected pooled review row count."
    )


# ============================================================
# 4. STRICT REVIEW SCHEMA QC
# ============================================================

embedding_cols = [
    f"emb_{i:03d}"
    for i in range(
        EXPECTED_EMBEDDING_DIM
    )
]


missing_embedding_cols = [
    c
    for c in embedding_cols
    if c not in reviews.columns
]


if missing_embedding_cols:
    raise RuntimeError(
        "Missing embedding columns."
    )


required_review_cols = (
    ORIGIN_KEY
    +
    ["review_count_365d"]
    +
    embedding_cols
)


reviews = reviews[
    required_review_cols
].copy()


duplicate_review_keys = int(
    reviews[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


missing_review_keys = int(
    reviews[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


membership_sum_before = int(
    reviews[
        "review_count_365d"
    ]
    .sum()
)


print(
    "Duplicate review origin keys:",
    duplicate_review_keys
)

print(
    "Missing review origin keys:",
    missing_review_keys
)

print(
    "Review memberships represented:",
    f"{membership_sum_before:,}"
)


if duplicate_review_keys != 0:
    raise RuntimeError(
        "Review representation keys are not unique."
    )

if missing_review_keys != 0:
    raise RuntimeError(
        "Review representation contains missing keys."
    )

if membership_sum_before != EXPECTED_MEMBERSHIPS:
    raise RuntimeError(
        "Review membership total mismatch."
    )


# ============================================================
# 5. ADD EXPLICIT AVAILABILITY FLAG
# ============================================================

reviews[
    "review_available"
] = np.int8(1)


# ============================================================
# 6. LEFT JOIN INTO COMPLETE BACKBONE
# ============================================================

print("\n" + "=" * 80)
print("REVIEW MERGE")
print("=" * 80)


rows_before = len(
    backbone
)


merged = backbone.merge(
    reviews,
    on=ORIGIN_KEY,
    how="left",
    validate="one_to_one",
    indicator="_review_merge"
)


rows_after = len(
    merged
)


merge_counts = (
    merged[
        "_review_merge"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "Rows before:",
    f"{rows_before:,}"
)

print(
    "Rows after:",
    f"{rows_after:,}"
)

print("\nMerge status:")
print(
    merge_counts
)


if rows_after != rows_before:
    raise RuntimeError(
        "Review merge changed backbone row count."
    )


# ============================================================
# 7. AVAILABILITY / COVERAGE QC
# ============================================================

matched = int(
    (
        merged[
            "_review_merge"
        ]
        ==
        "both"
    )
    .sum()
)


unmatched = int(
    (
        merged[
            "_review_merge"
        ]
        ==
        "left_only"
    )
    .sum()
)


right_only = int(
    (
        merged[
            "_review_merge"
        ]
        ==
        "right_only"
    )
    .sum()
)


print(
    "\nMatched review representations:",
    f"{matched:,}"
)

print(
    "No-review listing-origin rows:",
    f"{unmatched:,}"
)

print(
    "Right-only rows:",
    right_only
)


if matched != EXPECTED_WITH_REVIEW:
    raise RuntimeError(
        "Matched review count mismatch."
    )


if unmatched != EXPECTED_WITHOUT_REVIEW:
    raise RuntimeError(
        "No-review count mismatch."
    )


if right_only != 0:
    raise RuntimeError(
        "Unexpected right-only review rows."
    )


# ============================================================
# 8. FINALISE review_available
# ============================================================

merged[
    "review_available"
] = (
    merged[
        "review_available"
    ]
    .fillna(0)
    .astype("int8")
)


review_available_1 = int(
    (
        merged[
            "review_available"
        ]
        ==
        1
    )
    .sum()
)


review_available_0 = int(
    (
        merged[
            "review_available"
        ]
        ==
        0
    )
    .sum()
)


print(
    "\nreview_available = 1:",
    f"{review_available_1:,}"
)

print(
    "review_available = 0:",
    f"{review_available_0:,}"
)


# ============================================================
# 9. FINALISE review_count_365d
# ============================================================
#
# For no-review rows, count = 0 by definition.
# This is NOT semantic embedding imputation.
# ============================================================

missing_count_before = int(
    merged[
        "review_count_365d"
    ]
    .isna()
    .sum()
)


print(
    "Missing review_count_365d before fill:",
    f"{missing_count_before:,}"
)


merged[
    "review_count_365d"
] = (
    merged[
        "review_count_365d"
    ]
    .fillna(0)
)


# Require integer-valued counts
count_values = merged[
    "review_count_365d"
].to_numpy()


if not np.all(
    np.equal(
        count_values,
        np.floor(
            count_values
        )
    )
):
    raise RuntimeError(
        "review_count_365d contains non-integer values."
    )


merged[
    "review_count_365d"
] = (
    merged[
        "review_count_365d"
    ]
    .astype("int32")
)


membership_sum_after = int(
    merged[
        "review_count_365d"
    ]
    .sum()
)


print(
    "Membership total after merge:",
    f"{membership_sum_after:,}"
)


if membership_sum_after != EXPECTED_MEMBERSHIPS:
    raise RuntimeError(
        "Review membership total changed after merge."
    )


# ============================================================
# 10. EMBEDDING MISSINGNESS MUST MATCH review_available
# ============================================================

print("\n" + "=" * 80)
print("EMBEDDING MISSINGNESS QC")
print("=" * 80)


embedding_missing_count = (
    merged[
        embedding_cols
    ]
    .isna()
    .sum(
        axis=1
    )
)


fully_present = int(
    (
        embedding_missing_count
        ==
        0
    )
    .sum()
)


fully_missing = int(
    (
        embedding_missing_count
        ==
        EXPECTED_EMBEDDING_DIM
    )
    .sum()
)


partially_missing = int(
    (
        (
            embedding_missing_count
            >
            0
        )
        &
        (
            embedding_missing_count
            <
            EXPECTED_EMBEDDING_DIM
        )
    )
    .sum()
)


print(
    "Rows with all 384 embeddings present:",
    f"{fully_present:,}"
)

print(
    "Rows with all 384 embeddings missing:",
    f"{fully_missing:,}"
)

print(
    "Rows with partially missing embeddings:",
    f"{partially_missing:,}"
)


available_but_embedding_missing = int(
    (
        (
            merged[
                "review_available"
            ]
            ==
            1
        )
        &
        (
            embedding_missing_count
            !=
            0
        )
    )
    .sum()
)


unavailable_but_embedding_present = int(
    (
        (
            merged[
                "review_available"
            ]
            ==
            0
        )
        &
        (
            embedding_missing_count
            !=
            EXPECTED_EMBEDDING_DIM
        )
    )
    .sum()
)


print(
    "review_available=1 but incomplete embedding:",
    available_but_embedding_missing
)

print(
    "review_available=0 but embedding present:",
    unavailable_but_embedding_present
)


# ============================================================
# 11. NUMERICAL QC FOR AVAILABLE EMBEDDINGS
# ============================================================

available_mask = (
    merged[
        "review_available"
    ]
    ==
    1
)


available_emb = (
    merged.loc[
        available_mask,
        embedding_cols
    ]
    .to_numpy(
        dtype=np.float32,
        copy=False
    )
)


nan_available = int(
    np.isnan(
        available_emb
    ).sum()
)


inf_available = int(
    np.isinf(
        available_emb
    ).sum()
)


norms = np.linalg.norm(
    available_emb,
    axis=1
)


zero_norm_available = int(
    (
        norms
        ==
        0
    )
    .sum()
)


print(
    "\nNaN values among available embeddings:",
    nan_available
)

print(
    "Inf values among available embeddings:",
    inf_available
)

print(
    "Zero-norm available embeddings:",
    zero_norm_available
)


# ============================================================
# 12. COVERAGE BY TRANSITION AFTER MERGE
# ============================================================

transition_qc = (
    merged
    .groupby(
        "transition",
        sort=False
    )
    .agg(

        rows=(
            "listing_id",
            "size"
        ),

        with_review=(
            "review_available",
            "sum"
        ),

        review_memberships=(
            "review_count_365d",
            "sum"
        )
    )
    .reset_index()
)


transition_qc[
    "without_review"
] = (
    transition_qc[
        "rows"
    ]
    -
    transition_qc[
        "with_review"
    ]
)


transition_qc[
    "coverage_pct"
] = (
    transition_qc[
        "with_review"
    ]
    /
    transition_qc[
        "rows"
    ]
    *
    100
)


print("\n" + "=" * 80)
print("REVIEW COVERAGE AFTER MERGE")
print("=" * 80)

display(
    transition_qc
)


# ============================================================
# 13. LISTING-ORIGIN INTEGRITY
# ============================================================

final_duplicates = int(
    merged[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


final_missing_keys = int(
    merged[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("FINAL LISTING-ORIGIN INTEGRITY")
print("=" * 80)

print(
    "Rows:",
    f"{len(merged):,}"
)

print(
    "Duplicate listing-origin keys:",
    final_duplicates
)

print(
    "Missing listing-origin key values:",
    final_missing_keys
)


# ============================================================
# 14. REMOVE MERGE INDICATOR
# ============================================================

merged = merged.drop(
    columns=[
        "_review_merge"
    ]
)


# ============================================================
# 15. PRE-SAVE PASS
# ============================================================

pre_save_pass = all(
    [
        len(merged)
            ==
            EXPECTED_TOTAL,

        matched
            ==
            EXPECTED_WITH_REVIEW,

        unmatched
            ==
            EXPECTED_WITHOUT_REVIEW,

        review_available_1
            ==
            EXPECTED_WITH_REVIEW,

        review_available_0
            ==
            EXPECTED_WITHOUT_REVIEW,

        membership_sum_after
            ==
            EXPECTED_MEMBERSHIPS,

        fully_present
            ==
            EXPECTED_WITH_REVIEW,

        fully_missing
            ==
            EXPECTED_WITHOUT_REVIEW,

        partially_missing
            ==
            0,

        available_but_embedding_missing
            ==
            0,

        unavailable_but_embedding_present
            ==
            0,

        nan_available
            ==
            0,

        inf_available
            ==
            0,

        zero_norm_available
            ==
            0,

        final_duplicates
            ==
            0,

        final_missing_keys
            ==
            0
    ]
)


if not pre_save_pass:

    raise RuntimeError(
        "STEP 24.8D-1 pre-save QC failed. "
        "Integrated review backbone will NOT be frozen."
    )


# ============================================================
# 16. SAVE
# ============================================================

merged.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\nSaved:")
print(
    OUTPUT_PATH
)


# ============================================================
# 17. SAVE / RELOAD QC
# ============================================================

check = pd.read_parquet(
    OUTPUT_PATH
)


reload_duplicates = int(
    check[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


reload_review_available_1 = int(
    (
        check[
            "review_available"
        ]
        ==
        1
    )
    .sum()
)


reload_review_available_0 = int(
    (
        check[
            "review_available"
        ]
        ==
        0
    )
    .sum()
)


reload_memberships = int(
    check[
        "review_count_365d"
    ]
    .sum()
)


reload_emb_missing = (
    check[
        embedding_cols
    ]
    .isna()
    .sum(
        axis=1
    )
)


reload_fully_present = int(
    (
        reload_emb_missing
        ==
        0
    )
    .sum()
)


reload_fully_missing = int(
    (
        reload_emb_missing
        ==
        EXPECTED_EMBEDDING_DIM
    )
    .sum()
)


reload_partial = int(
    (
        (
            reload_emb_missing
            >
            0
        )
        &
        (
            reload_emb_missing
            <
            EXPECTED_EMBEDDING_DIM
        )
    )
    .sum()
)


print("\n" + "=" * 80)
print("SAVE / RELOAD QC")
print("=" * 80)

print(
    "Reload rows:",
    f"{len(check):,}"
)

print(
    "Reload columns:",
    len(check.columns)
)

print(
    "Duplicate listing-origin keys:",
    reload_duplicates
)

print(
    "review_available = 1:",
    f"{reload_review_available_1:,}"
)

print(
    "review_available = 0:",
    f"{reload_review_available_0:,}"
)

print(
    "Review memberships:",
    f"{reload_memberships:,}"
)

print(
    "Fully present embeddings:",
    f"{reload_fully_present:,}"
)

print(
    "Fully missing embeddings:",
    f"{reload_fully_missing:,}"
)

print(
    "Partially missing embeddings:",
    reload_partial
)


# ============================================================
# 18. FINAL STATUS
# ============================================================

FINAL_PASS = all(
    [
        len(check)
            ==
            EXPECTED_TOTAL,

        reload_duplicates
            ==
            0,

        reload_review_available_1
            ==
            EXPECTED_WITH_REVIEW,

        reload_review_available_0
            ==
            EXPECTED_WITHOUT_REVIEW,

        reload_memberships
            ==
            EXPECTED_MEMBERSHIPS,

        reload_fully_present
            ==
            EXPECTED_WITH_REVIEW,

        reload_fully_missing
            ==
            EXPECTED_WITHOUT_REVIEW,

        reload_partial
            ==
            0
    ]
)


print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if FINAL_PASS:

    print(
        "STEP 24.8D-1 STATUS: PASS — "
        "FROZEN REVIEW REPRESENTATION INTEGRATED"
    )

else:

    print(
        "STEP 24.8D-1 STATUS: FAIL — "
        "DO NOT PROCEED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8E-0 — OUTCOME SOURCE PREFLIGHT / FEASIBILITY QC
#
# Purpose
# -------
# Audit the 11 cleaned listing snapshots required to construct
# next-period outcomes for T1-T10.
#
# IMPORTANT:
# - This step creates NO outcomes.
# - This step modifies NO frozen files.
# - Aug-13-2026 is inspected ONLY as the future outcome source
#   for T10.
#
# Primary outcome feasibility:
#   next_period_persistence
#
# Secondary outcome feasibility:
#   relative advertised-price resilience
#
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8E-0 — OUTCOME SOURCE PREFLIGHT / FEASIBILITY QC")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

INTEGRATED_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context_reviews.parquet"
)


# ============================================================
# 2. SNAPSHOT / TRANSITION DEFINITIONS
# ============================================================

snapshots = [
    ("2025-10-05", "listings_2025-10-05.parquet"),
    ("2025-11-07", "listings_2025-11-07.parquet"),
    ("2025-12-11", "listings_2025-12-11.parquet"),
    ("2026-01-16", "listings_2026-01-16.parquet"),
    ("2026-02-13", "listings_2026-02-13.parquet"),
    ("2026-03-17", "listings_2026-03-17.parquet"),
    ("2026-04-16", "listings_2026-04-16.parquet"),
    ("2026-05-23", "listings_2026-05-23.parquet"),
    ("2026-06-19", "listings_2026-06-19.parquet"),
    ("2026-07-12", "listings_2026-07-12.parquet"),
    ("2026-08-13", "listings_2026-08-13.parquet"),
]


transitions = [
    ("T1",  "2025-10-05", "2025-11-07"),
    ("T2",  "2025-11-07", "2025-12-11"),
    ("T3",  "2025-12-11", "2026-01-16"),
    ("T4",  "2026-01-16", "2026-02-13"),
    ("T5",  "2026-02-13", "2026-03-17"),
    ("T6",  "2026-03-17", "2026-04-16"),
    ("T7",  "2026-04-16", "2026-05-23"),
    ("T8",  "2026-05-23", "2026-06-19"),
    ("T9",  "2026-06-19", "2026-07-12"),
    ("T10", "2026-07-12", "2026-08-13"),
]


expected_rows = {
    "2025-10-05": 47752,
    "2025-11-07": 48624,
    "2025-12-11": 49899,
    "2026-01-16": 46819,
    "2026-02-13": 48692,
    "2026-03-17": 49076,
    "2026-04-16": 48951,
    "2026-05-23": 49305,
    "2026-06-19": 50926,
    "2026-07-12": 51096,
    "2026-08-13": 51325,
}


# ============================================================
# 3. AUDIT EACH CLEANED SNAPSHOT
# ============================================================

snapshot_qc = []
snapshot_data = {}


print("\n" + "=" * 80)
print("CLEANED SNAPSHOT INVENTORY")
print("=" * 80)


for snapshot_date, filename in snapshots:

    path = LISTINGS_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing cleaned listing file:\n{path}"
        )

    df = pd.read_parquet(path)

    required = [
        "listing_id",
        "snapshot_date",
        "price",
        "room_type",
    ]

    missing_required = [
        c for c in required
        if c not in df.columns
    ]

    if missing_required:
        raise RuntimeError(
            f"{snapshot_date}: missing required columns "
            f"{missing_required}"
        )

    # --------------------------------------------------------
    # Normalize only for QC in memory
    # --------------------------------------------------------

    df["snapshot_date"] = pd.to_datetime(
        df["snapshot_date"]
    ).dt.normalize()

    expected_date = pd.Timestamp(
        snapshot_date
    ).normalize()

    # --------------------------------------------------------
    # Key integrity
    # --------------------------------------------------------

    duplicate_ids = int(
        df["listing_id"]
        .duplicated()
        .sum()
    )

    missing_ids = int(
        df["listing_id"]
        .isna()
        .sum()
    )

    wrong_snapshot_date = int(
        (
            df["snapshot_date"]
            != expected_date
        ).sum()
    )

    # --------------------------------------------------------
    # Price availability
    # --------------------------------------------------------

    price_numeric = pd.to_numeric(
        df["price"],
        errors="coerce"
    )

    price_available = int(
        price_numeric.notna().sum()
    )

    price_missing = int(
        price_numeric.isna().sum()
    )

    price_nonpositive = int(
        (
            price_numeric.notna()
            &
            (price_numeric <= 0)
        ).sum()
    )

    # --------------------------------------------------------
    # Room type availability
    # --------------------------------------------------------

    room_type_missing = int(
        df["room_type"]
        .isna()
        .sum()
    )

    room_type_blank = int(
        (
            df["room_type"]
            .fillna("")
            .astype(str)
            .str.strip()
            ==
            ""
        ).sum()
    )

    unique_room_types = int(
        df["room_type"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .nunique()
    )

    row_count_ok = (
        len(df)
        ==
        expected_rows[snapshot_date]
    )

    snapshot_qc.append(
        {
            "snapshot_date": snapshot_date,
            "rows": len(df),
            "expected_rows": expected_rows[snapshot_date],
            "row_count_ok": row_count_ok,
            "duplicate_listing_ids": duplicate_ids,
            "missing_listing_ids": missing_ids,
            "wrong_snapshot_date": wrong_snapshot_date,
            "price_available": price_available,
            "price_missing": price_missing,
            "price_nonpositive": price_nonpositive,
            "room_type_missing": room_type_missing,
            "room_type_blank": room_type_blank,
            "unique_room_types": unique_room_types,
        }
    )

    # Keep only fields required for transition feasibility.
    snapshot_data[snapshot_date] = (
        pd.DataFrame(
            {
                "listing_id": df["listing_id"].copy(),
                "price": price_numeric,
                "room_type": df["room_type"].copy(),
            }
        )
    )

    print(
        f"{snapshot_date}: "
        f"rows={len(df):,}, "
        f"duplicate IDs={duplicate_ids:,}, "
        f"missing IDs={missing_ids:,}, "
        f"price available={price_available:,}, "
        f"room_type missing={room_type_missing:,}"
    )

    del df, price_numeric
    gc.collect()


snapshot_qc = pd.DataFrame(snapshot_qc)


print("\nSnapshot QC:")
display(snapshot_qc)


# ============================================================
# 4. PRIMARY OUTCOME FEASIBILITY
#    next-period platform persistence
# ============================================================

print("\n" + "=" * 80)
print("PRIMARY OUTCOME FEASIBILITY — NEXT-PERIOD PERSISTENCE")
print("=" * 80)


transition_qc = []


for transition, anchor_date, future_date in transitions:

    anchor = snapshot_data[anchor_date]
    future = snapshot_data[future_date]

    anchor_ids = pd.Index(
        anchor["listing_id"]
    )

    future_ids = pd.Index(
        future["listing_id"]
    )

    # Persistence can be defined for every anchor listing:
    # appears in next snapshot = 1
    # absent from next snapshot  = 0
    persistence = anchor_ids.isin(
        future_ids
    )

    persistent_n = int(
        persistence.sum()
    )

    absent_n = int(
        (~persistence).sum()
    )

    persistence_rate = (
        persistent_n
        /
        len(anchor)
        *
        100
    )

    horizon_days = int(
        (
            pd.Timestamp(future_date)
            -
            pd.Timestamp(anchor_date)
        ).days
    )

    transition_qc.append(
        {
            "transition": transition,
            "anchor_date": anchor_date,
            "future_date": future_date,
            "horizon_days": horizon_days,
            "anchor_rows": len(anchor),
            "future_rows": len(future),
            "persistent": persistent_n,
            "absent_next_snapshot": absent_n,
            "persistence_rate_pct": persistence_rate,
            "primary_target_defined": len(anchor),
        }
    )


transition_qc = pd.DataFrame(
    transition_qc
)


display(
    transition_qc
)


primary_defined_total = int(
    transition_qc[
        "primary_target_defined"
    ].sum()
)


print(
    "\nTotal anchor listing-origin observations:",
    f"{primary_defined_total:,}"
)

print(
    "Expected:",
    "491,140"
)


# ============================================================
# 5. SECONDARY PRICE-RESILIENCE FEASIBILITY
#
# This does NOT construct percentile ranks.
# It only identifies transitions for which anchor and future
# advertised prices are actually available.
# ============================================================

print("\n" + "=" * 80)
print("SECONDARY OUTCOME FEASIBILITY — ADVERTISED PRICE")
print("=" * 80)


price_qc = []


for transition, anchor_date, future_date in transitions:

    anchor = snapshot_data[anchor_date].copy()
    future = snapshot_data[future_date].copy()

    anchor_price_available = int(
        anchor["price"]
        .notna()
        .sum()
    )

    future_price_available = int(
        future["price"]
        .notna()
        .sum()
    )

    # Match same listing across anchor and future.
    paired = anchor[
        [
            "listing_id",
            "price",
            "room_type"
        ]
    ].merge(
        future[
            [
                "listing_id",
                "price"
            ]
        ].rename(
            columns={
                "price":
                    "future_price"
            }
        ),
        on="listing_id",
        how="inner",
        validate="one_to_one"
    ).rename(
        columns={
            "price":
                "anchor_price"
        }
    )

    paired_persistent = len(
        paired
    )

    both_price_available = int(
        (
            paired[
                "anchor_price"
            ].notna()
            &
            paired[
                "future_price"
            ].notna()
        ).sum()
    )

    anchor_room_type_available = int(
        (
            paired[
                "room_type"
            ].notna()
            &
            (
                paired[
                    "room_type"
                ]
                .astype(str)
                .str.strip()
                !=
                ""
            )
        ).sum()
    )

    # This is only feasibility.
    # We are NOT calculating peer ranks yet.
    secondary_feasible = (
        both_price_available
        >
        0
    )

    price_qc.append(
        {
            "transition":
                transition,

            "anchor_date":
                anchor_date,

            "future_date":
                future_date,

            "anchor_price_available":
                anchor_price_available,

            "future_price_available":
                future_price_available,

            "persistent_pairs":
                paired_persistent,

            "persistent_pairs_with_both_prices":
                both_price_available,

            "persistent_pairs_with_room_type":
                anchor_room_type_available,

            "secondary_price_outcome_feasible":
                secondary_feasible,
        }
    )


price_qc = pd.DataFrame(
    price_qc
)


display(
    price_qc
)


# ============================================================
# 6. VERIFY T1-T10 ANCHOR KEYS AGAINST FROZEN INTEGRATED DATA
# ============================================================

print("\n" + "=" * 80)
print("ANCHOR BACKBONE CONSISTENCY")
print("=" * 80)


integrated_keys = pd.read_parquet(
    INTEGRATED_PATH,
    columns=[
        "transition",
        "snapshot_date",
        "listing_id"
    ]
)


integrated_keys[
    "transition"
] = (
    integrated_keys[
        "transition"
    ]
    .astype("string")
    .str.strip()
)


integrated_keys[
    "snapshot_date"
] = pd.to_datetime(
    integrated_keys[
        "snapshot_date"
    ]
).dt.normalize()


expected_anchor_parts = []


for transition, anchor_date, future_date in transitions:

    temp = snapshot_data[
        anchor_date
    ][
        ["listing_id"]
    ].copy()

    temp[
        "transition"
    ] = transition

    temp[
        "snapshot_date"
    ] = pd.Timestamp(
        anchor_date
    )

    expected_anchor_parts.append(
        temp[
            [
                "transition",
                "snapshot_date",
                "listing_id"
            ]
        ]
    )


expected_anchor_keys = pd.concat(
    expected_anchor_parts,
    ignore_index=True
)


del expected_anchor_parts
gc.collect()


anchor_key_check = (
    expected_anchor_keys
    .merge(
        integrated_keys,
        on=[
            "transition",
            "snapshot_date",
            "listing_id"
        ],
        how="outer",
        indicator=True,
        validate="one_to_one"
    )
)


anchor_key_counts = (
    anchor_key_check[
        "_merge"
    ]
    .value_counts()
)


print(
    anchor_key_counts
)


anchor_left_only = int(
    (
        anchor_key_check[
            "_merge"
        ]
        ==
        "left_only"
    ).sum()
)


anchor_right_only = int(
    (
        anchor_key_check[
            "_merge"
        ]
        ==
        "right_only"
    ).sum()
)


print(
    "Cleaned-anchor keys absent from integrated backbone:",
    anchor_left_only
)

print(
    "Integrated backbone keys absent from cleaned anchors:",
    anchor_right_only
)


# ============================================================
# 7. AUGUST LEAKAGE BOUNDARY CHECK
# ============================================================

print("\n" + "=" * 80)
print("T10 / AUGUST OUTCOME-ONLY BOUNDARY")
print("=" * 80)


t10_integrated = integrated_keys[
    integrated_keys[
        "transition"
    ]
    ==
    "T10"
]


t10_dates = (
    t10_integrated[
        "snapshot_date"
    ]
    .drop_duplicates()
    .sort_values()
    .dt.strftime(
        "%Y-%m-%d"
    )
    .tolist()
)


print(
    "T10 predictor snapshot date(s):",
    t10_dates
)

print(
    "Required T10 predictor anchor:",
    "2026-07-12"
)

print(
    "T10 future outcome snapshot:",
    "2026-08-13"
)

print(
    "August predictor rows in integrated backbone:",
    int(
        (
            integrated_keys[
                "snapshot_date"
            ]
            ==
            pd.Timestamp(
                "2026-08-13"
            )
        ).sum()
    )
)


# ============================================================
# 8. FINAL QC
# ============================================================

all_snapshot_rows_ok = bool(
    snapshot_qc[
        "row_count_ok"
    ].all()
)


all_snapshot_ids_unique = bool(
    (
        snapshot_qc[
            "duplicate_listing_ids"
        ]
        ==
        0
    ).all()
)


all_snapshot_ids_present = bool(
    (
        snapshot_qc[
            "missing_listing_ids"
        ]
        ==
        0
    ).all()
)


all_snapshot_dates_ok = bool(
    (
        snapshot_qc[
            "wrong_snapshot_date"
        ]
        ==
        0
    ).all()
)


primary_complete = (
    primary_defined_total
    ==
    491_140
)


backbone_keys_exact = (
    anchor_left_only
    ==
    0
    and
    anchor_right_only
    ==
    0
)


t10_boundary_ok = (
    t10_dates
    ==
    ["2026-07-12"]
    and
    int(
        (
            integrated_keys[
                "snapshot_date"
            ]
            ==
            pd.Timestamp(
                "2026-08-13"
            )
        ).sum()
    )
    ==
    0
)


FINAL_PASS = all(
    [
        all_snapshot_rows_ok,
        all_snapshot_ids_unique,
        all_snapshot_ids_present,
        all_snapshot_dates_ok,
        primary_complete,
        backbone_keys_exact,
        t10_boundary_ok
    ]
)


print("\n" + "=" * 80)
print("FINAL PREFLIGHT SUMMARY")
print("=" * 80)

print(
    "All 11 snapshot row counts correct:",
    all_snapshot_rows_ok
)

print(
    "All listing IDs unique within snapshots:",
    all_snapshot_ids_unique
)

print(
    "No missing listing IDs:",
    all_snapshot_ids_present
)

print(
    "All snapshot dates correct:",
    all_snapshot_dates_ok
)

print(
    "Primary persistence target definable for all 491,140 anchors:",
    primary_complete
)

print(
    "T1-T10 cleaned anchor keys exactly match integrated backbone:",
    backbone_keys_exact
)

print(
    "T10 / August outcome-only boundary correct:",
    t10_boundary_ok
)


print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 80)

if FINAL_PASS:

    print(
        "STEP 24.8E-0 STATUS: PASS — "
        "OUTCOME SOURCES READY FOR CONTROLLED TARGET CONSTRUCTION"
    )

else:

    print(
        "STEP 24.8E-0 STATUS: FAIL — "
        "DO NOT CONSTRUCT OUTCOMES YET"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8E-1 — CONSTRUCT AND FREEZE PRIMARY
#                 NEXT-PERIOD PERSISTENCE OUTCOME
#
# Definition:
#   next_period_persistence = 1
#       if anchor listing_id appears in the immediately
#       following cleaned snapshot
#
#   next_period_persistence = 0
#       otherwise
#
# IMPORTANT:
# - Outcome is defined for ALL 491,140 T1-T10 anchors.
# - August 2026 is used ONLY as T10 future outcome source.
# - No August predictors are introduced.
# - No rows are dropped.
# - No class balancing / resampling is performed here.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc


print("=" * 80)
print("STEP 24.8E-1 — CONSTRUCT PRIMARY NEXT-PERIOD PERSISTENCE OUTCOME")
print("=" * 80)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

INTEGRATED_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "integration"
    / "listing_origin_backbone_T1_T10_with_context_reviews.parquet"
)

OUTCOME_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "outcomes"
)

OUTCOME_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTCOME_DIR
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)


# ============================================================
# 2. FROZEN TRANSITION MAP
# ============================================================

transitions = [
    ("T1",  "2025-10-05", "2025-11-07"),
    ("T2",  "2025-11-07", "2025-12-11"),
    ("T3",  "2025-12-11", "2026-01-16"),
    ("T4",  "2026-01-16", "2026-02-13"),
    ("T5",  "2026-02-13", "2026-03-17"),
    ("T6",  "2026-03-17", "2026-04-16"),
    ("T7",  "2026-04-16", "2026-05-23"),
    ("T8",  "2026-05-23", "2026-06-19"),
    ("T9",  "2026-06-19", "2026-07-12"),
    ("T10", "2026-07-12", "2026-08-13"),
]


expected_anchor_rows = {
    "T1": 47752,
    "T2": 48624,
    "T3": 49899,
    "T4": 46819,
    "T5": 48692,
    "T6": 49076,
    "T7": 48951,
    "T8": 49305,
    "T9": 50926,
    "T10": 51096,
}


# Expected values independently established in E-0.
expected_persistent = {
    "T1": 46511,
    "T2": 47385,
    "T3": 44603,
    "T4": 45797,
    "T5": 47105,
    "T6": 47427,
    "T7": 47370,
    "T8": 48100,
    "T9": 49944,
    "T10": 49949,
}


expected_absent = {
    "T1": 1241,
    "T2": 1239,
    "T3": 5296,
    "T4": 1022,
    "T5": 1587,
    "T6": 1649,
    "T7": 1581,
    "T8": 1205,
    "T9": 982,
    "T10": 1147,
}


EXPECTED_TOTAL = 491_140


# ============================================================
# 3. LOAD FROZEN INTEGRATED BACKBONE KEYS
# ============================================================

backbone = pd.read_parquet(
    INTEGRATED_PATH,
    columns=[
        "transition",
        "snapshot_date",
        "listing_id"
    ]
)


backbone["transition"] = (
    backbone["transition"]
    .astype("string")
    .str.strip()
)


backbone["snapshot_date"] = pd.to_datetime(
    backbone["snapshot_date"]
).dt.normalize()


ORIGIN_KEY = [
    "transition",
    "snapshot_date",
    "listing_id"
]


print(
    "\nIntegrated backbone rows:",
    f"{len(backbone):,}"
)


backbone_duplicates = int(
    backbone[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


backbone_missing_keys = int(
    backbone[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Duplicate backbone origin keys:",
    backbone_duplicates
)

print(
    "Missing backbone origin keys:",
    backbone_missing_keys
)


if len(backbone) != EXPECTED_TOTAL:
    raise RuntimeError(
        "Unexpected integrated backbone row count."
    )


if backbone_duplicates != 0:
    raise RuntimeError(
        "Backbone origin keys are not unique."
    )


if backbone_missing_keys != 0:
    raise RuntimeError(
        "Backbone contains missing origin keys."
    )


# ============================================================
# 4. CONSTRUCT OUTCOME TRANSITION BY TRANSITION
# ============================================================

outcome_parts = []
qc_rows = []


print("\n" + "=" * 80)
print("CONSTRUCTING NEXT-PERIOD PERSISTENCE")
print("=" * 80)


for transition, anchor_date, future_date in transitions:

    print(
        f"\n{transition}: "
        f"{anchor_date} -> {future_date}"
    )

    # --------------------------------------------------------
    # Anchor listing-origin keys come from frozen backbone.
    # This ensures exact alignment with modelling observations.
    # --------------------------------------------------------

    anchor = (
        backbone[
            backbone[
                "transition"
            ]
            ==
            transition
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Load ONLY future listing IDs.
    #
    # For T10 this reads Aug-13-2026 only to define outcome.
    # --------------------------------------------------------

    future_path = (
        LISTINGS_DIR
        / f"listings_{future_date}.parquet"
    )


    if not future_path.exists():

        raise FileNotFoundError(
            f"Missing future snapshot:\n"
            f"{future_path}"
        )


    future = pd.read_parquet(
        future_path,
        columns=[
            "listing_id"
        ]
    )


    future_missing_ids = int(
        future[
            "listing_id"
        ]
        .isna()
        .sum()
    )


    future_duplicate_ids = int(
        future[
            "listing_id"
        ]
        .duplicated()
        .sum()
    )


    if future_missing_ids != 0:

        raise RuntimeError(
            f"{transition}: future snapshot "
            f"contains missing listing IDs."
        )


    if future_duplicate_ids != 0:

        raise RuntimeError(
            f"{transition}: future snapshot "
            f"contains duplicate listing IDs."
        )


    # --------------------------------------------------------
    # Define binary outcome
    # --------------------------------------------------------

    future_ids = pd.Index(
        future[
            "listing_id"
        ]
    )


    anchor[
        "next_period_persistence"
    ] = (
        anchor[
            "listing_id"
        ]
        .isin(
            future_ids
        )
        .astype(
            "int8"
        )
    )


    # Explicit outcome provenance
    anchor[
        "outcome_date"
    ] = pd.Timestamp(
        future_date
    )


    anchor[
        "outcome_horizon_days"
    ] = (
        pd.Timestamp(
            future_date
        )
        -
        pd.Timestamp(
            anchor_date
        )
    ).days


    # --------------------------------------------------------
    # QC
    # --------------------------------------------------------

    n_rows = len(
        anchor
    )


    persistent_n = int(
        anchor[
            "next_period_persistence"
        ]
        .sum()
    )


    absent_n = int(
        n_rows
        -
        persistent_n
    )


    persistence_rate = (
        persistent_n
        /
        n_rows
        *
        100
    )


    missing_target = int(
        anchor[
            "next_period_persistence"
        ]
        .isna()
        .sum()
    )


    target_values = sorted(
        anchor[
            "next_period_persistence"
        ]
        .unique()
        .tolist()
    )


    row_count_ok = (
        n_rows
        ==
        expected_anchor_rows[
            transition
        ]
    )


    persistent_ok = (
        persistent_n
        ==
        expected_persistent[
            transition
        ]
    )


    absent_ok = (
        absent_n
        ==
        expected_absent[
            transition
        ]
    )


    transition_pass = all(
        [
            row_count_ok,
            persistent_ok,
            absent_ok,
            missing_target == 0,
            set(target_values).issubset(
                {0, 1}
            )
        ]
    )


    qc_rows.append(
        {
            "transition":
                transition,

            "anchor_date":
                anchor_date,

            "outcome_date":
                future_date,

            "horizon_days":
                int(
                    anchor[
                        "outcome_horizon_days"
                    ].iloc[0]
                ),

            "rows":
                n_rows,

            "persistent":
                persistent_n,

            "absent_next_snapshot":
                absent_n,

            "persistence_rate_pct":
                persistence_rate,

            "missing_target":
                missing_target,

            "target_values":
                str(
                    target_values
                ),

            "status":
                (
                    "PASS"
                    if transition_pass
                    else "FAIL"
                )
        }
    )


    print(
        "  Rows:",
        f"{n_rows:,}"
    )

    print(
        "  Persistent:",
        f"{persistent_n:,}"
    )

    print(
        "  Absent next snapshot:",
        f"{absent_n:,}"
    )

    print(
        "  Persistence rate:",
        f"{persistence_rate:.4f}%"
    )


    # Keep only outcome table fields.
    outcome_parts.append(
        anchor[
            [
                "transition",
                "snapshot_date",
                "listing_id",
                "outcome_date",
                "outcome_horizon_days",
                "next_period_persistence"
            ]
        ]
    )


    del future, future_ids, anchor
    gc.collect()


# ============================================================
# 5. COMBINE OUTCOME TABLE
# ============================================================

outcomes = pd.concat(
    outcome_parts,
    ignore_index=True
)


del outcome_parts
gc.collect()


qc = pd.DataFrame(
    qc_rows
)


print("\n" + "=" * 80)
print("TRANSITION-LEVEL OUTCOME QC")
print("=" * 80)

display(
    qc
)


# ============================================================
# 6. GLOBAL OUTCOME QC
# ============================================================

print("\n" + "=" * 80)
print("GLOBAL OUTCOME QC")
print("=" * 80)


total_rows = len(
    outcomes
)


duplicate_keys = int(
    outcomes[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


missing_keys = int(
    outcomes[
        ORIGIN_KEY
    ]
    .isna()
    .sum()
    .sum()
)


missing_targets = int(
    outcomes[
        "next_period_persistence"
    ]
    .isna()
    .sum()
)


target_values_global = sorted(
    outcomes[
        "next_period_persistence"
    ]
    .unique()
    .tolist()
)


total_persistent = int(
    outcomes[
        "next_period_persistence"
    ]
    .sum()
)


total_absent = int(
    total_rows
    -
    total_persistent
)


overall_persistence_rate = (
    total_persistent
    /
    total_rows
    *
    100
)


print(
    "Rows:",
    f"{total_rows:,}"
)

print(
    "Duplicate listing-origin keys:",
    duplicate_keys
)

print(
    "Missing listing-origin key values:",
    missing_keys
)

print(
    "Missing targets:",
    missing_targets
)

print(
    "Target values:",
    target_values_global
)

print(
    "Persistent:",
    f"{total_persistent:,}"
)

print(
    "Absent next snapshot:",
    f"{total_absent:,}"
)

print(
    "Overall persistence rate:",
    f"{overall_persistence_rate:.4f}%"
)


# ============================================================
# 7. EXACT KEY ALIGNMENT WITH FROZEN BACKBONE
# ============================================================

key_check = (
    backbone[
        ORIGIN_KEY
    ]
    .merge(
        outcomes[
            ORIGIN_KEY
        ],
        on=ORIGIN_KEY,
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)


key_counts = (
    key_check[
        "_merge"
    ]
    .value_counts()
)


print("\n" + "=" * 80)
print("BACKBONE / OUTCOME KEY ALIGNMENT")
print("=" * 80)

print(
    key_counts
)


backbone_without_outcome = int(
    (
        key_check[
            "_merge"
        ]
        ==
        "left_only"
    ).sum()
)


outcome_without_backbone = int(
    (
        key_check[
            "_merge"
        ]
        ==
        "right_only"
    ).sum()
)


print(
    "Backbone rows without outcome:",
    backbone_without_outcome
)

print(
    "Outcome rows without backbone:",
    outcome_without_backbone
)


# ============================================================
# 8. TEMPORAL PROVENANCE QC
# ============================================================

expected_temporal = pd.DataFrame(
    transitions,
    columns=[
        "transition",
        "expected_anchor_date",
        "expected_outcome_date"
    ]
)


expected_temporal[
    "expected_anchor_date"
] = pd.to_datetime(
    expected_temporal[
        "expected_anchor_date"
    ]
)


expected_temporal[
    "expected_outcome_date"
] = pd.to_datetime(
    expected_temporal[
        "expected_outcome_date"
    ]
)


expected_temporal[
    "expected_horizon_days"
] = (
    expected_temporal[
        "expected_outcome_date"
    ]
    -
    expected_temporal[
        "expected_anchor_date"
    ]
).dt.days


observed_temporal = (
    outcomes[
        [
            "transition",
            "snapshot_date",
            "outcome_date",
            "outcome_horizon_days"
        ]
    ]
    .drop_duplicates()
)


temporal_check = (
    observed_temporal
    .merge(
        expected_temporal,
        on="transition",
        how="left",
        validate="one_to_one"
    )
)


temporal_check[
    "anchor_date_ok"
] = (
    temporal_check[
        "snapshot_date"
    ]
    ==
    temporal_check[
        "expected_anchor_date"
    ]
)


temporal_check[
    "outcome_date_ok"
] = (
    temporal_check[
        "outcome_date"
    ]
    ==
    temporal_check[
        "expected_outcome_date"
    ]
)


temporal_check[
    "horizon_ok"
] = (
    temporal_check[
        "outcome_horizon_days"
    ]
    ==
    temporal_check[
        "expected_horizon_days"
    ]
)


print("\n" + "=" * 80)
print("TEMPORAL PROVENANCE QC")
print("=" * 80)

display(
    temporal_check
)


temporal_pass = bool(
    (
        temporal_check[
            [
                "anchor_date_ok",
                "outcome_date_ok",
                "horizon_ok"
            ]
        ]
        .all()
        .all()
    )
)


# ============================================================
# 9. EXPLICIT T10 LEAKAGE BOUNDARY
# ============================================================

t10 = outcomes[
    outcomes[
        "transition"
    ]
    ==
    "T10"
]


t10_anchor_dates = (
    t10[
        "snapshot_date"
    ]
    .drop_duplicates()
    .dt.strftime(
        "%Y-%m-%d"
    )
    .tolist()
)


t10_outcome_dates = (
    t10[
        "outcome_date"
    ]
    .drop_duplicates()
    .dt.strftime(
        "%Y-%m-%d"
    )
    .tolist()
)


print("\n" + "=" * 80)
print("T10 OUTCOME BOUNDARY")
print("=" * 80)

print(
    "T10 predictor anchor date:",
    t10_anchor_dates
)

print(
    "T10 outcome date:",
    t10_outcome_dates
)


t10_boundary_ok = (
    t10_anchor_dates
    ==
    ["2026-07-12"]
    and
    t10_outcome_dates
    ==
    ["2026-08-13"]
)


# ============================================================
# 10. PRE-SAVE PASS
# ============================================================

all_transition_pass = bool(
    (
        qc[
            "status"
        ]
        ==
        "PASS"
    )
    .all()
)


PRE_SAVE_PASS = all(
    [
        total_rows
            ==
            EXPECTED_TOTAL,

        duplicate_keys
            ==
            0,

        missing_keys
            ==
            0,

        missing_targets
            ==
            0,

        set(
            target_values_global
        )
            ==
            {0, 1},

        backbone_without_outcome
            ==
            0,

        outcome_without_backbone
            ==
            0,

        all_transition_pass,

        temporal_pass,

        t10_boundary_ok
    ]
)


if not PRE_SAVE_PASS:

    raise RuntimeError(
        "STEP 24.8E-1 pre-save QC failed. "
        "Outcome table will NOT be frozen."
    )


# ============================================================
# 11. SAVE FROZEN PRIMARY OUTCOME
# ============================================================

outcomes.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\nSaved:")
print(
    OUTPUT_PATH
)


# ============================================================
# 12. SAVE / RELOAD QC
# ============================================================

check = pd.read_parquet(
    OUTPUT_PATH
)


reload_rows = len(
    check
)


reload_duplicates = int(
    check[
        ORIGIN_KEY
    ]
    .duplicated()
    .sum()
)


reload_missing_target = int(
    check[
        "next_period_persistence"
    ]
    .isna()
    .sum()
)


reload_target_values = sorted(
    check[
        "next_period_persistence"
    ]
    .unique()
    .tolist()
)


reload_persistent = int(
    check[
        "next_period_persistence"
    ]
    .sum()
)


reload_absent = int(
    reload_rows
    -
    reload_persistent
)


print("\n" + "=" * 80)
print("SAVE / RELOAD QC")
print("=" * 80)

print(
    "Reload rows:",
    f"{reload_rows:,}"
)

print(
    "Reload columns:",
    len(
        check.columns
    )
)

print(
    "Duplicate listing-origin keys:",
    reload_duplicates
)

print(
    "Missing targets:",
    reload_missing_target
)

print(
    "Target values:",
    reload_target_values
)

print(
    "Persistent:",
    f"{reload_persistent:,}"
)

print(
    "Absent next snapshot:",
    f"{reload_absent:,}"
)


# ============================================================
# 13. FINAL STATUS
# ============================================================

FINAL_PASS = all(
    [
        reload_rows
            ==
            EXPECTED_TOTAL,

        reload_duplicates
            ==
            0,

        reload_missing_target
            ==
            0,

        set(
            reload_target_values
        )
            ==
            {0, 1},

        reload_persistent
            ==
            total_persistent,

        reload_absent
            ==
            total_absent
    ]
)


print(
    "\nElapsed seconds:",
    round(
        time.time()
        -
        start,
        2
    )
)


print("\n" + "=" * 80)

if FINAL_PASS:

    print(
        "STEP 24.8E-1 STATUS: PASS — "
        "PRIMARY NEXT-PERIOD PERSISTENCE OUTCOME FROZEN"
    )

else:

    print(
        "STEP 24.8E-1 STATUS: FAIL — "
        "DO NOT PROCEED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 24.8E-2A
# SECONDARY ADVERTISED-PRICE RESILIENCE
# PEER-GROUP FEASIBILITY AUDIT
#
# AUDIT ONLY:
# - Does NOT construct percentile ranks
# - Does NOT construct price-resilience outcome
# - Does NOT modify frozen files
#
# Pre-specified peer hierarchy:
#   Primary: TA × room_type, if endpoint peer N >= 30
#   Fallback: TA-only
#
# IMPORTANT:
# T10 is reported for feasibility/provenance only.
# It must NOT be used to redesign the peer rule or threshold.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc

print("=" * 90)
print("STEP 24.8E-2A — SECONDARY PRICE-RESILIENCE PEER-GROUP FEASIBILITY AUDIT")
print("=" * 90)

start = time.time()


# ============================================================
# 1. PATHS
# ============================================================

LISTINGS_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "listings"
)

GEOGRAPHY_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "geography"
)


# ============================================================
# 2. TRANSITIONS
#
# Only transitions empirically shown in E-0 to have price
# available at BOTH anchor and future endpoints.
# ============================================================

eligible_transitions = [
    ("T1",  "2025-10-05", "2025-11-07"),
    ("T6",  "2026-03-17", "2026-04-16"),
    ("T7",  "2026-04-16", "2026-05-23"),
    ("T8",  "2026-05-23", "2026-06-19"),
    ("T9",  "2026-06-19", "2026-07-12"),
    ("T10", "2026-07-12", "2026-08-13"),
]

MIN_PEER_N = 30


# Expected number of persistent listings with price
# at BOTH endpoints, independently established in E-0.

expected_both_price = {
    "T1": 42386,
    "T6": 43077,
    "T7": 42825,
    "T8": 43639,
    "T9": 43991,
    "T10": 43871,
}


# ============================================================
# 3. FIND GEOGRAPHY FILE FOR EACH SNAPSHOT
#
# We do not hard-code the exact geography filename.
# We identify the unique parquet containing the snapshot date.
# ============================================================

def find_geography_file(snapshot_date):

    matches = sorted(
        GEOGRAPHY_DIR.glob(
            f"*{snapshot_date}*.parquet"
        )
    )

    if len(matches) != 1:

        print(
            f"\nGeography candidates for {snapshot_date}:"
        )

        for p in matches:
            print("  ", p.name)

        raise RuntimeError(
            f"Expected exactly 1 geography parquet for "
            f"{snapshot_date}, found {len(matches)}."
        )

    return matches[0]


# ============================================================
# 4. LOAD ONE SNAPSHOT WITH PRICE + ROOM TYPE + TA
# ============================================================

def load_endpoint(snapshot_date):

    listing_path = (
        LISTINGS_DIR
        / f"listings_{snapshot_date}.parquet"
    )

    if not listing_path.exists():
        raise FileNotFoundError(
            f"Missing listing snapshot:\n{listing_path}"
        )

    geo_path = find_geography_file(
        snapshot_date
    )

    # --------------------------------------------------------
    # Listings
    # --------------------------------------------------------

    listings = pd.read_parquet(
        listing_path,
        columns=[
            "listing_id",
            "price",
            "room_type"
        ]
    )

    if listings["listing_id"].isna().any():
        raise RuntimeError(
            f"{snapshot_date}: missing listing_id in listings."
        )

    if listings["listing_id"].duplicated().any():
        raise RuntimeError(
            f"{snapshot_date}: duplicate listing_id in listings."
        )

    listings["price"] = pd.to_numeric(
        listings["price"],
        errors="coerce"
    )

    listings["room_type"] = (
        listings["room_type"]
        .astype("string")
        .str.strip()
    )

    # --------------------------------------------------------
    # Geography
    # --------------------------------------------------------

    geo = pd.read_parquet(
        geo_path
    )

    required_geo = {
        "listing_id",
        "ta_code"
    }

    missing_geo_cols = (
        required_geo
        -
        set(geo.columns)
    )

    if missing_geo_cols:
        raise RuntimeError(
            f"{snapshot_date}: geography missing columns "
            f"{sorted(missing_geo_cols)}"
        )

    geo = geo[
        [
            "listing_id",
            "ta_code"
        ]
    ].copy()

    if geo["listing_id"].isna().any():
        raise RuntimeError(
            f"{snapshot_date}: missing listing_id in geography."
        )

    if geo["listing_id"].duplicated().any():
        raise RuntimeError(
            f"{snapshot_date}: duplicate listing_id in geography."
        )

    geo["ta_code"] = (
        geo["ta_code"]
        .astype("string")
        .str.strip()
    )

    # --------------------------------------------------------
    # Merge
    # --------------------------------------------------------

    endpoint = listings.merge(
        geo,
        on="listing_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    unmatched_geo = int(
        (
            endpoint["_merge"]
            !=
            "both"
        ).sum()
    )

    missing_ta = int(
        endpoint["ta_code"]
        .isna()
        .sum()
    )

    if unmatched_geo != 0 or missing_ta != 0:
        raise RuntimeError(
            f"{snapshot_date}: geography merge failed. "
            f"unmatched={unmatched_geo:,}, "
            f"missing TA={missing_ta:,}"
        )

    endpoint = endpoint.drop(
        columns="_merge"
    )

    return endpoint, geo_path.name


# ============================================================
# 5. TRANSITION-BY-TRANSITION AUDIT
# ============================================================

transition_summary = []
endpoint_summary = []
ta_change_summary = []

all_pair_audits = []


print("\n" + "=" * 90)
print("TRANSITION AUDIT")
print("=" * 90)


for transition, anchor_date, future_date in eligible_transitions:

    print(
        f"\n{'-' * 90}\n"
        f"{transition}: {anchor_date} -> {future_date}"
    )

    anchor, anchor_geo_file = load_endpoint(
        anchor_date
    )

    future, future_geo_file = load_endpoint(
        future_date
    )

    print(
        "Anchor geography:",
        anchor_geo_file
    )

    print(
        "Future geography:",
        future_geo_file
    )

    # --------------------------------------------------------
    # Persistent listings only
    # --------------------------------------------------------

    pairs = (
        anchor[
            [
                "listing_id",
                "price",
                "room_type",
                "ta_code"
            ]
        ]
        .rename(
            columns={
                "price": "anchor_price",
                "room_type": "anchor_room_type",
                "ta_code": "anchor_ta_code"
            }
        )
        .merge(
            future[
                [
                    "listing_id",
                    "price",
                    "room_type",
                    "ta_code"
                ]
            ].rename(
                columns={
                    "price": "future_price",
                    "room_type": "future_room_type",
                    "ta_code": "future_ta_code"
                }
            ),
            on="listing_id",
            how="inner",
            validate="one_to_one"
        )
    )

    persistent_pairs = len(
        pairs
    )

    # --------------------------------------------------------
    # Require price at BOTH endpoints
    # --------------------------------------------------------

    eligible = pairs[
        pairs["anchor_price"].notna()
        &
        pairs["future_price"].notna()
    ].copy()

    both_price_n = len(
        eligible
    )

    expected_n = expected_both_price[
        transition
    ]

    both_price_ok = (
        both_price_n
        ==
        expected_n
    )

    # --------------------------------------------------------
    # Basic price QC
    # --------------------------------------------------------

    nonpositive_anchor_price = int(
        (
            eligible["anchor_price"]
            <= 0
        ).sum()
    )

    nonpositive_future_price = int(
        (
            eligible["future_price"]
            <= 0
        ).sum()
    )

    # --------------------------------------------------------
    # Room-type QC
    # --------------------------------------------------------

    missing_anchor_room = int(
        eligible[
            "anchor_room_type"
        ]
        .isna()
        .sum()
    )

    missing_future_room = int(
        eligible[
            "future_room_type"
        ]
        .isna()
        .sum()
    )

    blank_anchor_room = int(
        (
            eligible[
                "anchor_room_type"
            ]
            .fillna("")
            .str.strip()
            ==
            ""
        ).sum()
    )

    blank_future_room = int(
        (
            eligible[
                "future_room_type"
            ]
            .fillna("")
            .str.strip()
            ==
            ""
        ).sum()
    )

    room_type_changed = (
        eligible[
            "anchor_room_type"
        ]
        !=
        eligible[
            "future_room_type"
        ]
    )

    room_type_changed_n = int(
        room_type_changed.fillna(
            False
        ).sum()
    )

    # --------------------------------------------------------
    # TA stability
    # --------------------------------------------------------

    ta_changed = (
        eligible[
            "anchor_ta_code"
        ]
        !=
        eligible[
            "future_ta_code"
        ]
    )

    ta_changed_n = int(
        ta_changed.fillna(
            False
        ).sum()
    )

    ta_999_anchor = int(
        (
            eligible[
                "anchor_ta_code"
            ]
            ==
            "999"
        ).sum()
    )

    ta_999_future = int(
        (
            eligible[
                "future_ta_code"
            ]
            ==
            "999"
        ).sum()
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Peer sizes are calculated among eligible persistent
    # listings with price at BOTH endpoints.
    #
    # We audit each endpoint separately using its own:
    #   TA + room_type
    #
    # No rank is calculated yet.
    # --------------------------------------------------------

    anchor_peer_sizes = (
        eligible
        .groupby(
            [
                "anchor_ta_code",
                "anchor_room_type"
            ],
            dropna=False
        )
        .size()
        .rename(
            "anchor_ta_room_n"
        )
        .reset_index()
    )

    future_peer_sizes = (
        eligible
        .groupby(
            [
                "future_ta_code",
                "future_room_type"
            ],
            dropna=False
        )
        .size()
        .rename(
            "future_ta_room_n"
        )
        .reset_index()
    )

    anchor_ta_sizes = (
        eligible
        .groupby(
            "anchor_ta_code",
            dropna=False
        )
        .size()
        .rename(
            "anchor_ta_n"
        )
        .reset_index()
    )

    future_ta_sizes = (
        eligible
        .groupby(
            "future_ta_code",
            dropna=False
        )
        .size()
        .rename(
            "future_ta_n"
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Attach peer sizes to each eligible listing
    # --------------------------------------------------------

    audit = (
        eligible
        .merge(
            anchor_peer_sizes,
            on=[
                "anchor_ta_code",
                "anchor_room_type"
            ],
            how="left",
            validate="many_to_one"
        )
        .merge(
            future_peer_sizes,
            on=[
                "future_ta_code",
                "future_room_type"
            ],
            how="left",
            validate="many_to_one"
        )
        .merge(
            anchor_ta_sizes,
            on="anchor_ta_code",
            how="left",
            validate="many_to_one"
        )
        .merge(
            future_ta_sizes,
            on="future_ta_code",
            how="left",
            validate="many_to_one"
        )
    )

    # --------------------------------------------------------
    # Primary peer availability
    # --------------------------------------------------------

    audit[
        "anchor_primary_peer_ok"
    ] = (
        audit[
            "anchor_ta_room_n"
        ]
        >=
        MIN_PEER_N
    )

    audit[
        "future_primary_peer_ok"
    ] = (
        audit[
            "future_ta_room_n"
        ]
        >=
        MIN_PEER_N
    )

    # --------------------------------------------------------
    # TA fallback availability
    # --------------------------------------------------------

    audit[
        "anchor_ta_fallback_ok"
    ] = (
        audit[
            "anchor_ta_n"
        ]
        >=
        MIN_PEER_N
    )

    audit[
        "future_ta_fallback_ok"
    ] = (
        audit[
            "future_ta_n"
        ]
        >=
        MIN_PEER_N
    )

    # Endpoint is rankable if:
    # primary TA×room peer is sufficient OR
    # TA-only fallback is sufficient.
    audit[
        "anchor_rankable"
    ] = (
        audit[
            "anchor_primary_peer_ok"
        ]
        |
        audit[
            "anchor_ta_fallback_ok"
        ]
    )

    audit[
        "future_rankable"
    ] = (
        audit[
            "future_primary_peer_ok"
        ]
        |
        audit[
            "future_ta_fallback_ok"
        ]
    )

    audit[
        "both_endpoints_rankable"
    ] = (
        audit[
            "anchor_rankable"
        ]
        &
        audit[
            "future_rankable"
        ]
    )

    # --------------------------------------------------------
    # How many actually require fallback?
    # --------------------------------------------------------

    audit[
        "anchor_requires_fallback"
    ] = (
        ~audit[
            "anchor_primary_peer_ok"
        ]
        &
        audit[
            "anchor_ta_fallback_ok"
        ]
    )

    audit[
        "future_requires_fallback"
    ] = (
        ~audit[
            "future_primary_peer_ok"
        ]
        &
        audit[
            "future_ta_fallback_ok"
        ]
    )

    anchor_primary_ok_n = int(
        audit[
            "anchor_primary_peer_ok"
        ].sum()
    )

    future_primary_ok_n = int(
        audit[
            "future_primary_peer_ok"
        ].sum()
    )

    anchor_fallback_n = int(
        audit[
            "anchor_requires_fallback"
        ].sum()
    )

    future_fallback_n = int(
        audit[
            "future_requires_fallback"
        ].sum()
    )

    both_rankable_n = int(
        audit[
            "both_endpoints_rankable"
        ].sum()
    )

    not_rankable_n = int(
        (
            ~audit[
                "both_endpoints_rankable"
            ]
        ).sum()
    )

    # --------------------------------------------------------
    # Group-level statistics
    # --------------------------------------------------------

    anchor_groups_total = len(
        anchor_peer_sizes
    )

    future_groups_total = len(
        future_peer_sizes
    )

    anchor_groups_lt30 = int(
        (
            anchor_peer_sizes[
                "anchor_ta_room_n"
            ]
            <
            MIN_PEER_N
        ).sum()
    )

    future_groups_lt30 = int(
        (
            future_peer_sizes[
                "future_ta_room_n"
            ]
            <
            MIN_PEER_N
        ).sum()
    )

    anchor_ta_groups_total = len(
        anchor_ta_sizes
    )

    future_ta_groups_total = len(
        future_ta_sizes
    )

    anchor_ta_groups_lt30 = int(
        (
            anchor_ta_sizes[
                "anchor_ta_n"
            ]
            <
            MIN_PEER_N
        ).sum()
    )

    future_ta_groups_lt30 = int(
        (
            future_ta_sizes[
                "future_ta_n"
            ]
            <
            MIN_PEER_N
        ).sum()
    )

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    transition_summary.append(
        {
            "transition":
                transition,

            "anchor_date":
                anchor_date,

            "future_date":
                future_date,

            "persistent_pairs":
                persistent_pairs,

            "both_price":
                both_price_n,

            "expected_both_price":
                expected_n,

            "both_price_ok":
                both_price_ok,

            "anchor_primary_peer_ok":
                anchor_primary_ok_n,

            "future_primary_peer_ok":
                future_primary_ok_n,

            "anchor_requires_fallback":
                anchor_fallback_n,

            "future_requires_fallback":
                future_fallback_n,

            "both_endpoints_rankable":
                both_rankable_n,

            "not_rankable":
                not_rankable_n,

            "ta_changed":
                ta_changed_n,

            "room_type_changed":
                room_type_changed_n,

            "anchor_999":
                ta_999_anchor,

            "future_999":
                ta_999_future,

            "nonpositive_anchor_price":
                nonpositive_anchor_price,

            "nonpositive_future_price":
                nonpositive_future_price
        }
    )

    endpoint_summary.append(
        {
            "transition":
                transition,

            "endpoint":
                "anchor",

            "date":
                anchor_date,

            "ta_room_groups":
                anchor_groups_total,

            "ta_room_groups_lt30":
                anchor_groups_lt30,

            "ta_groups":
                anchor_ta_groups_total,

            "ta_groups_lt30":
                anchor_ta_groups_lt30,

            "listings_primary_peer_ok":
                anchor_primary_ok_n,

            "listings_requiring_ta_fallback":
                anchor_fallback_n
        }
    )

    endpoint_summary.append(
        {
            "transition":
                transition,

            "endpoint":
                "future",

            "date":
                future_date,

            "ta_room_groups":
                future_groups_total,

            "ta_room_groups_lt30":
                future_groups_lt30,

            "ta_groups":
                future_ta_groups_total,

            "ta_groups_lt30":
                future_ta_groups_lt30,

            "listings_primary_peer_ok":
                future_primary_ok_n,

            "listings_requiring_ta_fallback":
                future_fallback_n
        }
    )

    # --------------------------------------------------------
    # TA-change detail
    # --------------------------------------------------------

    if ta_changed_n > 0:

        changed = (
            eligible.loc[
                ta_changed.fillna(False),
                [
                    "listing_id",
                    "anchor_ta_code",
                    "future_ta_code"
                ]
            ]
            .copy()
        )

        changed[
            "transition"
        ] = transition

        ta_change_summary.append(
            changed
        )

    # Keep compact listing-level audit only in memory.
    audit[
        "transition"
    ] = transition

    all_pair_audits.append(
        audit[
            [
                "transition",
                "listing_id",
                "anchor_ta_code",
                "future_ta_code",
                "anchor_room_type",
                "future_room_type",
                "anchor_ta_room_n",
                "future_ta_room_n",
                "anchor_ta_n",
                "future_ta_n",
                "anchor_primary_peer_ok",
                "future_primary_peer_ok",
                "anchor_requires_fallback",
                "future_requires_fallback",
                "both_endpoints_rankable"
            ]
        ]
    )

    print(
        f"Persistent pairs: {persistent_pairs:,}"
    )

    print(
        f"Both-price eligible: {both_price_n:,} "
        f"(expected {expected_n:,})"
    )

    print(
        f"TA changed: {ta_changed_n:,}"
    )

    print(
        f"Room type changed: {room_type_changed_n:,}"
    )

    print(
        f"Anchor TA×room fallback listings: "
        f"{anchor_fallback_n:,}"
    )

    print(
        f"Future TA×room fallback listings: "
        f"{future_fallback_n:,}"
    )

    print(
        f"Both endpoints rankable: "
        f"{both_rankable_n:,}"
    )

    print(
        f"Not rankable after TA fallback: "
        f"{not_rankable_n:,}"
    )

    del (
        anchor,
        future,
        pairs,
        eligible,
        audit,
        anchor_peer_sizes,
        future_peer_sizes,
        anchor_ta_sizes,
        future_ta_sizes
    )

    gc.collect()


# ============================================================
# 6. SUMMARY TABLES
# ============================================================

transition_summary = pd.DataFrame(
    transition_summary
)

endpoint_summary = pd.DataFrame(
    endpoint_summary
)


print("\n" + "=" * 90)
print("TRANSITION-LEVEL FEASIBILITY SUMMARY")
print("=" * 90)

display(
    transition_summary
)


print("\n" + "=" * 90)
print("ENDPOINT PEER-GROUP SUMMARY")
print("=" * 90)

display(
    endpoint_summary
)


# ============================================================
# 7. TA STABILITY SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("TA STABILITY")
print("=" * 90)


if ta_change_summary:

    ta_changes = pd.concat(
        ta_change_summary,
        ignore_index=True
    )

    print(
        "Total eligible listing-transition observations "
        "with TA change:",
        f"{len(ta_changes):,}"
    )

    print(
        "\nMost common TA changes:"
    )

    display(
        ta_changes
        .groupby(
            [
                "anchor_ta_code",
                "future_ta_code"
            ]
        )
        .size()
        .rename(
            "n"
        )
        .reset_index()
        .sort_values(
            "n",
            ascending=False
        )
        .head(30)
    )

else:

    ta_changes = pd.DataFrame()

    print(
        "No TA changes detected."
    )


# ============================================================
# 8. GLOBAL FEASIBILITY
# ============================================================

pair_audit = pd.concat(
    all_pair_audits,
    ignore_index=True
)


total_both_price = int(
    transition_summary[
        "both_price"
    ].sum()
)


total_rankable = int(
    transition_summary[
        "both_endpoints_rankable"
    ].sum()
)


total_not_rankable = int(
    transition_summary[
        "not_rankable"
    ].sum()
)


rankable_pct = (
    total_rankable
    /
    total_both_price
    *
    100
)


print("\n" + "=" * 90)
print("GLOBAL SECONDARY-OUTCOME FEASIBILITY")
print("=" * 90)

print(
    "Both-price eligible listing-transition observations:",
    f"{total_both_price:,}"
)

print(
    "Rankable at both endpoints under current hierarchy:",
    f"{total_rankable:,}"
)

print(
    "Not rankable after TA fallback:",
    f"{total_not_rankable:,}"
)

print(
    "Rankable percentage:",
    f"{rankable_pct:.4f}%"
)


# ============================================================
# 9. DEVELOPMENT VS LOCKED T10
#
# T1/T6-T9 may inform diagnostics.
# T10 is shown separately and must not be used to redesign
# threshold/grouping architecture.
# ============================================================

development = transition_summary[
    transition_summary[
        "transition"
    ]
    !=
    "T10"
]


locked_t10 = transition_summary[
    transition_summary[
        "transition"
    ]
    ==
    "T10"
]


print("\n" + "=" * 90)
print("DEVELOPMENT TRANSITIONS — T1, T6-T9")
print("=" * 90)

display(
    development
)


print("\n" + "=" * 90)
print("LOCKED HOLDOUT FEASIBILITY REPORT — T10")
print("=" * 90)

display(
    locked_t10
)


# ============================================================
# 10. QC CONDITIONS
#
# This PASS means the audit ran consistently.
# It does NOT yet freeze the final percentile-rank rule.
# ============================================================

all_expected_price_counts_match = bool(
    transition_summary[
        "both_price_ok"
    ].all()
)


no_nonpositive_prices = bool(
    (
        transition_summary[
            [
                "nonpositive_anchor_price",
                "nonpositive_future_price"
            ]
        ]
        ==
        0
    )
    .all()
    .all()
)


no_missing_peer_sizes = bool(
    pair_audit[
        [
            "anchor_ta_room_n",
            "future_ta_room_n",
            "anchor_ta_n",
            "future_ta_n"
        ]
    ]
    .notna()
    .all()
    .all()
)


audit_keys_unique = (
    pair_audit[
        [
            "transition",
            "listing_id"
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


AUDIT_PASS = all(
    [
        all_expected_price_counts_match,
        no_nonpositive_prices,
        no_missing_peer_sizes,
        audit_keys_unique
    ]
)


print("\n" + "=" * 90)
print("FINAL AUDIT QC")
print("=" * 90)

print(
    "All E-0 both-price counts reproduced:",
    all_expected_price_counts_match
)

print(
    "No non-positive prices among eligible observations:",
    no_nonpositive_prices
)

print(
    "No missing peer-size values:",
    no_missing_peer_sizes
)

print(
    "Unique transition + listing_id audit keys:",
    audit_keys_unique
)

print(
    "T10 excluded from design/tuning decisions:",
    True
)

print(
    "\nElapsed seconds:",
    round(
        time.time() - start,
        2
    )
)


print("\n" + "=" * 90)

if AUDIT_PASS:

    print(
        "STEP 24.8E-2A STATUS: PASS — "
        "PEER-GROUP FEASIBILITY AUDIT COMPLETE"
    )

else:

    print(
        "STEP 24.8E-2A STATUS: FAIL — "
        "DO NOT CONSTRUCT SECONDARY OUTCOME"
    )

print("=" * 90)

In [ ]:
# ============================================================
# STEP 24.8F-0
# CALENDAR FEATURE INVENTORY & 28-DAY PRICE PREFLIGHT
#
# PURPOSE:
# Verify whether the preprocessed calendar channel exists and
# whether median_price_28d is available for all 11 snapshots.
#
# NO outcome construction.
# NO percentile ranking.
# NO file modification.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 90)
print("STEP 24.8F-0 — CALENDAR FEATURE INVENTORY & 28-DAY PRICE PREFLIGHT")
print("=" * 90)

CALENDAR_DIR = PROJECT_ROOT / "02_intermediate" / "calendar"

snapshot_dates = [
    "2025-10-05",
    "2025-11-07",
    "2025-12-11",
    "2026-01-16",
    "2026-02-13",
    "2026-03-17",
    "2026-04-16",
    "2026-05-23",
    "2026-06-19",
    "2026-07-12",
    "2026-08-13",
]

print("\nCalendar directory:")
print(CALENDAR_DIR)
print("Exists:", CALENDAR_DIR.exists())

print("\n" + "=" * 90)
print("FILES FOUND")
print("=" * 90)

if not CALENDAR_DIR.exists():
    print("Calendar directory does not exist.")
    calendar_files = []
else:
    calendar_files = sorted(CALENDAR_DIR.rglob("*.parquet"))

print("Parquet files found:", len(calendar_files))

for p in calendar_files:
    print(" -", p.relative_to(PROJECT_ROOT))


print("\n" + "=" * 90)
print("SNAPSHOT-BY-SNAPSHOT INVENTORY")
print("=" * 90)

summary = []

for snapshot_date in snapshot_dates:

    matches = [
        p for p in calendar_files
        if snapshot_date in p.name
    ]

    print("\n" + "-" * 90)
    print("Snapshot:", snapshot_date)
    print("Matching parquet files:", len(matches))

    if len(matches) == 0:
        print("STATUS: NO MATCHING CALENDAR PARQUET")

        summary.append({
            "snapshot_date": snapshot_date,
            "files_found": 0,
            "rows": np.nan,
            "columns": np.nan,
            "listing_id_present": False,
            "median_price_28d_present": False,
            "median_price_28d_nonmissing": np.nan,
            "median_price_28d_missing": np.nan,
            "duplicate_listing_id": np.nan,
        })

        continue

    for p in matches:
        print("\nFILE:", p.relative_to(PROJECT_ROOT))

        df = pd.read_parquet(p)

        print("Rows:", f"{len(df):,}")
        print("Columns:", len(df.columns))
        print("Column names:")
        print(df.columns.tolist())

        listing_id_present = "listing_id" in df.columns
        median_price_present = "median_price_28d" in df.columns

        duplicate_listing_id = (
            int(df["listing_id"].duplicated().sum())
            if listing_id_present
            else np.nan
        )

        if median_price_present:

            price = pd.to_numeric(
                df["median_price_28d"],
                errors="coerce"
            )

            nonmissing = int(price.notna().sum())
            missing = int(price.isna().sum())
            nonpositive = int((price.dropna() <= 0).sum())

            print(
                "median_price_28d non-missing:",
                f"{nonmissing:,}"
            )

            print(
                "median_price_28d missing:",
                f"{missing:,}"
            )

            print(
                "median_price_28d non-positive:",
                f"{nonpositive:,}"
            )

            if nonmissing > 0:
                print(
                    "median_price_28d median:",
                    float(price.median())
                )

                print(
                    "median_price_28d min/max:",
                    float(price.min()),
                    "/",
                    float(price.max())
                )

        else:
            nonmissing = np.nan
            missing = np.nan
            nonpositive = np.nan

            print(
                "median_price_28d: COLUMN NOT FOUND"
            )

        print(
            "listing_id present:",
            listing_id_present
        )

        print(
            "duplicate listing_id:",
            duplicate_listing_id
        )

        summary.append({
            "snapshot_date": snapshot_date,
            "file": str(p.relative_to(PROJECT_ROOT)),
            "files_found": len(matches),
            "rows": len(df),
            "columns": len(df.columns),
            "listing_id_present": listing_id_present,
            "median_price_28d_present": median_price_present,
            "median_price_28d_nonmissing": nonmissing,
            "median_price_28d_missing": missing,
            "median_price_28d_nonpositive": nonpositive,
            "duplicate_listing_id": duplicate_listing_id,
        })

        del df


summary_df = pd.DataFrame(summary)

print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

display(summary_df)


print("\n" + "=" * 90)
print("KEY PREFLIGHT QUESTIONS")
print("=" * 90)

all_dates_have_file = all(
    any(d in p.name for p in calendar_files)
    for d in snapshot_dates
)

any_price28 = (
    summary_df["median_price_28d_present"].fillna(False).any()
    if len(summary_df)
    else False
)

print(
    "All 11 snapshots have at least one calendar parquet:",
    all_dates_have_file
)

print(
    "At least one file contains median_price_28d:",
    any_price28
)

print("\n" + "=" * 90)
print(
    "STEP 24.8F-0 COMPLETE — "
    "INVENTORY ONLY; DO NOT CONSTRUCT SECONDARY OUTCOME YET"
)
print("=" * 90)

In [ ]:
# ============================================================
# STEP 24.8F-1A
# RAW CALENDAR INVENTORY & SCHEMA QC
#
# INVENTORY ONLY:
# - no aggregation
# - no feature construction
# - no output modification
# ============================================================

from pathlib import Path
import pandas as pd
import gzip
import time

print("=" * 90)
print("STEP 24.8F-1A — RAW CALENDAR INVENTORY & SCHEMA QC")
print("=" * 90)

start = time.time()

RAW_AIRBNB_DIR = PROJECT_ROOT / "01_raw" / "airbnb"

snapshot_dates = [
    "2025-10-05",
    "2025-11-07",
    "2025-12-11",
    "2026-01-16",
    "2026-02-13",
    "2026-03-17",
    "2026-04-16",
    "2026-05-23",
    "2026-06-19",
    "2026-07-12",
    "2026-08-13",
]

results = []

print("\nRaw Airbnb directory:")
print(RAW_AIRBNB_DIR)
print("Exists:", RAW_AIRBNB_DIR.exists())

for snapshot_date in snapshot_dates:

    print("\n" + "-" * 90)
    print("SNAPSHOT:", snapshot_date)

    snapshot_dir = RAW_AIRBNB_DIR / snapshot_date

    # Search only inside the correct snapshot folder
    candidates = sorted(
        snapshot_dir.rglob("calendar.csv.gz")
    ) if snapshot_dir.exists() else []

    print("Snapshot folder exists:", snapshot_dir.exists())
    print("calendar.csv.gz files found:", len(candidates))

    if len(candidates) != 1:

        for p in candidates:
            print("  Candidate:", p)

        results.append({
            "snapshot_date": snapshot_date,
            "file_found": len(candidates),
            "schema_ok": False
        })

        continue

    path = candidates[0]

    print("File:", path.relative_to(PROJECT_ROOT))

    # --------------------------------------------------------
    # Read header only
    # --------------------------------------------------------

    header = pd.read_csv(
        path,
        compression="gzip",
        nrows=0
    )

    columns = header.columns.tolist()

    print("Columns:")
    print(columns)

    required = [
        "listing_id",
        "date",
        "available",
        "price",
        "minimum_nights"
    ]

    missing_required = [
        c for c in required
        if c not in columns
    ]

    schema_ok = len(missing_required) == 0

    print("Required columns present:", schema_ok)

    if missing_required:
        print("Missing required:", missing_required)

    # --------------------------------------------------------
    # Small sample only — do NOT load full raw calendar
    # --------------------------------------------------------

    sample = pd.read_csv(
        path,
        compression="gzip",
        usecols=[
            c for c in required
            if c in columns
        ],
        nrows=10000,
        low_memory=False
    )

    print("Sample rows read:", f"{len(sample):,}")

    if "listing_id" in sample.columns:
        print(
            "Sample missing listing_id:",
            int(sample["listing_id"].isna().sum())
        )

    if "date" in sample.columns:

        sample_dates = pd.to_datetime(
            sample["date"],
            errors="coerce"
        )

        print(
            "Sample invalid dates:",
            int(sample_dates.isna().sum())
        )

        if sample_dates.notna().any():
            print(
                "Sample date range:",
                sample_dates.min().date(),
                "->",
                sample_dates.max().date()
            )

    if "available" in sample.columns:
        print(
            "Sample available values:",
            sorted(
                sample["available"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )
        )

    if "price" in sample.columns:
        print(
            "Sample non-missing price:",
            int(sample["price"].notna().sum())
        )

        print(
            "Example price values:",
            sample["price"]
            .dropna()
            .astype(str)
            .head(5)
            .tolist()
        )

    if "minimum_nights" in sample.columns:
        print(
            "Sample non-missing minimum_nights:",
            int(
                sample["minimum_nights"]
                .notna()
                .sum()
            )
        )

    results.append({
        "snapshot_date": snapshot_date,
        "file_found": 1,
        "file": str(path.relative_to(PROJECT_ROOT)),
        "column_count": len(columns),
        "schema_ok": schema_ok,
        "missing_required": ", ".join(missing_required)
    })


results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("RAW CALENDAR INVENTORY SUMMARY")
print("=" * 90)

display(results_df)


all_files_exactly_one = bool(
    (results_df["file_found"] == 1).all()
)

all_schema_ok = bool(
    results_df["schema_ok"].all()
)

print("\n" + "=" * 90)
print("FINAL PREFLIGHT QC")
print("=" * 90)

print(
    "Exactly one calendar.csv.gz for every snapshot:",
    all_files_exactly_one
)

print(
    "Required schema valid for every snapshot:",
    all_schema_ok
)

print(
    "Expected snapshots:",
    len(snapshot_dates)
)

print(
    "Snapshots audited:",
    len(results_df)
)

print(
    "Elapsed seconds:",
    round(time.time() - start, 2)
)

print("\n" + "=" * 90)

if all_files_exactly_one and all_schema_ok:
    print(
        "STEP 24.8F-1A STATUS: PASS — "
        "RAW CALENDAR FILES READY FOR FULL TEMPORAL QC"
    )
else:
    print(
        "STEP 24.8F-1A STATUS: FAIL — "
        "DO NOT START CALENDAR AGGREGATION"
    )

print("=" * 90)

In [ ]:
# =====================================================================
# PRODUCTION BLOCK F — COMPLETE VERSION
# INSTALL DUCKDB IF NEEDED + AUDIT + BUILD CALENDAR FEATURES
#
# Project:
# 7th_NZ_STR_Construction_AI
#
# IMPORTANT
# - Raw data are READ ONLY
# - Processes all 11 snapshots
# - Uses DuckDB directly on calendar.csv.gz
# - Creates 4 weekly bins + 28-day summaries
# - Saves one Parquet per snapshot
# - Performs automatic QC
# - Does NOT yet construct secondary outcome
# =====================================================================

from pathlib import Path
import sys
import subprocess
import importlib.util
import time
import pandas as pd
import numpy as np


# =====================================================================
# 0. ENSURE DUCKDB EXISTS IN THIS JUPYTER KERNEL
# =====================================================================

print("=" * 100)
print("0. ENVIRONMENT CHECK")
print("=" * 100)

print("Python executable:")
print(sys.executable)

if importlib.util.find_spec("duckdb") is None:

    print("\nDuckDB is not installed in this Jupyter environment.")
    print("Installing DuckDB now...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "duckdb"
    ])

    print("\nDuckDB installation completed.")

import duckdb

print("DuckDB version:", duckdb.__version__)
print("Environment check: PASS")


# =====================================================================
# 1. PROJECT PATH
# =====================================================================

print("\n" + "=" * 100)
print("1. PROJECT PATH")
print("=" * 100)

# Use existing PROJECT_ROOT if already defined.
# Otherwise use the known project location.
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

RAW_AIRBNB_DIR = PROJECT_ROOT / "01_raw" / "airbnb"
CAL_OUT = PROJECT_ROOT / "02_intermediate" / "calendar"

CAL_OUT.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw Airbnb:", RAW_AIRBNB_DIR)
print("Calendar output:", CAL_OUT)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
    )

if not RAW_AIRBNB_DIR.exists():
    raise FileNotFoundError(
        f"Raw Airbnb folder does not exist: {RAW_AIRBNB_DIR}"
    )

print("Project path check: PASS")


# =====================================================================
# 2. FROZEN SNAPSHOT DESIGN
# =====================================================================

snapshot_dates = [
    "2025-10-05",
    "2025-11-07",
    "2025-12-11",
    "2026-01-16",
    "2026-02-13",
    "2026-03-17",
    "2026-04-16",
    "2026-05-23",
    "2026-06-19",
    "2026-07-12",
    "2026-08-13",
]

expected_listing_counts = {
    "2025-10-05": 47752,
    "2025-11-07": 48624,
    "2025-12-11": 49899,
    "2026-01-16": 46819,
    "2026-02-13": 48692,
    "2026-03-17": 49076,
    "2026-04-16": 48951,
    "2026-05-23": 49305,
    "2026-06-19": 50926,
    "2026-07-12": 51096,
    "2026-08-13": 51325,
}

print("\n" + "=" * 100)
print("2. RAW CALENDAR FILE CHECK")
print("=" * 100)

for snap in snapshot_dates:

    p = (
        RAW_AIRBNB_DIR
        / snap
        / "calendar.csv.gz"
    )

    if not p.exists():
        raise FileNotFoundError(
            f"Missing calendar file: {p}"
        )

    print("PASS:", snap)

print("\nAll 11 raw calendar files found.")


# =====================================================================
# 3. START DUCKDB
# =====================================================================

con = duckdb.connect(
    database=":memory:"
)

qc_rows = []

start_all = time.time()


# =====================================================================
# 4. PROCESS EACH SNAPSHOT
# =====================================================================

for i, snap in enumerate(snapshot_dates, start=1):

    snapshot_start = time.time()

    print("\n" + "=" * 100)
    print(
        f"SNAPSHOT {i}/11 — {snap}"
    )
    print("=" * 100)

    raw_path = (
        RAW_AIRBNB_DIR
        / snap
        / "calendar.csv.gz"
    )

    raw_sql = raw_path.as_posix()


    # -----------------------------------------------------------------
    # 4A. FULL RAW AUDIT
    # -----------------------------------------------------------------

    audit_sql = f"""
    WITH raw AS (

        SELECT

            TRY_CAST(
                listing_id AS BIGINT
            ) AS listing_id,

            TRY_CAST(
                date AS DATE
            ) AS calendar_date,

            available,

            TRY_CAST(
                REPLACE(
                    REPLACE(price, '$', ''),
                    ',',
                    ''
                )
                AS DOUBLE
            ) AS price_num,

            TRY_CAST(
                REPLACE(
                    REPLACE(adjusted_price, '$', ''),
                    ',',
                    ''
                )
                AS DOUBLE
            ) AS adjusted_price_num,

            TRY_CAST(
                minimum_nights AS DOUBLE
            ) AS minimum_nights_num

        FROM read_csv_auto(
            '{raw_sql}',
            header = true,
            all_varchar = true
        )
    )

    SELECT

        COUNT(*) AS raw_rows,

        COUNT(
            DISTINCT listing_id
        ) AS raw_listings,

        MIN(
            calendar_date
        ) AS min_date,

        MAX(
            calendar_date
        ) AS max_date,

        COUNT(*)
        FILTER (
            WHERE listing_id IS NULL
        ) AS missing_listing_id,

        COUNT(*)
        FILTER (
            WHERE calendar_date IS NULL
        ) AS invalid_date,

        COUNT(*)
        FILTER (
            WHERE price_num IS NOT NULL
        ) AS price_nonmissing,

        COUNT(*)
        FILTER (
            WHERE price_num IS NOT NULL
              AND price_num <= 0
        ) AS price_nonpositive,

        COUNT(*)
        FILTER (
            WHERE adjusted_price_num IS NOT NULL
        ) AS adjusted_price_nonmissing,

        COUNT(*)
        FILTER (
            WHERE adjusted_price_num IS NOT NULL
              AND adjusted_price_num <= 0
        ) AS adjusted_price_nonpositive,

        COUNT(*)
        FILTER (
            WHERE minimum_nights_num IS NOT NULL
        ) AS minimum_nights_nonmissing,

        COUNT(*)
        FILTER (
            WHERE available = 't'
        ) AS available_t,

        COUNT(*)
        FILTER (
            WHERE available = 'f'
        ) AS available_f,

        COUNT(*)
        FILTER (
            WHERE available NOT IN ('t','f')
               OR available IS NULL
        ) AS available_invalid

    FROM raw
    """

    audit = (
        con.execute(audit_sql)
        .df()
        .iloc[0]
    )

    if pd.isna(audit["min_date"]):
        raise RuntimeError(
            f"{snap}: no valid calendar dates."
        )

    first_date = pd.Timestamp(
        audit["min_date"]
    )

    nominal_date = pd.Timestamp(
        snap
    )

    start_offset = int(
        (
            first_date
            - nominal_date
        ).days
    )

    window_end = (
        first_date
        + pd.Timedelta(days=27)
    )

    print(
        "Raw rows:",
        f'{int(audit["raw_rows"]):,}'
    )

    print(
        "Raw listings:",
        f'{int(audit["raw_listings"]):,}'
    )

    print(
        "Raw date range:",
        audit["min_date"],
        "->",
        audit["max_date"]
    )

    print(
        "Nominal snapshot:",
        nominal_date.date()
    )

    print(
        "First observed calendar date:",
        first_date.date()
    )

    print(
        "Start offset:",
        start_offset,
        "days"
    )

    print(
        "Operational 28-day window:",
        first_date.date(),
        "->",
        window_end.date()
    )

    print(
        "Raw price non-missing:",
        f'{int(audit["price_nonmissing"]):,}'
    )

    print(
        "Raw adjusted_price non-missing:",
        f'{int(audit["adjusted_price_nonmissing"]):,}'
    )

    print(
        "Invalid available values:",
        f'{int(audit["available_invalid"]):,}'
    )


    # -----------------------------------------------------------------
    # 4B. CALENDAR AGGREGATION
    #
    # 4 x 7-day bins:
    # W1 = days 0-6
    # W2 = days 7-13
    # W3 = days 14-20
    # W4 = days 21-27
    #
    # IMPORTANT:
    # available='t' -> open configuration
    # available='f' is NOT interpreted as booking.
    # -----------------------------------------------------------------

    agg_sql = f"""
    WITH raw AS (

        SELECT

            TRY_CAST(
                listing_id AS BIGINT
            ) AS listing_id,

            TRY_CAST(
                date AS DATE
            ) AS calendar_date,

            available,

            TRY_CAST(
                REPLACE(
                    REPLACE(price, '$', ''),
                    ',',
                    ''
                )
                AS DOUBLE
            ) AS price_num,

            TRY_CAST(
                minimum_nights AS DOUBLE
            ) AS minimum_nights_num

        FROM read_csv_auto(
            '{raw_sql}',
            header = true,
            all_varchar = true
        )
    ),

    windowed AS (

        SELECT

            *,

            DATE_DIFF(
                'day',
                DATE '{first_date.date()}',
                calendar_date
            ) AS day_index

        FROM raw

        WHERE
            calendar_date
            BETWEEN
                DATE '{first_date.date()}'
            AND
                DATE '{window_end.date()}'
    ),

    prepared AS (

        SELECT

            *,

            CAST(
                FLOOR(
                    day_index / 7.0
                ) + 1
                AS INTEGER
            ) AS week_no,

            CASE

                WHEN available = 't'
                    THEN 1.0

                WHEN available = 'f'
                    THEN 0.0

                ELSE NULL

            END AS open_flag,

            CASE

                WHEN EXTRACT(
                    'dow'
                    FROM calendar_date
                ) IN (0, 6)
                    THEN 1

                ELSE 0

            END AS weekend_flag

        FROM windowed
    ),

    agg AS (

        SELECT

            listing_id,

            COUNT(
                DISTINCT calendar_date
            ) AS calendar_days_28d,

            /* =============================
               WEEK 1
               ============================= */

            MEDIAN(price_num)
            FILTER (
                WHERE week_no = 1
            ) AS median_price_w1,

            AVG(open_flag)
            FILTER (
                WHERE week_no = 1
            ) AS open_ratio_w1,

            CASE

                WHEN
                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 1
                    ) > 0

                THEN

                    STDDEV_SAMP(price_num)
                    FILTER (
                        WHERE week_no = 1
                    )

                    /

                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 1
                    )

                ELSE NULL

            END AS price_cv_w1,

            MEDIAN(minimum_nights_num)
            FILTER (
                WHERE week_no = 1
            ) AS median_minimum_nights_w1,


            /* =============================
               WEEK 2
               ============================= */

            MEDIAN(price_num)
            FILTER (
                WHERE week_no = 2
            ) AS median_price_w2,

            AVG(open_flag)
            FILTER (
                WHERE week_no = 2
            ) AS open_ratio_w2,

            CASE

                WHEN
                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 2
                    ) > 0

                THEN

                    STDDEV_SAMP(price_num)
                    FILTER (
                        WHERE week_no = 2
                    )

                    /

                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 2
                    )

                ELSE NULL

            END AS price_cv_w2,

            MEDIAN(minimum_nights_num)
            FILTER (
                WHERE week_no = 2
            ) AS median_minimum_nights_w2,


            /* =============================
               WEEK 3
               ============================= */

            MEDIAN(price_num)
            FILTER (
                WHERE week_no = 3
            ) AS median_price_w3,

            AVG(open_flag)
            FILTER (
                WHERE week_no = 3
            ) AS open_ratio_w3,

            CASE

                WHEN
                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 3
                    ) > 0

                THEN

                    STDDEV_SAMP(price_num)
                    FILTER (
                        WHERE week_no = 3
                    )

                    /

                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 3
                    )

                ELSE NULL

            END AS price_cv_w3,

            MEDIAN(minimum_nights_num)
            FILTER (
                WHERE week_no = 3
            ) AS median_minimum_nights_w3,


            /* =============================
               WEEK 4
               ============================= */

            MEDIAN(price_num)
            FILTER (
                WHERE week_no = 4
            ) AS median_price_w4,

            AVG(open_flag)
            FILTER (
                WHERE week_no = 4
            ) AS open_ratio_w4,

            CASE

                WHEN
                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 4
                    ) > 0

                THEN

                    STDDEV_SAMP(price_num)
                    FILTER (
                        WHERE week_no = 4
                    )

                    /

                    AVG(price_num)
                    FILTER (
                        WHERE week_no = 4
                    )

                ELSE NULL

            END AS price_cv_w4,

            MEDIAN(minimum_nights_num)
            FILTER (
                WHERE week_no = 4
            ) AS median_minimum_nights_w4,


            /* =============================
               28-DAY SUMMARY
               ============================= */

            MEDIAN(
                price_num
            ) AS median_price_28d,

            AVG(
                open_flag
            ) AS open_ratio_28d,

            CASE

                WHEN AVG(price_num) > 0

                THEN
                    STDDEV_SAMP(price_num)
                    /
                    AVG(price_num)

                ELSE NULL

            END AS price_cv_28d,

            MEDIAN(
                minimum_nights_num
            ) AS median_minimum_nights_28d,

            MEDIAN(price_num)
            FILTER (
                WHERE weekend_flag = 1
            ) AS weekend_median_price,

            MEDIAN(price_num)
            FILTER (
                WHERE weekend_flag = 0
            ) AS weekday_median_price

        FROM prepared

        GROUP BY listing_id
    )

    SELECT

        *,

        CASE

            WHEN
                weekday_median_price > 0
                AND weekend_median_price IS NOT NULL

            THEN

                (
                    weekend_median_price
                    - weekday_median_price
                )
                /
                weekday_median_price

            ELSE NULL

        END AS weekend_price_premium

    FROM agg
    """

    cal = con.execute(
        agg_sql
    ).df()


    # -----------------------------------------------------------------
    # 4C. ADD PROVENANCE
    # -----------------------------------------------------------------

    cal.insert(
        0,
        "snapshot_date",
        nominal_date
    )

    cal.insert(
        1,
        "calendar_window_start",
        first_date
    )

    cal.insert(
        2,
        "calendar_window_end",
        window_end
    )

    cal.insert(
        3,
        "calendar_start_offset_days",
        start_offset
    )


    # -----------------------------------------------------------------
    # 4D. AUTOMATIC QC
    # -----------------------------------------------------------------

    duplicate_ids = int(
        cal["listing_id"]
        .duplicated()
        .sum()
    )

    missing_ids = int(
        cal["listing_id"]
        .isna()
        .sum()
    )

    full_28d = int(
        (
            cal["calendar_days_28d"]
            == 28
        ).sum()
    )

    partial_28d = int(
        (
            (
                cal["calendar_days_28d"]
                > 0
            )
            &
            (
                cal["calendar_days_28d"]
                < 28
            )
        ).sum()
    )

    median_price_available = int(
        cal["median_price_28d"]
        .notna()
        .sum()
    )

    open_ratio_available = int(
        cal["open_ratio_28d"]
        .notna()
        .sum()
    )

    minimum_nights_available = int(
        cal[
            "median_minimum_nights_28d"
        ]
        .notna()
        .sum()
    )

    price_positive = int(
        (
            cal["median_price_28d"]
            .dropna()
            > 0
        ).sum()
    )

    expected = (
        expected_listing_counts[snap]
    )

    # HARD GATES
    if duplicate_ids != 0:

        raise RuntimeError(
            f"{snap}: duplicate listing_id "
            "after calendar aggregation."
        )

    if missing_ids != 0:

        raise RuntimeError(
            f"{snap}: missing listing_id "
            "after calendar aggregation."
        )

    if (
        cal["calendar_days_28d"]
        > 28
    ).any():

        raise RuntimeError(
            f"{snap}: listing with >28 "
            "distinct calendar days."
        )

    valid_open = (
        cal["open_ratio_28d"]
        .dropna()
        .between(0, 1)
        .all()
    )

    if not valid_open:

        raise RuntimeError(
            f"{snap}: open_ratio_28d "
            "outside [0,1]."
        )


    # -----------------------------------------------------------------
    # 4E. SAVE PARQUET
    # -----------------------------------------------------------------

    out_path = (
        CAL_OUT
        /
        f"calendar_features_{snap}.parquet"
    )

    cal.to_parquet(
        out_path,
        index=False
    )


    # -----------------------------------------------------------------
    # 4F. SAVE/RELOAD VALIDATION
    # -----------------------------------------------------------------

    reloaded = pd.read_parquet(
        out_path
    )

    if len(reloaded) != len(cal):

        raise RuntimeError(
            f"{snap}: save/reload row mismatch."
        )

    if (
        reloaded["listing_id"]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            f"{snap}: duplicates after reload."
        )

    print("\nAGGREGATION QC")
    print("-" * 50)

    print(
        "Aggregated rows:",
        f"{len(cal):,}"
    )

    print(
        "Cleaned listings expected:",
        f"{expected:,}"
    )

    print(
        "Duplicate listing IDs:",
        duplicate_ids
    )

    print(
        "Full 28-day listings:",
        f"{full_28d:,}"
    )

    print(
        "Partial 28-day listings:",
        f"{partial_28d:,}"
    )

    print(
        "median_price_28d available:",
        f"{median_price_available:,}"
    )

    print(
        "open_ratio_28d available:",
        f"{open_ratio_available:,}"
    )

    print(
        "minimum_nights_28d available:",
        f"{minimum_nights_available:,}"
    )

    print(
        "Saved:",
        out_path.relative_to(
            PROJECT_ROOT
        )
    )


    # -----------------------------------------------------------------
    # 4G. STORE QC
    # -----------------------------------------------------------------

    qc_rows.append({

        "snapshot_date":
            snap,

        "raw_rows":
            int(audit["raw_rows"]),

        "raw_listings":
            int(audit["raw_listings"]),

        "cleaned_listing_expected":
            expected,

        "raw_min_date":
            audit["min_date"],

        "raw_max_date":
            audit["max_date"],

        "window_start":
            first_date.date(),

        "window_end":
            window_end.date(),

        "start_offset_days":
            start_offset,

        "aggregated_rows":
            len(cal),

        "full_28d":
            full_28d,

        "partial_28d":
            partial_28d,

        "duplicate_listing_id":
            duplicate_ids,

        "missing_listing_id":
            missing_ids,

        "raw_price_nonmissing":
            int(
                audit[
                    "price_nonmissing"
                ]
            ),

        "raw_price_nonpositive":
            int(
                audit[
                    "price_nonpositive"
                ]
            ),

        "raw_adjusted_price_nonmissing":
            int(
                audit[
                    "adjusted_price_nonmissing"
                ]
            ),

        "median_price_28d_nonmissing":
            median_price_available,

        "median_price_28d_positive":
            price_positive,

        "open_ratio_28d_nonmissing":
            open_ratio_available,

        "minimum_nights_28d_nonmissing":
            minimum_nights_available,

        "runtime_seconds":
            round(
                time.time()
                - snapshot_start,
                2
            )
    })

    del cal
    del reloaded


# =====================================================================
# 5. CLOSE DATABASE
# =====================================================================

con.close()


# =====================================================================
# 6. GLOBAL QC TABLE
# =====================================================================

qc = pd.DataFrame(
    qc_rows
)

qc_path = (
    CAL_OUT
    /
    "calendar_production_qc.csv"
)

qc.to_csv(
    qc_path,
    index=False
)


print("\n\n" + "=" * 100)
print("GLOBAL CALENDAR PRODUCTION QC")
print("=" * 100)

display(qc)


# =====================================================================
# 7. GLOBAL INTEGRITY CHECKS
# =====================================================================

saved_files = sorted(
    CAL_OUT.glob(
        "calendar_features_*.parquet"
    )
)

print("\n" + "=" * 100)
print("GLOBAL INTEGRITY RESULTS")
print("=" * 100)

print(
    "Calendar feature files:",
    len(saved_files),
    "/ 11"
)

print(
    "Total aggregated rows:",
    f'{int(qc["aggregated_rows"].sum()):,}'
)

print(
    "Total full 28-day listing-snapshots:",
    f'{int(qc["full_28d"].sum()):,}'
)

print(
    "Total partial 28-day listing-snapshots:",
    f'{int(qc["partial_28d"].sum()):,}'
)

print(
    "Snapshots with usable median_price_28d:",
    int(
        (
            qc[
                "median_price_28d_nonmissing"
            ]
            > 0
        ).sum()
    ),
    "/ 11"
)

print(
    "Snapshots with raw price data:",
    int(
        (
            qc[
                "raw_price_nonmissing"
            ]
            > 0
        ).sum()
    ),
    "/ 11"
)

print(
    "Snapshots with raw adjusted_price data:",
    int(
        (
            qc[
                "raw_adjusted_price_nonmissing"
            ]
            > 0
        ).sum()
    ),
    "/ 11"
)

print(
    "Start-offset range:",
    int(
        qc[
            "start_offset_days"
        ].min()
    ),
    "to",
    int(
        qc[
            "start_offset_days"
        ].max()
    ),
    "days"
)

print(
    "Global duplicate IDs after aggregation:",
    int(
        qc[
            "duplicate_listing_id"
        ].sum()
    )
)

print(
    "Global missing IDs after aggregation:",
    int(
        qc[
            "missing_listing_id"
        ].sum()
    )
)

print(
    "QC summary:",
    qc_path.relative_to(
        PROJECT_ROOT
    )
)

print(
    "Total runtime:",
    round(
        (
            time.time()
            - start_all
        ) / 60,
        2
    ),
    "minutes"
)


# =====================================================================
# 8. SCIENTIFIC DECISION FLAGS
# =====================================================================

print("\n" + "=" * 100)
print("SCIENTIFIC DECISION FLAGS")
print("=" * 100)

price_snapshots = int(
    (
        qc[
            "median_price_28d_nonmissing"
        ]
        > 0
    ).sum()
)

if price_snapshots == 11:

    print(
        "PRICE STATUS: Calendar-derived "
        "median_price_28d is available "
        "in all 11 snapshots."
    )

elif price_snapshots > 0:

    print(
        "PRICE STATUS: Calendar-derived "
        "median_price_28d is only available "
        f"in {price_snapshots}/11 snapshots."
    )

    print(
        "Do NOT impute missing snapshot prices."
    )

else:

    print(
        "PRICE STATUS: No usable calendar-derived "
        "median_price_28d was found."
    )

    print(
        "Do NOT construct the secondary "
        "price-resilience outcome yet."
    )


max_offset = int(
    qc[
        "start_offset_days"
    ].max()
)

if max_offset == 0:

    print(
        "WINDOW STATUS: Calendar begins "
        "on every nominal snapshot date."
    )

else:

    print(
        "WINDOW STATUS: Raw calendar begins "
        f"0–{max_offset} days after nominal "
        "snapshot dates."
    )

    print(
        "Offsets are explicitly preserved "
        "for methodological review."
    )


# =====================================================================
# 9. FINAL STATUS
# =====================================================================

print("\n" + "=" * 100)

if (
    len(saved_files) == 11
    and qc["duplicate_listing_id"].sum() == 0
    and qc["missing_listing_id"].sum() == 0
):

    print(
        "PRODUCTION BLOCK F STATUS: PASS"
    )

    print(
        "ALL 11 CALENDAR SNAPSHOTS "
        "PROCESSED AND SAVED."
    )

else:

    print(
        "PRODUCTION BLOCK F STATUS: FAIL"
    )

    print(
        "DO NOT PROCEED TO MODELLING."
    )

print("=" * 100)

In [ ]:
# =====================================================================
# PRODUCTION BLOCK G
#
# FINAL SUPPORTED CALENDAR CHANNEL
# + CALENDAR INTEGRATION
# + SNAPSHOT ADVERTISED-PRICE RESILIENCE OUTCOME
#
# IMPORTANT:
# 1. Calendar price is NOT used.
# 2. First 28 OBSERVED dates are selected separately for each listing.
# 3. Calendar rows are restricted to the cleaned-listing universe.
# 4. available='t' means open calendar configuration only.
# 5. available='f' is NOT interpreted as booked/occupied.
# 6. Secondary price uses CLEANED LISTINGS price.
# 7. Peer reference = ALL valid-price listings at each endpoint.
# 8. Peer = TA x room_type; fallback TA if peer N < 30.
# 9. T2-T5 secondary price target remains unavailable.
# 10. T10 is application/holdout only, never used to redesign rules.
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import time
import re
import json

print("=" * 110)
print("PRODUCTION BLOCK G")
print("FINAL CALENDAR + INTEGRATION + SECONDARY PRICE RESILIENCE")
print("=" * 110)

START = time.time()

# ---------------------------------------------------------------------
# 0. PROJECT PATHS
# ---------------------------------------------------------------------

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

RAW_AIRBNB = PROJECT_ROOT / "01_raw" / "airbnb"
LISTING_DIR = PROJECT_ROOT / "02_intermediate" / "listings"
CAL_DIR = PROJECT_ROOT / "02_intermediate" / "calendar"
INTEGRATION_DIR = PROJECT_ROOT / "02_intermediate" / "integration"
OUTCOME_DIR = PROJECT_ROOT / "02_intermediate" / "outcomes"

CAL_FINAL_DIR = CAL_DIR / "final_supported"

CAL_FINAL_DIR.mkdir(parents=True, exist_ok=True)
OUTCOME_DIR.mkdir(parents=True, exist_ok=True)

BACKBONE_PATH = (
    INTEGRATION_DIR
    / "listing_origin_backbone_T1_T10_with_context_reviews.parquet"
)

if not BACKBONE_PATH.exists():
    raise FileNotFoundError(
        f"Frozen integrated backbone not found:\n{BACKBONE_PATH}"
    )

print("\nFrozen backbone:")
print(BACKBONE_PATH)


# ---------------------------------------------------------------------
# 1. FROZEN TEMPORAL MAP
# ---------------------------------------------------------------------

transition_map = {
    "T1":  ("2025-10-05", "2025-11-07"),
    "T2":  ("2025-11-07", "2025-12-11"),
    "T3":  ("2025-12-11", "2026-01-16"),
    "T4":  ("2026-01-16", "2026-02-13"),
    "T5":  ("2026-02-13", "2026-03-17"),
    "T6":  ("2026-03-17", "2026-04-16"),
    "T7":  ("2026-04-16", "2026-05-23"),
    "T8":  ("2026-05-23", "2026-06-19"),
    "T9":  ("2026-06-19", "2026-07-12"),
    "T10": ("2026-07-12", "2026-08-13"),
}

all_dates = sorted(
    set(
        [x[0] for x in transition_map.values()]
        +
        [x[1] for x in transition_map.values()]
    )
)

expected_listing_counts = {
    "2025-10-05": 47752,
    "2025-11-07": 48624,
    "2025-12-11": 49899,
    "2026-01-16": 46819,
    "2026-02-13": 48692,
    "2026-03-17": 49076,
    "2026-04-16": 48951,
    "2026-05-23": 49305,
    "2026-06-19": 50926,
    "2026-07-12": 51096,
    "2026-08-13": 51325,
}

feasible_price_transitions = {
    "T1", "T6", "T7", "T8", "T9", "T10"
}

development_price_transitions = {
    "T1", "T6", "T7", "T8", "T9"
}

PEER_MIN_N = 30

print("\nTemporal design: PASS")
print("Secondary price feasible:", sorted(feasible_price_transitions))
print("Development price transitions:", sorted(development_price_transitions))
print("Locked T10: application only")


# =====================================================================
# PART A — REBUILD FINAL CALENDAR FEATURES
# =====================================================================

print("\n" + "=" * 110)
print("PART A — FINAL LISTING-SPECIFIC CALENDAR CONSTRUCTION")
print("=" * 110)

con = duckdb.connect(database=":memory:")

calendar_qc = []

for idx, snap in enumerate(all_dates, start=1):

    t0 = time.time()

    print("\n" + "-" * 110)
    print(f"CALENDAR {idx}/11 — {snap}")

    raw_calendar = (
        RAW_AIRBNB
        / snap
        / "calendar.csv.gz"
    )

    listing_path = (
        LISTING_DIR
        / f"listings_{snap}.parquet"
    )

    if not raw_calendar.exists():
        raise FileNotFoundError(raw_calendar)

    if not listing_path.exists():
        raise FileNotFoundError(listing_path)

    raw_sql = raw_calendar.as_posix()
    listing_sql = listing_path.as_posix()

    # -----------------------------------------------------------------
    # Select first 28 DISTINCT OBSERVED dates PER LISTING.
    #
    # Restrict to cleaned listing universe before aggregation.
    # -----------------------------------------------------------------

    query = f"""
    WITH cleaned_listing_universe AS (

        SELECT DISTINCT
            TRY_CAST(listing_id AS BIGINT) AS listing_id

        FROM read_parquet('{listing_sql}')
    ),

    raw AS (

        SELECT
            TRY_CAST(c.listing_id AS BIGINT) AS listing_id,
            TRY_CAST(c.date AS DATE) AS calendar_date,
            c.available,

            TRY_CAST(
                c.minimum_nights AS DOUBLE
            ) AS minimum_nights_num

        FROM read_csv_auto(
            '{raw_sql}',
            header=true,
            all_varchar=true
        ) c

        INNER JOIN cleaned_listing_universe l
            ON TRY_CAST(c.listing_id AS BIGINT) = l.listing_id

        WHERE
            TRY_CAST(c.date AS DATE) IS NOT NULL
    ),

    date_ranked AS (

        SELECT
            *,
            DENSE_RANK() OVER (
                PARTITION BY listing_id
                ORDER BY calendar_date
            ) AS observed_day_rank

        FROM raw
    ),

    first28 AS (

        SELECT
            *,
            CAST(
                FLOOR(
                    (observed_day_rank - 1) / 7.0
                ) + 1
                AS INTEGER
            ) AS week_no,

            CASE
                WHEN available = 't' THEN 1.0
                WHEN available = 'f' THEN 0.0
                ELSE NULL
            END AS open_flag

        FROM date_ranked

        WHERE observed_day_rank <= 28
    ),

    aggregated AS (

        SELECT

            listing_id,

            MIN(calendar_date)
                AS calendar_first_observed_date,

            MAX(calendar_date)
                AS calendar_28th_observed_date,

            COUNT(DISTINCT calendar_date)
                AS calendar_observed_days_28d,

            /* Week 1 */

            AVG(open_flag)
                FILTER (WHERE week_no = 1)
                AS open_ratio_w1,

            MEDIAN(minimum_nights_num)
                FILTER (WHERE week_no = 1)
                AS median_minimum_nights_w1,

            /* Week 2 */

            AVG(open_flag)
                FILTER (WHERE week_no = 2)
                AS open_ratio_w2,

            MEDIAN(minimum_nights_num)
                FILTER (WHERE week_no = 2)
                AS median_minimum_nights_w2,

            /* Week 3 */

            AVG(open_flag)
                FILTER (WHERE week_no = 3)
                AS open_ratio_w3,

            MEDIAN(minimum_nights_num)
                FILTER (WHERE week_no = 3)
                AS median_minimum_nights_w3,

            /* Week 4 */

            AVG(open_flag)
                FILTER (WHERE week_no = 4)
                AS open_ratio_w4,

            MEDIAN(minimum_nights_num)
                FILTER (WHERE week_no = 4)
                AS median_minimum_nights_w4,

            /* 28-day summaries */

            AVG(open_flag)
                AS open_ratio_28d,

            MEDIAN(minimum_nights_num)
                AS median_minimum_nights_28d

        FROM first28

        GROUP BY listing_id
    )

    SELECT *
    FROM aggregated
    ORDER BY listing_id
    """

    cal = con.execute(query).df()

    expected = expected_listing_counts[snap]

    # -------------------------------------------------------------
    # HARD QC
    # -------------------------------------------------------------

    duplicate = int(
        cal["listing_id"].duplicated().sum()
    )

    missing_id = int(
        cal["listing_id"].isna().sum()
    )

    full28 = int(
        (cal["calendar_observed_days_28d"] == 28).sum()
    )

    partial = int(
        (
            cal["calendar_observed_days_28d"]
            .between(1, 27)
        ).sum()
    )

    invalid_gt28 = int(
        (cal["calendar_observed_days_28d"] > 28).sum()
    )

    if duplicate != 0:
        raise RuntimeError(
            f"{snap}: duplicate calendar listing IDs."
        )

    if missing_id != 0:
        raise RuntimeError(
            f"{snap}: missing calendar listing IDs."
        )

    if invalid_gt28 != 0:
        raise RuntimeError(
            f"{snap}: >28 observed dates after ranking."
        )

    if len(cal) != expected:
        raise RuntimeError(
            f"{snap}: final calendar rows {len(cal):,} "
            f"!= cleaned listings {expected:,}"
        )

    open_cols = [
        "open_ratio_w1",
        "open_ratio_w2",
        "open_ratio_w3",
        "open_ratio_w4",
        "open_ratio_28d",
    ]

    for c in open_cols:

        values = cal[c].dropna()

        if not values.between(0, 1).all():
            raise RuntimeError(
                f"{snap}: {c} outside [0,1]."
            )

    # Provenance
    cal.insert(
        0,
        "snapshot_date",
        pd.Timestamp(snap)
    )

    cal["calendar_first_offset_days"] = (
        pd.to_datetime(
            cal["calendar_first_observed_date"]
        )
        -
        pd.Timestamp(snap)
    ).dt.days

    cal["calendar_28th_span_days"] = (
        pd.to_datetime(
            cal["calendar_28th_observed_date"]
        )
        -
        pd.to_datetime(
            cal["calendar_first_observed_date"]
        )
    ).dt.days + 1

    # -------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------

    out = (
        CAL_FINAL_DIR
        / f"calendar_supported_{snap}.parquet"
    )

    cal.to_parquet(
        out,
        index=False
    )

    reload = pd.read_parquet(out)

    if len(reload) != expected:
        raise RuntimeError(
            f"{snap}: save/reload calendar mismatch."
        )

    if reload["listing_id"].duplicated().any():
        raise RuntimeError(
            f"{snap}: duplicate after calendar reload."
        )

    calendar_qc.append({
        "snapshot_date": snap,
        "expected_cleaned_listings": expected,
        "final_calendar_rows": len(cal),
        "duplicate_ids": duplicate,
        "full_28_observed_dates": full28,
        "partial_lt28": partial,
        "open_ratio_28d_nonmissing":
            int(cal["open_ratio_28d"].notna().sum()),
        "minimum_nights_28d_nonmissing":
            int(
                cal[
                    "median_minimum_nights_28d"
                ].notna().sum()
            ),
        "first_offset_min":
            int(
                cal[
                    "calendar_first_offset_days"
                ].min()
            ),
        "first_offset_median":
            float(
                cal[
                    "calendar_first_offset_days"
                ].median()
            ),
        "first_offset_max":
            int(
                cal[
                    "calendar_first_offset_days"
                ].max()
            ),
        "runtime_sec":
            round(time.time() - t0, 2)
    })

    print(
        f"Rows {len(cal):,} | "
        f"full28 {full28:,} | "
        f"partial {partial:,} | "
        f"duplicates {duplicate}"
    )

    del cal, reload


calendar_qc = pd.DataFrame(calendar_qc)

calendar_qc_path = (
    CAL_FINAL_DIR
    / "calendar_supported_final_qc.csv"
)

calendar_qc.to_csv(
    calendar_qc_path,
    index=False
)

print("\nFINAL CALENDAR QC")
display(calendar_qc)


# =====================================================================
# PART B — INTEGRATE FINAL CALENDAR INTO T1-T10 BACKBONE
# =====================================================================

print("\n" + "=" * 110)
print("PART B — INTEGRATE SUPPORTED CALENDAR CHANNEL")
print("=" * 110)

backbone = pd.read_parquet(
    BACKBONE_PATH
)

print(
    "Frozen backbone:",
    f"{len(backbone):,}",
    "rows x",
    len(backbone.columns),
    "columns"
)

if len(backbone) != 491_140:
    raise RuntimeError(
        "Frozen backbone must contain exactly 491,140 rows."
    )

if backbone[
    ["transition", "listing_id"]
].duplicated().any():
    raise RuntimeError(
        "Duplicate listing-origin keys in backbone."
    )

calendar_parts = []

for transition, (anchor, future) in transition_map.items():

    p = (
        CAL_FINAL_DIR
        / f"calendar_supported_{anchor}.parquet"
    )

    x = pd.read_parquet(p)

    x["transition"] = transition

    calendar_parts.append(x)

calendar_all = pd.concat(
    calendar_parts,
    ignore_index=True
)

calendar_merge_cols = [
    "transition",
    "listing_id",
    "calendar_first_observed_date",
    "calendar_28th_observed_date",
    "calendar_observed_days_28d",
    "calendar_first_offset_days",
    "calendar_28th_span_days",
    "open_ratio_w1",
    "median_minimum_nights_w1",
    "open_ratio_w2",
    "median_minimum_nights_w2",
    "open_ratio_w3",
    "median_minimum_nights_w3",
    "open_ratio_w4",
    "median_minimum_nights_w4",
    "open_ratio_28d",
    "median_minimum_nights_28d",
]

calendar_all = calendar_all[
    calendar_merge_cols
]

if calendar_all[
    ["transition", "listing_id"]
].duplicated().any():
    raise RuntimeError(
        "Duplicate calendar listing-origin keys."
    )

before = len(backbone)

integrated = backbone.merge(
    calendar_all,
    on=["transition", "listing_id"],
    how="left",
    validate="one_to_one",
    indicator="_calendar_merge"
)

after = len(integrated)

if before != after:
    raise RuntimeError(
        "Calendar merge changed backbone row count."
    )

calendar_unmatched = int(
    (
        integrated["_calendar_merge"]
        != "both"
    ).sum()
)

print("Rows before:", f"{before:,}")
print("Rows after :", f"{after:,}")
print("Calendar unmatched:", calendar_unmatched)

if calendar_unmatched != 0:
    raise RuntimeError(
        "Some backbone rows failed calendar merge."
    )

integrated.drop(
    columns="_calendar_merge",
    inplace=True
)

CAL_INTEGRATED_PATH = (
    INTEGRATION_DIR
    / "listing_origin_backbone_T1_T10_final_multimodal.parquet"
)

integrated.to_parquet(
    CAL_INTEGRATED_PATH,
    index=False
)

print(
    "Saved final multimodal backbone:",
    CAL_INTEGRATED_PATH.relative_to(PROJECT_ROOT)
)


# =====================================================================
# PART C — BUILD ENDPOINT PRICE REFERENCE TABLES
# =====================================================================

print("\n" + "=" * 110)
print("PART C — CONTEMPORANEOUS SNAPSHOT PRICE REFERENCE")
print("=" * 110)

# -------------------------------------------------------------
# Geography:
# use T1-T10 integrated backbone wherever possible.
#
# For Aug 2026, geography is discovered from 02_intermediate.
# -------------------------------------------------------------

# Anchor endpoint information from integrated backbone
anchor_endpoint = integrated[
    [
        "transition",
        "snapshot_date",
        "listing_id",
        "ta_code",
        "room_type",
    ]
].copy()

anchor_endpoint["snapshot_date"] = pd.to_datetime(
    anchor_endpoint["snapshot_date"]
).dt.strftime("%Y-%m-%d")


# -------------------------------------------------------------
# Helper: robustly parse listing price
# -------------------------------------------------------------

def parse_price(series):

    return pd.to_numeric(
        series.astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )


# -------------------------------------------------------------
# Helper: discover geography file for a date
# -------------------------------------------------------------

def find_geography_for_date(snapshot_date):

    candidates = []

    geo_root = (
        PROJECT_ROOT
        / "02_intermediate"
    )

    for p in geo_root.rglob("*.parquet"):

        name = p.name.lower()

        if (
            snapshot_date in p.name
            and (
                "geo" in name
                or "ta" in name
            )
        ):
            candidates.append(p)

    valid = []

    for p in candidates:

        try:
            cols = pd.read_parquet(
                p
            ).columns.tolist()

            if (
                "listing_id" in cols
                and "ta_code" in cols
            ):
                valid.append(p)

        except Exception:
            pass

    return valid


# -------------------------------------------------------------
# Build endpoint tables for all 11 snapshots
# -------------------------------------------------------------

endpoint_tables = {}
endpoint_qc = []

# Map anchor-date -> TA from existing frozen backbone
anchor_geo_lookup = {}

for snap in all_dates[:-1]:

    temp = anchor_endpoint[
        anchor_endpoint["snapshot_date"] == snap
    ][
        [
            "listing_id",
            "ta_code"
        ]
    ].drop_duplicates()

    anchor_geo_lookup[snap] = temp


for snap in all_dates:

    listing_path = (
        LISTING_DIR
        / f"listings_{snap}.parquet"
    )

    ls = pd.read_parquet(
        listing_path
    )

    required = {
        "listing_id",
        "price",
        "room_type"
    }

    missing = required - set(ls.columns)

    if missing:
        raise RuntimeError(
            f"{snap}: missing listing columns {missing}"
        )

    ep = ls[
        [
            "listing_id",
            "price",
            "room_type"
        ]
    ].copy()

    ep["price_num"] = parse_price(
        ep["price"]
    )

    # Geography
    if snap in anchor_geo_lookup:

        geo = anchor_geo_lookup[snap]

    else:

        # Aug 13 outcome endpoint
        geo_candidates = find_geography_for_date(
            snap
        )

        print(
            "\nAug geography candidates:",
            [str(x.relative_to(PROJECT_ROOT))
             for x in geo_candidates]
        )

        if len(geo_candidates) != 1:

            raise RuntimeError(
                f"{snap}: expected exactly one usable "
                f"geography parquet, found "
                f"{len(geo_candidates)}.\n"
                f"Candidates: {geo_candidates}"
            )

        geo = pd.read_parquet(
            geo_candidates[0]
        )[
            [
                "listing_id",
                "ta_code"
            ]
        ].copy()

        geo = geo.drop_duplicates(
            subset="listing_id"
        )

    ep = ep.merge(
        geo,
        on="listing_id",
        how="left",
        validate="one_to_one"
    )

    missing_geo = int(
        ep["ta_code"].isna().sum()
    )

    if missing_geo != 0:
        raise RuntimeError(
            f"{snap}: {missing_geo} listings missing TA."
        )

    valid_price = (
        ep["price_num"].notna()
        &
        (ep["price_num"] > 0)
    )

    ref = ep.loc[
        valid_price
    ].copy()

    # Peer sizes based on ALL valid-price listings
    ref["peer_n_ta_room"] = (
        ref.groupby(
            ["ta_code", "room_type"]
        )["listing_id"]
        .transform("size")
    )

    ref["peer_n_ta"] = (
        ref.groupby(
            ["ta_code"]
        )["listing_id"]
        .transform("size")
    )

    # ---------------------------------------------------------
    # Midrank empirical percentile:
    #
    # (average rank - 0.5) / N
    #
    # Advantages:
    # - defined for N >= 1
    # - lies strictly inside (0,1)
    # - symmetric
    # - ties receive same midpoint percentile
    # ---------------------------------------------------------

    ref["rank_ta_room"] = (
        ref.groupby(
            ["ta_code", "room_type"]
        )["price_num"]
        .rank(
            method="average"
        )
    )

    ref["pct_ta_room"] = (
        ref["rank_ta_room"] - 0.5
    ) / ref["peer_n_ta_room"]

    ref["rank_ta"] = (
        ref.groupby(
            ["ta_code"]
        )["price_num"]
        .rank(
            method="average"
        )
    )

    ref["pct_ta"] = (
        ref["rank_ta"] - 0.5
    ) / ref["peer_n_ta"]

    # Primary peer hierarchy
    ref["price_peer_fallback"] = (
        ref["peer_n_ta_room"] < PEER_MIN_N
    ).astype("int8")

    ref["price_peer_type"] = np.where(
        ref["peer_n_ta_room"] >= PEER_MIN_N,
        "TA_room_type",
        np.where(
            ref["peer_n_ta"] >= PEER_MIN_N,
            "TA_only",
            "unrankable"
        )
    )

    ref["price_peer_n_used"] = np.where(
        ref["peer_n_ta_room"] >= PEER_MIN_N,
        ref["peer_n_ta_room"],
        np.where(
            ref["peer_n_ta"] >= PEER_MIN_N,
            ref["peer_n_ta"],
            np.nan
        )
    )

    ref["price_percentile"] = np.where(
        ref["peer_n_ta_room"] >= PEER_MIN_N,
        ref["pct_ta_room"],
        np.where(
            ref["peer_n_ta"] >= PEER_MIN_N,
            ref["pct_ta"],
            np.nan
        )
    )

    if not ref[
        "price_percentile"
    ].dropna().between(0, 1).all():

        raise RuntimeError(
            f"{snap}: percentile outside [0,1]."
        )

    endpoint_tables[snap] = ref[
        [
            "listing_id",
            "price_num",
            "room_type",
            "ta_code",
            "peer_n_ta_room",
            "peer_n_ta",
            "price_peer_fallback",
            "price_peer_type",
            "price_peer_n_used",
            "price_percentile",
        ]
    ].copy()

    endpoint_qc.append({
        "snapshot_date": snap,
        "listing_rows": len(ep),
        "valid_positive_price": len(ref),
        "TA_room_type_primary":
            int(
                (
                    ref["price_peer_type"]
                    == "TA_room_type"
                ).sum()
            ),
        "TA_only_fallback":
            int(
                (
                    ref["price_peer_type"]
                    == "TA_only"
                ).sum()
            ),
        "unrankable":
            int(
                (
                    ref["price_peer_type"]
                    == "unrankable"
                ).sum()
            ),
        "code999_valid_price":
            int(
                (
                    ref["ta_code"]
                    .astype(str)
                    == "999"
                ).sum()
            )
    })


endpoint_qc = pd.DataFrame(
    endpoint_qc
)

print("\nENDPOINT PRICE REFERENCE QC")
display(endpoint_qc)


# =====================================================================
# PART D — CONSTRUCT SECONDARY PRICE RESILIENCE OUTCOME
# =====================================================================

print("\n" + "=" * 110)
print("PART D — SECONDARY SNAPSHOT ADVERTISED-PRICE RESILIENCE")
print("=" * 110)

secondary_parts = []
secondary_qc = []

for transition, (anchor, future) in transition_map.items():

    anchor_ids = integrated.loc[
        integrated["transition"] == transition,
        ["listing_id"]
    ].copy()

    anchor_ids["transition"] = transition
    anchor_ids["anchor_date"] = pd.Timestamp(anchor)
    anchor_ids["future_date"] = pd.Timestamp(future)

    # Persistent only
    future_listing_ids = set(
        pd.read_parquet(
            LISTING_DIR
            / f"listings_{future}.parquet",
            columns=["listing_id"]
        )["listing_id"]
        .tolist()
    )

    anchor_ids["persistent_next"] = (
        anchor_ids["listing_id"]
        .isin(future_listing_ids)
        .astype("int8")
    )

    if transition not in feasible_price_transitions:

        # Structurally unavailable secondary outcome
        temp = anchor_ids.copy()

        temp["secondary_price_available"] = 0
        temp["anchor_price"] = np.nan
        temp["future_price"] = np.nan
        temp["anchor_price_percentile"] = np.nan
        temp["future_price_percentile"] = np.nan
        temp["anchor_peer_type"] = pd.NA
        temp["future_peer_type"] = pd.NA
        temp["anchor_peer_n"] = np.nan
        temp["future_peer_n"] = np.nan
        temp["price_resilience_delta"] = np.nan

        secondary_parts.append(temp)

        secondary_qc.append({
            "transition": transition,
            "anchor_date": anchor,
            "future_date": future,
            "anchor_rows": len(temp),
            "persistent": int(
                temp["persistent_next"].sum()
            ),
            "both_price_rankable": 0,
            "secondary_available": 0,
            "structurally_unavailable": True
        })

        continue

    a = endpoint_tables[anchor].rename(
        columns={
            "price_num":
                "anchor_price",

            "price_percentile":
                "anchor_price_percentile",

            "price_peer_type":
                "anchor_peer_type",

            "price_peer_n_used":
                "anchor_peer_n",

            "ta_code":
                "anchor_ta_code_price",

            "room_type":
                "anchor_room_type_price",
        }
    )

    f = endpoint_tables[future].rename(
        columns={
            "price_num":
                "future_price",

            "price_percentile":
                "future_price_percentile",

            "price_peer_type":
                "future_peer_type",

            "price_peer_n_used":
                "future_peer_n",

            "ta_code":
                "future_ta_code_price",

            "room_type":
                "future_room_type_price",
        }
    )

    keep_a = [
        "listing_id",
        "anchor_price",
        "anchor_price_percentile",
        "anchor_peer_type",
        "anchor_peer_n",
        "anchor_ta_code_price",
        "anchor_room_type_price",
    ]

    keep_f = [
        "listing_id",
        "future_price",
        "future_price_percentile",
        "future_peer_type",
        "future_peer_n",
        "future_ta_code_price",
        "future_room_type_price",
    ]

    temp = anchor_ids.merge(
        a[keep_a],
        on="listing_id",
        how="left",
        validate="one_to_one"
    )

    temp = temp.merge(
        f[keep_f],
        on="listing_id",
        how="left",
        validate="one_to_one"
    )

    valid_secondary = (
        (temp["persistent_next"] == 1)
        &
        temp["anchor_price"].notna()
        &
        temp["future_price"].notna()
        &
        temp["anchor_price_percentile"].notna()
        &
        temp["future_price_percentile"].notna()
    )

    temp["secondary_price_available"] = (
        valid_secondary.astype("int8")
    )

    temp["price_resilience_delta"] = np.where(
        valid_secondary,
        (
            temp["future_price_percentile"]
            -
            temp["anchor_price_percentile"]
        ),
        np.nan
    )

    # Valid delta must be [-1, 1]
    if not temp[
        "price_resilience_delta"
    ].dropna().between(-1, 1).all():

        raise RuntimeError(
            f"{transition}: price delta outside [-1,1]."
        )

    secondary_parts.append(temp)

    secondary_qc.append({
        "transition": transition,
        "anchor_date": anchor,
        "future_date": future,
        "anchor_rows": len(temp),
        "persistent":
            int(temp["persistent_next"].sum()),
        "anchor_valid_price":
            int(temp["anchor_price"].notna().sum()),
        "future_valid_price":
            int(temp["future_price"].notna().sum()),
        "both_price_rankable":
            int(valid_secondary.sum()),
        "secondary_available":
            int(
                temp[
                    "secondary_price_available"
                ].sum()
            ),
        "anchor_TA_only_fallback":
            int(
                (
                    temp.loc[
                        valid_secondary,
                        "anchor_peer_type"
                    ]
                    == "TA_only"
                ).sum()
            ),
        "future_TA_only_fallback":
            int(
                (
                    temp.loc[
                        valid_secondary,
                        "future_peer_type"
                    ]
                    == "TA_only"
                ).sum()
            ),
        "delta_mean":
            float(
                temp[
                    "price_resilience_delta"
                ].mean()
            ),
        "delta_median":
            float(
                temp[
                    "price_resilience_delta"
                ].median()
            ),
        "structurally_unavailable":
            False
    })


secondary = pd.concat(
    secondary_parts,
    ignore_index=True
)

secondary_qc = pd.DataFrame(
    secondary_qc
)


# =====================================================================
# PART E — FINAL SECONDARY QC
# =====================================================================

print("\n" + "=" * 110)
print("PART E — SECONDARY OUTCOME QC")
print("=" * 110)

if len(secondary) != 491_140:
    raise RuntimeError(
        f"Secondary table rows = {len(secondary):,}; "
        "expected 491,140."
    )

if secondary[
    ["transition", "listing_id"]
].duplicated().any():
    raise RuntimeError(
        "Duplicate transition-listing keys "
        "in secondary outcome."
    )

# T2-T5 must be entirely unavailable
for t in ["T2", "T3", "T4", "T5"]:

    z = secondary.loc[
        secondary["transition"] == t
    ]

    if (
        z["secondary_price_available"]
        .sum()
        != 0
    ):
        raise RuntimeError(
            f"{t}: secondary target should "
            "be structurally unavailable."
        )

# T10 rule applied, not tuned
print(
    "Peer minimum N:",
    PEER_MIN_N,
    "(pre-specified; not selected using T10)"
)

print("\nSECONDARY TRANSITION QC")
display(secondary_qc)


# =====================================================================
# PART F — SAVE/FREEZE SECONDARY OUTCOME
# =====================================================================

SECONDARY_PATH = (
    OUTCOME_DIR
    / "secondary_snapshot_advertised_price_resilience_T1_T10_frozen.parquet"
)

secondary.to_parquet(
    SECONDARY_PATH,
    index=False
)

secondary_reload = pd.read_parquet(
    SECONDARY_PATH
)

if len(secondary_reload) != 491_140:
    raise RuntimeError(
        "Secondary save/reload row mismatch."
    )

if secondary_reload[
    ["transition", "listing_id"]
].duplicated().any():
    raise RuntimeError(
        "Secondary duplicates after reload."
    )


# =====================================================================
# PART G — SAVE QC REPORTS
# =====================================================================

endpoint_qc_path = (
    OUTCOME_DIR
    / "secondary_price_endpoint_reference_qc.csv"
)

secondary_qc_path = (
    OUTCOME_DIR
    / "secondary_price_transition_qc.csv"
)

endpoint_qc.to_csv(
    endpoint_qc_path,
    index=False
)

secondary_qc.to_csv(
    secondary_qc_path,
    index=False
)


# =====================================================================
# PART H — FINAL SUMMARY
# =====================================================================

print("\n" + "=" * 110)
print("FINAL PRODUCTION SUMMARY")
print("=" * 110)

print(
    "Final supported calendar files:",
    len(
        list(
            CAL_FINAL_DIR.glob(
                "calendar_supported_*.parquet"
            )
        )
    ),
    "/ 11"
)

print(
    "Calendar integrated backbone rows:",
    f"{len(integrated):,}"
)

print(
    "Calendar unmatched rows:",
    calendar_unmatched
)

print(
    "Secondary outcome rows:",
    f"{len(secondary):,}"
)

print(
    "Secondary available observations:",
    f'{int(secondary["secondary_price_available"].sum()):,}'
)

print(
    "Secondary missing/unavailable observations:",
    f'{int((secondary["secondary_price_available"] == 0).sum()):,}'
)

print(
    "Secondary outcome min/max:",
    secondary["price_resilience_delta"].min(),
    "/",
    secondary["price_resilience_delta"].max()
)

print(
    "\nFinal multimodal backbone:"
)
print(
    CAL_INTEGRATED_PATH.relative_to(
        PROJECT_ROOT
    )
)

print(
    "\nFrozen secondary outcome:"
)
print(
    SECONDARY_PATH.relative_to(
        PROJECT_ROOT
    )
)

print(
    "\nCalendar QC:"
)
print(
    calendar_qc_path.relative_to(
        PROJECT_ROOT
    )
)

print(
    "\nEndpoint reference QC:"
)
print(
    endpoint_qc_path.relative_to(
        PROJECT_ROOT
    )
)

print(
    "\nSecondary transition QC:"
)
print(
    secondary_qc_path.relative_to(
        PROJECT_ROOT
    )
)

print(
    "\nTotal runtime:",
    round(
        (time.time() - START) / 60,
        2
    ),
    "minutes"
)

print("\n" + "=" * 110)

print(
    "PRODUCTION BLOCK G STATUS: PASS"
)

print(
    "SUPPORTED CALENDAR CHANNEL + "
    "SECONDARY SNAPSHOT ADVERTISED-PRICE "
    "RESILIENCE CONSTRUCTED."
)

print("=" * 110)

con.close()

In [ ]:
# ======================================================================
# PRODUCTION BLOCK H
# FINAL MODELLING PANEL + FEATURE REGISTRY + LEAKAGE AUDIT
#
# DOES NOT FIT ANY MODEL.
# DOES NOT USE T10 TO SELECT / IMPUTE / SCALE FEATURES.
# DOES NOT GLOBALLY IMPUTE MISSING VALUES.
# ======================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import hashlib
from collections import Counter

print("=" * 110)
print("PRODUCTION BLOCK H")
print("MODELLING PANEL + FEATURE REGISTRY + LEAKAGE AUDIT")
print("=" * 110)

# ----------------------------------------------------------------------
# 0. PATHS
# ----------------------------------------------------------------------

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

INT_DIR = PROJECT_ROOT / "02_intermediate" / "integration"
OUT_DIR = PROJECT_ROOT / "02_intermediate" / "outcomes"
MODEL_DIR = PROJECT_ROOT / "02_intermediate" / "modelling"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BACKBONE_PATH = (
    INT_DIR /
    "listing_origin_backbone_T1_T10_final_multimodal.parquet"
)

PRIMARY_PATH = (
    OUT_DIR /
    "primary_persistence_outcome_T1_T10_frozen.parquet"
)

SECONDARY_PATH = (
    OUT_DIR /
    "secondary_snapshot_advertised_price_resilience_T1_T10_frozen.parquet"
)

for p in [BACKBONE_PATH, PRIMARY_PATH, SECONDARY_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("Input files: PASS")


# ----------------------------------------------------------------------
# 1. LOAD FROZEN DATA
# ----------------------------------------------------------------------

X = pd.read_parquet(BACKBONE_PATH)
P = pd.read_parquet(PRIMARY_PATH)
S = pd.read_parquet(SECONDARY_PATH)

print("\nBackbone :", X.shape)
print("Primary  :", P.shape)
print("Secondary:", S.shape)

if len(X) != 491_140:
    raise RuntimeError("Backbone must contain 491,140 rows.")

# ----------------------------------------------------------------------
# 2. IDENTIFY PRIMARY TARGET COLUMN SAFELY
# ----------------------------------------------------------------------

print("\nPrimary columns:")
print(P.columns.tolist())

primary_candidates = [
    c for c in P.columns
    if c.lower() in {
        "persistent_next",
        "next_period_persistence",
        "platform_persistence",
        "persistence"
    }
]

if len(primary_candidates) != 1:
    raise RuntimeError(
        "Cannot uniquely identify primary target. "
        f"Candidates found: {primary_candidates}"
    )

PRIMARY_TARGET = primary_candidates[0]

print("Primary target:", PRIMARY_TARGET)


# ----------------------------------------------------------------------
# 3. NORMALISE KEY TYPES
# ----------------------------------------------------------------------

for df in [X, P, S]:
    if "listing_id" in df.columns:
        df["listing_id"] = pd.to_numeric(
            df["listing_id"], errors="raise"
        ).astype("int64")

if "transition" not in X.columns:
    raise RuntimeError("Backbone has no transition column.")

if X[["transition", "listing_id"]].duplicated().any():
    raise RuntimeError("Duplicate keys in backbone.")

if P[["transition", "listing_id"]].duplicated().any():
    raise RuntimeError("Duplicate keys in primary outcome.")

if S[["transition", "listing_id"]].duplicated().any():
    raise RuntimeError("Duplicate keys in secondary outcome.")


# ----------------------------------------------------------------------
# 4. MERGE OUTCOMES — ONLY TARGET FIELDS
# ----------------------------------------------------------------------

primary_keep = [
    "transition",
    "listing_id",
    PRIMARY_TARGET
]

secondary_keep = [
    "transition",
    "listing_id",
    "secondary_price_available",
    "price_resilience_delta"
]

panel = X.merge(
    P[primary_keep],
    on=["transition", "listing_id"],
    how="left",
    validate="one_to_one"
)

panel = panel.merge(
    S[secondary_keep],
    on=["transition", "listing_id"],
    how="left",
    validate="one_to_one"
)

if len(panel) != 491_140:
    raise RuntimeError("Outcome merge changed panel row count.")

if panel[PRIMARY_TARGET].isna().any():
    raise RuntimeError("Primary target contains missing values.")

if not set(panel[PRIMARY_TARGET].unique()).issubset({0, 1}):
    raise RuntimeError("Primary target is not binary.")

secondary_n = int(
    panel["price_resilience_delta"].notna().sum()
)

if secondary_n != 259_699:
    raise RuntimeError(
        f"Secondary usable N={secondary_n:,}; expected 259,699."
    )

print("\nOutcome merge: PASS")
print("Primary N:", panel[PRIMARY_TARGET].notna().sum())
print("Secondary usable N:", secondary_n)


# ----------------------------------------------------------------------
# 5. ADD EXPLICIT SEASONAL CYCLICAL FEATURES
#
# Only anchor month is used.
# No future month information is used.
# ----------------------------------------------------------------------

if "snapshot_date" not in panel.columns:
    raise RuntimeError("snapshot_date missing from backbone.")

panel["snapshot_date"] = pd.to_datetime(
    panel["snapshot_date"]
)

panel["anchor_month"] = panel["snapshot_date"].dt.month

panel["month_sin"] = np.sin(
    2 * np.pi * panel["anchor_month"] / 12.0
)

panel["month_cos"] = np.cos(
    2 * np.pi * panel["anchor_month"] / 12.0
)

if "forecast_horizon_days" not in panel.columns:
    raise RuntimeError(
        "forecast_horizon_days missing; "
        "do not reconstruct silently."
    )


# ----------------------------------------------------------------------
# 6. DEVELOPMENT / LOCKED-HOLDOUT FLAG
# ----------------------------------------------------------------------

panel["evaluation_role"] = np.where(
    panel["transition"] == "T10",
    "LOCKED_T10",
    "DEVELOPMENT"
)

DEV = panel["transition"] != "T10"

print("\nDevelopment rows:", DEV.sum())
print("Locked T10 rows :", (~DEV).sum())


# ----------------------------------------------------------------------
# 7. EXPLICIT LEAKAGE / NON-PREDICTOR RULES
# ----------------------------------------------------------------------

exact_blacklist = {
    # identifiers / evaluation metadata
    "listing_id",
    "transition",
    "snapshot_date",
    "evaluation_role",
    "anchor_month",

    # outcomes
    PRIMARY_TARGET,
    "secondary_price_available",
    "price_resilience_delta",

    # calendar provenance rather than substantive predictors
    "calendar_first_observed_date",
    "calendar_28th_observed_date",
    "calendar_observed_days_28d",
    "calendar_first_offset_days",
    "calendar_28th_span_days",
}

# Strings that strongly indicate future/outcome leakage.
# Conservative: flagged columns are quarantined for review.
leakage_patterns = [
    r"(^|_)future($|_)",
    r"(^|_)target($|_)",
    r"(^|_)outcome($|_)",
    r"resilience_delta",
    r"price_percentile",
    r"peer_n",
    r"peer_type",
    r"persistent_next",
    r"next_period",
    r"future_price",
    r"future_ta",
    r"future_room",
]

def matches_any(name, patterns):
    s = name.lower()
    return any(re.search(p, s) for p in patterns)


# ----------------------------------------------------------------------
# 8. FEATURE-FAMILY CLASSIFICATION
# ----------------------------------------------------------------------

calendar_exact = {
    "open_ratio_w1",
    "open_ratio_w2",
    "open_ratio_w3",
    "open_ratio_w4",
    "median_minimum_nights_w1",
    "median_minimum_nights_w2",
    "median_minimum_nights_w3",
    "median_minimum_nights_w4",
    "open_ratio_28d",
    "median_minimum_nights_28d",
}

construction_exact = {
    "construction_dwelling_3m_log",
    "construction_dwelling_12m_log",
    "construction_momentum_3m_vs_prior9m",
    "construction_multiunit_share_12m",
}

construction_robustness = {
    "construction_dwelling_6m_log"
}

rental_primary = {
    "rental_geomean_rent_log",
    "rental_geomean_rent_change_12m_log",
}

rental_robustness = {
    "rental_geomean_rent_change_3m_log"
}

seasonality_exact = {
    "month_sin",
    "month_cos",
    "forecast_horizon_days",
    "historical_seasonality_index_horizon",
}

# Semantic embedding naming is detected, not guessed as a fixed dimension.
def is_review_embedding(c):
    s = c.lower()
    return (
        re.match(r"^(review_)?emb(ed(ding)?)?[_-]?\d+$", s)
        or re.match(r"^review_embedding_\d+$", s)
        or re.match(r"^review_emb_\d+$", s)
        or re.match(r"^emb_\d+$", s)
    )

def classify_feature(c):

    s = c.lower()

    if c in exact_blacklist:
        return "NON_PREDICTOR"

    if matches_any(c, leakage_patterns):
        return "LEAKAGE_QUARANTINE"

    if c in calendar_exact:
        return "CALENDAR"

    if c in construction_exact:
        return "CONSTRUCTION_PRIMARY"

    if c in construction_robustness:
        return "CONSTRUCTION_ROBUSTNESS"

    if c in rental_primary:
        return "RENTAL_MARKET_PRIMARY"

    if c in rental_robustness:
        return "RENTAL_MARKET_ROBUSTNESS"

    if c in seasonality_exact:
        return "SEASONALITY"

    if is_review_embedding(c):
        return "REVIEW_SEMANTIC"

    # Review-history / availability metadata
    if (
        s.startswith("review_")
        or "review_count" in s
    ):
        return "REVIEW_HISTORY"

    # Geography
    if s in {
        "ta_code",
        "ta_name",
        "territorial_authority",
        "territorial_authority_code"
    }:
        return "GEOGRAPHY"

    # Known listing/platform variables are left for schema-based review.
    # We intentionally DO NOT auto-classify all remaining columns.
    return "UNCLASSIFIED"


# ----------------------------------------------------------------------
# 9. BUILD REGISTRY WITH DEVELOPMENT-ONLY STATISTICS
# ----------------------------------------------------------------------

rows = []

for c in panel.columns:

    family = classify_feature(c)

    dev_s = panel.loc[DEV, c]

    dtype = str(panel[c].dtype)

    n_missing_dev = int(dev_s.isna().sum())
    missing_rate_dev = float(dev_s.isna().mean())

    try:
        nunique_dev = int(dev_s.nunique(dropna=True))
    except Exception:
        nunique_dev = -1

    rows.append({
        "feature": c,
        "dtype": dtype,
        "family": family,
        "development_missing_n": n_missing_dev,
        "development_missing_rate": missing_rate_dev,
        "development_nunique": nunique_dev,
        "is_exact_blacklist": c in exact_blacklist,
        "is_leakage_pattern":
            matches_any(c, leakage_patterns),
    })

registry = pd.DataFrame(rows)


# ----------------------------------------------------------------------
# 10. DETECT EMBEDDING FAMILY
# ----------------------------------------------------------------------

embedding_cols = registry.loc[
    registry["family"] == "REVIEW_SEMANTIC",
    "feature"
].tolist()

print("\nDetected review embedding dimensions:", len(embedding_cols))

if len(embedding_cols) not in {0, 384}:
    print(
        "WARNING: expected 384 MiniLM dimensions, "
        f"detected {len(embedding_cols)}."
    )


# ----------------------------------------------------------------------
# 11. IDENTIFY AUTOMATICALLY SAFE CORE FEATURES
#
# Property/platform is intentionally NOT automatically frozen yet.
# We inspect remaining columns once.
# ----------------------------------------------------------------------

safe_families = {
    "CALENDAR",
    "CONSTRUCTION_PRIMARY",
    "RENTAL_MARKET_PRIMARY",
    "SEASONALITY",
    "REVIEW_SEMANTIC",
    "REVIEW_HISTORY",
    "GEOGRAPHY",
}

registry["candidate_core"] = (
    registry["family"].isin(safe_families)
    &
    ~registry["is_exact_blacklist"]
    &
    ~registry["is_leakage_pattern"]
)

# Robustness variables are not primary predictors.
registry["robustness_only"] = registry["family"].isin({
    "CONSTRUCTION_ROBUSTNESS",
    "RENTAL_MARKET_ROBUSTNESS"
})


# ----------------------------------------------------------------------
# 12. CONSTANT / ALL-MISSING CHECK — DEVELOPMENT ONLY
# ----------------------------------------------------------------------

registry["all_missing_development"] = (
    registry["development_missing_rate"] == 1.0
)

registry["constant_development"] = (
    registry["development_nunique"] <= 1
)

registry["candidate_core"] = (
    registry["candidate_core"]
    &
    ~registry["all_missing_development"]
    &
    ~registry["constant_development"]
)


# ----------------------------------------------------------------------
# 13. UNCLASSIFIED COLUMNS
# ----------------------------------------------------------------------

unclassified = registry.loc[
    registry["family"] == "UNCLASSIFIED",
    [
        "feature",
        "dtype",
        "development_missing_rate",
        "development_nunique"
    ]
].copy()

print("\n" + "=" * 110)
print("UNCLASSIFIED COLUMNS — MUST BE REVIEWED ONCE")
print("=" * 110)

pd.set_option("display.max_rows", 200)
display(unclassified)


# ----------------------------------------------------------------------
# 14. LEAKAGE QUARANTINE REPORT
# ----------------------------------------------------------------------

leakage = registry.loc[
    registry["family"].isin(
        ["LEAKAGE_QUARANTINE", "NON_PREDICTOR"]
    ),
    [
        "feature",
        "family",
        "dtype"
    ]
].copy()

print("\n" + "=" * 110)
print("LEAKAGE / NON-PREDICTOR QUARANTINE")
print("=" * 110)

display(leakage)


# ----------------------------------------------------------------------
# 15. FAMILY SUMMARY
# ----------------------------------------------------------------------

family_summary = (
    registry.groupby("family")
    .agg(
        n_columns=("feature", "size"),
        candidate_core=("candidate_core", "sum"),
        mean_dev_missing=("development_missing_rate", "mean"),
        max_dev_missing=("development_missing_rate", "max")
    )
    .reset_index()
    .sort_values("family")
)

print("\n" + "=" * 110)
print("FEATURE FAMILY SUMMARY")
print("=" * 110)

display(family_summary)


# ----------------------------------------------------------------------
# 16. PRIMARY OUTCOME DISTRIBUTION BY TRANSITION
# ----------------------------------------------------------------------

primary_qc = (
    panel.groupby("transition", sort=False)
    [PRIMARY_TARGET]
    .agg(["size", "sum", "mean"])
    .reset_index()
)

primary_qc.columns = [
    "transition",
    "n",
    "persistent_n",
    "persistent_rate"
]

primary_qc["absent_n"] = (
    primary_qc["n"] - primary_qc["persistent_n"]
)

print("\nPRIMARY OUTCOME BY TRANSITION")
display(primary_qc)


# ----------------------------------------------------------------------
# 17. SECONDARY OUTCOME DISTRIBUTION
# ----------------------------------------------------------------------

secondary_qc = (
    panel.groupby("transition", sort=False)
    .agg(
        n=("listing_id", "size"),
        secondary_n=("price_resilience_delta", "count"),
        secondary_mean=("price_resilience_delta", "mean"),
        secondary_sd=("price_resilience_delta", "std"),
        secondary_median=("price_resilience_delta", "median")
    )
    .reset_index()
)

print("\nSECONDARY OUTCOME BY TRANSITION")
display(secondary_qc)


# ----------------------------------------------------------------------
# 18. SAVE MODELLING PANEL
#
# IMPORTANT:
# This is the analytical master panel.
# It contains metadata and outcomes.
# Model code MUST use the final predictor whitelist, not panel.columns.
# ----------------------------------------------------------------------

PANEL_PATH = (
    MODEL_DIR /
    "modelling_panel_T1_T10.parquet"
)

panel.to_parquet(
    PANEL_PATH,
    index=False
)


# ----------------------------------------------------------------------
# 19. SAVE REGISTRY / REPORTS
# ----------------------------------------------------------------------

REGISTRY_PATH = MODEL_DIR / "feature_registry.csv"
registry.to_csv(REGISTRY_PATH, index=False)

FAMILY_PATH = MODEL_DIR / "feature_family_summary.csv"
family_summary.to_csv(FAMILY_PATH, index=False)

PRIMARY_QC_PATH = MODEL_DIR / "primary_outcome_by_transition.csv"
primary_qc.to_csv(PRIMARY_QC_PATH, index=False)

SECONDARY_QC_PATH = MODEL_DIR / "secondary_outcome_by_transition.csv"
secondary_qc.to_csv(SECONDARY_QC_PATH, index=False)

UNCLASSIFIED_PATH = MODEL_DIR / "unclassified_features.csv"
unclassified.to_csv(UNCLASSIFIED_PATH, index=False)

LEAKAGE_PATH = MODEL_DIR / "leakage_quarantine.csv"
leakage.to_csv(LEAKAGE_PATH, index=False)


# ----------------------------------------------------------------------
# 20. CANDIDATE CORE WHITELIST
#
# Not yet FINAL because property/platform columns require one explicit
# schema review rather than automatic guessing.
# ----------------------------------------------------------------------

candidate_core = registry.loc[
    registry["candidate_core"],
    "feature"
].tolist()

CANDIDATE_PATH = (
    MODEL_DIR /
    "predictor_whitelist_candidate.txt"
)

with open(CANDIDATE_PATH, "w", encoding="utf-8") as f:
    for c in candidate_core:
        f.write(c + "\n")


# ----------------------------------------------------------------------
# 21. DATASET FINGERPRINT
# ----------------------------------------------------------------------

fingerprint_text = (
    f"rows={len(panel)}|"
    f"cols={len(panel.columns)}|"
    f"primary={PRIMARY_TARGET}|"
    f"secondary_n={secondary_n}|"
    f"features={','.join(panel.columns)}"
)

fingerprint = hashlib.sha256(
    fingerprint_text.encode("utf-8")
).hexdigest()

manifest = {
    "panel_rows": int(len(panel)),
    "panel_columns": int(len(panel.columns)),
    "primary_target": PRIMARY_TARGET,
    "primary_nonmissing":
        int(panel[PRIMARY_TARGET].notna().sum()),
    "secondary_nonmissing": secondary_n,
    "development_transitions":
        [f"T{i}" for i in range(1, 10)],
    "locked_test_transition": "T10",
    "candidate_core_features":
        len(candidate_core),
    "review_embedding_dimensions":
        len(embedding_cols),
    "unclassified_columns":
        len(unclassified),
    "dataset_schema_sha256":
        fingerprint,
    "global_imputation_performed": False,
    "global_scaling_performed": False,
    "T10_used_for_feature_selection": False
}

MANIFEST_PATH = MODEL_DIR / "modelling_manifest.json"

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )


# ----------------------------------------------------------------------
# 22. RELOAD HARD QC
# ----------------------------------------------------------------------

check = pd.read_parquet(PANEL_PATH)

if len(check) != 491_140:
    raise RuntimeError("Saved panel row mismatch.")

if check[
    ["transition", "listing_id"]
].duplicated().any():
    raise RuntimeError("Saved panel contains duplicate keys.")

if int(check[PRIMARY_TARGET].notna().sum()) != 491_140:
    raise RuntimeError("Primary target corrupted after save.")

if int(
    check["price_resilience_delta"].notna().sum()
) != 259_699:
    raise RuntimeError("Secondary target corrupted after save.")


# ----------------------------------------------------------------------
# 23. FINAL DECISION GATES
# ----------------------------------------------------------------------

hard_leakage_in_candidates = registry.loc[
    registry["candidate_core"]
    &
    (
        registry["is_exact_blacklist"]
        |
        registry["is_leakage_pattern"]
    )
]

if len(hard_leakage_in_candidates) != 0:
    raise RuntimeError(
        "Leakage variable entered candidate predictor list."
    )

print("\n" + "=" * 110)
print("FINAL BLOCK H SUMMARY")
print("=" * 110)

print("Panel rows:", f"{len(panel):,}")
print("Panel columns:", len(panel.columns))
print("Primary usable:", f"{panel[PRIMARY_TARGET].notna().sum():,}")
print("Secondary usable:", f"{secondary_n:,}")
print("Review embedding dimensions detected:", len(embedding_cols))
print("Candidate core predictors:", len(candidate_core))
print("Unclassified columns requiring one review:", len(unclassified))
print("Global imputation performed: NO")
print("Global scaling performed: NO")
print("T10 used for feature selection: NO")

print("\nSaved:")
print(PANEL_PATH.relative_to(PROJECT_ROOT))
print(REGISTRY_PATH.relative_to(PROJECT_ROOT))
print(FAMILY_PATH.relative_to(PROJECT_ROOT))
print(UNCLASSIFIED_PATH.relative_to(PROJECT_ROOT))
print(LEAKAGE_PATH.relative_to(PROJECT_ROOT))
print(CANDIDATE_PATH.relative_to(PROJECT_ROOT))
print(MANIFEST_PATH.relative_to(PROJECT_ROOT))

print("\nDataset/schema fingerprint:")
print(fingerprint)

print("\n" + "=" * 110)

if len(unclassified) == 0:
    print("PRODUCTION BLOCK H STATUS: PASS — READY TO FREEZE FEATURE REGISTRY")
else:
    print(
        "PRODUCTION BLOCK H STATUS: CONDITIONAL PASS\n"
        "DATA PANEL IS VALID; UNCLASSIFIED PROPERTY/PLATFORM COLUMNS "
        "MUST BE REVIEWED ONCE BEFORE FINAL FEATURE FREEZE."
    )

print("=" * 110)

In [ ]:
# ======================================================================
# PRODUCTION BLOCK H2
# FINAL FEATURE CLASSIFICATION + PREDICTOR FREEZE
# ======================================================================

from pathlib import Path
import pandas as pd
import json

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)
MODEL_DIR = PROJECT_ROOT / "02_intermediate" / "modelling"

REGISTRY_PATH = MODEL_DIR / "feature_registry.csv"

registry = pd.read_csv(REGISTRY_PATH)

# ----------------------------------------------------------------------
# 1. FINAL PROPERTY / PLATFORM FEATURES
# ----------------------------------------------------------------------

property_features = {
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities_count",
}

platform_features = {
    "host_is_superhost",
    "host_listings_count",
    "host_has_profile_pic",
    "host_identity_verified",

    "minimum_nights",
    "maximum_nights",

    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "reviews_per_month",

    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",

    "listing_review_age_days",
    "days_since_last_review",
}

# ----------------------------------------------------------------------
# 2. EXPLICIT FINAL EXCLUSIONS
# ----------------------------------------------------------------------

final_exclude = {
    "last_scraped",
    "latitude",
    "longitude",
    "amenities",
    "price",
    "price_raw",
    "first_review",
    "last_review",
    "ta_name_ascii",
    "latest_available_consent_period",
    "no_dwelling_12m",
    "eligible_rental_month",
    "rental_current_available",
    "rental_change_12m_available",
    "anchor_date",
    "seasonality_source",
}

# ----------------------------------------------------------------------
# 3. ASSIGN PROPERTY / PLATFORM
# ----------------------------------------------------------------------

registry.loc[
    registry["feature"].isin(property_features),
    "family"
] = "PROPERTY"

registry.loc[
    registry["feature"].isin(platform_features),
    "family"
] = "PLATFORM"

registry.loc[
    registry["feature"].isin(final_exclude),
    "family"
] = "FINAL_EXCLUDE"

# ----------------------------------------------------------------------
# 4. GEOGRAPHY — KEEP ONE REPRESENTATION ONLY
# ----------------------------------------------------------------------

geo = registry.loc[
    registry["family"] == "GEOGRAPHY",
    "feature"
].tolist()

print("Detected GEOGRAPHY variables:", geo)

# Prefer TA code because it is stable and compact.
ta_code_candidates = [
    c for c in geo
    if "code" in c.lower()
]

if len(ta_code_candidates) == 1:
    KEEP_GEO = ta_code_candidates[0]

elif len(geo) == 1:
    KEEP_GEO = geo[0]

else:
    raise RuntimeError(
        "Geography representation requires inspection. "
        f"Detected: {geo}"
    )

for c in geo:
    if c != KEEP_GEO:
        registry.loc[
            registry["feature"] == c,
            "family"
        ] = "FINAL_EXCLUDE"

print("Final geography predictor:", KEEP_GEO)

# ----------------------------------------------------------------------
# 5. FINAL PRIMARY FEATURE FAMILIES
# ----------------------------------------------------------------------

PRIMARY_FAMILIES = {
    "PROPERTY",
    "PLATFORM",
    "GEOGRAPHY",
    "CALENDAR",
    "CONSTRUCTION_PRIMARY",
    "RENTAL_MARKET_PRIMARY",
    "SEASONALITY",
    "REVIEW_HISTORY",
    "REVIEW_SEMANTIC",
}

registry["final_primary_predictor"] = (
    registry["family"].isin(PRIMARY_FAMILIES)
    &
    ~registry["all_missing_development"].astype(bool)
    &
    ~registry["constant_development"].astype(bool)
)

# ----------------------------------------------------------------------
# 6. HARD LEAKAGE GATE
# ----------------------------------------------------------------------

forbidden = {
    "listing_id",
    "transition",
    "snapshot_date",
    "future_date",
    "next_period_persistence",
    "secondary_price_available",
    "price_resilience_delta",
    "evaluation_role",
    "anchor_month",
}

final_features = registry.loc[
    registry["final_primary_predictor"],
    "feature"
].tolist()

bad = sorted(set(final_features) & forbidden)

if bad:
    raise RuntimeError(
        f"FORBIDDEN VARIABLES IN FINAL FEATURES: {bad}"
    )

# Price must NOT enter predictor matrix.
if "price" in final_features or "price_raw" in final_features:
    raise RuntimeError("Raw price entered final predictor set.")

# ----------------------------------------------------------------------
# 7. CHECK UNCLASSIFIED
# ----------------------------------------------------------------------

remaining_unknown = registry.loc[
    registry["family"] == "UNCLASSIFIED",
    "feature"
].tolist()

if remaining_unknown:
    print("\nRemaining unclassified:")
    print(remaining_unknown)
    raise RuntimeError(
        "Some variables remain unclassified. Do not model yet."
    )

# ----------------------------------------------------------------------
# 8. FAMILY COUNTS
# ----------------------------------------------------------------------

summary = (
    registry.loc[registry["final_primary_predictor"]]
    .groupby("family")
    .size()
    .rename("n_features")
    .reset_index()
)

print("\nFINAL PRIMARY FEATURE FAMILIES")
display(summary)

print("\nTotal final predictors:", len(final_features))

# ----------------------------------------------------------------------
# 9. SAVE FINAL FREEZE
# ----------------------------------------------------------------------

registry.to_csv(
    MODEL_DIR / "feature_registry_FINAL_FROZEN.csv",
    index=False
)

with open(
    MODEL_DIR / "predictor_whitelist_FINAL_FROZEN.txt",
    "w",
    encoding="utf-8"
) as f:
    for c in final_features:
        f.write(c + "\n")

with open(
    MODEL_DIR / "feature_groups_FINAL_FROZEN.json",
    "w",
    encoding="utf-8"
) as f:

    groups = {}

    for family in sorted(PRIMARY_FAMILIES):
        groups[family] = registry.loc[
            registry["final_primary_predictor"]
            & (registry["family"] == family),
            "feature"
        ].tolist()

    json.dump(groups, f, indent=2)

# ----------------------------------------------------------------------
# 10. MODEL CONTRACT
# ----------------------------------------------------------------------

contract = {
    "development_transitions": [
        "T1","T2","T3","T4","T5","T6","T7","T8","T9"
    ],

    "locked_final_test": "T10",

    "primary_target":
        "next_period_persistence",

    "secondary_target":
        "price_resilience_delta",

    "global_imputation_allowed": False,
    "global_scaling_allowed": False,

    "preprocessing_rule":
        "Fit imputation, encoding and scaling on each training fold only.",

    "t10_rule":
        "T10 must not inform feature design, preprocessing, "
        "hyperparameters, architecture or thresholds.",

    "raw_price_predictor": False,

    "raw_coordinates_predictor": False,

    "raw_amenities_predictor": False,

    "final_predictor_count":
        len(final_features),
}

with open(
    MODEL_DIR / "MODEL_DATA_CONTRACT_FROZEN.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(contract, f, indent=2)

print("\n" + "=" * 100)
print("PRODUCTION BLOCK H2 STATUS: PASS")
print("=" * 100)

print("Final predictors:", len(final_features))
print("Final geography:", KEEP_GEO)
print("Remaining unclassified: 0")
print("Raw price predictor: NO")
print("Raw coordinates: NO")
print("Global imputation: NO")
print("T10 used for design: NO")

print("\nSaved:")
print("feature_registry_FINAL_FROZEN.csv")
print("predictor_whitelist_FINAL_FROZEN.txt")
print("feature_groups_FINAL_FROZEN.json")
print("MODEL_DATA_CONTRACT_FROZEN.json")

print("=" * 100)

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK I
# DEVELOPMENT MODELLING PACKAGE — PRIMARY OUTCOME
#
# Rolling-origin development:
#   Train T1-T3 -> Test T4
#   Train T1-T4 -> Test T5
#   Train T1-T5 -> Test T6
#   Train T1-T6 -> Test T7
#   Train T1-T7 -> Test T8
#   Train T1-T8 -> Test T9
#
# T10 IS NEVER LOADED INTO A MODEL FIT OR EVALUATION.
#
# Models:
#   M0 prevalence baseline
#   M1 logistic regression — structured
#   M2 XGBoost — structured
#   M3 XGBoost — full multimodal
#
# Primary evaluation event:
#   disappearance = 1 - next_period_persistence
#
# Saves:
#   rolling metrics
#   macro / weighted summaries
#   predictions
#   calibration bins
#   feature/model metadata
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
import warnings
import joblib

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    balanced_accuracy_score,
    log_loss
)

from xgboost import XGBClassifier

# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

MODEL_DATA_DIR = PROJECT_ROOT / "02_intermediate" / "modelling"
RESULT_DIR = PROJECT_ROOT / "05_results" / "development_modelling"
PRED_DIR = RESULT_DIR / "predictions"
MODEL_SAVE_DIR = RESULT_DIR / "models"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

PANEL_PATH = MODEL_DATA_DIR / "modelling_panel_T1_T10.parquet"

WHITELIST_PATH = (
    MODEL_DATA_DIR /
    "predictor_whitelist_FINAL_FROZEN.txt"
)

GROUP_PATH = (
    MODEL_DATA_DIR /
    "feature_groups_FINAL_FROZEN.json"
)

CONTRACT_PATH = (
    MODEL_DATA_DIR /
    "MODEL_DATA_CONTRACT_FROZEN.json"
)

for p in [
    PANEL_PATH,
    WHITELIST_PATH,
    GROUP_PATH,
    CONTRACT_PATH
]:
    if not p.exists():
        raise FileNotFoundError(p)

print("=" * 110)
print("PRODUCTION BLOCK I")
print("FULL DEVELOPMENT MODELLING — T4 TO T9 ONLY")
print("=" * 110)


# --------------------------------------------------------------------------------------------------
# 1. LOAD FROZEN CONTRACT
# --------------------------------------------------------------------------------------------------

with open(
    WHITELIST_PATH,
    "r",
    encoding="utf-8"
) as f:
    FULL_FEATURES = [
        x.strip()
        for x in f.readlines()
        if x.strip()
    ]

with open(
    GROUP_PATH,
    "r",
    encoding="utf-8"
) as f:
    FEATURE_GROUPS = json.load(f)

with open(
    CONTRACT_PATH,
    "r",
    encoding="utf-8"
) as f:
    CONTRACT = json.load(f)

print("Frozen predictors:", len(FULL_FEATURES))

if len(FULL_FEATURES) != 437:
    raise RuntimeError(
        f"Expected 437 frozen predictors; found {len(FULL_FEATURES)}"
    )

if CONTRACT["locked_final_test"] != "T10":
    raise RuntimeError("Frozen contract does not identify T10 as locked test.")


# --------------------------------------------------------------------------------------------------
# 2. LOAD PANEL
# --------------------------------------------------------------------------------------------------

panel = pd.read_parquet(PANEL_PATH)

print("Panel:", panel.shape)

if len(panel) != 491_140:
    raise RuntimeError("Unexpected modelling panel row count.")

if "next_period_persistence" not in panel.columns:
    raise RuntimeError("Primary outcome missing.")

missing_features = [
    c for c in FULL_FEATURES
    if c not in panel.columns
]

if missing_features:
    raise RuntimeError(
        f"Frozen features missing from panel: {missing_features}"
    )


# --------------------------------------------------------------------------------------------------
# 3. ABSOLUTE T10 FIREWALL
# --------------------------------------------------------------------------------------------------

development = panel.loc[
    panel["transition"].isin(
        ["T1","T2","T3","T4","T5","T6","T7","T8","T9"]
    )
].copy()

if "T10" in development["transition"].unique():
    raise RuntimeError("T10 LEAKAGE.")

print("Development rows:", f"{len(development):,}")
print(
    "Development transitions:",
    development["transition"].unique().tolist()
)

# Define disappearance as evaluation-positive event.
development["y_disappear"] = (
    1 - development["next_period_persistence"].astype(int)
)

print(
    "Development disappearance rate:",
    development["y_disappear"].mean()
)


# --------------------------------------------------------------------------------------------------
# 4. DEFINE STRUCTURED-ONLY FEATURES
# --------------------------------------------------------------------------------------------------

# Review semantic embeddings excluded.
# Everything else in frozen primary predictor list retained.

review_semantic = set(
    FEATURE_GROUPS.get("REVIEW_SEMANTIC", [])
)

STRUCTURED_FEATURES = [
    c for c in FULL_FEATURES
    if c not in review_semantic
]

print("Structured predictors:", len(STRUCTURED_FEATURES))
print("Semantic predictors  :", len(review_semantic))
print("Full predictors      :", len(FULL_FEATURES))

if len(review_semantic) != 384:
    raise RuntimeError(
        f"Expected 384 review semantic features; found {len(review_semantic)}"
    )


# --------------------------------------------------------------------------------------------------
# 5. ROLLING FOLDS
# --------------------------------------------------------------------------------------------------

FOLDS = [
    {
        "fold": "DEV_T4",
        "train": ["T1","T2","T3"],
        "test": "T4"
    },
    {
        "fold": "DEV_T5",
        "train": ["T1","T2","T3","T4"],
        "test": "T5"
    },
    {
        "fold": "DEV_T6",
        "train": ["T1","T2","T3","T4","T5"],
        "test": "T6"
    },
    {
        "fold": "DEV_T7",
        "train": ["T1","T2","T3","T4","T5","T6"],
        "test": "T7"
    },
    {
        "fold": "DEV_T8",
        "train": ["T1","T2","T3","T4","T5","T6","T7"],
        "test": "T8"
    },
    {
        "fold": "DEV_T9",
        "train": ["T1","T2","T3","T4","T5","T6","T7","T8"],
        "test": "T9"
    },
]


# --------------------------------------------------------------------------------------------------
# 6. PREPROCESSING
# --------------------------------------------------------------------------------------------------

def split_feature_types(df, feature_list):

    categorical = []
    numerical = []

    for c in feature_list:

        if (
            pd.api.types.is_object_dtype(df[c])
            or
            isinstance(df[c].dtype, pd.CategoricalDtype)
            or
            pd.api.types.is_string_dtype(df[c])
        ):
            categorical.append(c)
        else:
            numerical.append(c)

    return numerical, categorical


def make_preprocessor(train_df, feature_list, scale_numeric=False):

    num_cols, cat_cols = split_feature_types(
        train_df,
        feature_list
    )

    if scale_numeric:

        numeric_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

    else:

        numeric_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            )
        ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols)
        ],
        remainder="drop",
        sparse_threshold=0.3
    )

    return preprocessor, num_cols, cat_cols


# --------------------------------------------------------------------------------------------------
# 7. METRICS
# --------------------------------------------------------------------------------------------------

def classification_metrics(
    y_true,
    p_disappear,
    threshold=0.5
):

    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p_disappear).astype(float)

    p = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )

    pred = (p >= threshold).astype(int)

    return {
        "roc_auc":
            roc_auc_score(y_true, p),

        "pr_auc_disappearance":
            average_precision_score(y_true, p),

        "brier":
            brier_score_loss(y_true, p),

        "balanced_accuracy_0p5":
            balanced_accuracy_score(y_true, pred),

        "log_loss":
            log_loss(y_true, p),

        "observed_disappearance_rate":
            float(np.mean(y_true)),

        "predicted_disappearance_rate":
            float(np.mean(p))
    }


# --------------------------------------------------------------------------------------------------
# 8. CALIBRATION TABLE
# --------------------------------------------------------------------------------------------------

def calibration_table(
    y_true,
    p,
    n_bins=10
):

    d = pd.DataFrame({
        "y": np.asarray(y_true),
        "p": np.asarray(p)
    })

    # Rank-based bins are more stable under strong imbalance.
    try:
        d["bin"] = pd.qcut(
            d["p"],
            q=n_bins,
            duplicates="drop"
        )
    except Exception:
        d["bin"] = pd.cut(
            d["p"],
            bins=n_bins,
            duplicates="drop"
        )

    out = (
        d.groupby(
            "bin",
            observed=True
        )
        .agg(
            n=("y", "size"),
            mean_predicted=("p", "mean"),
            observed_rate=("y", "mean")
        )
        .reset_index(drop=True)
    )

    out["abs_calibration_error"] = (
        out["mean_predicted"]
        -
        out["observed_rate"]
    ).abs()

    return out


# --------------------------------------------------------------------------------------------------
# 9. MODEL CONFIGURATION
#
# Fixed development specification.
# No T10 tuning.
# --------------------------------------------------------------------------------------------------

XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 5,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,

    "objective": "binary:logistic",
    "eval_metric": "logloss",

    "tree_method": "hist",
    "n_jobs": -1,

    "random_state": 2026
}

LOGISTIC_PARAMS = {
    "solver": "saga",
    "penalty": "l2",
    "C": 1.0,
    "max_iter": 300,
    "n_jobs": -1,
    "random_state": 2026
}


# --------------------------------------------------------------------------------------------------
# 10. STORAGE
# --------------------------------------------------------------------------------------------------

all_metrics = []
all_predictions = []
all_calibration = []
runtime_records = []


# --------------------------------------------------------------------------------------------------
# 11. ROLLING DEVELOPMENT LOOP
# --------------------------------------------------------------------------------------------------

for fold_spec in FOLDS:

    fold = fold_spec["fold"]
    train_transitions = fold_spec["train"]
    test_transition = fold_spec["test"]

    print("\n")
    print("=" * 110)
    print(f"{fold}")
    print(
        f"TRAIN {'+'.join(train_transitions)} "
        f"-> TEST {test_transition}"
    )
    print("=" * 110)

    train_df = development.loc[
        development["transition"].isin(
            train_transitions
        )
    ].copy()

    test_df = development.loc[
        development["transition"] == test_transition
    ].copy()

    y_train = train_df[
        "y_disappear"
    ].astype(int).values

    y_test = test_df[
        "y_disappear"
    ].astype(int).values

    print("Train N:", f"{len(train_df):,}")
    print("Test N :", f"{len(test_df):,}")

    print(
        "Train disappearance:",
        round(y_train.mean(), 5)
    )

    print(
        "Test disappearance :",
        round(y_test.mean(), 5)
    )


    # ==============================================================================================
    # M0 — TRAINING PREVALENCE BASELINE
    # ==============================================================================================

    start = time.time()

    train_prevalence = float(
        y_train.mean()
    )

    p0 = np.full(
        len(test_df),
        train_prevalence,
        dtype=float
    )

    m = classification_metrics(
        y_test,
        p0
    )

    m.update({
        "fold": fold,
        "test_transition": test_transition,
        "model": "M0_PREVALENCE",
        "feature_set": "NONE",
        "train_n": len(train_df),
        "test_n": len(test_df),
        "runtime_sec": time.time() - start
    })

    all_metrics.append(m)

    pred0 = pd.DataFrame({
        "fold": fold,
        "transition": test_transition,
        "listing_id": test_df["listing_id"].values,
        "y_disappear": y_test,
        "p_disappear": p0,
        "model": "M0_PREVALENCE"
    })

    all_predictions.append(pred0)

    cal = calibration_table(
        y_test,
        p0
    )

    cal["fold"] = fold
    cal["model"] = "M0_PREVALENCE"

    all_calibration.append(cal)


    # ==============================================================================================
    # M1 — LOGISTIC REGRESSION, STRUCTURED
    # ==============================================================================================

    print("\nM1 Logistic structured...")

    start = time.time()

    prep_log, _, _ = make_preprocessor(
        train_df,
        STRUCTURED_FEATURES,
        scale_numeric=True
    )

    logistic = LogisticRegression(
        **LOGISTIC_PARAMS
    )

    pipe_log = Pipeline([
        ("preprocess", prep_log),
        ("model", logistic)
    ])

    pipe_log.fit(
        train_df[STRUCTURED_FEATURES],
        y_train
    )

    p1 = pipe_log.predict_proba(
        test_df[STRUCTURED_FEATURES]
    )[:, 1]

    m = classification_metrics(
        y_test,
        p1
    )

    m.update({
        "fold": fold,
        "test_transition": test_transition,
        "model": "M1_LOGISTIC_STRUCTURED",
        "feature_set": "STRUCTURED",
        "train_n": len(train_df),
        "test_n": len(test_df),
        "runtime_sec": time.time() - start
    })

    all_metrics.append(m)

    pred1 = pd.DataFrame({
        "fold": fold,
        "transition": test_transition,
        "listing_id": test_df["listing_id"].values,
        "y_disappear": y_test,
        "p_disappear": p1,
        "model": "M1_LOGISTIC_STRUCTURED"
    })

    all_predictions.append(pred1)

    cal = calibration_table(
        y_test,
        p1
    )

    cal["fold"] = fold
    cal["model"] = "M1_LOGISTIC_STRUCTURED"

    all_calibration.append(cal)

    # Save last development logistic pipeline as checkpoint.
    joblib.dump(
        pipe_log,
        MODEL_SAVE_DIR /
        f"{fold}_M1_logistic_structured.joblib"
    )


    # ==============================================================================================
    # M2 — XGBOOST STRUCTURED
    # ==============================================================================================

    print("M2 XGBoost structured...")

    start = time.time()

    prep_xgb_s, _, _ = make_preprocessor(
        train_df,
        STRUCTURED_FEATURES,
        scale_numeric=False
    )

    xgb_s = XGBClassifier(
        **XGB_PARAMS
    )

    pipe_xgb_s = Pipeline([
        ("preprocess", prep_xgb_s),
        ("model", xgb_s)
    ])

    pipe_xgb_s.fit(
        train_df[STRUCTURED_FEATURES],
        y_train
    )

    p2 = pipe_xgb_s.predict_proba(
        test_df[STRUCTURED_FEATURES]
    )[:, 1]

    m = classification_metrics(
        y_test,
        p2
    )

    m.update({
        "fold": fold,
        "test_transition": test_transition,
        "model": "M2_XGB_STRUCTURED",
        "feature_set": "STRUCTURED",
        "train_n": len(train_df),
        "test_n": len(test_df),
        "runtime_sec": time.time() - start
    })

    all_metrics.append(m)

    pred2 = pd.DataFrame({
        "fold": fold,
        "transition": test_transition,
        "listing_id": test_df["listing_id"].values,
        "y_disappear": y_test,
        "p_disappear": p2,
        "model": "M2_XGB_STRUCTURED"
    })

    all_predictions.append(pred2)

    cal = calibration_table(
        y_test,
        p2
    )

    cal["fold"] = fold
    cal["model"] = "M2_XGB_STRUCTURED"

    all_calibration.append(cal)

    joblib.dump(
        pipe_xgb_s,
        MODEL_SAVE_DIR /
        f"{fold}_M2_xgb_structured.joblib"
    )


    # ==============================================================================================
    # M3 — XGBOOST FULL MULTIMODAL
    # ==============================================================================================

    print("M3 XGBoost full multimodal...")

    start = time.time()

    prep_xgb_f, _, _ = make_preprocessor(
        train_df,
        FULL_FEATURES,
        scale_numeric=False
    )

    xgb_f = XGBClassifier(
        **XGB_PARAMS
    )

    pipe_xgb_f = Pipeline([
        ("preprocess", prep_xgb_f),
        ("model", xgb_f)
    ])

    pipe_xgb_f.fit(
        train_df[FULL_FEATURES],
        y_train
    )

    p3 = pipe_xgb_f.predict_proba(
        test_df[FULL_FEATURES]
    )[:, 1]

    m = classification_metrics(
        y_test,
        p3
    )

    m.update({
        "fold": fold,
        "test_transition": test_transition,
        "model": "M3_XGB_FULL",
        "feature_set": "FULL",
        "train_n": len(train_df),
        "test_n": len(test_df),
        "runtime_sec": time.time() - start
    })

    all_metrics.append(m)

    pred3 = pd.DataFrame({
        "fold": fold,
        "transition": test_transition,
        "listing_id": test_df["listing_id"].values,
        "y_disappear": y_test,
        "p_disappear": p3,
        "model": "M3_XGB_FULL"
    })

    all_predictions.append(pred3)

    cal = calibration_table(
        y_test,
        p3
    )

    cal["fold"] = fold
    cal["model"] = "M3_XGB_FULL"

    all_calibration.append(cal)

    joblib.dump(
        pipe_xgb_f,
        MODEL_SAVE_DIR /
        f"{fold}_M3_xgb_full.joblib"
    )

    print("Fold complete.")


# --------------------------------------------------------------------------------------------------
# 12. COMBINE RESULTS
# --------------------------------------------------------------------------------------------------

metrics = pd.DataFrame(
    all_metrics
)

predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

calibration = pd.concat(
    all_calibration,
    ignore_index=True
)


# --------------------------------------------------------------------------------------------------
# 13. MACRO SUMMARY
# --------------------------------------------------------------------------------------------------

metric_cols = [
    "roc_auc",
    "pr_auc_disappearance",
    "brier",
    "balanced_accuracy_0p5",
    "log_loss"
]

macro = (
    metrics
    .groupby(
        ["model", "feature_set"]
    )[metric_cols]
    .mean()
    .reset_index()
)

macro = macro.rename(
    columns={
        c: f"macro_{c}"
        for c in metric_cols
    }
)


# --------------------------------------------------------------------------------------------------
# 14. TEST-N WEIGHTED SUMMARY
# --------------------------------------------------------------------------------------------------

weighted_rows = []

for (model, feature_set), g in metrics.groupby(
    ["model", "feature_set"]
):

    weights = g["test_n"].values.astype(float)

    row = {
        "model": model,
        "feature_set": feature_set
    }

    for c in metric_cols:
        row[
            f"weighted_{c}"
        ] = np.average(
            g[c],
            weights=weights
        )

    weighted_rows.append(row)

weighted = pd.DataFrame(
    weighted_rows
)


# --------------------------------------------------------------------------------------------------
# 15. COMBINED SUMMARY
# --------------------------------------------------------------------------------------------------

summary = macro.merge(
    weighted,
    on=["model", "feature_set"],
    how="outer"
)

summary = summary.sort_values(
    "macro_pr_auc_disappearance",
    ascending=False
)


# --------------------------------------------------------------------------------------------------
# 16. INCREMENTAL FULL-vs-STRUCTURED COMPARISON
# --------------------------------------------------------------------------------------------------

pivot = metrics.pivot(
    index="test_transition",
    columns="model",
    values=[
        "roc_auc",
        "pr_auc_disappearance",
        "brier"
    ]
)

incremental_rows = []

for transition in [
    "T4","T5","T6","T7","T8","T9"
]:

    g = metrics.loc[
        metrics["test_transition"] == transition
    ].set_index("model")

    if (
        "M2_XGB_STRUCTURED" in g.index
        and
        "M3_XGB_FULL" in g.index
    ):

        incremental_rows.append({
            "transition": transition,

            "delta_roc_auc_full_minus_structured":
                g.loc["M3_XGB_FULL", "roc_auc"]
                -
                g.loc["M2_XGB_STRUCTURED", "roc_auc"],

            "delta_pr_auc_full_minus_structured":
                g.loc[
                    "M3_XGB_FULL",
                    "pr_auc_disappearance"
                ]
                -
                g.loc[
                    "M2_XGB_STRUCTURED",
                    "pr_auc_disappearance"
                ],

            "delta_brier_full_minus_structured":
                g.loc["M3_XGB_FULL", "brier"]
                -
                g.loc["M2_XGB_STRUCTURED", "brier"]
        })

incremental = pd.DataFrame(
    incremental_rows
)


# --------------------------------------------------------------------------------------------------
# 17. SAVE
# --------------------------------------------------------------------------------------------------

metrics.to_csv(
    RESULT_DIR /
    "primary_development_metrics_by_window.csv",
    index=False
)

summary.to_csv(
    RESULT_DIR /
    "primary_development_metrics_summary.csv",
    index=False
)

incremental.to_csv(
    RESULT_DIR /
    "full_vs_structured_incremental.csv",
    index=False
)

calibration.to_csv(
    RESULT_DIR /
    "primary_development_calibration.csv",
    index=False
)

predictions.to_parquet(
    PRED_DIR /
    "primary_development_predictions_T4_T9.parquet",
    index=False
)


# --------------------------------------------------------------------------------------------------
# 18. DEVELOPMENT DECISION REPORT
# --------------------------------------------------------------------------------------------------

decision_report = {
    "T10_evaluated": False,

    "development_test_windows": [
        "T4","T5","T6","T7","T8","T9"
    ],

    "models": [
        "M0_PREVALENCE",
        "M1_LOGISTIC_STRUCTURED",
        "M2_XGB_STRUCTURED",
        "M3_XGB_FULL"
    ],

    "full_predictors":
        len(FULL_FEATURES),

    "structured_predictors":
        len(STRUCTURED_FEATURES),

    "semantic_predictors":
        len(review_semantic),

    "positive_evaluation_event":
        "next-period disappearance",

    "primary_scientific_outcome":
        "next-period platform persistence",

    "threshold":
        0.5,

    "hyperparameters_selected_using_T10":
        False,

    "global_imputation":
        False,

    "preprocessing":
        "fit separately within each rolling training fold"
}

with open(
    RESULT_DIR /
    "development_modelling_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision_report,
        f,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# 19. FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 110)
print("PRIMARY DEVELOPMENT METRICS — BY WINDOW")
print("=" * 110)

display(
    metrics[
        [
            "test_transition",
            "model",
            "test_n",
            "observed_disappearance_rate",
            "roc_auc",
            "pr_auc_disappearance",
            "brier",
            "balanced_accuracy_0p5",
            "predicted_disappearance_rate",
            "runtime_sec"
        ]
    ].sort_values(
        ["test_transition", "model"]
    )
)

print("\n")
print("=" * 110)
print("MACRO + WEIGHTED DEVELOPMENT SUMMARY")
print("=" * 110)

display(summary)

print("\n")
print("=" * 110)
print("FULL vs STRUCTURED XGBOOST")
print("=" * 110)

display(incremental)

print("\n")
print("=" * 110)
print("PRODUCTION BLOCK I STATUS: PASS")
print("=" * 110)

print("T10 evaluated: NO")
print("Development windows evaluated: T4-T9")
print("Frozen full predictors:", len(FULL_FEATURES))
print("Structured predictors:", len(STRUCTURED_FEATURES))
print("Semantic predictors:", len(review_semantic))

print("\nSaved:")
print(
    RESULT_DIR /
    "primary_development_metrics_by_window.csv"
)
print(
    RESULT_DIR /
    "primary_development_metrics_summary.csv"
)
print(
    RESULT_DIR /
    "full_vs_structured_incremental.csv"
)
print(
    RESULT_DIR /
    "primary_development_calibration.csv"
)
print(
    PRED_DIR /
    "primary_development_predictions_T4_T9.parquet"
)

print("\nIMPORTANT: DO NOT RUN T10 YET.")
print("=" * 110)

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK I-B
# DEVELOPMENT ABLATION + COMPACT MLP + CALIBRATION + THRESHOLD
# + MODEL FREEZE RECOMMENDATION
#
# STRICT RULES
# --------------------------------------------------------------------------------------------------
# - Development only: T1-T9
# - Evaluation windows: T4-T9
# - T10 is NEVER evaluated or used for model/threshold/calibration selection
# - Frozen 437-feature registry is not modified
# - No global imputation/scaling
# - Preprocessing is fit within each rolling training fold
# - Checkpoint at MODEL x FOLD level
# - Hyperparameters are fixed for this block; no T10 tuning
#
# Scientific outcome:
#   next-period platform persistence
#
# Classification orientation for evaluation:
#   disappearance = 1 - persistence
#
# Main model-selection metrics:
#   1. PR-AUC for disappearance
#   2. ROC-AUC
#   3. Brier score / calibration
#   4. temporal stability
#
# Threshold:
#   selected ONLY from pooled development predictions T4-T9
#   using maximum Youden J = sensitivity + specificity - 1
#
# IMPORTANT:
#   T10 rows are filtered out immediately after loading.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
import warnings
import gc
import hashlib
import joblib

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    balanced_accuracy_score,
    log_loss,
    confusion_matrix,
    roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier


# ==================================================================================================
# 0. PROJECT PATHS
# ==================================================================================================

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

MODEL_DATA_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "modelling"
)

DEV_RESULT_DIR = (
    PROJECT_ROOT /
    "05_results" /
    "development_modelling"
)

IB_DIR = (
    PROJECT_ROOT /
    "05_results" /
    "development_modelling_IB"
)

CHECKPOINT_DIR = IB_DIR / "checkpoints"
PRED_DIR = IB_DIR / "predictions"
MODEL_DIR = IB_DIR / "models"
TABLE_DIR = IB_DIR / "tables"
FREEZE_DIR = IB_DIR / "freeze"

for d in [
    IB_DIR,
    CHECKPOINT_DIR,
    PRED_DIR,
    MODEL_DIR,
    TABLE_DIR,
    FREEZE_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


PANEL_PATH = (
    MODEL_DATA_DIR /
    "modelling_panel_T1_T10.parquet"
)

WHITELIST_PATH = (
    MODEL_DATA_DIR /
    "predictor_whitelist_FINAL_FROZEN.txt"
)

GROUP_PATH = (
    MODEL_DATA_DIR /
    "feature_groups_FINAL_FROZEN.json"
)

CONTRACT_PATH = (
    MODEL_DATA_DIR /
    "MODEL_DATA_CONTRACT_FROZEN.json"
)

BLOCK_I_PRED_PATH = (
    DEV_RESULT_DIR /
    "predictions" /
    "primary_development_predictions_T4_T9.parquet"
)

BLOCK_I_METRIC_PATH = (
    DEV_RESULT_DIR /
    "primary_development_metrics_by_window.csv"
)


required_files = [
    PANEL_PATH,
    WHITELIST_PATH,
    GROUP_PATH,
    CONTRACT_PATH,
    BLOCK_I_PRED_PATH,
    BLOCK_I_METRIC_PATH
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(
            f"Required input missing:\n{p}"
        )


print("=" * 120)
print("PRODUCTION BLOCK I-B")
print("DEVELOPMENT ABLATION + MLP + CALIBRATION + THRESHOLD + FREEZE RECOMMENDATION")
print("=" * 120)


# ==================================================================================================
# 1. LOAD FROZEN FEATURE CONTRACT
# ==================================================================================================

with open(
    WHITELIST_PATH,
    "r",
    encoding="utf-8"
) as f:
    FULL_FEATURES = [
        x.strip()
        for x in f
        if x.strip()
    ]

with open(
    GROUP_PATH,
    "r",
    encoding="utf-8"
) as f:
    GROUPS = json.load(f)

with open(
    CONTRACT_PATH,
    "r",
    encoding="utf-8"
) as f:
    CONTRACT = json.load(f)


if len(FULL_FEATURES) != 437:
    raise RuntimeError(
        f"Expected 437 frozen predictors; found {len(FULL_FEATURES)}."
    )

if CONTRACT.get(
    "locked_final_test"
) != "T10":
    raise RuntimeError(
        "Frozen contract does not identify T10 as locked test."
    )


# ==================================================================================================
# 2. FEATURE GROUPS
# ==================================================================================================

def group(name):
    return list(
        GROUPS.get(name, [])
    )


PROPERTY = group("PROPERTY")
PLATFORM = group("PLATFORM")
GEOGRAPHY = group("GEOGRAPHY")

RENTAL = group(
    "RENTAL_MARKET_PRIMARY"
)

CONSTRUCTION = group(
    "CONSTRUCTION_PRIMARY"
)

SEASONALITY = group(
    "SEASONALITY"
)

CALENDAR = group(
    "CALENDAR"
)

REVIEW_HISTORY = group(
    "REVIEW_HISTORY"
)

REVIEW_SEMANTIC = group(
    "REVIEW_SEMANTIC"
)


def unique_ordered(items):

    seen = set()
    out = []

    for x in items:
        if x not in seen:
            out.append(x)
            seen.add(x)

    return out


BASE_PROPERTY_PLATFORM = unique_ordered(
    PROPERTY +
    PLATFORM +
    GEOGRAPHY
)

STRUCTURED = [
    c for c in FULL_FEATURES
    if c not in set(REVIEW_SEMANTIC)
]


# --------------------------------------------------------------------------------------------------
# Progressive scientific sequence
#
# BASE:
#   property + platform + geography
#
# MARKET:
#   BASE + local residential rental market
#
# CONSTRUCTION:
#   + construction/development pipeline
#
# SEASON:
#   + historical accommodation seasonality
#
# CALENDAR:
#   + short-horizon calendar configuration
#
# STRUCTURED:
#   + review-history variables, but no semantic embeddings
#
# FULL:
#   all frozen predictors including semantic embeddings
# --------------------------------------------------------------------------------------------------

FEATURE_SETS = {

    "A0_BASE":
        unique_ordered(
            BASE_PROPERTY_PLATFORM
        ),

    "A1_BASE_MARKET":
        unique_ordered(
            BASE_PROPERTY_PLATFORM +
            RENTAL
        ),

    "A2_BASE_MARKET_CONSTRUCTION":
        unique_ordered(
            BASE_PROPERTY_PLATFORM +
            RENTAL +
            CONSTRUCTION
        ),

    "A3_PLUS_SEASONALITY":
        unique_ordered(
            BASE_PROPERTY_PLATFORM +
            RENTAL +
            CONSTRUCTION +
            SEASONALITY
        ),

    "A4_PLUS_CALENDAR":
        unique_ordered(
            BASE_PROPERTY_PLATFORM +
            RENTAL +
            CONSTRUCTION +
            SEASONALITY +
            CALENDAR
        ),

    "A5_STRUCTURED_FULL":
        unique_ordered(
            STRUCTURED
        ),

    "A6_FULL_MULTIMODAL":
        unique_ordered(
            FULL_FEATURES
        ),

    # Required leave-one-family-out ablations

    "A7_MINUS_CONSTRUCTION":
        [
            c for c in FULL_FEATURES
            if c not in set(CONSTRUCTION)
        ],

    "A8_MINUS_SEASONALITY":
        [
            c for c in FULL_FEATURES
            if c not in set(SEASONALITY)
        ],

    "A9_MINUS_CALENDAR":
        [
            c for c in FULL_FEATURES
            if c not in set(CALENDAR)
        ],

    # Remove both review history and review semantic representation.
    "A10_MINUS_REVIEWS":
        [
            c for c in FULL_FEATURES
            if c not in set(
                REVIEW_HISTORY +
                REVIEW_SEMANTIC
            )
        ],
}


# ==================================================================================================
# 3. HARD FEATURE QC
# ==================================================================================================

for name, cols in FEATURE_SETS.items():

    bad = [
        c for c in cols
        if c not in FULL_FEATURES
    ]

    if bad:
        raise RuntimeError(
            f"{name} contains non-frozen features: {bad}"
        )


FORBIDDEN = {
    "transition",
    "listing_id",
    "snapshot_date",
    "future_date",
    "next_period_persistence",
    "secondary_price_available",
    "price_resilience_delta",
    "evaluation_role",
    "price",
    "price_raw",
    "latitude",
    "longitude"
}

for name, cols in FEATURE_SETS.items():

    leaked = sorted(
        set(cols) &
        FORBIDDEN
    )

    if leaked:
        raise RuntimeError(
            f"Leakage in {name}: {leaked}"
        )


print("\nFEATURE SET SIZES")

for name, cols in FEATURE_SETS.items():
    print(
        f"{name:30s}: {len(cols):4d}"
    )

print(
    f"\nReview semantic dimensions: "
    f"{len(REVIEW_SEMANTIC)}"
)

if len(REVIEW_SEMANTIC) != 384:
    raise RuntimeError(
        "Expected 384 semantic dimensions."
    )


# ==================================================================================================
# 4. LOAD PANEL — IMMEDIATE T10 FIREWALL
# ==================================================================================================

panel = pd.read_parquet(
    PANEL_PATH
)

if len(panel) != 491_140:
    raise RuntimeError(
        f"Unexpected panel N={len(panel)}"
    )

# Keep T1-T9 ONLY.
dev = panel.loc[
    panel["transition"].isin(
        [
            "T1","T2","T3",
            "T4","T5","T6",
            "T7","T8","T9"
        ]
    )
].copy()

# Destroy reference to complete panel immediately.
del panel
gc.collect()

if (
    dev["transition"]
    .eq("T10")
    .any()
):
    raise RuntimeError(
        "T10 FIREWALL FAILURE."
    )

if len(dev) != 440_044:
    raise RuntimeError(
        f"Expected 440,044 development rows; found {len(dev):,}"
    )

dev[
    "y_disappear"
] = (
    1 -
    dev[
        "next_period_persistence"
    ].astype(int)
)


print("\nDevelopment rows:",
      f"{len(dev):,}")

print(
    "Development disappearance rate:",
    round(
        dev["y_disappear"].mean(),
        6
    )
)

print("T10 loaded into development: NO")


# ==================================================================================================
# 5. ROLLING ORIGIN FOLDS
# ==================================================================================================

FOLDS = [

    (
        "DEV_T4",
        ["T1","T2","T3"],
        "T4"
    ),

    (
        "DEV_T5",
        ["T1","T2","T3","T4"],
        "T5"
    ),

    (
        "DEV_T6",
        ["T1","T2","T3","T4","T5"],
        "T6"
    ),

    (
        "DEV_T7",
        ["T1","T2","T3","T4","T5","T6"],
        "T7"
    ),

    (
        "DEV_T8",
        ["T1","T2","T3","T4","T5","T6","T7"],
        "T8"
    ),

    (
        "DEV_T9",
        ["T1","T2","T3","T4","T5","T6","T7","T8"],
        "T9"
    )
]


# ==================================================================================================
# 6. PREPROCESSING
# ==================================================================================================

def feature_types(
    df,
    features
):

    num = []
    cat = []

    for c in features:

        dtype = df[c].dtype

        if (
            pd.api.types.is_object_dtype(dtype)
            or
            isinstance(
                dtype,
                pd.CategoricalDtype
            )
            or
            pd.api.types.is_string_dtype(dtype)
        ):
            cat.append(c)

        else:
            num.append(c)

    return num, cat


def make_preprocessor(
    train_df,
    features,
    scale_numeric=False
):

    num_cols, cat_cols = feature_types(
        train_df,
        features
    )

    num_steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]

    if scale_numeric:
        num_steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    num_pipe = Pipeline(
        num_steps
    )

    cat_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ])

    prep = ColumnTransformer(
        transformers=[
            (
                "num",
                num_pipe,
                num_cols
            ),
            (
                "cat",
                cat_pipe,
                cat_cols
            )
        ],
        remainder="drop",
        sparse_threshold=0.3
    )

    return prep


# ==================================================================================================
# 7. FIXED XGBOOST SPECIFICATION
#
# Same specification as Block I.
# We do NOT retune individual ablations.
# ==================================================================================================

XGB_PARAMS = {

    "n_estimators": 500,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.80,

    "colsample_bytree": 0.80,

    "min_child_weight": 5,

    "reg_lambda": 1.0,

    "reg_alpha": 0.0,

    "objective":
        "binary:logistic",

    "eval_metric":
        "logloss",

    "tree_method":
        "hist",

    "n_jobs":
        -1,

    "random_state":
        2026
}


# ==================================================================================================
# 8. FIXED COMPACT MLP SPECIFICATION
#
# MLP is NOT tuned against T10.
#
# It is deliberately compact:
#   128 -> 32
#   ReLU
#   L2 alpha = 1e-4
#   early stopping on training-fold internal validation
#
# For computational feasibility and scientific relevance:
#   MLP is tested on:
#       structured feature set
#       full multimodal feature set
#
# This directly tests whether semantic embeddings are more useful
# in a neural architecture than in XGBoost.
# ==================================================================================================

MLP_PARAMS = {

    "hidden_layer_sizes":
        (128, 32),

    "activation":
        "relu",

    "solver":
        "adam",

    "alpha":
        1e-4,

    "batch_size":
        512,

    "learning_rate_init":
        1e-3,

    "max_iter":
        80,

    "early_stopping":
        True,

    "validation_fraction":
        0.10,

    "n_iter_no_change":
        7,

    "random_state":
        2026,

    "verbose":
        False
}


# ==================================================================================================
# 9. CHECKPOINT UTILITIES
# ==================================================================================================

def safe_name(x):

    return (
        x.replace(
            "/",
            "_"
        )
        .replace(
            "\\",
            "_"
        )
        .replace(
            " ",
            "_"
        )
    )


def checkpoint_path(
    model_name,
    feature_name,
    fold
):

    return (
        CHECKPOINT_DIR /
        (
            safe_name(
                f"{model_name}__{feature_name}__{fold}"
            )
            + ".parquet"
        )
    )


def save_checkpoint(
    df,
    path
):

    tmp = path.with_suffix(
        ".tmp.parquet"
    )

    df.to_parquet(
        tmp,
        index=False
    )

    tmp.replace(
        path
    )


# ==================================================================================================
# 10. METRICS
# ==================================================================================================

def ranking_probability_metrics(
    y,
    p
):

    y = np.asarray(
        y
    ).astype(int)

    p = np.asarray(
        p
    ).astype(float)

    p = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )

    return {

        "roc_auc":
            roc_auc_score(
                y,
                p
            ),

        "pr_auc_disappearance":
            average_precision_score(
                y,
                p
            ),

        "brier":
            brier_score_loss(
                y,
                p
            ),

        "log_loss":
            log_loss(
                y,
                p
            ),

        "observed_rate":
            float(
                y.mean()
            ),

        "predicted_rate":
            float(
                p.mean()
            )
    }


def threshold_metrics(
    y,
    p,
    threshold
):

    y = np.asarray(
        y
    ).astype(int)

    p = np.asarray(
        p
    ).astype(float)

    pred = (
        p >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred,
            labels=[0,1]
        )
        .ravel()
    )

    sensitivity = (
        tp /
        (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn /
        (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision = (
        tp /
        (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    balanced = (
        sensitivity +
        specificity
    ) / 2

    return {

        "threshold":
            threshold,

        "sensitivity_disappearance":
            sensitivity,

        "specificity":
            specificity,

        "precision_disappearance":
            precision,

        "balanced_accuracy":
            balanced,

        "predicted_positive_rate":
            pred.mean(),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn)
    }


# ==================================================================================================
# 11. RUN XGBOOST ABLATIONS WITH CHECKPOINTS
# ==================================================================================================

XGB_ABLATIONS = [
    "A0_BASE",
    "A1_BASE_MARKET",
    "A2_BASE_MARKET_CONSTRUCTION",
    "A3_PLUS_SEASONALITY",
    "A4_PLUS_CALENDAR",
    "A5_STRUCTURED_FULL",
    "A6_FULL_MULTIMODAL",
    "A7_MINUS_CONSTRUCTION",
    "A8_MINUS_SEASONALITY",
    "A9_MINUS_CALENDAR",
    "A10_MINUS_REVIEWS"
]


def run_xgb_job(
    fold_name,
    train_transitions,
    test_transition,
    feature_name
):

    cp = checkpoint_path(
        "XGB",
        feature_name,
        fold_name
    )

    if cp.exists():

        print(
            f"  CHECKPOINT: "
            f"{feature_name} / {fold_name}"
        )

        return pd.read_parquet(
            cp
        )

    features = FEATURE_SETS[
        feature_name
    ]

    train = dev.loc[
        dev[
            "transition"
        ].isin(
            train_transitions
        )
    ]

    test = dev.loc[
        dev[
            "transition"
        ].eq(
            test_transition
        )
    ]

    y_train = (
        train[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    y_test = (
        test[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    prep = make_preprocessor(
        train,
        features,
        scale_numeric=False
    )

    model = XGBClassifier(
        **XGB_PARAMS
    )

    pipe = Pipeline([
        (
            "preprocess",
            prep
        ),
        (
            "model",
            model
        )
    ])

    start = time.time()

    pipe.fit(
        train[features],
        y_train
    )

    p = pipe.predict_proba(
        test[features]
    )[:,1]

    runtime = (
        time.time() -
        start
    )

    result = pd.DataFrame({

        "fold":
            fold_name,

        "transition":
            test_transition,

        "listing_id":
            test[
                "listing_id"
            ].values,

        "y_disappear":
            y_test,

        "p_disappear":
            p.astype(
                np.float32
            ),

        "model":
            "XGB",

        "feature_set":
            feature_name,

        "runtime_sec":
            runtime
    })

    save_checkpoint(
        result,
        cp
    )

    del pipe
    del model
    del prep

    gc.collect()

    return result


xgb_predictions = []

print("\n")
print("=" * 120)
print("I-B1 — XGBOOST SCIENTIFIC ABLATIONS")
print("=" * 120)

for feature_name in XGB_ABLATIONS:

    print(
        f"\nFEATURE SET: {feature_name}"
    )

    for (
        fold_name,
        train_transitions,
        test_transition
    ) in FOLDS:

        print(
            f"  {fold_name}..."
        )

        out = run_xgb_job(
            fold_name,
            train_transitions,
            test_transition,
            feature_name
        )

        xgb_predictions.append(
            out
        )


xgb_predictions = pd.concat(
    xgb_predictions,
    ignore_index=True
)


# ==================================================================================================
# 12. MLP — STRUCTURED AND FULL MULTIMODAL
# ==================================================================================================

MLP_FEATURES = {
    "MLP_STRUCTURED":
        FEATURE_SETS[
            "A5_STRUCTURED_FULL"
        ],

    "MLP_FULL_MULTIMODAL":
        FEATURE_SETS[
            "A6_FULL_MULTIMODAL"
        ]
}


def run_mlp_job(
    fold_name,
    train_transitions,
    test_transition,
    feature_name
):

    cp = checkpoint_path(
        "MLP",
        feature_name,
        fold_name
    )

    if cp.exists():

        print(
            f"  CHECKPOINT: "
            f"{feature_name} / {fold_name}"
        )

        return pd.read_parquet(
            cp
        )

    features = MLP_FEATURES[
        feature_name
    ]

    train = dev.loc[
        dev[
            "transition"
        ].isin(
            train_transitions
        )
    ]

    test = dev.loc[
        dev[
            "transition"
        ].eq(
            test_transition
        )
    ]

    y_train = (
        train[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    y_test = (
        test[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    prep = make_preprocessor(
        train,
        features,
        scale_numeric=True
    )

    mlp = MLPClassifier(
        **MLP_PARAMS
    )

    pipe = Pipeline([
        (
            "preprocess",
            prep
        ),
        (
            "model",
            mlp
        )
    ])

    start = time.time()

    pipe.fit(
        train[features],
        y_train
    )

    p = pipe.predict_proba(
        test[features]
    )[:,1]

    runtime = (
        time.time() -
        start
    )

    result = pd.DataFrame({

        "fold":
            fold_name,

        "transition":
            test_transition,

        "listing_id":
            test[
                "listing_id"
            ].values,

        "y_disappear":
            y_test,

        "p_disappear":
            p.astype(
                np.float32
            ),

        "model":
            "MLP",

        "feature_set":
            feature_name,

        "runtime_sec":
            runtime
    })

    save_checkpoint(
        result,
        cp
    )

    del pipe
    del mlp
    del prep

    gc.collect()

    return result


mlp_predictions = []

print("\n")
print("=" * 120)
print("I-B2 — COMPACT MLP")
print("=" * 120)

for feature_name in MLP_FEATURES:

    print(
        f"\n{feature_name}"
    )

    for (
        fold_name,
        train_transitions,
        test_transition
    ) in FOLDS:

        print(
            f"  {fold_name}..."
        )

        out = run_mlp_job(
            fold_name,
            train_transitions,
            test_transition,
            feature_name
        )

        mlp_predictions.append(
            out
        )


mlp_predictions = pd.concat(
    mlp_predictions,
    ignore_index=True
)


# ==================================================================================================
# 13. COMBINE DEVELOPMENT PREDICTIONS
# ==================================================================================================

all_new_predictions = pd.concat(
    [
        xgb_predictions,
        mlp_predictions
    ],
    ignore_index=True
)

all_new_predictions.to_parquet(
    PRED_DIR /
    "IB_all_development_predictions.parquet",
    index=False
)


# ==================================================================================================
# 14. WINDOW-LEVEL PERFORMANCE
# ==================================================================================================

metric_rows = []

for (
    model,
    feature_set,
    transition
), g in all_new_predictions.groupby(
    [
        "model",
        "feature_set",
        "transition"
    ]
):

    m = ranking_probability_metrics(
        g["y_disappear"],
        g["p_disappear"]
    )

    m.update({

        "model":
            model,

        "feature_set":
            feature_set,

        "transition":
            transition,

        "test_n":
            len(g),

        "runtime_sec":
            g[
                "runtime_sec"
            ].iloc[0]
    })

    metric_rows.append(
        m
    )


window_metrics = pd.DataFrame(
    metric_rows
)

window_metrics.to_csv(
    TABLE_DIR /
    "IB_metrics_by_window.csv",
    index=False
)


# ==================================================================================================
# 15. MACRO + WEIGHTED SUMMARY
# ==================================================================================================

metric_cols = [
    "roc_auc",
    "pr_auc_disappearance",
    "brier",
    "log_loss"
]


macro = (
    window_metrics
    .groupby(
        [
            "model",
            "feature_set"
        ]
    )[metric_cols]
    .mean()
    .reset_index()
)

macro = macro.rename(
    columns={
        c:
            "macro_" + c
        for c in metric_cols
    }
)


weighted_rows = []

for (
    model,
    feature_set
), g in window_metrics.groupby(
    [
        "model",
        "feature_set"
    ]
):

    weights = (
        g[
            "test_n"
        ]
        .astype(float)
        .values
    )

    row = {

        "model":
            model,

        "feature_set":
            feature_set
    }

    for c in metric_cols:

        row[
            "weighted_" + c
        ] = np.average(
            g[c],
            weights=weights
        )

    weighted_rows.append(
        row
    )


weighted = pd.DataFrame(
    weighted_rows
)


summary = macro.merge(
    weighted,
    on=[
        "model",
        "feature_set"
    ],
    how="outer"
)


# Temporal stability:
# SD and worst-window metrics.

stability = (
    window_metrics
    .groupby(
        [
            "model",
            "feature_set"
        ]
    )
    .agg(
        sd_roc_auc=(
            "roc_auc",
            "std"
        ),
        sd_pr_auc=(
            "pr_auc_disappearance",
            "std"
        ),
        min_roc_auc=(
            "roc_auc",
            "min"
        ),
        min_pr_auc=(
            "pr_auc_disappearance",
            "min"
        ),
        max_brier=(
            "brier",
            "max"
        )
    )
    .reset_index()
)


summary = summary.merge(
    stability,
    on=[
        "model",
        "feature_set"
    ],
    how="left"
)


summary = summary.sort_values(
    [
        "macro_pr_auc_disappearance",
        "macro_roc_auc"
    ],
    ascending=[
        False,
        False
    ]
)


summary.to_csv(
    TABLE_DIR /
    "IB_model_summary.csv",
    index=False
)


# ==================================================================================================
# 16. SCIENTIFIC INCREMENT TABLE
# ==================================================================================================

sequence = [
    "A0_BASE",
    "A1_BASE_MARKET",
    "A2_BASE_MARKET_CONSTRUCTION",
    "A3_PLUS_SEASONALITY",
    "A4_PLUS_CALENDAR",
    "A5_STRUCTURED_FULL",
    "A6_FULL_MULTIMODAL"
]


increment_rows = []

for previous, current in zip(
    sequence[:-1],
    sequence[1:]
):

    prev = summary.loc[
        (
            summary[
                "model"
            ] == "XGB"
        )
        &
        (
            summary[
                "feature_set"
            ] == previous
        )
    ]

    curr = summary.loc[
        (
            summary[
                "model"
            ] == "XGB"
        )
        &
        (
            summary[
                "feature_set"
            ] == current
        )
    ]

    if (
        len(prev) == 1
        and
        len(curr) == 1
    ):

        increment_rows.append({

            "from":
                previous,

            "to":
                current,

            "delta_macro_roc_auc":
                float(
                    curr[
                        "macro_roc_auc"
                    ].iloc[0]
                    -
                    prev[
                        "macro_roc_auc"
                    ].iloc[0]
                ),

            "delta_macro_pr_auc":
                float(
                    curr[
                        "macro_pr_auc_disappearance"
                    ].iloc[0]
                    -
                    prev[
                        "macro_pr_auc_disappearance"
                    ].iloc[0]
                ),

            "delta_macro_brier":
                float(
                    curr[
                        "macro_brier"
                    ].iloc[0]
                    -
                    prev[
                        "macro_brier"
                    ].iloc[0]
                )
        })


increments = pd.DataFrame(
    increment_rows
)

increments.to_csv(
    TABLE_DIR /
    "IB_progressive_feature_increments.csv",
    index=False
)


# ==================================================================================================
# 17. LEAVE-ONE-FAMILY-OUT COMPARISON
# ==================================================================================================

loo_map = {

    "construction":
        "A7_MINUS_CONSTRUCTION",

    "seasonality":
        "A8_MINUS_SEASONALITY",

    "calendar":
        "A9_MINUS_CALENDAR",

    "reviews":
        "A10_MINUS_REVIEWS"
}


full_row = summary.loc[
    (
        summary[
            "model"
        ] == "XGB"
    )
    &
    (
        summary[
            "feature_set"
        ] == "A6_FULL_MULTIMODAL"
    )
].iloc[0]


loo_rows = []

for family, feature_name in loo_map.items():

    r = summary.loc[
        (
            summary[
                "model"
            ] == "XGB"
        )
        &
        (
            summary[
                "feature_set"
            ] == feature_name
        )
    ].iloc[0]

    loo_rows.append({

        "removed_family":
            family,

        "full_macro_roc_auc":
            full_row[
                "macro_roc_auc"
            ],

        "minus_macro_roc_auc":
            r[
                "macro_roc_auc"
            ],

        "delta_full_minus_removed_roc":
            full_row[
                "macro_roc_auc"
            ]
            -
            r[
                "macro_roc_auc"
            ],

        "full_macro_pr_auc":
            full_row[
                "macro_pr_auc_disappearance"
            ],

        "minus_macro_pr_auc":
            r[
                "macro_pr_auc_disappearance"
            ],

        "delta_full_minus_removed_pr":
            full_row[
                "macro_pr_auc_disappearance"
            ]
            -
            r[
                "macro_pr_auc_disappearance"
            ],

        "full_macro_brier":
            full_row[
                "macro_brier"
            ],

        "minus_macro_brier":
            r[
                "macro_brier"
            ],

        "delta_full_minus_removed_brier":
            full_row[
                "macro_brier"
            ]
            -
            r[
                "macro_brier"
            ]
    })


loo = pd.DataFrame(
    loo_rows
)

loo.to_csv(
    TABLE_DIR /
    "IB_leave_one_family_out.csv",
    index=False
)


# ==================================================================================================
# 18. AUTOMATIC DEVELOPMENT MODEL RECOMMENDATION
#
# Selection principle:
#
#   Primary = macro PR-AUC
#   Secondary = macro ROC-AUC
#   Calibration safeguard = Brier
#
# We do NOT use threshold performance to choose the ranking model.
#
# A model is not automatically preferred merely because it is more complex.
# ==================================================================================================

candidate_summary = summary.copy()

best_pr = (
    candidate_summary[
        "macro_pr_auc_disappearance"
    ]
    .max()
)

# Candidates within 1% relative of best macro PR-AUC.
pr_floor = (
    best_pr *
    0.99
)

near_best = candidate_summary.loc[
    candidate_summary[
        "macro_pr_auc_disappearance"
    ] >= pr_floor
].copy()

# Among near-best PR candidates:
# prefer higher ROC, then lower Brier.
near_best = near_best.sort_values(
    [
        "macro_roc_auc",
        "macro_brier"
    ],
    ascending=[
        False,
        True
    ]
)

recommended = (
    near_best
    .iloc[0]
)


RECOMMENDED_MODEL = (
    recommended[
        "model"
    ]
)

RECOMMENDED_FEATURE_SET = (
    recommended[
        "feature_set"
    ]
)


print("\n")
print("=" * 120)
print("DEVELOPMENT MODEL RECOMMENDATION")
print("=" * 120)

print(
    "Recommended model:",
    RECOMMENDED_MODEL
)

print(
    "Recommended feature set:",
    RECOMMENDED_FEATURE_SET
)

print(
    "Macro PR-AUC:",
    recommended[
        "macro_pr_auc_disappearance"
    ]
)

print(
    "Macro ROC-AUC:",
    recommended[
        "macro_roc_auc"
    ]
)

print(
    "Macro Brier:",
    recommended[
        "macro_brier"
    ]
)


# ==================================================================================================
# 19. EXTRACT POOLED DEVELOPMENT PREDICTIONS FOR RECOMMENDED MODEL
# ==================================================================================================

selected_pred = (
    all_new_predictions.loc[
        (
            all_new_predictions[
                "model"
            ] == RECOMMENDED_MODEL
        )
        &
        (
            all_new_predictions[
                "feature_set"
            ] == RECOMMENDED_FEATURE_SET
        )
    ]
    .copy()
)


expected_test_transitions = {
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9"
}

if set(
    selected_pred[
        "transition"
    ].unique()
) != expected_test_transitions:

    raise RuntimeError(
        "Selected model does not contain all T4-T9 predictions."
    )


# ==================================================================================================
# 20. THRESHOLD SELECTION — DEVELOPMENT ONLY
#
# Youden J maximizes:
#   sensitivity + specificity - 1
#
# This is equivalent to maximizing balanced accuracy
# over ROC thresholds and does not depend on T10.
# ==================================================================================================

y_pool = (
    selected_pred[
        "y_disappear"
    ]
    .astype(int)
    .values
)

p_pool = (
    selected_pred[
        "p_disappear"
    ]
    .astype(float)
    .values
)


fpr, tpr, thresholds = roc_curve(
    y_pool,
    p_pool
)

youden = (
    tpr -
    fpr
)

valid = np.isfinite(
    thresholds
)

best_index = np.argmax(
    np.where(
        valid,
        youden,
        -np.inf
    )
)

FROZEN_THRESHOLD = float(
    thresholds[
        best_index
    ]
)


pooled_threshold_metrics = (
    threshold_metrics(
        y_pool,
        p_pool,
        FROZEN_THRESHOLD
    )
)


# ==================================================================================================
# 21. THRESHOLD PERFORMANCE BY WINDOW
# ==================================================================================================

threshold_rows = []

for transition, g in selected_pred.groupby(
    "transition"
):

    tm = threshold_metrics(
        g[
            "y_disappear"
        ],
        g[
            "p_disappear"
        ],
        FROZEN_THRESHOLD
    )

    tm[
        "transition"
    ] = transition

    threshold_rows.append(
        tm
    )


threshold_by_window = pd.DataFrame(
    threshold_rows
)

threshold_by_window.to_csv(
    TABLE_DIR /
    "IB_frozen_threshold_by_window.csv",
    index=False
)


# ==================================================================================================
# 22. CALIBRATION DIAGNOSTICS — RAW MODEL
# ==================================================================================================

def expected_calibration_error(
    y,
    p,
    bins=10
):

    d = pd.DataFrame({
        "y":
            np.asarray(y),
        "p":
            np.asarray(p)
    })

    try:

        d[
            "bin"
        ] = pd.qcut(
            d["p"],
            q=bins,
            duplicates="drop"
        )

    except Exception:

        d[
            "bin"
        ] = pd.cut(
            d["p"],
            bins=bins,
            duplicates="drop"
        )

    cal = (
        d.groupby(
            "bin",
            observed=True
        )
        .agg(
            n=(
                "y",
                "size"
            ),
            mean_pred=(
                "p",
                "mean"
            ),
            observed=(
                "y",
                "mean"
            )
        )
        .reset_index(
            drop=True
        )
    )

    cal[
        "weight"
    ] = (
        cal[
            "n"
        ]
        /
        len(d)
    )

    cal[
        "abs_error"
    ] = (
        cal[
            "mean_pred"
        ]
        -
        cal[
            "observed"
        ]
    ).abs()

    ece = (
        cal[
            "weight"
        ]
        *
        cal[
            "abs_error"
        ]
    ).sum()

    return float(ece), cal


raw_ece, raw_calibration_table = (
    expected_calibration_error(
        y_pool,
        p_pool,
        bins=10
    )
)

raw_calibration_table.to_csv(
    TABLE_DIR /
    "IB_raw_calibration_bins.csv",
    index=False
)


# ==================================================================================================
# 23. CALIBRATION STRATEGY ASSESSMENT
#
# IMPORTANT:
# We do NOT fit a calibrator on pooled predictions and evaluate it
# on those same predictions.
#
# Instead, use leave-one-development-window-out calibration:
#
#   Fit calibrator on five rolling prediction windows
#   evaluate on the sixth.
#
# Compare:
#   RAW
#   Platt/logistic
#   Isotonic
#
# The calibration METHOD can then be frozen before T10.
# ==================================================================================================

calibration_eval_rows = []


for heldout_transition in sorted(
    expected_test_transitions
):

    cal_train = selected_pred.loc[
        selected_pred[
            "transition"
        ] != heldout_transition
    ].copy()

    cal_test = selected_pred.loc[
        selected_pred[
            "transition"
        ] == heldout_transition
    ].copy()

    y_cal_train = (
        cal_train[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    p_cal_train = (
        cal_train[
            "p_disappear"
        ]
        .astype(float)
        .values
    )

    y_cal_test = (
        cal_test[
            "y_disappear"
        ]
        .astype(int)
        .values
    )

    p_cal_test = (
        cal_test[
            "p_disappear"
        ]
        .astype(float)
        .values
    )


    # ----------------------------------------------------------------------------------------------
    # RAW
    # ----------------------------------------------------------------------------------------------

    raw_m = ranking_probability_metrics(
        y_cal_test,
        p_cal_test
    )

    raw_m.update({
        "transition":
            heldout_transition,

        "calibration_method":
            "RAW"
    })

    calibration_eval_rows.append(
        raw_m
    )


    # ----------------------------------------------------------------------------------------------
    # PLATT SCALING
    #
    # Calibrate using logit of raw probability.
    # ----------------------------------------------------------------------------------------------

    eps = 1e-6

    train_logit = np.log(
        np.clip(
            p_cal_train,
            eps,
            1-eps
        )
        /
        (
            1 -
            np.clip(
                p_cal_train,
                eps,
                1-eps
            )
        )
    ).reshape(-1,1)

    test_logit = np.log(
        np.clip(
            p_cal_test,
            eps,
            1-eps
        )
        /
        (
            1 -
            np.clip(
                p_cal_test,
                eps,
                1-eps
            )
        )
    ).reshape(-1,1)


    platt = LogisticRegression(
        solver="lbfgs",
        max_iter=1000
    )

    platt.fit(
        train_logit,
        y_cal_train
    )

    p_platt = (
        platt.predict_proba(
            test_logit
        )[:,1]
    )

    pm = ranking_probability_metrics(
        y_cal_test,
        p_platt
    )

    pm.update({
        "transition":
            heldout_transition,

        "calibration_method":
            "PLATT"
    })

    calibration_eval_rows.append(
        pm
    )


    # ----------------------------------------------------------------------------------------------
    # ISOTONIC
    # ----------------------------------------------------------------------------------------------

    iso = IsotonicRegression(
        y_min=0,
        y_max=1,
        out_of_bounds="clip"
    )

    iso.fit(
        p_cal_train,
        y_cal_train
    )

    p_iso = iso.predict(
        p_cal_test
    )

    im = ranking_probability_metrics(
        y_cal_test,
        p_iso
    )

    im.update({
        "transition":
            heldout_transition,

        "calibration_method":
            "ISOTONIC"
    })

    calibration_eval_rows.append(
        im
    )


calibration_eval = pd.DataFrame(
    calibration_eval_rows
)

calibration_eval.to_csv(
    TABLE_DIR /
    "IB_calibration_method_by_window.csv",
    index=False
)


calibration_summary = (
    calibration_eval
    .groupby(
        "calibration_method"
    )
    .agg(
        macro_brier=(
            "brier",
            "mean"
        ),
        macro_log_loss=(
            "log_loss",
            "mean"
        ),
        macro_roc_auc=(
            "roc_auc",
            "mean"
        ),
        macro_pr_auc=(
            "pr_auc_disappearance",
            "mean"
        )
    )
    .reset_index()
)


# Calibration selection is based on probability accuracy:
# primary Brier, secondary log loss.

calibration_summary = (
    calibration_summary
    .sort_values(
        [
            "macro_brier",
            "macro_log_loss"
        ],
        ascending=[
            True,
            True
        ]
    )
)


BEST_CALIBRATION_METHOD = (
    calibration_summary
    .iloc[0][
        "calibration_method"
    ]
)


calibration_summary.to_csv(
    TABLE_DIR /
    "IB_calibration_method_summary.csv",
    index=False
)


# ==================================================================================================
# 24. CALIBRATION SAFEGUARD
#
# Do not force calibration merely because one method wins by a
# numerically trivial amount.
#
# Require at least 0.5% relative Brier improvement over RAW.
# Otherwise freeze RAW probabilities.
# ==================================================================================================

raw_brier = float(
    calibration_summary.loc[
        calibration_summary[
            "calibration_method"
        ] == "RAW",
        "macro_brier"
    ].iloc[0]
)

best_brier = float(
    calibration_summary[
        "macro_brier"
    ].min()
)


relative_brier_improvement = (
    (
        raw_brier -
        best_brier
    )
    /
    raw_brier
    if raw_brier > 0
    else 0
)


if (
    BEST_CALIBRATION_METHOD != "RAW"
    and
    relative_brier_improvement >= 0.005
):

    FROZEN_CALIBRATION = (
        BEST_CALIBRATION_METHOD
    )

else:

    FROZEN_CALIBRATION = "RAW"


# ==================================================================================================
# 25. FREEZE RECOMMENDATION
# ==================================================================================================

freeze_recommendation = {

    "status":
        "DEVELOPMENT_SPECIFICATION_READY_FOR_REVIEW",

    "T10_evaluated":
        False,

    "development_windows":
        [
            "T4","T5","T6",
            "T7","T8","T9"
        ],

    "scientific_outcome":
        "next-period platform persistence",

    "evaluation_positive_class":
        "next-period disappearance",

    "recommended_model":
        RECOMMENDED_MODEL,

    "recommended_feature_set":
        RECOMMENDED_FEATURE_SET,

    "recommended_feature_count":
        int(
            len(
                FEATURE_SETS[
                    RECOMMENDED_FEATURE_SET
                ]
            )
            if RECOMMENDED_MODEL == "XGB"
            else
            len(
                MLP_FEATURES[
                    RECOMMENDED_FEATURE_SET
                ]
            )
        ),

    "development_macro_pr_auc":
        float(
            recommended[
                "macro_pr_auc_disappearance"
            ]
        ),

    "development_macro_roc_auc":
        float(
            recommended[
                "macro_roc_auc"
            ]
        ),

    "development_macro_brier":
        float(
            recommended[
                "macro_brier"
            ]
        ),

    "frozen_threshold_candidate":
        FROZEN_THRESHOLD,

    "threshold_selection":
        (
            "Pooled T4-T9 rolling-origin development predictions; "
            "maximum Youden J / balanced accuracy."
        ),

    "pooled_threshold_performance":
        {
            k:
                (
                    float(v)
                    if isinstance(
                        v,
                        (
                            np.floating,
                            float
                        )
                    )
                    else
                    int(v)
                    if isinstance(
                        v,
                        (
                            np.integer,
                            int
                        )
                    )
                    else
                    v
                )
            for k, v in
            pooled_threshold_metrics.items()
        },

    "calibration_method_candidate":
        FROZEN_CALIBRATION,

    "calibration_selection":
        (
            "Leave-one-development-window-out comparison using "
            "Brier score and log loss; calibrated method retained "
            "only if relative Brier improvement over RAW >= 0.5%."
        ),

    "raw_macro_calibration_brier":
        raw_brier,

    "best_macro_calibration_brier":
        best_brier,

    "relative_brier_improvement":
        relative_brier_improvement,

    "xgb_parameters":
        XGB_PARAMS,

    "mlp_parameters":
        {
            k:
                (
                    list(v)
                    if isinstance(
                        v,
                        tuple
                    )
                    else v
                )
            for k, v in
            MLP_PARAMS.items()
        },

    "global_imputation":
        False,

    "global_scaling":
        False,

    "fold_specific_preprocessing":
        True,

    "raw_price_predictor":
        False,

    "raw_coordinates_predictor":
        False,

    "T10_rule":
        (
            "Do not evaluate T10 until this development specification "
            "has been reviewed and explicitly frozen."
        )
}


with open(
    FREEZE_DIR /
    "MODEL_FREEZE_RECOMMENDATION.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze_recommendation,
        f,
        indent=2
    )


# ==================================================================================================
# 26. FEATURE SET MANIFEST
# ==================================================================================================

feature_manifest = {

    name:
        {
            "n_features":
                len(cols),

            "features":
                cols
        }

    for name, cols
    in FEATURE_SETS.items()
}


with open(
    FREEZE_DIR /
    "IB_feature_set_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_manifest,
        f,
        indent=2
    )


# ==================================================================================================
# 27. DATA / CONFIG FINGERPRINT
# ==================================================================================================

fingerprint_payload = json.dumps(
    {
        "full_features":
            FULL_FEATURES,

        "feature_sets":
            FEATURE_SETS,

        "xgb":
            XGB_PARAMS,

        "mlp":
            {
                k:
                    (
                        list(v)
                        if isinstance(
                            v,
                            tuple
                        )
                        else v
                    )
                for k, v
                in MLP_PARAMS.items()
            },

        "folds":
            FOLDS
    },
    sort_keys=True
).encode(
    "utf-8"
)


fingerprint = hashlib.sha256(
    fingerprint_payload
).hexdigest()


with open(
    FREEZE_DIR /
    "IB_configuration_fingerprint.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        fingerprint
    )


# ==================================================================================================
# 28. OUTPUT TABLES
# ==================================================================================================

print("\n")
print("=" * 120)
print("TABLE I-B1 — DEVELOPMENT MODEL / ABLATION SUMMARY")
print("=" * 120)

display(
    summary[
        [
            "model",
            "feature_set",
            "macro_roc_auc",
            "macro_pr_auc_disappearance",
            "macro_brier",
            "weighted_roc_auc",
            "weighted_pr_auc_disappearance",
            "weighted_brier",
            "sd_roc_auc",
            "sd_pr_auc",
            "min_roc_auc",
            "min_pr_auc"
        ]
    ]
)


print("\n")
print("=" * 120)
print("TABLE I-B2 — PROGRESSIVE FEATURE CONTRIBUTIONS")
print("=" * 120)

display(
    increments
)


print("\n")
print("=" * 120)
print("TABLE I-B3 — LEAVE-ONE-FAMILY-OUT")
print("=" * 120)

display(
    loo
)


print("\n")
print("=" * 120)
print("TABLE I-B4 — FROZEN THRESHOLD PERFORMANCE BY WINDOW")
print("=" * 120)

display(
    threshold_by_window[
        [
            "transition",
            "threshold",
            "sensitivity_disappearance",
            "specificity",
            "precision_disappearance",
            "balanced_accuracy",
            "predicted_positive_rate"
        ]
    ]
)


print("\n")
print("=" * 120)
print("TABLE I-B5 — CALIBRATION METHOD COMPARISON")
print("=" * 120)

display(
    calibration_summary
)


# ==================================================================================================
# 29. FINAL RECOMMENDATION
# ==================================================================================================

print("\n")
print("=" * 120)
print("MODEL FREEZE RECOMMENDATION")
print("=" * 120)

print(
    "Recommended model:",
    RECOMMENDED_MODEL
)

print(
    "Recommended feature set:",
    RECOMMENDED_FEATURE_SET
)

print(
    "Development macro PR-AUC:",
    round(
        float(
            recommended[
                "macro_pr_auc_disappearance"
            ]
        ),
        6
    )
)

print(
    "Development macro ROC-AUC:",
    round(
        float(
            recommended[
                "macro_roc_auc"
            ]
        ),
        6
    )
)

print(
    "Development macro Brier:",
    round(
        float(
            recommended[
                "macro_brier"
            ]
        ),
        6
    )
)

print(
    "Candidate frozen threshold:",
    round(
        FROZEN_THRESHOLD,
        8
    )
)

print(
    "Pooled development balanced accuracy:",
    round(
        pooled_threshold_metrics[
            "balanced_accuracy"
        ],
        6
    )
)

print(
    "Pooled disappearance sensitivity:",
    round(
        pooled_threshold_metrics[
            "sensitivity_disappearance"
        ],
        6
    )
)

print(
    "Pooled specificity:",
    round(
        pooled_threshold_metrics[
            "specificity"
        ],
        6
    )
)

print(
    "Calibration candidate:",
    FROZEN_CALIBRATION
)

print(
    "Relative Brier improvement from calibration:",
    round(
        relative_brier_improvement,
        6
    )
)

print(
    "T10 evaluated: NO"
)


print("\n")
print("=" * 120)
print("PRODUCTION BLOCK I-B STATUS: PASS — DEVELOPMENT ONLY")
print("=" * 120)

print(
    "Checkpointing: ENABLED"
)

print(
    "If interrupted, rerun this cell; completed model-fold jobs are reused."
)

print(
    "\nDO NOT RUN T10 YET."
)

print(
    "Review the five tables above and MODEL_FREEZE_RECOMMENDATION.json first."
)

print("=" * 120)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

names = [
    "IB_all_development_predictions.parquet",
    "IB_model_summary.csv",
    "IB_frozen_threshold_by_window.csv",
    "IB_calibration_method_summary.csv",
]

print("=" * 100)
print("SEARCHING FOR BLOCK I-B OUTPUT FILES")
print("=" * 100)

found = {}

for name in names:
    matches = list(PROJECT_ROOT.rglob(name))
    found[name] = matches

    print(f"\n{name}")
    if matches:
        for p in matches:
            print("  FOUND:", p)
    else:
        print("  NOT FOUND")

print("\nDONE")

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK I-C
# FINAL DEVELOPMENT INTEGRITY AUDIT + PRIMARY MODEL FREEZE
#
# PURPOSE
#   1. Verify T4-T9 completeness for every model/feature-set represented in I-B predictions
#   2. Check duplicate / missing listing-origin predictions
#   3. Confirm A5_STRUCTURED_FULL contains exactly 53 frozen predictors
#   4. Confirm T10 has NOT entered development predictions / development tables
#   5. Confirm frozen threshold ~= 0.02679064
#   6. Confirm PLATT is the selected calibration method
#   7. Create FINAL_PRIMARY_MODEL_SPEC_FROZEN.json
#   8. Create SHA256 fingerprint for the frozen specification
#
# IMPORTANT
#   - NO model fitting
#   - NO hyperparameter tuning
#   - NO T10 evaluation
#   - READ-ONLY audit of development outputs, except final freeze files
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib


print("=" * 120)
print("PRODUCTION BLOCK I-C")
print("FINAL DEVELOPMENT INTEGRITY AUDIT + PRIMARY MODEL FREEZE")
print("=" * 120)


# ==================================================================================================
# 0. PATHS — CORRECTED TO ACTUAL BLOCK I-B OUTPUT STRUCTURE
# ==================================================================================================

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(
        r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

IB_DIR = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
)

PRED_DIR = IB_DIR / "predictions"
TABLE_DIR = IB_DIR / "tables"
FREEZE_DIR = IB_DIR / "freeze"

MODELLING_DIR = (
    PROJECT_ROOT
    / "02_intermediate"
    / "modelling"
)

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# Actual Block I-B outputs
# --------------------------------------------------------------------------------------------------

PRED_PATH = (
    PRED_DIR
    / "IB_all_development_predictions.parquet"
)

SUMMARY_PATH = (
    TABLE_DIR
    / "IB_model_summary.csv"
)

THRESHOLD_PATH = (
    TABLE_DIR
    / "IB_frozen_threshold_by_window.csv"
)

CALIB_PATH = (
    TABLE_DIR
    / "IB_calibration_method_summary.csv"
)


# --------------------------------------------------------------------------------------------------
# Frozen Block H/H2 specifications
# --------------------------------------------------------------------------------------------------

FEATURE_GROUPS_PATH = (
    MODELLING_DIR
    / "feature_groups_FINAL_FROZEN.json"
)

WHITELIST_PATH = (
    MODELLING_DIR
    / "predictor_whitelist_FINAL_FROZEN.txt"
)

REGISTRY_PATH = (
    MODELLING_DIR
    / "feature_registry_FINAL_FROZEN.csv"
)

CONTRACT_PATH = (
    MODELLING_DIR
    / "MODEL_DATA_CONTRACT_FROZEN.json"
)


# --------------------------------------------------------------------------------------------------
# Block I-C outputs
# --------------------------------------------------------------------------------------------------

OUT_JSON = (
    FREEZE_DIR
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.json"
)

OUT_HASH = (
    FREEZE_DIR
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.sha256"
)

OUT_QC = (
    FREEZE_DIR
    / "FINAL_PRIMARY_MODEL_FREEZE_QC.csv"
)


# ==================================================================================================
# 1. HARD INPUT FILE CHECK
# ==================================================================================================

required_files = [
    PRED_PATH,
    SUMMARY_PATH,
    THRESHOLD_PATH,
    CALIB_PATH,
    FEATURE_GROUPS_PATH,
    WHITELIST_PATH,
    REGISTRY_PATH,
    CONTRACT_PATH,
]

print("\nRequired input files:")

for p in required_files:
    status = "FOUND" if p.exists() else "MISSING"
    print(f"  {status:<7}: {p}")


missing_files = [
    str(p)
    for p in required_files
    if not p.exists()
]


if missing_files:

    raise FileNotFoundError(
        "\nBLOCK I-C STOPPED.\n"
        "Missing required frozen/development files:\n"
        + "\n".join(missing_files)
    )


print("\nINPUT FILE CHECK: PASS")


# ==================================================================================================
# 2. LOAD DEVELOPMENT OUTPUTS
# ==================================================================================================

pred = pd.read_parquet(PRED_PATH)

summary = pd.read_csv(
    SUMMARY_PATH
)

threshold_df = pd.read_csv(
    THRESHOLD_PATH
)

calib = pd.read_csv(
    CALIB_PATH
)


with open(
    FEATURE_GROUPS_PATH,
    "r",
    encoding="utf-8"
) as f:

    feature_groups = json.load(f)


with open(
    CONTRACT_PATH,
    "r",
    encoding="utf-8"
) as f:

    data_contract = json.load(f)


with open(
    WHITELIST_PATH,
    "r",
    encoding="utf-8"
) as f:

    whitelist = [
        x.strip()
        for x in f.readlines()
        if x.strip()
    ]


registry = pd.read_csv(
    REGISTRY_PATH
)


print("\nLoaded:")

print(
    f"  Development predictions : "
    f"{pred.shape}"
)

print(
    f"  Model summary           : "
    f"{summary.shape}"
)

print(
    f"  Threshold table         : "
    f"{threshold_df.shape}"
)

print(
    f"  Calibration summary     : "
    f"{calib.shape}"
)

print(
    f"  Frozen whitelist        : "
    f"{len(whitelist):,} predictors"
)


# ==================================================================================================
# 3. HELPER — FIND COLUMN
# ==================================================================================================

def find_col(
    df,
    candidates,
    required=True
):

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for cand in candidates:

        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    if required:

        raise KeyError(
            f"\nCould not find any of:\n"
            f"{candidates}\n\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )

    return None


# ==================================================================================================
# 4. DETECT PREDICTION COLUMNS
# ==================================================================================================

model_col = find_col(
    pred,
    ["model"]
)

feature_col = find_col(
    pred,
    ["feature_set"]
)

transition_col = find_col(
    pred,
    [
        "transition",
        "test_transition",
        "window"
    ]
)


id_col = find_col(
    pred,
    [
        "id",
        "listing_id",
        "listingid"
    ],
    required=False
)


y_col = find_col(
    pred,
    [
        "y_true",
        "target",
        "disappearance",
        "y_disappearance",
        "actual_disappearance",
        "true_disappearance"
    ],
    required=False
)


p_col = find_col(
    pred,
    [
        "p_disappear",
        "p_disappearance",
        "pred_proba",
        "prediction",
        "probability",
        "y_prob",
        "y_score"
    ],
    required=False
)


print("\nDetected prediction columns:")

print(
    f"  model       : {model_col}"
)

print(
    f"  feature_set : {feature_col}"
)

print(
    f"  transition  : {transition_col}"
)

print(
    f"  listing ID  : {id_col}"
)

print(
    f"  y_true      : {y_col}"
)

print(
    f"  probability : {p_col}"
)


# ==================================================================================================
# 5. I-C1 — DEVELOPMENT TRANSITION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C1 — DEVELOPMENT TRANSITION AUDIT"
)

print("=" * 120)


EXPECTED_TRANSITIONS = [
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9"
]


pred[transition_col] = (
    pred[transition_col]
    .astype(str)
    .str.upper()
)


observed_transitions = sorted(
    pred[transition_col]
    .dropna()
    .unique()
    .tolist()
)


print(
    "Expected:",
    EXPECTED_TRANSITIONS
)

print(
    "Observed:",
    observed_transitions
)


unexpected_transitions = sorted(
    set(observed_transitions)
    - set(EXPECTED_TRANSITIONS)
)


missing_global_transitions = sorted(
    set(EXPECTED_TRANSITIONS)
    - set(observed_transitions)
)


t10_rows = int(
    pred[transition_col]
    .eq("T10")
    .sum()
)


print(
    f"\nT10 rows in development predictions: "
    f"{t10_rows:,}"
)


if t10_rows != 0:

    raise RuntimeError(
        "HARD FAIL: T10 is present in "
        "development predictions."
    )


if unexpected_transitions:

    raise RuntimeError(
        "HARD FAIL: Unexpected development "
        f"transitions: {unexpected_transitions}"
    )


if missing_global_transitions:

    raise RuntimeError(
        "HARD FAIL: Missing development "
        f"transitions: {missing_global_transitions}"
    )


print(
    "DEVELOPMENT TRANSITION AUDIT: PASS"
)


# ==================================================================================================
# 6. I-C2 — MODEL × FEATURE SET × FOLD COMPLETENESS
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C2 — MODEL × FEATURE-SET × FOLD COMPLETENESS"
)

print("=" * 120)


config_rows = []


for (model, fs), g in pred.groupby(
    [
        model_col,
        feature_col
    ],
    dropna=False
):

    folds = sorted(
        g[transition_col]
        .dropna()
        .unique()
        .tolist()
    )


    missing_folds = sorted(
        set(EXPECTED_TRANSITIONS)
        - set(folds)
    )


    extra_folds = sorted(
        set(folds)
        - set(EXPECTED_TRANSITIONS)
    )


    complete = (
        len(missing_folds) == 0
        and len(extra_folds) == 0
        and len(folds) == 6
    )


    config_rows.append(
        {
            "model": model,
            "feature_set": fs,
            "n_rows": len(g),
            "n_transitions": len(folds),
            "transitions": ",".join(folds),
            "missing_folds":
                ",".join(missing_folds),
            "extra_folds":
                ",".join(extra_folds),
            "complete_T4_T9":
                complete
        }
    )


fold_qc = pd.DataFrame(
    config_rows
)


print(
    fold_qc[
        [
            "model",
            "feature_set",
            "n_rows",
            "n_transitions",
            "transitions",
            "complete_T4_T9"
        ]
    ].to_string(
        index=False
    )
)


bad_fold_configs = fold_qc.loc[
    ~fold_qc["complete_T4_T9"]
]


if len(bad_fold_configs) > 0:

    print(
        "\nINCOMPLETE CONFIGURATIONS:"
    )

    print(
        bad_fold_configs.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "HARD FAIL: At least one "
        "model/feature-set does not contain "
        "exactly T4-T9."
    )


print(
    "\nALL MODEL / FEATURE-SET CONFIGURATIONS "
    "HAVE T4-T9: PASS"
)


# ==================================================================================================
# 7. I-C3 — DUPLICATE / MISSING PREDICTION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C3 — DUPLICATE / MISSING PREDICTION AUDIT"
)

print("=" * 120)


duplicate_rows = 0


if id_col is not None:

    dup_keys = [
        model_col,
        feature_col,
        transition_col,
        id_col
    ]


    duplicate_rows = int(
        pred.duplicated(
            dup_keys,
            keep=False
        ).sum()
    )


    print(
        "Duplicate model × feature-set × "
        "transition × listing rows: "
        f"{duplicate_rows:,}"
    )


    if duplicate_rows > 0:

        raise RuntimeError(
            "HARD FAIL: Duplicate "
            "listing-level development predictions."
        )


else:

    print(
        "WARNING: listing ID column not detected; "
        "exact listing-level duplicate check "
        "cannot be performed."
    )


missing_transition = int(
    pred[transition_col]
    .isna()
    .sum()
)


missing_model = int(
    pred[model_col]
    .isna()
    .sum()
)


missing_feature = int(
    pred[feature_col]
    .isna()
    .sum()
)


print(
    f"Missing transition : "
    f"{missing_transition:,}"
)

print(
    f"Missing model      : "
    f"{missing_model:,}"
)

print(
    f"Missing feature set: "
    f"{missing_feature:,}"
)


if (
    missing_transition > 0
    or missing_model > 0
    or missing_feature > 0
):

    raise RuntimeError(
        "HARD FAIL: Missing model / "
        "feature-set / transition identifiers."
    )


missing_y = None
missing_p = None
invalid_p = None


if y_col is not None:

    missing_y = int(
        pred[y_col]
        .isna()
        .sum()
    )

    print(
        f"Missing y_true     : "
        f"{missing_y:,}"
    )

    if missing_y > 0:

        raise RuntimeError(
            "HARD FAIL: Missing "
            "development outcome values."
        )


if p_col is not None:

    missing_p = int(
        pred[p_col]
        .isna()
        .sum()
    )


    invalid_p = int(
        (
            (~pred[p_col].between(0, 1))
            & pred[p_col].notna()
        ).sum()
    )


    print(
        f"Missing probability: "
        f"{missing_p:,}"
    )

    print(
        f"Probability outside [0,1]: "
        f"{invalid_p:,}"
    )


    if (
        missing_p > 0
        or invalid_p > 0
    ):

        raise RuntimeError(
            "HARD FAIL: Missing or invalid "
            "development probabilities."
        )


print(
    "DUPLICATE / MISSING PREDICTION AUDIT: PASS"
)


# ==================================================================================================
# 8. I-C4 — A5_STRUCTURED_FULL FEATURE AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C4 — A5_STRUCTURED_FULL FEATURE AUDIT"
)

print("=" * 120)


print(
    "Available frozen feature groups:"
)


for k, v in feature_groups.items():

    if isinstance(v, list):

        print(
            f"  {k:<35}: "
            f"{len(v):>4}"
        )

    else:

        print(
            f"  {k:<35}: "
            f"{type(v).__name__}"
        )


# --------------------------------------------------------------------------------------------------
# Frozen structured model specification
#
# PROPERTY                  = 7
# PLATFORM                  = 16
# GEOGRAPHY                 = 1
# RENTAL_MARKET_PRIMARY     = 2
# CONSTRUCTION_PRIMARY      = 4
# SEASONALITY               = 4
# CALENDAR                  = 10
# REVIEW_HISTORY            = 9
#
# TOTAL                     = 53
# --------------------------------------------------------------------------------------------------

STRUCTURED_GROUPS = [
    "PROPERTY",
    "PLATFORM",
    "GEOGRAPHY",
    "RENTAL_MARKET_PRIMARY",
    "CONSTRUCTION_PRIMARY",
    "SEASONALITY",
    "CALENDAR",
    "REVIEW_HISTORY",
]


missing_groups = [
    g
    for g in STRUCTURED_GROUPS
    if g not in feature_groups
]


if missing_groups:

    raise KeyError(
        "HARD FAIL: Missing frozen "
        f"feature groups: {missing_groups}"
    )


A5_FEATURES = []


for g in STRUCTURED_GROUPS:

    values = feature_groups[g]

    if not isinstance(
        values,
        list
    ):

        raise TypeError(
            f"HARD FAIL: feature group "
            f"{g} is not a list."
        )

    A5_FEATURES.extend(
        values
    )


# Check duplicate predictor names
a5_duplicates = sorted(
    {
        x
        for x in A5_FEATURES
        if A5_FEATURES.count(x) > 1
    }
)


print(
    "\nA5 structured groups:"
)


for g in STRUCTURED_GROUPS:

    print(
        f"  {g:<30}: "
        f"{len(feature_groups[g]):>4}"
    )


print(
    f"\nA5 predictor count: "
    f"{len(A5_FEATURES)}"
)


print(
    "A5 duplicate predictor names: "
    f"{len(a5_duplicates)}"
)


if len(A5_FEATURES) != 53:

    raise RuntimeError(
        "HARD FAIL: A5 must contain "
        "exactly 53 predictors; "
        f"found {len(A5_FEATURES)}."
    )


if a5_duplicates:

    raise RuntimeError(
        "HARD FAIL: Duplicate A5 "
        f"predictors: {a5_duplicates}"
    )


not_in_whitelist = [
    x
    for x in A5_FEATURES
    if x not in whitelist
]


if not_in_whitelist:

    raise RuntimeError(
        "HARD FAIL: A5 contains predictors "
        "not present in frozen whitelist:\n"
        f"{not_in_whitelist}"
    )


semantic_features = (
    feature_groups.get(
        "REVIEW_SEMANTIC",
        []
    )
)


semantic_overlap = (
    set(A5_FEATURES)
    .intersection(
        set(semantic_features)
    )
)


print(
    "Semantic dimensions inside A5: "
    f"{len(semantic_overlap)}"
)


if semantic_overlap:

    raise RuntimeError(
        "HARD FAIL: REVIEW_SEMANTIC "
        "predictors entered A5."
    )


print(
    "A5_STRUCTURED_FULL = "
    "53 predictors: PASS"
)


# ==================================================================================================
# 9. I-C5 — DEVELOPMENT MODEL SELECTION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C5 — DEVELOPMENT MODEL SELECTION AUDIT"
)

print("=" * 120)


summary_model_col = find_col(
    summary,
    ["model"]
)


summary_feature_col = find_col(
    summary,
    ["feature_set"]
)


summary_pr_col = find_col(
    summary,
    [
        "macro_pr_auc_disappearance",
        "macro_pr_auc"
    ]
)


summary_roc_col = find_col(
    summary,
    [
        "macro_roc_auc"
    ]
)


summary_brier_col = find_col(
    summary,
    [
        "macro_brier"
    ]
)


selected = summary.loc[
    (
        summary[summary_model_col]
        .astype(str)
        .str.upper()
        == "XGB"
    )
    &
    (
        summary[summary_feature_col]
        .astype(str)
        == "A5_STRUCTURED_FULL"
    )
].copy()


if len(selected) != 1:

    raise RuntimeError(
        "HARD FAIL: Could not uniquely "
        "identify XGB + A5_STRUCTURED_FULL "
        "in I-B summary."
    )


selected_row = (
    selected.iloc[0]
)


macro_pr = float(
    selected_row[
        summary_pr_col
    ]
)


macro_roc = float(
    selected_row[
        summary_roc_col
    ]
)


macro_brier = float(
    selected_row[
        summary_brier_col
    ]
)


print(
    "Selected development model:"
)

print(
    "  Model       : XGB"
)

print(
    "  Feature set : A5_STRUCTURED_FULL"
)

print(
    f"  Macro PR-AUC: "
    f"{macro_pr:.9f}"
)

print(
    f"  Macro ROC   : "
    f"{macro_roc:.9f}"
)

print(
    f"  Macro Brier : "
    f"{macro_brier:.9f}"
)


# Expected values from completed Block I-B
EXPECTED_PR = (
    0.07492441218063363
)

EXPECTED_ROC = (
    0.7383562060548434
)

EXPECTED_BRIER = (
    0.029242723859104754
)

METRIC_TOL = 1e-6


if not np.isclose(
    macro_pr,
    EXPECTED_PR,
    atol=METRIC_TOL
):

    raise RuntimeError(
        "HARD FAIL: Development "
        "PR-AUC changed."
    )


if not np.isclose(
    macro_roc,
    EXPECTED_ROC,
    atol=METRIC_TOL
):

    raise RuntimeError(
        "HARD FAIL: Development "
        "ROC-AUC changed."
    )


if not np.isclose(
    macro_brier,
    EXPECTED_BRIER,
    atol=METRIC_TOL
):

    raise RuntimeError(
        "HARD FAIL: Development "
        "Brier changed."
    )


print(
    "DEVELOPMENT MODEL SELECTION AUDIT: PASS"
)


# ==================================================================================================
# 10. I-C6 — THRESHOLD AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C6 — THRESHOLD AUDIT"
)

print("=" * 120)


threshold_col = find_col(
    threshold_df,
    ["threshold"]
)


threshold_values = (
    pd.to_numeric(
        threshold_df[
            threshold_col
        ],
        errors="coerce"
    )
    .dropna()
    .unique()
)


print(
    "Threshold values found:",
    threshold_values
)


if len(
    threshold_values
) != 1:

    raise RuntimeError(
        "HARD FAIL: Threshold file "
        "contains more than one "
        "development threshold."
    )


observed_threshold = float(
    threshold_values[0]
)


EXPECTED_THRESHOLD = (
    0.02679064
)


print(
    f"Expected threshold : "
    f"{EXPECTED_THRESHOLD:.8f}"
)

print(
    f"Observed threshold : "
    f"{observed_threshold:.12f}"
)


if not np.isclose(
    observed_threshold,
    EXPECTED_THRESHOLD,
    atol=1e-7
):

    raise RuntimeError(
        "HARD FAIL: Development threshold "
        "does not match I-B freeze "
        "recommendation."
    )


print(
    "THRESHOLD AUDIT: PASS"
)


# ==================================================================================================
# 11. I-C7 — CALIBRATION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C7 — CALIBRATION AUDIT"
)

print("=" * 120)


cal_method_col = find_col(
    calib,
    [
        "calibration_method",
        "method"
    ]
)


cal_brier_col = find_col(
    calib,
    [
        "macro_brier"
    ]
)


cal_logloss_col = find_col(
    calib,
    [
        "macro_log_loss",
        "macro_logloss"
    ],
    required=False
)


calib[
    cal_method_col
] = (
    calib[
        cal_method_col
    ]
    .astype(str)
    .str.upper()
)


display_cols = [
    cal_method_col,
    cal_brier_col
]


if cal_logloss_col:

    display_cols.append(
        cal_logloss_col
    )


print(
    calib[
        display_cols
    ].to_string(
        index=False
    )
)


raw_row = calib.loc[
    calib[
        cal_method_col
    ]
    == "RAW"
]


platt_row = calib.loc[
    calib[
        cal_method_col
    ]
    == "PLATT"
]


if (
    len(raw_row) != 1
    or len(platt_row) != 1
):

    raise RuntimeError(
        "HARD FAIL: RAW and PLATT "
        "calibration rows must each "
        "appear exactly once."
    )


raw_brier = float(
    raw_row.iloc[0][
        cal_brier_col
    ]
)


platt_brier = float(
    platt_row.iloc[0][
        cal_brier_col
    ]
)


relative_brier_improvement = (
    raw_brier
    - platt_brier
) / raw_brier


print(
    f"\nRAW Brier                  : "
    f"{raw_brier:.9f}"
)

print(
    f"PLATT Brier                : "
    f"{platt_brier:.9f}"
)

print(
    "Relative Brier improvement : "
    f"{relative_brier_improvement:.6%}"
)


if (
    platt_brier
    >= raw_brier
):

    raise RuntimeError(
        "HARD FAIL: PLATT does not "
        "improve Brier score."
    )


if (
    relative_brier_improvement
    < 0.005
):

    raise RuntimeError(
        "HARD FAIL: PLATT improvement "
        "is below the pre-specified "
        "0.5% relative Brier safeguard."
    )


SELECTED_CALIBRATION = (
    "PLATT"
)


print(
    "SELECTED CALIBRATION: PLATT"
)

print(
    "CALIBRATION AUDIT: PASS"
)


# ==================================================================================================
# 12. I-C8 — T10 DEVELOPMENT-LEAKAGE SAFETY
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C8 — T10 DEVELOPMENT-LEAKAGE SAFETY"
)

print("=" * 120)


t10_hits = []


# Search all I-B CSV outputs.
# Only inspect columns explicitly representing transition/window.
for csv_path in IB_DIR.rglob(
    "*.csv"
):

    try:

        temp = pd.read_csv(
            csv_path
        )

    except Exception:

        continue


    transition_like_cols = [
        c
        for c in temp.columns
        if str(c).lower()
        in {
            "transition",
            "test_transition",
            "window"
        }
    ]


    for c in transition_like_cols:

        vals = (
            temp[c]
            .astype(str)
            .str.upper()
        )


        n = int(
            vals.eq(
                "T10"
            ).sum()
        )


        if n > 0:

            t10_hits.append(
                {
                    "file":
                        str(
                            csv_path.relative_to(
                                IB_DIR
                            )
                        ),

                    "column":
                        c,

                    "n_T10_rows":
                        n
                }
            )


if t10_hits:

    print(
        pd.DataFrame(
            t10_hits
        ).to_string(
            index=False
        )
    )

    raise RuntimeError(
        "HARD FAIL: T10 found in "
        "I-B development artifact(s)."
    )


print(
    "T10 in development predictions       : NO"
)

print(
    "T10 in transition-bearing I-B CSVs   : NO"
)

print(
    "T10 DEVELOPMENT-LEAKAGE SAFETY: PASS"
)


# ==================================================================================================
# 13. HASH FUNCTIONS
# ==================================================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_json_object(
    obj
):

    payload = json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        payload
    ).hexdigest()


# Exact ordered A5 feature specification hash
a5_feature_hash = (
    sha256_json_object(
        A5_FEATURES
    )
)


# Hash important upstream frozen specifications
upstream_hashes = {

    "feature_groups_FINAL_FROZEN.json":
        sha256_file(
            FEATURE_GROUPS_PATH
        ),

    "predictor_whitelist_FINAL_FROZEN.txt":
        sha256_file(
            WHITELIST_PATH
        ),

    "feature_registry_FINAL_FROZEN.csv":
        sha256_file(
            REGISTRY_PATH
        ),

    "MODEL_DATA_CONTRACT_FROZEN.json":
        sha256_file(
            CONTRACT_PATH
        ),

    "IB_model_summary.csv":
        sha256_file(
            SUMMARY_PATH
        ),

    "IB_frozen_threshold_by_window.csv":
        sha256_file(
            THRESHOLD_PATH
        ),

    "IB_calibration_method_summary.csv":
        sha256_file(
            CALIB_PATH
        ),
}


# ==================================================================================================
# 14. I-C9 — CREATE FINAL PRIMARY MODEL SPECIFICATION
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "I-C9 — CREATE FINAL PRIMARY MODEL FREEZE"
)

print("=" * 120)


freeze_spec = {

    "freeze_name":
        "FINAL_PRIMARY_MODEL_SPEC_FROZEN",

    "freeze_stage":
        "development_complete_pre_T10",


    # ----------------------------------------------------------------------------------------------
    # Outcome
    # ----------------------------------------------------------------------------------------------

    "outcome": {

        "primary_target_original":
            "next_period_persistence",

        "modelled_positive_event":
            "next_period_disappearance",

        "positive_event_definition":
            "1 - next_period_persistence",

        "interpretation":
            (
                "next-period platform disappearance; "
                "not business failure or business survival"
            )
    },


    # ----------------------------------------------------------------------------------------------
    # Development design
    # ----------------------------------------------------------------------------------------------

    "development_design": {

        "development_transitions":
            EXPECTED_TRANSITIONS,

        "locked_final_transition":
            "T10",

        "t10_used_for_model_selection":
            False,

        "t10_used_for_feature_selection":
            False,

        "t10_used_for_threshold_selection":
            False,

        "t10_used_for_calibration_selection":
            False,

        "rolling_origin_rule":
            (
                "expanding-window development "
                "evaluation through T9"
            )
    },


    # ----------------------------------------------------------------------------------------------
    # Final primary model
    # ----------------------------------------------------------------------------------------------

    "final_primary_model": {

        "algorithm":
            "XGBoost",

        "feature_set":
            "A5_STRUCTURED_FULL",

        "n_predictors":
            len(
                A5_FEATURES
            ),

        "predictors":
            A5_FEATURES,

        "review_semantic_embeddings_in_final_model":
            False,

        "review_semantic_embeddings_role":
            (
                "development multimodal comparator "
                "and ablation only"
            )
    },


    # ----------------------------------------------------------------------------------------------
    # Fixed hyperparameters from development protocol
    # ----------------------------------------------------------------------------------------------

    "xgboost_hyperparameters": {

        "n_estimators":
            500,

        "max_depth":
            6,

        "learning_rate":
            0.05,

        "subsample":
            0.8,

        "colsample_bytree":
            0.8,

        "min_child_weight":
            5,

        "reg_lambda":
            1.0,

        "tree_method":
            "hist"
    },


    # ----------------------------------------------------------------------------------------------
    # Development performance
    # ----------------------------------------------------------------------------------------------

    "development_performance": {

        "macro_pr_auc_disappearance":
            macro_pr,

        "macro_roc_auc":
            macro_roc,

        "macro_brier_raw":
            macro_brier
    },


    # ----------------------------------------------------------------------------------------------
    # Frozen classification threshold
    # ----------------------------------------------------------------------------------------------

    "classification_threshold": {

        "probability_orientation":
            "disappearance",

        "threshold":
            observed_threshold,

        "selection_data":
            (
                "pooled development rolling-origin "
                "predictions T4-T9"
            ),

        "selection_rule":
            (
                "maximum Youden J / "
                "maximum balanced accuracy"
            ),

        "t10_used":
            False
    },


    # ----------------------------------------------------------------------------------------------
    # Frozen calibration choice
    # ----------------------------------------------------------------------------------------------

    "calibration": {

        "selected_method":
            SELECTED_CALIBRATION,

        "raw_macro_brier":
            raw_brier,

        "platt_macro_brier":
            platt_brier,

        "relative_brier_improvement":
            relative_brier_improvement,

        "minimum_required_relative_brier_improvement":
            0.005,

        "selection_data":
            (
                "development rolling-origin "
                "predictions only"
            ),

        "t10_used":
            False
    },


    # ----------------------------------------------------------------------------------------------
    # Preprocessing
    # ----------------------------------------------------------------------------------------------

    "preprocessing": {

        "rule":
            (
                "training-data-only preprocessing; "
                "no global imputation"
            ),

        "final_refit_scope":
            (
                "T1-T9 only before locked "
                "T10 evaluation"
            ),

        "t10_preprocessing_fit":
            False
    },


    # ----------------------------------------------------------------------------------------------
    # Feature specification
    # ----------------------------------------------------------------------------------------------

    "feature_specification": {

        "structured_groups":
            STRUCTURED_GROUPS,

        "a5_feature_sha256":
            a5_feature_hash
    },


    # ----------------------------------------------------------------------------------------------
    # Upstream hashes
    # ----------------------------------------------------------------------------------------------

    "upstream_artifact_sha256":
        upstream_hashes,


    # ----------------------------------------------------------------------------------------------
    # Integrity checks
    # ----------------------------------------------------------------------------------------------

    "integrity_checks": {

        "all_model_feature_sets_complete_T4_T9":
            True,

        "t10_in_development_predictions":
            False,

        "duplicate_prediction_rows":
            duplicate_rows,

        "a5_predictor_count":
            len(
                A5_FEATURES
            ),

        "a5_semantic_overlap":
            len(
                semantic_overlap
            ),

        "selected_calibration":
            SELECTED_CALIBRATION
    },


    # ----------------------------------------------------------------------------------------------
    # Scientific interpretation constraints
    # ----------------------------------------------------------------------------------------------

    "scientific_interpretation_constraints": [

        (
            "Do not interpret platform persistence "
            "as business survival."
        ),

        (
            "Do not interpret platform disappearance "
            "as business failure."
        ),

        (
            "Do not claim occupancy or bookings "
            "from calendar availability."
        ),

        (
            "Do not claim realized revenue, "
            "profitability, yield, or ROI."
        ),

        (
            "Do not interpret model associations "
            "as causal effects."
        )
    ]
}


# ==================================================================================================
# 15. WRITE CANONICAL FREEZE JSON
# ==================================================================================================

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze_spec,
        f,
        indent=2,
        ensure_ascii=False,
        sort_keys=True
    )


freeze_hash = (
    sha256_file(
        OUT_JSON
    )
)


# ==================================================================================================
# 16. WRITE SHA256 FILE
# ==================================================================================================

with open(
    OUT_HASH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        f"{freeze_hash}  "
        f"{OUT_JSON.name}\n"
    )


# ==================================================================================================
# 17. SAVE QC TABLE
# ==================================================================================================

qc_rows = [

    {
        "check":
            "all_model_feature_sets_complete_T4_T9",

        "result":
            "PASS"
    },

    {
        "check":
            "T10_absent_from_development_predictions",

        "result":
            "PASS"
    },

    {
        "check":
            "T10_absent_from_transition_bearing_IB_CSVs",

        "result":
            "PASS"
    },

    {
        "check":
            "duplicate_prediction_rows",

        "result":
            duplicate_rows
    },

    {
        "check":
            "A5_STRUCTURED_FULL_predictor_count",

        "result":
            len(
                A5_FEATURES
            )
    },

    {
        "check":
            "A5_semantic_overlap",

        "result":
            len(
                semantic_overlap
            )
    },

    {
        "check":
            "frozen_threshold",

        "result":
            observed_threshold
    },

    {
        "check":
            "selected_calibration",

        "result":
            SELECTED_CALIBRATION
    },

    {
        "check":
            "relative_Brier_improvement",

        "result":
            relative_brier_improvement
    },

    {
        "check":
            "final_freeze_sha256",

        "result":
            freeze_hash
    }
]


pd.DataFrame(
    qc_rows
).to_csv(
    OUT_QC,
    index=False
)


# ==================================================================================================
# 18. FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "FINAL PRIMARY MODEL FREEZE"
)

print("=" * 120)


print("\nMODEL")

print(
    "  Algorithm            : XGBoost"
)

print(
    "  Feature set          : A5_STRUCTURED_FULL"
)

print(
    f"  Predictor count      : "
    f"{len(A5_FEATURES)}"
)

print(
    "  Semantic embeddings  : "
    "EXCLUDED from final primary model"
)


print("\nDEVELOPMENT PERFORMANCE")

print(
    f"  Macro ROC-AUC        : "
    f"{macro_roc:.6f}"
)

print(
    f"  Macro PR-AUC         : "
    f"{macro_pr:.6f}"
)

print(
    f"  Raw Macro Brier      : "
    f"{macro_brier:.6f}"
)


print("\nDECISION RULE")

print(
    "  Disappearance threshold : "
    f"{observed_threshold:.8f}"
)

print(
    "  Calibration             : "
    f"{SELECTED_CALIBRATION}"
)

print(
    "  Calibration Brier gain  : "
    f"{relative_brier_improvement:.2%}"
)


print("\nLEAKAGE CONTROL")

print(
    "  Development folds       : "
    "T4-T9"
)

print(
    "  T10 used                : NO"
)

print(
    "  Final T10 status        : "
    "STILL LOCKED"
)


print("\nFINGERPRINTS")

print(
    "  A5 feature hash:"
)

print(
    f"    {a5_feature_hash}"
)

print(
    "\n  Final freeze hash:"
)

print(
    f"    {freeze_hash}"
)


print("\nSAVED")

print(
    f"  {OUT_JSON}"
)

print(
    f"  {OUT_HASH}"
)

print(
    f"  {OUT_QC}"
)


print("\n" + "=" * 120)

print(
    "PRODUCTION BLOCK I-C STATUS: PASS — "
    "PRIMARY DEVELOPMENT MODEL OFFICIALLY FROZEN"
)

print("=" * 120)


print(
    "\nDO NOT MODIFY FEATURES, MODEL HYPERPARAMETERS, "
    "THRESHOLD, OR CALIBRATION AFTER THIS POINT."
)

print(
    "T10 remains locked. Review this audit before running "
    "the final T1-T9 -> T10 evaluation."
)

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK J0
# PRE-FLIGHT INSPECTION BEFORE THE ONE-TIME LOCKED T10 EVALUATION
# NO MODEL FITTING — NO T10 EVALUATION
# ==================================================================================================

from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

PANEL_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "modelling"
    / "modelling_panel_T1_T10.parquet"
)

PRED_PATH = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
    / "predictions"
    / "IB_all_development_predictions.parquet"
)

FREEZE_PATH = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
    / "freeze"
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.json"
)

GROUPS_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "modelling"
    / "feature_groups_FINAL_FROZEN.json"
)

print("=" * 110)
print("PRODUCTION BLOCK J0 — PRE-FLIGHT")
print("NO T10 EVALUATION")
print("=" * 110)

for p in [PANEL_PATH, PRED_PATH, FREEZE_PATH, GROUPS_PATH]:
    print(("FOUND   " if p.exists() else "MISSING "), p)

assert PANEL_PATH.exists()
assert PRED_PATH.exists()
assert FREEZE_PATH.exists()
assert GROUPS_PATH.exists()

# --------------------------------------------------------------------------------
# Read schemas / frozen specifications
# --------------------------------------------------------------------------------

panel = pd.read_parquet(PANEL_PATH)

pred = pd.read_parquet(PRED_PATH)

with open(FREEZE_PATH, "r", encoding="utf-8") as f:
    freeze = json.load(f)

with open(GROUPS_PATH, "r", encoding="utf-8") as f:
    groups = json.load(f)

print("\n" + "=" * 110)
print("PANEL")
print("=" * 110)

print("Shape:", panel.shape)
print("\nColumns:")
print(panel.columns.tolist())

print("\n" + "=" * 110)
print("DEVELOPMENT PREDICTIONS")
print("=" * 110)

print("Shape:", pred.shape)
print("\nColumns:")
print(pred.columns.tolist())

print("\nUnique models:")
print(pred["model"].value_counts(dropna=False))

print("\nUnique feature sets:")
print(pred["feature_set"].value_counts(dropna=False))

print("\nTransitions:")
print(pred["transition"].value_counts(dropna=False).sort_index())

# --------------------------------------------------------------------------------
# Specifically inspect frozen A5 development predictions
# --------------------------------------------------------------------------------

a5 = pred.loc[
    (pred["model"].astype(str).str.upper() == "XGB")
    &
    (pred["feature_set"].astype(str) == "A5_STRUCTURED_FULL")
].copy()

print("\n" + "=" * 110)
print("A5 XGB DEVELOPMENT PREDICTIONS")
print("=" * 110)

print("Shape:", a5.shape)
print("Columns:", a5.columns.tolist())

print("\nRows by transition:")
print(a5["transition"].value_counts().sort_index())

print("\nFirst 5 rows:")
print(a5.head().to_string(index=False))

print("\nMissing values:")
print(a5.isna().sum().to_string())

# --------------------------------------------------------------------------------
# Frozen specification
# --------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FROZEN PRIMARY SPECIFICATION")
print("=" * 110)

print("Algorithm:")
print(freeze["final_primary_model"]["algorithm"])

print("\nFeature set:")
print(freeze["final_primary_model"]["feature_set"])

print("\nNumber predictors:")
print(freeze["final_primary_model"]["n_predictors"])

print("\nThreshold:")
print(freeze["classification_threshold"]["threshold"])

print("\nCalibration:")
print(freeze["calibration"]["selected_method"])

print("\nHyperparameters:")
print(
    json.dumps(
        freeze["xgboost_hyperparameters"],
        indent=2
    )
)

print("\nA5 predictors:")
for i, col in enumerate(
    freeze["final_primary_model"]["predictors"],
    start=1
):
    print(f"{i:02d}. {col}")

# --------------------------------------------------------------------------------
# Identify key panel columns
# --------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("KEY PANEL COLUMNS / CANDIDATES")
print("=" * 110)

keywords = [
    "transition",
    "persistence",
    "listing",
    "target",
    "disappear",
    "snapshot",
    "anchor"
]

for key in keywords:
    matches = [
        c for c in panel.columns
        if key.lower() in c.lower()
    ]

    print(f"\n{key.upper()}:")
    print(matches)

# --------------------------------------------------------------------------------
# Check feature existence — does NOT evaluate T10
# --------------------------------------------------------------------------------

frozen_features = freeze["final_primary_model"]["predictors"]

missing_features = [
    c for c in frozen_features
    if c not in panel.columns
]

print("\n" + "=" * 110)
print("FEATURE INTEGRITY")
print("=" * 110)

print("Frozen feature count:", len(frozen_features))
print("Missing frozen features:", len(missing_features))

if missing_features:
    print(missing_features)

assert len(frozen_features) == 53
assert len(missing_features) == 0

# --------------------------------------------------------------------------------
# Do NOT inspect T10 outcome values here.
# Only verify the transition label exists.
# --------------------------------------------------------------------------------

transition_candidates = [
    c for c in panel.columns
    if c.lower() in [
        "transition",
        "transition_id",
        "origin_transition"
    ]
]

print("\nTransition-column candidates:")
print(transition_candidates)

if transition_candidates:
    tc = transition_candidates[0]

    print("\nPanel row counts by transition:")
    print(
        panel[tc]
        .value_counts(dropna=False)
        .sort_index()
    )

print("\n" + "=" * 110)
print("PRODUCTION BLOCK J0 STATUS: PASS")
print("T10 HAS NOT BEEN EVALUATED.")
print("=" * 110)

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK J00
# RECOVER EXACT BLOCK I-B IMPLEMENTATION BEFORE OPENING LOCKED T10
#
# PURPOSE:
#   - Find the exact code used for Block I-B
#   - Recover XGB preprocessing + PLATT calibration implementation
#   - DOES NOT evaluate T10
#   - DOES NOT fit any model
#   - DOES NOT change frozen specification
# ==================================================================================================

from pathlib import Path
import json
import re

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

print("=" * 120)
print("PRODUCTION BLOCK J00")
print("RECOVER EXACT I-B IMPLEMENTATION — T10 REMAINS LOCKED")
print("=" * 120)

# --------------------------------------------------------------------------------------------------
# 1. Search likely code locations
# --------------------------------------------------------------------------------------------------

search_roots = [
    PROJECT_ROOT / "04_code",
    PROJECT_ROOT,
]

extensions = {".py", ".ipynb"}

files = []

for root in search_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if (
            p.is_file()
            and p.suffix.lower() in extensions
            and ".ipynb_checkpoints" not in str(p)
        ):
            files.append(p)

# Remove duplicates
files = list(dict.fromkeys(files))

print(f"\nPython/notebook files found: {len(files):,}")

# --------------------------------------------------------------------------------------------------
# 2. Extract readable code/text
# --------------------------------------------------------------------------------------------------

def read_code_file(path):
    try:
        if path.suffix.lower() == ".py":
            return path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        elif path.suffix.lower() == ".ipynb":
            obj = json.loads(
                path.read_text(
                    encoding="utf-8",
                    errors="ignore"
                )
            )

            blocks = []

            for i, cell in enumerate(obj.get("cells", [])):
                source = "".join(cell.get("source", []))

                if source.strip():
                    blocks.append(
                        f"\n\n# ===== NOTEBOOK CELL {i} =====\n{source}"
                    )

            return "".join(blocks)

    except Exception:
        return ""

    return ""


# --------------------------------------------------------------------------------------------------
# 3. Find files likely containing I-B
# --------------------------------------------------------------------------------------------------

strong_terms = [
    "PRODUCTION BLOCK I-B",
    "A5_STRUCTURED_FULL",
    "IB_calibration_method_summary",
    "IB_all_development_predictions",
    "MLP_FULL_MULTIMODAL",
]

candidate_files = []

for p in files:

    txt = read_code_file(p)

    score = sum(
        term.lower() in txt.lower()
        for term in strong_terms
    )

    if score >= 2:
        candidate_files.append(
            (score, p, txt)
        )

candidate_files.sort(
    key=lambda x: x[0],
    reverse=True
)

print("\n" + "=" * 120)
print("LIKELY BLOCK I-B SOURCE FILES")
print("=" * 120)

if not candidate_files:
    print(
        "\nNo saved .py/.ipynb containing Block I-B was found."
    )

else:
    for score, p, _ in candidate_files[:10]:
        print(f"[score={score}] {p}")


# --------------------------------------------------------------------------------------------------
# 4. Search specifically for preprocessing/calibration implementation
# --------------------------------------------------------------------------------------------------

patterns = [
    "LogisticRegression",
    "IsotonicRegression",
    "PLATT",
    "platt",
    "calibr",
    "SimpleImputer",
    "OneHotEncoder",
    "ColumnTransformer",
    "XGBClassifier",
    "median",
    "most_frequent",
    "add_indicator",
]

print("\n" + "=" * 120)
print("EXACT I-B PREPROCESSING / CALIBRATION CODE")
print("=" * 120)

found_any = False

for score, p, txt in candidate_files[:10]:

    lines = txt.splitlines()

    hit_indices = []

    for i, line in enumerate(lines):

        if any(
            pat.lower() in line.lower()
            for pat in patterns
        ):
            hit_indices.append(i)

    if not hit_indices:
        continue

    found_any = True

    print("\n\n")
    print("#" * 120)
    print("FILE:")
    print(p)
    print("#" * 120)

    # Merge nearby hits into ranges
    ranges = []

    for idx in hit_indices:

        start = max(0, idx - 12)
        end = min(len(lines), idx + 20)

        if not ranges or start > ranges[-1][1] + 1:
            ranges.append([start, end])
        else:
            ranges[-1][1] = max(
                ranges[-1][1],
                end
            )

    for start, end in ranges:

        print(
            f"\n--- lines approximately {start+1} to {end} ---"
        )

        for j in range(start, end):
            print(
                f"{j+1:05d}: {lines[j]}"
            )


# --------------------------------------------------------------------------------------------------
# 5. Also inspect frozen JSON for any stored calibration implementation
# --------------------------------------------------------------------------------------------------

FREEZE_PATH = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
    / "freeze"
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.json"
)

print("\n" + "=" * 120)
print("FROZEN JSON — CALIBRATION / PREPROCESSING CONTENT")
print("=" * 120)

with open(
    FREEZE_PATH,
    "r",
    encoding="utf-8"
) as f:
    freeze = json.load(f)


def walk_json(obj, path="root"):

    if isinstance(obj, dict):

        for k, v in obj.items():

            new_path = f"{path}.{k}"

            if any(
                word in str(k).lower()
                for word in [
                    "calib",
                    "platt",
                    "preprocess",
                    "imput",
                    "categor",
                    "numeric",
                    "threshold"
                ]
            ):
                print(
                    f"\n{new_path}:"
                )
                print(
                    json.dumps(
                        v,
                        indent=2,
                        default=str
                    )
                )

            walk_json(
                v,
                new_path
            )

    elif isinstance(obj, list):

        for i, v in enumerate(obj):
            walk_json(
                v,
                f"{path}[{i}]"
            )


walk_json(freeze)


# --------------------------------------------------------------------------------------------------
# 6. HARD SAFETY CONFIRMATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("SAFETY CHECK")
print("=" * 120)

print("No T10 model fitting performed.")
print("No T10 predictions generated.")
print("No T10 metrics calculated.")
print("Frozen model specification unchanged.")

if found_any:
    print(
        "\nSTATUS: SOURCE CODE FOUND — COPY THE OUTPUT ABOVE BACK TO CHAT."
    )
else:
    print(
        "\nSTATUS: SAVED I-B SOURCE NOT FOUND."
    )
    print(
        "If Block I-B was executed only as an unsaved notebook cell, "
        "send/copy that original I-B cell from your notebook."
    )

print("=" * 120)

In [ ]:
# ==================================================================================================
# PRODUCTION BLOCK J — FINAL CORRECTED VERSION
# FINAL LOCKED T10 PRIMARY EVALUATION
#
# FROZEN DESIGN
#   Positive event : next-period disappearance = 1
#   Model          : XGBoost
#   Feature set    : A5_STRUCTURED_FULL
#   Predictors     : 53
#   Final training : T1-T9
#   Locked test    : T10
#   Calibration    : PLATT
#                    LogisticRegression on logit(raw XGB probability)
#                    fitted ONLY from development OOF predictions T4-T9
#   Threshold      : 0.0267906449735164
#                    applied to RAW XGBoost disappearance probability
#
# IMPORTANT
#   - No T10 tuning.
#   - No feature redesign.
#   - No model redesign.
#   - No threshold redesign.
#   - No calibration redesign.
#   - After successful execution, T10 is officially OPEN.
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import warnings
import joblib

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ==================================================================================================
# 0. PROJECT ROOT
# ==================================================================================================

def find_project_root():

    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
    ]

    for candidate in candidates:

        if (
            (candidate / "02_intermediate").exists()
            and
            (candidate / "05_results").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root()

print("=" * 120)
print("PRODUCTION BLOCK J")
print("FINAL LOCKED T10 PRIMARY EVALUATION")
print("=" * 120)

print("\nPROJECT ROOT:")
print(PROJECT_ROOT)


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PANEL_PATH = (
    PROJECT_ROOT
    / "02_intermediate"
    / "modelling"
    / "modelling_panel_T1_T10.parquet"
)

DEV_PRED_PATH = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
    / "predictions"
    / "IB_all_development_predictions.parquet"
)

FREEZE_DIR = (
    PROJECT_ROOT
    / "05_results"
    / "development_modelling_IB"
    / "freeze"
)

FREEZE_PATH = (
    FREEZE_DIR
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.json"
)

FREEZE_SHA_PATH = (
    FREEZE_DIR
    / "FINAL_PRIMARY_MODEL_SPEC_FROZEN.sha256"
)

OUT_DIR = (
    PROJECT_ROOT
    / "05_results"
    / "final_locked_T10_primary"
)

TABLE_DIR = OUT_DIR / "tables"
PRED_DIR = OUT_DIR / "predictions"
MODEL_DIR = OUT_DIR / "models"
PROV_DIR = OUT_DIR / "provenance"

for folder in [
    OUT_DIR,
    TABLE_DIR,
    PRED_DIR,
    MODEL_DIR,
    PROV_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 2. HELPER FUNCTIONS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def logit_probability(
    p,
    eps=1e-6
):

    p = np.asarray(
        p,
        dtype=float
    )

    p = np.clip(
        p,
        eps,
        1 - eps
    )

    return np.log(
        p / (1 - p)
    ).reshape(-1, 1)


def probability_metrics(
    y,
    p
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )

    p = np.clip(
        p,
        1e-15,
        1 - 1e-15
    )

    return {
        "roc_auc":
            float(
                roc_auc_score(
                    y,
                    p
                )
            ),

        "pr_auc":
            float(
                average_precision_score(
                    y,
                    p
                )
            ),

        "brier":
            float(
                brier_score_loss(
                    y,
                    p
                )
            ),

        "log_loss":
            float(
                log_loss(
                    y,
                    p,
                    labels=[0, 1]
                )
            )
    }


def classification_metrics_at_threshold(
    y,
    p,
    threshold
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )

    pred = (
        p >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred,
            labels=[0, 1]
        )
        .ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    balanced_accuracy = (
        sensitivity
        + specificity
    ) / 2

    return {
        "threshold":
            float(threshold),

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "sensitivity_disappearance":
            float(sensitivity),

        "specificity":
            float(specificity),

        "balanced_accuracy":
            float(balanced_accuracy),

        "precision_disappearance":
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "recall_disappearance":
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "f1_disappearance":
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "predicted_positive_rate":
            float(
                pred.mean()
            )
    }


def make_calibration_table(
    y,
    p,
    n_bins=10
):

    df = pd.DataFrame({
        "y":
            np.asarray(
                y,
                dtype=int
            ),

        "p":
            np.asarray(
                p,
                dtype=float
            )
    })

    df["bin_internal"] = pd.qcut(
        df["p"],
        q=n_bins,
        duplicates="drop"
    )

    out = (
        df
        .groupby(
            "bin_internal",
            observed=True
        )
        .agg(
            n=(
                "y",
                "size"
            ),

            mean_predicted_probability=(
                "p",
                "mean"
            ),

            observed_disappearance_rate=(
                "y",
                "mean"
            ),

            min_probability=(
                "p",
                "min"
            ),

            max_probability=(
                "p",
                "max"
            )
        )
        .reset_index(
            drop=True
        )
    )

    out.insert(
        0,
        "bin",
        np.arange(
            1,
            len(out) + 1
        )
    )

    return out


# ==================================================================================================
# 3. REQUIRED FILE GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("J1 — REQUIRED FILES")
print("=" * 120)

required_files = [
    PANEL_PATH,
    DEV_PRED_PATH,
    FREEZE_PATH,
    FREEZE_SHA_PATH
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"HARD FAIL — missing required file:\n{path}"
        )

    print(
        "FOUND:",
        path
    )


# ==================================================================================================
# 4. VERIFY COMPLETE FROZEN SPECIFICATION HASH
# ==================================================================================================

print("\n" + "=" * 120)
print("J2 — FROZEN SPECIFICATION INTEGRITY")
print("=" * 120)

actual_freeze_hash = (
    sha256_file(
        FREEZE_PATH
    )
)

sha_text = (
    FREEZE_SHA_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

recorded_freeze_hash = (
    sha_text.split()[0]
)

EXPECTED_BLOCK_IC_HASH = (
    "d830c39d141c08c55801ac63819813c8"
    "a37ba9e4a5b4f1ebd487e1d11557b761"
)

print(
    "Actual freeze hash  :",
    actual_freeze_hash
)

print(
    "Recorded freeze hash:",
    recorded_freeze_hash
)

if (
    actual_freeze_hash
    != recorded_freeze_hash
):

    raise RuntimeError(
        "HARD FAIL — frozen specification differs from SHA256 record."
    )

if (
    actual_freeze_hash
    != EXPECTED_BLOCK_IC_HASH
):

    raise RuntimeError(
        "HARD FAIL — frozen specification differs from Block I-C freeze."
    )

print(
    "FREEZE SPECIFICATION HASH GATE: PASS"
)


# ==================================================================================================
# 5. LOAD FROZEN SPECIFICATION
# ==================================================================================================

with open(
    FREEZE_PATH,
    "r",
    encoding="utf-8"
) as f:

    freeze = json.load(f)


if (
    freeze[
        "final_primary_model"
    ][
        "algorithm"
    ]
    != "XGBoost"
):

    raise RuntimeError(
        "HARD FAIL — frozen algorithm is not XGBoost."
    )


if (
    freeze[
        "final_primary_model"
    ][
        "feature_set"
    ]
    != "A5_STRUCTURED_FULL"
):

    raise RuntimeError(
        "HARD FAIL — frozen feature set is not A5_STRUCTURED_FULL."
    )


if (
    int(
        freeze[
            "final_primary_model"
        ][
            "n_predictors"
        ]
    )
    != 53
):

    raise RuntimeError(
        "HARD FAIL — frozen predictor count is not 53."
    )


if (
    freeze[
        "calibration"
    ][
        "selected_method"
    ]
    != "PLATT"
):

    raise RuntimeError(
        "HARD FAIL — frozen calibration method is not PLATT."
    )


FEATURES = list(
    freeze[
        "final_primary_model"
    ][
        "predictors"
    ]
)

THRESHOLD = float(
    freeze[
        "classification_threshold"
    ][
        "threshold"
    ]
)

XGB_PARAMS = dict(
    freeze[
        "xgboost_hyperparameters"
    ]
)


print(
    "\nAlgorithm   : XGBoost"
)

print(
    "Feature set : A5_STRUCTURED_FULL"
)

print(
    "Predictors  :",
    len(FEATURES)
)

print(
    "Threshold   :",
    THRESHOLD
)

print(
    "Calibration : PLATT"
)


# ==================================================================================================
# 6. A5 FEATURE INTEGRITY
#
# IMPORTANT:
# Do NOT recompute the historical feature-list SHA using a new serialization convention.
#
# The authoritative predictor list is contained inside the complete frozen JSON.
# The complete frozen JSON has already passed the original Block I-C SHA256 integrity check above.
# ==================================================================================================

print("\n" + "=" * 120)
print("J3 — A5 FEATURE INTEGRITY")
print("=" * 120)


if len(FEATURES) != 53:

    raise RuntimeError(
        f"HARD FAIL — expected 53 frozen predictors; found {len(FEATURES)}."
    )


if len(set(FEATURES)) != 53:

    raise RuntimeError(
        "HARD FAIL — duplicate predictors in frozen A5 specification."
    )


if any(
    (not isinstance(col, str))
    or
    (not col.strip())
    for col in FEATURES
):

    raise RuntimeError(
        "HARD FAIL — invalid predictor name in frozen specification."
    )


print(
    "Frozen A5 predictor count:",
    len(FEATURES)
)

print(
    "Duplicate predictor count:",
    len(FEATURES)
    - len(set(FEATURES))
)

print(
    "Authoritative feature source: FINAL_PRIMARY_MODEL_SPEC_FROZEN.json"
)

print(
    "Complete frozen specification SHA256:",
    actual_freeze_hash
)

print(
    "A5 FEATURE INTEGRITY GATE: PASS"
)


# ==================================================================================================
# 7. LOAD MODELLING PANEL
# ==================================================================================================

print("\n" + "=" * 120)
print("J4 — LOAD MODELLING PANEL")
print("=" * 120)


panel = pd.read_parquet(
    PANEL_PATH
)


print(
    "Panel shape:",
    panel.shape
)


required_panel_columns = (
    [
        "transition",
        "listing_id"
    ]
    + FEATURES
)


missing_panel_columns = [
    col
    for col in required_panel_columns
    if col not in panel.columns
]


if missing_panel_columns:

    raise RuntimeError(
        "HARD FAIL — required panel columns missing:\n"
        + str(missing_panel_columns)
    )


print(
    "All frozen predictors found in panel: PASS"
)


# ==================================================================================================
# 8. PRIMARY OUTCOME
# ==================================================================================================

print("\n" + "=" * 120)
print("J5 — PRIMARY OUTCOME")
print("=" * 120)


if (
    "next_period_persistence"
    in panel.columns
):

    if (
        panel[
            "next_period_persistence"
        ]
        .isna()
        .any()
    ):

        raise RuntimeError(
            "HARD FAIL — missing next_period_persistence."
        )

    panel[
        "y_disappear"
    ] = (
        1
        - panel[
            "next_period_persistence"
        ].astype(int)
    )


elif (
    "y_disappear"
    in panel.columns
):

    if (
        panel[
            "y_disappear"
        ]
        .isna()
        .any()
    ):

        raise RuntimeError(
            "HARD FAIL — missing y_disappear."
        )

    panel[
        "y_disappear"
    ] = (
        panel[
            "y_disappear"
        ]
        .astype(int)
    )


else:

    raise RuntimeError(
        "HARD FAIL — primary outcome not found."
    )


if not set(
    panel[
        "y_disappear"
    ].unique()
).issubset(
    {0, 1}
):

    raise RuntimeError(
        "HARD FAIL — y_disappear contains values other than 0/1."
    )


print(
    "Positive event: next-period disappearance = 1"
)

print(
    "PRIMARY OUTCOME GATE: PASS"
)


# ==================================================================================================
# 9. FINAL TEMPORAL SPLIT
# ==================================================================================================

print("\n" + "=" * 120)
print("J6 — FINAL TEMPORAL SPLIT")
print("=" * 120)


TRAIN_TRANSITIONS = [
    "T1",
    "T2",
    "T3",
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9"
]

TEST_TRANSITION = "T10"


train_df = (
    panel.loc[
        panel[
            "transition"
        ].isin(
            TRAIN_TRANSITIONS
        )
    ]
    .copy()
)


test_df = (
    panel.loc[
        panel[
            "transition"
        ]
        == TEST_TRANSITION
    ]
    .copy()
)


print(
    "T1-T9 training rows:",
    f"{len(train_df):,}"
)

print(
    "T10 locked rows:",
    f"{len(test_df):,}"
)


if len(train_df) != 440_044:

    raise RuntimeError(
        f"HARD FAIL — expected 440,044 T1-T9 rows; "
        f"found {len(train_df):,}."
    )


if len(test_df) != 51_096:

    raise RuntimeError(
        f"HARD FAIL — expected 51,096 T10 rows; "
        f"found {len(test_df):,}."
    )


if (
    train_df[
        [
            "transition",
            "listing_id"
        ]
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "HARD FAIL — duplicate T1-T9 listing-origin rows."
    )


if (
    test_df[
        [
            "transition",
            "listing_id"
        ]
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "HARD FAIL — duplicate T10 listing-origin rows."
    )


y_train = (
    train_df[
        "y_disappear"
    ]
    .astype(int)
    .to_numpy()
)


y_test = (
    test_df[
        "y_disappear"
    ]
    .astype(int)
    .to_numpy()
)


print(
    "FINAL TEMPORAL SPLIT GATE: PASS"
)


# ==================================================================================================
# 10. OPEN LOCKED T10 OUTCOME
# ==================================================================================================

print("\n" + "=" * 120)
print("J7 — LOCKED T10 OUTCOME QC")
print("T10 IS NOW OFFICIALLY OPEN")
print("=" * 120)


n_t10 = int(
    len(y_test)
)

n_disappear = int(
    y_test.sum()
)

n_persist = int(
    n_t10
    - n_disappear
)

disappearance_rate = (
    n_disappear
    / n_t10
)

persistence_rate = (
    n_persist
    / n_t10
)


print(
    f"N                        : {n_t10:,}"
)

print(
    f"Persistent               : {n_persist:,}"
)

print(
    f"Disappeared              : {n_disappear:,}"
)

print(
    f"Persistence rate         : {persistence_rate:.6%}"
)

print(
    f"Disappearance prevalence : {disappearance_rate:.6%}"
)


if n_t10 != 51_096:

    raise RuntimeError(
        "HARD FAIL — unexpected T10 sample size."
    )


if n_persist != 49_949:

    raise RuntimeError(
        "HARD FAIL — unexpected T10 persistence count."
    )


if n_disappear != 1_147:

    raise RuntimeError(
        "HARD FAIL — unexpected T10 disappearance count."
    )


print(
    "T10 OUTCOME GATE: PASS"
)


# ==================================================================================================
# 11. PREPROCESSING — TRAINING DATA ONLY
# ==================================================================================================

print("\n" + "=" * 120)
print("J8 — TRAINING-ONLY PREPROCESSING")
print("=" * 120)


X_train = (
    train_df[
        FEATURES
    ]
    .copy()
)


X_test = (
    test_df[
        FEATURES
    ]
    .copy()
)


categorical_features = []

numeric_features = []


for col in FEATURES:

    dtype = X_train[col].dtype

    if (
        pd.api.types.is_object_dtype(dtype)
        or
        pd.api.types.is_string_dtype(dtype)
        or
        isinstance(
            dtype,
            pd.CategoricalDtype
        )
    ):

        categorical_features.append(
            col
        )

    else:

        numeric_features.append(
            col
        )


print(
    "Numeric predictors:",
    len(numeric_features)
)

print(
    "Categorical predictors:",
    len(categorical_features)
)

print(
    "Categorical variables:",
    categorical_features
)


if (
    len(numeric_features)
    + len(categorical_features)
    != 53
):

    raise RuntimeError(
        "HARD FAIL — numeric + categorical predictors do not total 53."
    )


numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),

        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],

    remainder="drop",

    sparse_threshold=0.3
)


print(
    "PREPROCESSING CONFIGURATION: PASS"
)


# ==================================================================================================
# 12. FINAL XGBOOST — FIT T1-T9 ONLY
# ==================================================================================================

print("\n" + "=" * 120)
print("J9 — FINAL XGBOOST FIT")
print("=" * 120)


print(
    "Frozen XGBoost parameters:"
)

print(
    json.dumps(
        XGB_PARAMS,
        indent=2
    )
)


final_xgb = XGBClassifier(
    **XGB_PARAMS
)


final_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            preprocessor
        ),

        (
            "model",
            final_xgb
        )
    ]
)


print(
    "\nFitting final XGBoost on T1-T9..."
)


final_pipeline.fit(
    X_train,
    y_train
)


print(
    "FINAL XGBOOST FIT: COMPLETE"
)


# ==================================================================================================
# 13. FINAL PLATT CALIBRATOR
#     DEVELOPMENT OOF T4-T9 ONLY
# ==================================================================================================

print("\n" + "=" * 120)
print("J10 — FINAL PLATT CALIBRATOR")
print("=" * 120)


dev_pred = pd.read_parquet(
    DEV_PRED_PATH
)


required_dev_columns = [
    "model",
    "feature_set",
    "transition",
    "listing_id",
    "y_disappear",
    "p_disappear"
]


missing_dev_columns = [
    col
    for col in required_dev_columns
    if col not in dev_pred.columns
]


if missing_dev_columns:

    raise RuntimeError(
        "HARD FAIL — missing development prediction columns:\n"
        + str(missing_dev_columns)
    )


selected_dev = (
    dev_pred.loc[
        (
            dev_pred[
                "model"
            ]
            .astype(str)
            .str.upper()
            == "XGB"
        )
        &
        (
            dev_pred[
                "feature_set"
            ]
            .astype(str)
            == "A5_STRUCTURED_FULL"
        )
    ]
    .copy()
)


EXPECTED_DEV_TRANSITIONS = {
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9"
}


observed_dev_transitions = set(
    selected_dev[
        "transition"
    ]
    .astype(str)
    .unique()
)


print(
    "Development calibration transitions:",
    sorted(
        observed_dev_transitions
    )
)

print(
    "Development OOF calibration rows:",
    f"{len(selected_dev):,}"
)


if (
    observed_dev_transitions
    != EXPECTED_DEV_TRANSITIONS
):

    raise RuntimeError(
        "HARD FAIL — Platt calibration input must contain exactly T4-T9."
    )


if len(selected_dev) != 293_769:

    raise RuntimeError(
        f"HARD FAIL — expected 293,769 A5 OOF rows; "
        f"found {len(selected_dev):,}."
    )


if (
    selected_dev[
        [
            "transition",
            "listing_id"
        ]
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "HARD FAIL — duplicate development OOF predictions."
    )


if (
    selected_dev[
        [
            "y_disappear",
            "p_disappear"
        ]
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "HARD FAIL — missing development calibration values."
    )


if "T10" in observed_dev_transitions:

    raise RuntimeError(
        "HARD FAIL — T10 entered calibration fitting."
    )


p_dev_raw = (
    selected_dev[
        "p_disappear"
    ]
    .astype(float)
    .to_numpy()
)


y_dev = (
    selected_dev[
        "y_disappear"
    ]
    .astype(int)
    .to_numpy()
)


if (
    np.any(
        p_dev_raw < 0
    )
    or
    np.any(
        p_dev_raw > 1
    )
):

    raise RuntimeError(
        "HARD FAIL — development probabilities outside [0,1]."
    )


dev_logit = logit_probability(
    p_dev_raw,
    eps=1e-6
)


final_platt = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)


final_platt.fit(
    dev_logit,
    y_dev
)


platt_coefficient = float(
    final_platt
    .coef_
    .ravel()[0]
)


platt_intercept = float(
    final_platt
    .intercept_[0]
)


print(
    "Platt fit scope: development OOF T4-T9 ONLY"
)

print(
    "T10 used for Platt fitting: NO"
)

print(
    "Platt coefficient:",
    platt_coefficient
)

print(
    "Platt intercept:",
    platt_intercept
)


# ==================================================================================================
# 14. ONE-TIME T10 PREDICTION
# ==================================================================================================

print("\n" + "=" * 120)
print("J11 — ONE-TIME LOCKED T10 PREDICTION")
print("=" * 120)


p_t10_raw = (
    final_pipeline
    .predict_proba(
        X_test
    )[:, 1]
)


if (
    np.isnan(
        p_t10_raw
    )
    .any()
):

    raise RuntimeError(
        "HARD FAIL — NaN raw T10 probabilities."
    )


if (
    np.any(
        p_t10_raw < 0
    )
    or
    np.any(
        p_t10_raw > 1
    )
):

    raise RuntimeError(
        "HARD FAIL — raw T10 probabilities outside [0,1]."
    )


t10_logit = logit_probability(
    p_t10_raw,
    eps=1e-6
)


p_t10_calibrated = (
    final_platt
    .predict_proba(
        t10_logit
    )[:, 1]
)


if (
    np.isnan(
        p_t10_calibrated
    )
    .any()
):

    raise RuntimeError(
        "HARD FAIL — NaN calibrated T10 probabilities."
    )


p_t10_persistence = (
    1
    - p_t10_calibrated
)


# IMPORTANT:
# The frozen threshold was selected from RAW development probabilities.
# Therefore it is applied to RAW T10 probabilities.

pred_t10_disappear = (
    p_t10_raw
    >= THRESHOLD
).astype(int)


pred_t10_persist = (
    1
    - pred_t10_disappear
)


print(
    "T10 predictions generated:",
    f"{len(p_t10_raw):,}"
)

print(
    "Frozen threshold:",
    THRESHOLD
)

print(
    "Threshold probability scale: RAW"
)

print(
    "ONE-TIME T10 PREDICTION: COMPLETE"
)


# ==================================================================================================
# 15. FINAL T10 PROBABILITY METRICS
# ==================================================================================================

print("\n" + "=" * 120)
print("J12 — FINAL T10 PROBABILITY METRICS")
print("=" * 120)


raw_metrics = probability_metrics(
    y_test,
    p_t10_raw
)


calibrated_metrics = probability_metrics(
    y_test,
    p_t10_calibrated
)


probability_results = pd.DataFrame(
    [
        {
            "transition":
                "T10",

            "probability":
                "RAW",

            **raw_metrics
        },

        {
            "transition":
                "T10",

            "probability":
                "PLATT_CALIBRATED",

            **calibrated_metrics
        }
    ]
)


# ==================================================================================================
# 16. FINAL FROZEN-THRESHOLD METRICS
# ==================================================================================================

threshold_metrics = (
    classification_metrics_at_threshold(
        y_test,
        p_t10_raw,
        THRESHOLD
    )
)


threshold_results = pd.DataFrame(
    [
        {
            "transition":
                "T10",

            "probability_used_for_threshold":
                "RAW",

            **threshold_metrics
        }
    ]
)


# ==================================================================================================
# 17. CALIBRATION DECILES
# ==================================================================================================

calibration_raw = (
    make_calibration_table(
        y_test,
        p_t10_raw,
        n_bins=10
    )
)


calibration_raw[
    "probability"
] = "RAW"


calibration_platt = (
    make_calibration_table(
        y_test,
        p_t10_calibrated,
        n_bins=10
    )
)


calibration_platt[
    "probability"
] = "PLATT_CALIBRATED"


calibration_table = pd.concat(
    [
        calibration_raw,
        calibration_platt
    ],
    ignore_index=True
)


# ==================================================================================================
# 18. T10 OUTCOME SUMMARY
# ==================================================================================================

outcome_qc = pd.DataFrame(
    [
        {
            "transition":
                "T10",

            "n":
                n_t10,

            "persistent_n":
                n_persist,

            "disappeared_n":
                n_disappear,

            "persistence_rate":
                float(
                    persistence_rate
                ),

            "disappearance_rate":
                float(
                    disappearance_rate
                )
        }
    ]
)


# ==================================================================================================
# 19. LISTING-LEVEL FINAL T10 PREDICTIONS
# ==================================================================================================

prediction_output = pd.DataFrame(
    {
        "transition":
            np.repeat(
                "T10",
                n_t10
            ),

        "listing_id":
            test_df[
                "listing_id"
            ]
            .to_numpy(),

        "y_disappear":
            y_test,

        "next_period_persistence":
            1
            - y_test,

        "p_disappear_raw":
            p_t10_raw,

        "p_disappear_calibrated":
            p_t10_calibrated,

        "p_persistence_calibrated":
            p_t10_persistence,

        "frozen_threshold_raw":
            np.repeat(
                THRESHOLD,
                n_t10
            ),

        "predicted_disappear_frozen_threshold":
            pred_t10_disappear,

        "predicted_persist_frozen_threshold":
            pred_t10_persist
    }
)


if len(
    prediction_output
) != 51_096:

    raise RuntimeError(
        "HARD FAIL — final T10 prediction row count incorrect."
    )


if (
    prediction_output[
        "listing_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "HARD FAIL — duplicate listing IDs in final T10 predictions."
    )


# ==================================================================================================
# 20. SAVE OUTPUT TABLES
# ==================================================================================================

PRED_OUT = (
    PRED_DIR
    / "J_final_locked_T10_primary_predictions.parquet"
)


OUTCOME_OUT = (
    TABLE_DIR
    / "J_T10_outcome_QC.csv"
)


PROBABILITY_OUT = (
    TABLE_DIR
    / "J_T10_probability_metrics.csv"
)


THRESHOLD_OUT = (
    TABLE_DIR
    / "J_T10_frozen_threshold_metrics.csv"
)


CALIBRATION_OUT = (
    TABLE_DIR
    / "J_T10_calibration_deciles.csv"
)


prediction_output.to_parquet(
    PRED_OUT,
    index=False
)


outcome_qc.to_csv(
    OUTCOME_OUT,
    index=False
)


probability_results.to_csv(
    PROBABILITY_OUT,
    index=False
)


threshold_results.to_csv(
    THRESHOLD_OUT,
    index=False
)


calibration_table.to_csv(
    CALIBRATION_OUT,
    index=False
)


# ==================================================================================================
# 21. SAVE FINAL MODEL + PLATT CALIBRATOR
# ==================================================================================================

FINAL_MODEL_PATH = (
    MODEL_DIR
    / "J_final_XGB_A5_T1_T9_pipeline.joblib"
)


FINAL_PLATT_PATH = (
    MODEL_DIR
    / "J_final_PLATT_development_OOF_T4_T9.joblib"
)


joblib.dump(
    final_pipeline,
    FINAL_MODEL_PATH
)


joblib.dump(
    final_platt,
    FINAL_PLATT_PATH
)


# ==================================================================================================
# 22. FINAL PROVENANCE
# ==================================================================================================

provenance = {

    "block":
        "J",

    "status":
        "FINAL LOCKED T10 PRIMARY EVALUATION",

    "outcome":
        "next-period platform disappearance",

    "positive_event":
        1,

    "model":
        "XGBoost",

    "feature_set":
        "A5_STRUCTURED_FULL",

    "n_predictors":
        53,

    "feature_specification_source":
        "FINAL_PRIMARY_MODEL_SPEC_FROZEN.json",

    "complete_frozen_specification_sha256":
        actual_freeze_hash,

    "train_transitions":
        TRAIN_TRANSITIONS,

    "test_transition":
        "T10",

    "training_rows":
        int(
            len(train_df)
        ),

    "test_rows":
        int(
            len(test_df)
        ),

    "xgboost_parameters":
        XGB_PARAMS,

    "preprocessing":
        {
            "numeric":
                "median imputation with missing indicators",

            "categorical":
                "most-frequent imputation plus one-hot encoding",

            "fit_scope":
                "T1-T9 only"
        },

    "calibration":
        {
            "method":
                "PLATT",

            "input":
                "logit(raw XGBoost disappearance probability)",

            "epsilon":
                1e-6,

            "fit_scope":
                "development OOF T4-T9 only",

            "n":
                int(
                    len(selected_dev)
                ),

            "coefficient":
                platt_coefficient,

            "intercept":
                platt_intercept,

            "t10_used_for_fitting":
                False
        },

    "threshold":
        {
            "value":
                float(
                    THRESHOLD
                ),

            "probability_scale":
                "RAW",

            "selection_scope":
                "development T4-T9 only",

            "t10_used_for_selection":
                False
        },

    "t10_outcome":
        {
            "n":
                n_t10,

            "persistent_n":
                n_persist,

            "disappeared_n":
                n_disappear,

            "persistence_rate":
                float(
                    persistence_rate
                ),

            "disappearance_rate":
                float(
                    disappearance_rate
                )
        },

    "t10_raw_probability_metrics":
        raw_metrics,

    "t10_calibrated_probability_metrics":
        calibrated_metrics,

    "t10_frozen_threshold_metrics":
        threshold_metrics,

    "rules":
        {
            "t10_used_for_feature_selection":
                False,

            "t10_used_for_model_selection":
                False,

            "t10_used_for_hyperparameter_selection":
                False,

            "t10_used_for_threshold_selection":
                False,

            "t10_used_for_calibration_selection":
                False,

            "post_t10_retuning_permitted":
                False
        }
}


PROVENANCE_PATH = (
    PROV_DIR
    / "J_FINAL_LOCKED_T10_PROVENANCE.json"
)


with open(
    PROVENANCE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        provenance,
        f,
        indent=2
    )


# ==================================================================================================
# 23. OUTPUT ARTIFACT HASHES
# ==================================================================================================

artifact_paths = [
    PRED_OUT,
    OUTCOME_OUT,
    PROBABILITY_OUT,
    THRESHOLD_OUT,
    CALIBRATION_OUT,
    FINAL_MODEL_PATH,
    FINAL_PLATT_PATH,
    PROVENANCE_PATH
]


artifact_hash_rows = []


for path in artifact_paths:

    if not path.exists():

        raise RuntimeError(
            f"HARD FAIL — expected output not written:\n{path}"
        )

    artifact_hash_rows.append(
        {
            "file":
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),

            "sha256":
                sha256_file(
                    path
                )
        }
    )


artifact_hashes = pd.DataFrame(
    artifact_hash_rows
)


ARTIFACT_HASH_PATH = (
    PROV_DIR
    / "J_output_artifact_sha256.csv"
)


artifact_hashes.to_csv(
    ARTIFACT_HASH_PATH,
    index=False
)


# ==================================================================================================
# 24. FINAL RESULTS DISPLAY
# ==================================================================================================

print("\n" + "=" * 120)
print("TABLE J1 — FINAL LOCKED T10 OUTCOME")
print("=" * 120)

print(
    outcome_qc.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("TABLE J2 — FINAL LOCKED T10 PROBABILITY PERFORMANCE")
print("=" * 120)

print(
    probability_results.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("TABLE J3 — FINAL FROZEN-THRESHOLD PERFORMANCE")
print("=" * 120)

print(
    threshold_results.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("TABLE J4 — FINAL T10 CALIBRATION DECILES")
print("=" * 120)

print(
    calibration_table.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("FINAL PLATT PARAMETERS")
print("=" * 120)

print(
    "Coefficient:",
    platt_coefficient
)

print(
    "Intercept:",
    platt_intercept
)


print("\n" + "=" * 120)
print("SAVED OUTPUTS")
print("=" * 120)

print(
    "\nPredictions:"
)

print(
    PRED_OUT
)

print(
    "\nOutcome QC:"
)

print(
    OUTCOME_OUT
)

print(
    "\nProbability metrics:"
)

print(
    PROBABILITY_OUT
)

print(
    "\nThreshold metrics:"
)

print(
    THRESHOLD_OUT
)

print(
    "\nCalibration:"
)

print(
    CALIBRATION_OUT
)

print(
    "\nFinal model:"
)

print(
    FINAL_MODEL_PATH
)

print(
    "\nFinal Platt calibrator:"
)

print(
    FINAL_PLATT_PATH
)

print(
    "\nProvenance:"
)

print(
    PROVENANCE_PATH
)

print(
    "\nArtifact hashes:"
)

print(
    ARTIFACT_HASH_PATH
)


print("\n" + "=" * 120)
print(
    "PRODUCTION BLOCK J STATUS: PASS — "
    "FINAL LOCKED T10 PRIMARY EVALUATION COMPLETE"
)
print("=" * 120)

print(
    "\nIMPORTANT: T10 IS NOW OPEN."
)

print(
    "DO NOT RETUNE THE PRIMARY MODEL, FEATURES, "
    "HYPERPARAMETERS, THRESHOLD, OR CALIBRATION."
)

print(
    "\nNEXT STEP: BLOCK K — "
    "SECONDARY ADVERTISED-PRICE RESILIENCE."
)

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/block_K_secondary.py" --stage dev

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/block_K_secondary.py" --stage final

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/production_L_to_O.py"

In [ ]:
%pip install numpy pandas matplotlib pyarrow

In [ ]:
from matplotlib import font_manager

fonts = sorted({
    f.name
    for f in font_manager.fontManager.ttflist
})

[x for x in fonts if "Latin Modern" in x or "LM Roman" in x]

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/generate_manuscript_figures_WINDOWS_TIMES_FIXED.py"

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/generate_manuscript_figures_WINDOWS_TIMES_DISPLAY.py"

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/generate_F0_research_story.py"

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

input_file = (
    root / "05_results"
    / "final_locked_T10_primary"
    / "predictions"
    / "J_final_locked_T10_primary_predictions.parquet"
)

output_file = input_file.with_name(
    "J_final_locked_T10_primary_predictions_AUDIT.csv"
)

# Read the frozen predictions
df = pd.read_parquet(input_file)

# Verify the data
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Transitions:", df["transition"].unique())

# Export to CSV without modifying the original
df.to_csv(output_file, index=False)

print("\nCSV successfully saved:")
print(output_file)